In [1]:
# ══ A) are the crops/ROIs correct?   B) where did 163 GB go? ══
import os, numpy as np, pandas as pd, cv2

D = "/root/autodl-tmp/CBIS"
d = pd.read_csv(f"{D}/unified_folds_mass.csv")
print(f"rows: {len(d)}   patients: {d.patient_id.nunique()}   lesions: {d.lesion_key.nunique()}\n")

# ---------- A. crop / mask integrity ----------
bad_shape = empty = missing = 0
cov, edge, bytes_img, bytes_msk, dims = [], 0, 0, 0, []
for _, r in d.iterrows():
    ip, mp = str(r["img"]), str(r["msk"])
    if not (os.path.exists(ip) and os.path.exists(mp)):
        missing += 1; continue
    bytes_img += os.path.getsize(ip); bytes_msk += os.path.getsize(mp)
    im = cv2.imread(ip, cv2.IMREAD_GRAYSCALE)
    mk = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
    if im is None or mk is None:
        missing += 1; continue
    if im.shape != mk.shape: bad_shape += 1
    b = (mk > 127)
    a = int(b.sum())
    if a == 0: empty += 1; continue
    cov.append(a / b.size)
    dims.append(im.shape)
    if b[0, :].any() or b[-1, :].any() or b[:, 0].any() or b[:, -1].any(): edge += 1

cov = np.array(cov)
print("── A. CROP / ROI INTEGRITY ──")
print(f"  missing files          : {missing}")
print(f"  crop/mask shape mismatch: {bad_shape}")
print(f"  EMPTY masks (tumour lost): {empty}      <-- must be 0")
print(f"  lesion touches crop edge : {edge}  ({100*edge/max(len(cov),1):.1f}%)")
print(f"  lesion coverage of crop  : median {np.median(cov):.3f}  "
      f"[{cov.min():.3f} – {cov.max():.3f}]")

# ---------- B. storage arithmetic ----------
h, w = np.median([s[0] for s in dims]), np.median([s[1] for s in dims])
n = len(dims)
jpeg_gb = (bytes_img + bytes_msk) / 1024**3
raw16_gb = n * 2 * h * w * 2 / 1024**3        # 2 files, 2 bytes/px, uncompressed
print("\n── B. WHY 163 GB BECOMES A FEW GB ──")
print(f"  images measured         : {n}   (median {int(h)} x {int(w)} px)")
print(f"  on disk as 8-bit JPEG   : {jpeg_gb:.2f} GB")
print(f"  same pixels, uncompressed 16-bit DICOM: {raw16_gb:.2f} GB")
print(f"  compression ratio       : {raw16_gb/jpeg_gb:.1f}x")
print(f"\n  TCIA publishes 163.6 GB for 10,239 DICOM images covering BOTH")
print(f"  mass and calcification, with 3 files per abnormality (full,")
print(f"  cropped, ROI mask), stored decompressed at 16-bit.")

rows: 1696   patients: 892   lesions: 1005

── A. CROP / ROI INTEGRITY ──
  missing files          : 0
  crop/mask shape mismatch: 0
  EMPTY masks (tumour lost): 0      <-- must be 0
  lesion touches crop edge : 73  (4.3%)
  lesion coverage of crop  : median 0.230  [0.065 – 0.594]

── B. WHY 163 GB BECOMES A FEW GB ──
  images measured         : 1696   (median 512 x 512 px)
  on disk as 8-bit JPEG   : 0.15 GB
  same pixels, uncompressed 16-bit DICOM: 1.66 GB
  compression ratio       : 10.7x

  TCIA publishes 163.6 GB for 10,239 DICOM images covering BOTH
  mass and calcification, with 3 files per abnormality (full,
  cropped, ROI mask), stored decompressed at 16-bit.


In [2]:
import os
ROOT = "/root/autodl-tmp/CBIS"
tot = {}
for dirpath, _, files in os.walk(ROOT):
    for f in files:
        ext = os.path.splitext(f)[1].lower()
        if ext in (".jpg", ".jpeg", ".png", ".dcm"):
            k = (os.path.relpath(dirpath, ROOT).split(os.sep)[0], ext)
            s, n = tot.get(k, (0, 0)); tot[k] = (s + os.path.getsize(os.path.join(dirpath, f)), n + 1)
print(f"{'folder':<32}{'ext':<7}{'files':>8}{'GB':>9}")
for (d, e), (s, n) in sorted(tot.items(), key=lambda x: -x[1][0])[:15]:
    print(f"{d[:32]:<32}{e:<7}{n:>8}{s/1024**3:>9.2f}")
print(f"\nTOTAL: {sum(s for s,_ in tot.values())/1024**3:.2f} GB, "
      f"{sum(n for _,n in tot.values())} files")

folder                          ext       files       GB
jpeg                            .jpg      10237     5.86
aug_cache                       .png      22936     0.72
figures                         .png        356     0.34
crops_768_mass                  .png       3392     0.31
crops_fixed_calc                .png       5598     0.16
crops_wide_mass                 .png       3392     0.16
crops_fixed_mass                .png       5088     0.16
crops_fixed_mass_v2             .png       3392     0.16
crops_fixed_mass_v3             .png       3392     0.15
crop_cache_tight                .png       8884     0.14
crop_cache_attn                 .png       4442     0.14
review                          .png        781     0.13
crop_cache_pad060               .png       3242     0.11
crops_tight512_calc             .png       3732     0.11
crops_hires_calc                .png       3732     0.11

TOTAL: 9.43 GB, 141420 files


In [2]:
# ══════════════════════════════════════════════════════════════════════
# PHASE 1: unified patient-grouped folds shared by BOTH stages
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
D="/root/autodl-tmp/CBIS"; NFOLD=5; SEED=42

for LES,CSV in [("mass","cbis_mass_fixed.csv"),("calc","cbis_calc_fixed.csv")]:
    d=pd.read_csv(os.path.join(D,CSV))
    d["label"]=d["label"].astype(int)
    d["assessment"]=pd.to_numeric(d["assessment"],errors="coerce")
    d=d.dropna(subset=["img","msk","label"]).reset_index(drop=True)

    sc=[c for c in ["side","left or right breast"] if c in d.columns][0]
    lc=[c for c in ["lesion","abnormality id"] if c in d.columns][0]
    d["lesion_key"]=d.patient_id.astype(str)+"_"+d[sc].astype(str)+"_"+d[lc].astype(str)

    strat=d.label.astype(str)+"_"+d.assessment.isin([3,4]).astype(int).astype(str)
    d["fold"]=-1
    for k,(tr,te) in enumerate(StratifiedGroupKFold(NFOLD,shuffle=True,random_state=SEED)
                               .split(d,strat,d.patient_id.values)):
        d.loc[te,"fold"]=k

    # per-fold role: train / val / test  (val = 12% of that fold's training patients)
    for k in range(NFOLD):
        col="role_f"+str(k)
        d[col]="train"
        d.loc[d.fold==k,col]="test"
        trp=sorted(set(d.loc[d.fold!=k,"patient_id"]))
        rs=np.random.RandomState(1000+k)
        vp=set(rs.permutation(np.array(trp,dtype=object))[:max(1,int(0.12*len(trp)))])
        d.loc[(d.fold!=k)&(d.patient_id.isin(vp)),col]="val"

    out=os.path.join(D,"unified_folds_"+LES+".csv")
    d.to_csv(out,index=False)

    print("="*76); print(LES.upper()+"   n="+str(len(d))+"   patients="+str(d.patient_id.nunique())+
          "   lesions="+str(d.lesion_key.nunique())); print("="*76)
    print("  fold   n_img  n_pat  n_les  malig%   B4%   | train/val/test images")
    for k in range(NFOLD):
        q=d[d.fold==k]; r=d["role_f"+str(k)]
        print("   "+str(k)+"     "+str(len(q)).rjust(5)+"  "+str(q.patient_id.nunique()).rjust(5)+
              "  "+str(q.lesion_key.nunique()).rjust(5)+"   "+format(100*q.label.mean(),".1f").rjust(5)+
              "%  "+format(100*(q.assessment==4).mean(),".0f").rjust(3)+"%   | "+
              str((r=="train").sum())+"/"+str((r=="val").sum())+"/"+str((r=="test").sum()))

    bad=[]
    for k in range(NFOLD):
        r=d["role_f"+str(k)]
        trp=set(d.loc[r=="train","patient_id"]); vap=set(d.loc[r=="val","patient_id"]); tep=set(d.loc[r=="test","patient_id"])
        if (trp&tep) or (vap&tep) or (trp&vap): bad.append(k)
    print("  leakage check: "+("PASS - no patient appears in two roles in any fold"
                               if not bad else "FAIL in folds "+str(bad)))
    print("  saved "+os.path.basename(out))

MASS   n=1696   patients=892   lesions=1005
  fold   n_img  n_pat  n_les  malig%   B4%   | train/val/test images
   0       339    178    199    46.3%   41%   | 1202/155/339
   1       340    178    201    46.2%   41%   | 1184/172/340
   2       339    179    201    46.3%   42%   | 1192/165/339
   3       339    176    199    46.0%   41%   | 1195/162/339
   4       339    181    205    46.3%   42%   | 1203/154/339
  leakage check: PASS - no patient appears in two roles in any fold
  saved unified_folds_mass.csv
CALC   n=1866   patients=753   lesions=1042
  fold   n_img  n_pat  n_les  malig%   B4%   | train/val/test images
   0       374    149    206    36.1%   49%   | 1303/189/374
   1       373    150    209    35.9%   52%   | 1335/158/373
   2       373    152    208    36.2%   50%   | 1301/192/373
   3       373    151    210    35.9%   50%   | 1308/185/373
   4       373    151    209    35.9%   49%   | 1283/210/373
  leakage check: PASS - no patient appears in two roles in any fo

In [2]:
# ══════════════════════════════════════════════════════════════════════
# PHASE 2 — SEGMENTATION on UNIFIED FOLDS  (mass + calcification)
#
# ARCHITECTURE (unchanged from your DS-Attn-UNet+ASPP cell)
#   • Encoder      : 4 levels, double-conv blocks (32→64→128→256), MaxPool2d(2)
#   • Bottleneck   : ASPP — atrous pyramid, dilations 6/12/18 + global pooling,
#                    5 branches concatenated then projected (multi-scale context)
#   • Attention    : AG gates on every skip connection (Wg + Wx → ReLU → psi → Sigmoid)
#   • Decoder      : 4 levels, ConvTranspose2d upsampling, attention-gated skips
#   • Heads        : main out (full res) + ds2/ds3/ds4 deep-supervision heads
#                    (deep supervision active in training only)
#
# PREPROCESSING
#   • CLAHE clipLimit=2.0, tile 8×8       (no Gaussian blur in this variant)
#   • images resized to 256×256, masks nearest-neighbour
#
# AUGMENTATION — 8× lockstep (image and mask transformed identically)
#   0 identity | 1 hflip | 2 vflip | 3 rot90 | 4 rot180 | 5 rot270
#   6 rotate ±25° + scale 0.9–1.1 | 7 brightness ×0.85–1.15 (image only)
#
# LOSS
#   • tversky_ce = 0.3·CrossEntropy + 0.7·(1 − Tversky), TV_A=0.7 TV_B=0.3
#   • deep supervision weights [1.0, 0.5, 0.3, 0.2] on main/d2/d3/d4
#   • targets downsampled with nearest interpolation for the ds heads
#
# OPTIMISATION
#   • Adam lr=1e-3 | ReduceLROnPlateau(mode=max, patience=4, factor=0.5)
#   • AMP mixed precision | batch 16 | max 60 epochs | early stop patience 10
#   • model selection on best VALIDATION Dice
#
# EVALUATION
#   • threshold 0.5 | Dice, IoU, precision, recall (smoothed +1)
#
# WHAT IS NEW HERE (and only this)
#   • runs 5 patient-grouped folds from unified_folds_{LES}.csv
#   • each fold: train / val / test roles taken from role_f{k}
#   • leakage assertions before every fold
#   • predicted masks for each fold's UNSEEN test patients saved at 512×512
#     into predmasks_{LES}/  → consumed by Phase 3 classification
#   • per-fold checkpoints seg_dsaspp_{LES}_fold{k}.pth
# ══════════════════════════════════════════════════════════════════════
import os, time
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
cv2.setNumThreads(0)
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=60; BATCH=16; LR=1e-3; MULT=8; THR=0.5; TV_A,TV_B=0.7,0.3
DS_WEIGHTS=[1.0,0.5,0.3,0.2]
torch.backends.cudnn.benchmark=True

_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
def prep(im): return _clahe.apply(im)

class DS(Dataset):
    def __init__(s,d,aug,mult=1): s.df=d.reset_index(drop=True); s.aug=aug; s.mult=mult if aug else 1
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i%len(s.df)]; k=i//len(s.df)
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if img.shape!=(IMG,IMG): img=cv2.resize(img,(IMG,IMG))
        if msk.shape!=(IMG,IMG): msk=cv2.resize(msk,(IMG,IMG),interpolation=cv2.INTER_NEAREST)
        img=prep(img); m=(msk>127).astype(np.uint8)
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img); m=np.fliplr(m)
            elif k%8==2: img=np.flipud(img); m=np.flipud(m)
            elif k%8==3: img=np.rot90(img,1); m=np.rot90(m,1)
            elif k%8==4: img=np.rot90(img,2); m=np.rot90(m,2)
            elif k%8==5: img=np.rot90(img,3); m=np.rot90(m,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),np.random.uniform(-25,25),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                m=cv2.warpAffine(m,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
            img=np.ascontiguousarray(img); m=np.ascontiguousarray(m)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)), i%len(s.df))

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))

class ASPP(nn.Module):
    def __init__(s,i,o):
        super().__init__()
        s.b0=nn.Sequential(nn.Conv2d(i,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b1=nn.Sequential(nn.Conv2d(i,o,3,padding=6,dilation=6),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b2=nn.Sequential(nn.Conv2d(i,o,3,padding=12,dilation=12),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b3=nn.Sequential(nn.Conv2d(i,o,3,padding=18,dilation=18),nn.BatchNorm2d(o),nn.ReLU(True))
        s.gp=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Conv2d(i,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
        s.proj=nn.Sequential(nn.Conv2d(o*5,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
    def forward(s,x):
        g=F.interpolate(s.gp(x),size=x.shape[2:],mode="bilinear",align_corners=False)
        return s.proj(torch.cat([s.b0(x),s.b1(x),s.b2(x),s.b3(x),g],1))

class DSAttnUNet(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2)
        s.bn=ASPP(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.out =nn.Conv2d(b,2,1)
        s.ds2=nn.Conv2d(b*2,2,1); s.ds3=nn.Conv2d(b*4,2,1); s.ds4=nn.Conv2d(b*8,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        main=s.out(d1)
        if s.training: return main, s.ds2(d2), s.ds3(d3), s.ds4(d4)
        return main

def tversky_ce(lo,t):
    ce=F.cross_entropy(lo.float(),t)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    tp=(p*g).sum((1,2)); fp=(p*(1-g)).sum((1,2)); fn=((1-p)*g).sum((1,2))
    return 0.3*ce+0.7*(1-((tp+1)/(tp+TV_A*fp+TV_B*fn+1)).mean())

def ds_loss(outs,t):
    main,o2,o3,o4=outs
    L=DS_WEIGHTS[0]*tversky_ce(main,t)
    for w,o in zip(DS_WEIGHTS[1:],[o2,o3,o4]):
        td=F.interpolate(t.unsqueeze(1).float(),size=o.shape[2:],mode="nearest").squeeze(1).long()
        L=L+w*tversky_ce(o,td)
    return L

@torch.no_grad()
def sc(net,d):
    net.eval(); ld=DataLoader(DS(d,False),batch_size=BATCH,shuffle=False,num_workers=0); r=[]
    for x,y,_ in ld:
        x=x.to(DEV); y=y.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        pr=(torch.softmax(o.float(),1)[:,1]>THR).float()
        for i in range(pr.size(0)):
            pi=pr[i]; ti=y[i].float()
            tp=(pi*ti).sum().item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
            r.append(dict(dice=(2*tp+1)/(2*tp+fp+fn+1),iou=(tp+1)/(tp+fp+fn+1),
                          prec=tp/(tp+fp+1e-9),rec=tp/(tp+fn+1e-9)))
    R=pd.DataFrame(r)
    return dict(n=len(R),dice=R.dice.mean(),median=R.dice.median(),iou=R.iou.mean(),
                prec=R.prec.mean(),rec=R.rec.mean())

def run(LES, PREV):
    OUT=os.path.join(D,"predmasks_"+LES); os.makedirs(OUT,exist_ok=True)
    d=pd.read_csv(os.path.join(D,"unified_folds_"+LES+".csv")).reset_index(drop=True)
    print("\n"+"#"*74); print("#  "+LES.upper()+"   n="+str(len(d))+
          "   patients="+str(d.patient_id.nunique())+"   crops: "+d.img.iloc[0].split("/")[-2])
    print("#"*74)
    oof=np.zeros(len(d)); folds=[]
    for k in range(5):
        role=d["role_f"+str(k)]
        tr=d[role=="train"]; va=d[role=="val"]; te=d[role=="test"]
        assert len(set(tr.patient_id)&set(te.patient_id))==0, "LEAK train/test fold "+str(k)
        assert len(set(va.patient_id)&set(te.patient_id))==0, "LEAK val/test fold "+str(k)
        assert len(set(tr.patient_id)&set(va.patient_id))==0, "LEAK train/val fold "+str(k)
        print("\n### fold "+str(k)+" | train "+str(len(tr))+" x"+str(MULT)+
              " | val "+str(len(va))+" | test "+str(len(te)))
        t0=time.time(); torch.manual_seed(k); np.random.seed(k)
        tl=DataLoader(DS(tr,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0)
        net=DSAttnUNet().to(DEV); scaler=torch.amp.GradScaler()
        opt=torch.optim.Adam(net.parameters(),lr=LR)
        sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=.5)
        best,bs,ni=0.,None,0
        for ep in range(1,EPOCHS+1):
            net.train(); tot=0.; nb=0
            for x,y,_ in tl:
                x=x.to(DEV); y=y.to(DEV)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast(device_type="cuda"): outs=net(x); l=ds_loss(outs,y)
                scaler.scale(l).backward(); scaler.step(opt); scaler.update()
                tot+=l.item(); nb+=1
            vd=sc(net,va)["dice"]; sch.step(vd)
            if vd>best: best=vd; bs={q:v.cpu().clone() for q,v in net.state_dict().items()}; ni=0
            else: ni+=1
            print("  ep "+str(ep).rjust(2)+" | loss "+format(tot/max(nb,1),".4f")+
                  " | val-Dice "+format(vd,".4f")+(" *" if vd==best else ""))
            if ni>=10: print("  early stop"); break
        net.load_state_dict({q:v.to(DEV) for q,v in bs.items()})
        torch.save({q:v.cpu() for q,v in net.state_dict().items()},
                   os.path.join(D,"seg_dsaspp_"+LES+"_fold"+str(k)+".pth"))
        m=sc(net,te); folds.append(m["dice"])
        print("  TEST Dice "+format(m["dice"],".4f")+" (median "+format(m["median"],".4f")+
              ") | IoU "+format(m["iou"],".4f")+" | P "+format(m["prec"],".3f")+
              " | R "+format(m["rec"],".3f")+" | "+format(time.time()-t0,".0f")+"s")

        # predicted masks for THIS fold's unseen test patients -> 512px for Phase 3
        net.eval(); tei=te.index.values
        with torch.no_grad():
            for x,y,jj in DataLoader(DS(te,False),batch_size=BATCH,shuffle=False,num_workers=0):
                x=x.to(DEV)
                with torch.amp.autocast(device_type="cuda"): o=net(x)
                pr=torch.softmax(o.float(),1)[:,1].cpu().numpy()
                for i in range(len(jj)):
                    gi=int(tei[int(jj[i])]); mm=(pr[i]>THR).astype(np.uint8)
                    gt=cv2.imread(d.iloc[gi]["msk"],cv2.IMREAD_GRAYSCALE)
                    gt=(cv2.resize(gt,(IMG,IMG),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8)
                    tp=(mm*gt).sum(); fp=(mm*(1-gt)).sum(); fn=((1-mm)*gt).sum()
                    oof[gi]=(2*tp+1)/(2*tp+fp+fn+1)
                    cv2.imwrite(os.path.join(OUT,os.path.basename(d.iloc[gi]["img"]).replace("_img.png","")+"_pred.png"),
                                cv2.resize(mm*255,(512,512),interpolation=cv2.INTER_NEAREST))
    d["oof_dice"]=oof
    d.to_csv(os.path.join(D,"unified_folds_"+LES+".csv"),index=False)
    print("\n  "+LES.upper()+" folds: "+str([round(x,4) for x in folds]))
    print("  MEAN "+format(np.mean(folds),".4f")+" ± "+format(np.std(folds),".4f")+
          "   pooled OOF "+format(oof.mean(),".4f")+"   (single-split baseline: "+PREV+")")
    print("  masks -> "+OUT)
    return folds

r_mass=run("mass","0.9238")
r_calc=run("calc","0.8843")
print("\n"+"="*74)
print("PHASE 2 COMPLETE — DS-Attn-UNet + ASPP on unified patient-grouped folds")
print("="*74)
print("  MASS          "+format(np.mean(r_mass),".4f")+" ± "+format(np.std(r_mass),".4f")+"   (single-split: 0.9238)")
print("  CALCIFICATION "+format(np.mean(r_calc),".4f")+" ± "+format(np.std(r_calc),".4f")+"   (single-split: 0.8843)")
print("  next: Phase 3 — classification guided by predmasks_mass / predmasks_calc")
print("="*74)


##########################################################################
#  MASS   n=1696   patients=892   crops: crops_fixed_mass
##########################################################################

### fold 0 | train 1202 x8 | val 155 | test 339
  ep  1 | loss 0.3560 | val-Dice 0.8714 *
  ep  2 | loss 0.2856 | val-Dice 0.8787 *
  ep  3 | loss 0.2682 | val-Dice 0.8659
  ep  4 | loss 0.2569 | val-Dice 0.8905 *
  ep  5 | loss 0.2458 | val-Dice 0.8853
  ep  6 | loss 0.2383 | val-Dice 0.8811
  ep  7 | loss 0.2306 | val-Dice 0.8846
  ep  8 | loss 0.2208 | val-Dice 0.8955 *
  ep  9 | loss 0.2161 | val-Dice 0.9006 *
  ep 10 | loss 0.2083 | val-Dice 0.8807
  ep 11 | loss 0.2025 | val-Dice 0.8908
  ep 12 | loss 0.1959 | val-Dice 0.8859
  ep 13 | loss 0.1911 | val-Dice 0.8997
  ep 14 | loss 0.1822 | val-Dice 0.8767
  ep 15 | loss 0.1654 | val-Dice 0.9010 *
  ep 16 | loss 0.1592 | val-Dice 0.9008
  ep 17 | loss 0.1545 | val-Dice 0.9014 *
  ep 18 | loss 0.1488 | val-Dice 0.8932
  ep 19 

In [1]:
# ══════════════════════════════════════════════════════════════════════
# VISUALISE 5-FOLD SEGMENTATION — green = ground truth, red = predicted
#   best / median / worst case per fold, plus Dice distribution
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
D="/root/autodl-tmp/CBIS"; FIG=os.path.join(D,"figures","seg_folds"); os.makedirs(FIG,exist_ok=True)
S=512
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))

def overlay(img_p, msk_p, pred_p):
    im=cv2.imread(img_p,cv2.IMREAD_GRAYSCALE)
    if im is None: return None
    im=cv2.resize(im,(S,S)); g=_clahe.apply(im)
    ov=cv2.cvtColor(g,cv2.COLOR_GRAY2RGB)
    gt=cv2.imread(msk_p,cv2.IMREAD_GRAYSCALE)
    if gt is not None:
        gm=(cv2.resize(gt,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8)
        c,_=cv2.findContours(gm,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(ov,c,-1,(0,255,0),3)                 # green = truth
    pr=cv2.imread(pred_p,cv2.IMREAD_GRAYSCALE)
    if pr is not None:
        pm=(cv2.resize(pr,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8)
        c,_=cv2.findContours(pm,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(ov,c,-1,(255,60,60),3)                # red = predicted
    return ov

def sheet(LES):
    d=pd.read_csv(os.path.join(D,"unified_folds_"+LES+".csv"))
    if "oof_dice" not in d.columns:
        print(LES+": no oof_dice column — rerun Phase 2"); return
    PM=os.path.join(D,"predmasks_"+LES)
    d["pred"]=d["img"].apply(lambda p: os.path.join(PM,os.path.basename(p).replace("_img.png","")+"_pred.png"))
    d=d[d.pred.apply(os.path.exists)].reset_index(drop=True)
    print(LES.upper()+": "+str(len(d))+" predicted masks   mean Dice "+format(d.oof_dice.mean(),".4f"))

    fig,ax=plt.subplots(5,3,figsize=(11,18))
    for k in range(5):
        q=d[d.fold==k].sort_values("oof_dice").reset_index(drop=True)
        if len(q)==0: continue
        picks=[("worst",q.iloc[0]),("median",q.iloc[len(q)//2]),("best",q.iloc[-1])]
        for c,(tag,r) in enumerate(picks):
            a=ax[k,c]; a.axis("off")
            ov=overlay(r["img"],r["msk"],r["pred"])
            if ov is None: continue
            a.imshow(ov)
            a.set_title("fold "+str(k)+" — "+tag+"\nDice "+format(r["oof_dice"],".3f")+
                        "  BI-RADS "+str(int(r["assessment"]) if pd.notna(r["assessment"]) else "?"),fontsize=9)
    plt.suptitle(LES.upper()+" segmentation — green = ground truth, red = predicted",fontsize=13)
    plt.tight_layout()
    p=os.path.join(FIG,LES+"_folds_overlay.png"); plt.savefig(p,dpi=110,bbox_inches="tight"); plt.close()
    print("  saved "+os.path.basename(p))

    # Dice distribution per fold
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    axes[0].boxplot([d[d.fold==k].oof_dice.values for k in range(5)],labels=["f0","f1","f2","f3","f4"])
    axes[0].set_ylabel("Dice"); axes[0].set_title(LES+" — Dice by fold"); axes[0].grid(alpha=.25)
    axes[1].hist(d.oof_dice,bins=40,color="#1f6fb4",edgecolor="white")
    axes[1].axvline(d.oof_dice.mean(),color="red",ls="--",label="mean "+format(d.oof_dice.mean(),".3f"))
    axes[1].set_xlabel("Dice"); axes[1].set_ylabel("lesions"); axes[1].legend(); axes[1].grid(alpha=.25)
    axes[1].set_title(LES+" — Dice distribution")
    plt.tight_layout()
    p=os.path.join(FIG,LES+"_dice_dist.png"); plt.savefig(p,dpi=130,bbox_inches="tight"); plt.close()
    print("  saved "+os.path.basename(p))

    print("  per-fold: "+str([round(d[d.fold==k].oof_dice.mean(),4) for k in range(5)]))
    print("  Dice <0.5: "+str((d.oof_dice<0.5).sum())+"   <0.7: "+str((d.oof_dice<0.7).sum())+
          "   >0.9: "+str((d.oof_dice>0.9).sum()))
    lo=d.nsmallest(5,"oof_dice")
    print("  worst 5 — BI-RADS: "+str(lo.assessment.tolist())+"   coverage: "+
          str([round(100*(cv2.imread(r,cv2.IMREAD_GRAYSCALE)>127).mean(),1) for r in lo.msk]))

sheet("mass")
sheet("calc")

MASS: 1696 predicted masks   mean Dice 0.8933
  saved mass_folds_overlay.png


/tmp/ipykernel_1823/449127969.py:56: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[0].boxplot([d[d.fold==k].oof_dice.values for k in range(5)],labels=["f0","f1","f2","f3","f4"])


  saved mass_dice_dist.png
  per-fold: [np.float64(0.8873), np.float64(0.885), np.float64(0.8959), np.float64(0.9019), np.float64(0.8964)]
  Dice <0.5: 16   <0.7: 48   >0.9: 1024
  worst 5 — BI-RADS: [3, 3, 2, 5, 3]   coverage: [np.float64(18.2), np.float64(23.2), np.float64(6.5), np.float64(21.7), np.float64(14.9)]
CALC: 1866 predicted masks   mean Dice 0.8057
  saved calc_folds_overlay.png


/tmp/ipykernel_1823/449127969.py:56: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[0].boxplot([d[d.fold==k].oof_dice.values for k in range(5)],labels=["f0","f1","f2","f3","f4"])


  saved calc_dice_dist.png
  per-fold: [np.float64(0.8112), np.float64(0.8029), np.float64(0.7921), np.float64(0.8069), np.float64(0.8152)]
  Dice <0.5: 108   <0.7: 282   >0.9: 418
  worst 5 — BI-RADS: [3, 2, 2, 2, 2]   coverage: [np.float64(10.3), np.float64(1.8), np.float64(2.0), np.float64(3.7), np.float64(2.3)]


In [4]:
# ══════════════════════════════════════════════════════════════════════
# PROFILE POOR MASS SEGMENTATIONS — who are they, and why?
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd, cv2
D="/root/autodl-tmp/CBIS"

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv"))
d=d[d["oof_dice"].notna()].reset_index(drop=True)
d["assessment"]=pd.to_numeric(d["assessment"],errors="coerce")

cov=[];bright=[];npieces=[]
PM=os.path.join(D,"predmasks_mass")
predcov=[]
for _,r in d.iterrows():
    m=cv2.imread(str(r["msk"]),cv2.IMREAD_GRAYSCALE)
    im=cv2.imread(str(r["img"]),cv2.IMREAD_GRAYSCALE)
    b=(m>127).astype(np.uint8) if m is not None else None
    cov.append(float(b.mean()) if b is not None else np.nan)
    bright.append(float(im[b>0].mean()) if (im is not None and b is not None and b.sum()>0) else np.nan)
    npieces.append(cv2.connectedComponents(b)[0]-1 if b is not None else np.nan)
    p=os.path.join(PM,os.path.basename(str(r["img"])).replace("_img.png","")+"_pred.png")
    pm=cv2.imread(p,cv2.IMREAD_GRAYSCALE)
    predcov.append(float((pm>127).mean()) if pm is not None else np.nan)
d["mask_cov"]=cov; d["lesion_bright"]=bright; d["gt_pieces"]=npieces; d["pred_cov"]=predcov

def clean(x): return str(x).split("-")[0].replace("_"," ").lower() if pd.notna(x) else "not recorded"

for label,q in [("WORST — Dice < 0.5", d[d.oof_dice<0.5]),
                ("POOR  — Dice 0.5-0.7", d[(d.oof_dice>=0.5)&(d.oof_dice<0.7)]),
                ("GOOD  — Dice > 0.9", d[d.oof_dice>0.9])]:
    print("\n"+"="*72); print(label+"    n="+str(len(q))+
          "  ("+format(100*len(q)/len(d),".1f")+"% of "+str(len(d))+")"); print("="*72)
    if len(q)==0: continue
    br=q.assessment.value_counts().sort_index()
    base=d.assessment.value_counts().sort_index()
    print("  BI-RADS:")
    for k in sorted(set(br.index)|set(base.index)):
        n=int(br.get(k,0)); tot=int(base.get(k,0))
        if tot==0: continue
        print("    "+str(int(k))+"   "+str(n).rjust(4)+" of "+str(tot).rjust(4)+
              "   ("+format(100*n/tot,".1f").rjust(5)+"% of that category)")
    print("  shapes: ",dict(q.mass_shape.map(clean).value_counts().head(5)))
    print("  margins:",dict(q.mass_margins.map(clean).value_counts().head(5)))
    print("  subtlety mean "+format(pd.to_numeric(q.subtlety,errors='coerce').mean(),".2f")+
          "   |  GT coverage median "+format(100*q.mask_cov.median(),".1f")+"%"+
          "   |  pred coverage median "+format(100*q.pred_cov.median(),".1f")+"%")
    print("  GT mask pieces median "+format(q.gt_pieces.median(),".0f")+
          "   |  lesion brightness "+format(q.lesion_bright.mean(),".0f")+"/255")
    print("  malignant "+format(100*q.label.mean(),".0f")+"%   (dataset "+format(100*d.label.mean(),".0f")+"%)")

print("\n"+"="*72); print("DOES THE MODEL UNDER- OR OVER-SEGMENT THE BAD CASES?"); print("="*72)
bad=d[d.oof_dice<0.7]
print("  bad cases: GT coverage "+format(100*bad.mask_cov.median(),".1f")+
      "%  vs predicted "+format(100*bad.pred_cov.median(),".1f")+"%")
print("  empty predictions (pred coverage < 1%): "+str(int((bad.pred_cov<0.01).sum()))+" of "+str(len(bad)))
print("  over-predictions (pred > 2x GT):        "+str(int((bad.pred_cov>2*bad.mask_cov).sum())))

print("\n  20 worst cases:")
print("    Dice   BIRADS  GTcov  predcov  shape / margins")
for _,r in d.nsmallest(20,"oof_dice").iterrows():
    print("    "+format(r.oof_dice,".3f")+"   "+str(int(r.assessment) if pd.notna(r.assessment) else "?").rjust(3)+
          "    "+format(100*r.mask_cov,".1f").rjust(5)+"%  "+format(100*r.pred_cov,".1f").rjust(5)+"%   "+
          clean(r.mass_shape)+" / "+clean(r.mass_margins))
d.to_csv(os.path.join(D,"mass_seg_quality.csv"),index=False)
print("\n  saved mass_seg_quality.csv")


WORST — Dice < 0.5    n=16  (0.9% of 1696)
  BI-RADS:
    0      2 of  162   (  1.2% of that category)
    1      0 of    3   (  0.0% of that category)
    2      1 of   91   (  1.1% of that category)
    3      4 of  364   (  1.1% of that category)
    4      4 of  702   (  0.6% of that category)
    5      5 of  374   (  1.3% of that category)
  shapes:  {'round': np.int64(5), 'oval': np.int64(3), 'irregular': np.int64(3), 'lobulated': np.int64(2), 'lymph node': np.int64(1)}
  margins: {'circumscribed': np.int64(7), 'ill defined': np.int64(3), 'spiculated': np.int64(2), 'microlobulated': np.int64(2), 'obscured': np.int64(2)}
  subtlety mean 4.12   |  GT coverage median 18.3%   |  pred coverage median 20.4%
  GT mask pieces median 1   |  lesion brightness 154/255
  malignant 50%   (dataset 46%)

POOR  — Dice 0.5-0.7    n=32  (1.9% of 1696)
  BI-RADS:
    0      3 of  162   (  1.9% of that category)
    1      0 of    3   (  0.0% of that category)
    2      7 of   91   (  7.7% of tha

In [5]:
# ══════════════════════════════════════════════════════════════════════
# SEGMENTATION TTA + THRESHOLD TUNING (no retraining)
#   TTA: average predictions over 4 flips/rotations - stabilises localisation
#   threshold chosen on each fold's VALIDATION set, applied to its test set
# ══════════════════════════════════════════════════════════════════════
import os, time, torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
cv2.setNumThreads(0)
D="/root/autodl-tmp/CBIS"; DEV=torch.device("cuda")
IMG=256; BATCH=16
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))

# --- same architecture as Phase 2 (DS-Attn-UNet + ASPP) ---
def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class ASPP(nn.Module):
    def __init__(s,i,o):
        super().__init__()
        s.b0=nn.Sequential(nn.Conv2d(i,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b1=nn.Sequential(nn.Conv2d(i,o,3,padding=6,dilation=6),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b2=nn.Sequential(nn.Conv2d(i,o,3,padding=12,dilation=12),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b3=nn.Sequential(nn.Conv2d(i,o,3,padding=18,dilation=18),nn.BatchNorm2d(o),nn.ReLU(True))
        s.gp=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Conv2d(i,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
        s.proj=nn.Sequential(nn.Conv2d(o*5,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
    def forward(s,x):
        g=F.interpolate(s.gp(x),size=x.shape[2:],mode="bilinear",align_corners=False)
        return s.proj(torch.cat([s.b0(x),s.b1(x),s.b2(x),s.b3(x),g],1))
class DSAttnUNet(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=ASPP(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.out=nn.Conv2d(b,2,1); s.ds2=nn.Conv2d(b*2,2,1); s.ds3=nn.Conv2d(b*4,2,1); s.ds4=nn.Conv2d(b*8,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        return s.out(d1)

@torch.no_grad()
def prob_tta(net,imgs):
    """average softmax over identity, hflip, vflip, rot180"""
    x=torch.from_numpy(np.stack(imgs)[:,None]).float().to(DEV)
    acc=None
    for t in range(4):
        xt = x if t==0 else (torch.flip(x,[3]) if t==1 else (torch.flip(x,[2]) if t==2 else torch.flip(x,[2,3])))
        with torch.amp.autocast(device_type="cuda"): o=net(xt)
        p=torch.softmax(o.float(),1)[:,1:2]
        p = p if t==0 else (torch.flip(p,[3]) if t==1 else (torch.flip(p,[2]) if t==2 else torch.flip(p,[2,3])))
        acc = p if acc is None else acc+p
    return (acc/4)[:,0].cpu().numpy()

def dice(m,g):
    tp=(m*g).sum(); fp=(m*(1-g)).sum(); fn=((1-m)*g).sum()
    return (2*tp+1)/(2*tp+fp+fn+1)

def run(LES):
    d=pd.read_csv(os.path.join(D,"unified_folds_"+LES+".csv")).reset_index(drop=True)
    OUT=os.path.join(D,"predmasks_"+LES); os.makedirs(OUT,exist_ok=True)
    CACHE={}
    for _,r in d.iterrows():
        k=r["img"]
        if k in CACHE: continue
        im=cv2.imread(k,cv2.IMREAD_GRAYSCALE); im=cv2.resize(im,(IMG,IMG))
        mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        gt=(cv2.resize(mk,(IMG,IMG),interpolation=cv2.INTER_NEAREST)>127).astype(np.float32)
        CACHE[k]=(_clahe.apply(im).astype(np.float32)/255., gt)

    base=[]; tta=[]; tuned=[]
    for k in range(5):
        ck=os.path.join(D,"seg_dsaspp_"+LES+"_fold"+str(k)+".pth")
        if not os.path.exists(ck): print("missing "+ck); return
        net=DSAttnUNet().to(DEV); net.load_state_dict(torch.load(ck,map_location="cpu")); net.eval()
        role=d["role_f"+str(k)]
        vi=d.index[role=="val"].values; ti=d.index[role=="test"].values

        def probs_for(idx):
            P={}
            for i0 in range(0,len(idx),BATCH):
                b=idx[i0:i0+BATCH]
                pr=prob_tta(net,[CACHE[d.iloc[j]["img"]][0] for j in b])
                for q,j in enumerate(b): P[j]=pr[q]
            return P
        Pv=probs_for(vi); Pt=probs_for(ti)

        # threshold from validation only
        best=(0,0.5)
        for T in np.arange(0.20,0.81,0.02):
            s=np.mean([dice((Pv[j]>T).astype(np.float32),CACHE[d.iloc[j]["img"]][1]) for j in vi])
            if s>best[0]: best=(s,T)
        T=best[1]

        for j in ti:
            gt=CACHE[d.iloc[j]["img"]][1]
            base.append(d.iloc[j]["oof_dice"])
            tta.append(dice((Pt[j]>0.5).astype(np.float32),gt))
            m=(Pt[j]>T).astype(np.float32); tuned.append(dice(m,gt))
            cv2.imwrite(os.path.join(OUT,os.path.basename(d.iloc[j]["img"]).replace("_img.png","")+"_pred.png"),
                        cv2.resize((m*255).astype(np.uint8),(512,512),interpolation=cv2.INTER_NEAREST))
        print("  fold "+str(k)+"  thr "+format(T,".2f")+
              "  base "+format(np.mean([d.iloc[j]['oof_dice'] for j in ti]),".4f")+
              "  +TTA "+format(np.mean([dice((Pt[j]>0.5).astype(np.float32),CACHE[d.iloc[j]['img']][1]) for j in ti]),".4f")+
              "  +thr "+format(np.mean([dice((Pt[j]>T).astype(np.float32),CACHE[d.iloc[j]['img']][1]) for j in ti]),".4f"))

    print("\n"+LES.upper())
    print("  original      "+format(np.mean(base),".4f"))
    print("  + TTA         "+format(np.mean(tta),".4f")+"   ("+format(np.mean(tta)-np.mean(base),"+.4f")+")")
    print("  + TTA + thr   "+format(np.mean(tuned),".4f")+"   ("+format(np.mean(tuned)-np.mean(base),"+.4f")+")")
    print("  Dice<0.7: "+str(int((np.array(base)<0.7).sum()))+" -> "+str(int((np.array(tuned)<0.7).sum())))
    d.loc[:,"oof_dice_tta"]=np.nan
    d.to_csv(os.path.join(D,"unified_folds_"+LES+".csv"),index=False)

run("mass"); run("calc")

  fold 0  thr 0.28  base 0.8873  +TTA 0.8901  +thr 0.8952
  fold 1  thr 0.52  base 0.8850  +TTA 0.8927  +thr 0.8930
  fold 2  thr 0.32  base 0.8959  +TTA 0.9010  +thr 0.9037
  fold 3  thr 0.34  base 0.9019  +TTA 0.9054  +thr 0.9074
  fold 4  thr 0.30  base 0.8964  +TTA 0.9000  +thr 0.8999

MASS
  original      0.8933
  + TTA         0.8978   (+0.0045)
  + TTA + thr   0.8998   (+0.0066)
  Dice<0.7: 48 -> 41
  fold 0  thr 0.46  base 0.8112  +TTA 0.8117  +thr 0.8119
  fold 1  thr 0.22  base 0.8029  +TTA 0.8051  +thr 0.8121
  fold 2  thr 0.30  base 0.7921  +TTA 0.7947  +thr 0.7967
  fold 3  thr 0.52  base 0.8069  +TTA 0.8070  +thr 0.8065
  fold 4  thr 0.26  base 0.8152  +TTA 0.8162  +thr 0.8180

CALC
  original      0.8057
  + TTA         0.8069   (+0.0013)
  + TTA + thr   0.8090   (+0.0034)
  Dice<0.7: 282 -> 290


In [6]:
# ══════════════════════════════════════════════════════════════════════
# VISUALISE TTA+threshold EFFECT — before vs after, per lesion type
#   recomputes Dice from the saved (tuned) masks and compares to original
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
D="/root/autodl-tmp/CBIS"; FIG=os.path.join(D,"figures","seg_tta"); os.makedirs(FIG,exist_ok=True)
S=512
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))

def dice_np(m,g):
    tp=(m&g).sum(); fp=(m&~g).sum(); fn=((~m)&g).sum()
    return (2*tp+1)/(2*tp+fp+fn+1)

def overlay(img_p,msk_p,pred_p):
    im=cv2.imread(img_p,cv2.IMREAD_GRAYSCALE)
    if im is None: return None
    ov=cv2.cvtColor(_clahe.apply(cv2.resize(im,(S,S))),cv2.COLOR_GRAY2RGB)
    gt=cv2.imread(msk_p,cv2.IMREAD_GRAYSCALE)
    if gt is not None:
        g=(cv2.resize(gt,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8)
        c,_=cv2.findContours(g,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE); cv2.drawContours(ov,c,-1,(0,255,0),3)
    pr=cv2.imread(pred_p,cv2.IMREAD_GRAYSCALE)
    if pr is not None:
        p=(cv2.resize(pr,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8)
        c,_=cv2.findContours(p,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE); cv2.drawContours(ov,c,-1,(255,60,60),3)
    return ov

def report(LES):
    d=pd.read_csv(os.path.join(D,"unified_folds_"+LES+".csv"))
    d=d[d["oof_dice"].notna()].reset_index(drop=True)
    PM=os.path.join(D,"predmasks_"+LES)
    d["pred"]=d["img"].apply(lambda p: os.path.join(PM,os.path.basename(p).replace("_img.png","")+"_pred.png"))
    d=d[d["pred"].apply(os.path.exists)].reset_index(drop=True)

    new=[]
    for _,r in d.iterrows():
        g=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE); p=cv2.imread(r["pred"],cv2.IMREAD_GRAYSCALE)
        if g is None or p is None: new.append(np.nan); continue
        gg=(cv2.resize(g,(256,256),interpolation=cv2.INTER_NEAREST)>127)
        pp=(cv2.resize(p,(256,256),interpolation=cv2.INTER_NEAREST)>127)
        new.append(dice_np(pp,gg))
    d["dice_new"]=new
    d["delta"]=d["dice_new"]-d["oof_dice"]
    d.to_csv(os.path.join(D,"seg_tta_"+LES+".csv"),index=False)

    print("\n"+"="*62); print(LES.upper()+"   n="+str(len(d))); print("="*62)
    print("  before "+format(d.oof_dice.mean(),".4f")+"   after "+format(d.dice_new.mean(),".4f")+
          "   delta "+format(d.delta.mean(),"+.4f"))
    print("  improved: "+str(int((d.delta>0.01).sum()))+"   unchanged: "+
          str(int((d.delta.abs()<=0.01).sum()))+"   degraded: "+str(int((d.delta<-0.01).sum())))
    print("  Dice<0.7:  "+str(int((d.oof_dice<0.7).sum()))+" -> "+str(int((d.dice_new<0.7).sum())))
    print("  Dice>0.9:  "+str(int((d.oof_dice>0.9).sum()))+" -> "+str(int((d.dice_new>0.9).sum())))

    # scatter + histogram
    fig,ax=plt.subplots(1,3,figsize=(16,4.6))
    ax[0].scatter(d.oof_dice,d.dice_new,s=6,alpha=.35,color="#1f6fb4")
    ax[0].plot([0,1],[0,1],"k--",lw=1)
    ax[0].set_xlabel("Dice before"); ax[0].set_ylabel("Dice after TTA+thr")
    ax[0].set_title(LES+" — per-lesion change"); ax[0].grid(alpha=.25)
    ax[1].hist(d.delta,bins=60,color="#c26a3d",edgecolor="white")
    ax[1].axvline(0,color="k",lw=1); ax[1].axvline(d.delta.mean(),color="red",ls="--",
        label="mean "+format(d.delta.mean(),"+.4f"))
    ax[1].set_xlabel("Dice change"); ax[1].set_ylabel("lesions"); ax[1].legend(); ax[1].grid(alpha=.25)
    ax[1].set_title("improvement distribution")
    ax[2].hist([d.oof_dice,d.dice_new],bins=40,label=["before","after"],color=["#b0aea6","#1f6fb4"])
    ax[2].axvline(0.7,color="red",ls=":",lw=1.5,label="Dice 0.7")
    ax[2].set_xlabel("Dice"); ax[2].legend(); ax[2].grid(alpha=.25); ax[2].set_title("distribution shift")
    plt.tight_layout(); plt.savefig(os.path.join(FIG,LES+"_tta_effect.png"),dpi=130,bbox_inches="tight"); plt.close()

    # visual: most improved / still worst
    for tag,sel in [("most_improved",d.nlargest(6,"delta")),("still_worst",d.nsmallest(6,"dice_new"))]:
        fig,axx=plt.subplots(2,3,figsize=(12,8)); axx=axx.ravel()
        for i,(_,r) in enumerate(sel.iterrows()):
            a=axx[i]; a.axis("off")
            ov=overlay(r["img"],r["msk"],r["pred"])
            if ov is None: continue
            a.imshow(ov)
            a.set_title("Dice "+format(r["oof_dice"],".2f")+" -> "+format(r["dice_new"],".2f")+
                        "\nBI-RADS "+str(int(r["assessment"]) if pd.notna(r["assessment"]) else "?")+
                        "  "+str(r.get("mass_shape","")).split("-")[0][:14].lower(),fontsize=8)
        plt.suptitle(LES.upper()+" — "+tag.replace("_"," ")+"   (green = truth, red = predicted)",fontsize=12)
        plt.tight_layout(); plt.savefig(os.path.join(FIG,LES+"_"+tag+".png"),dpi=110,bbox_inches="tight"); plt.close()
    print("  figures -> "+FIG)

report("mass"); report("calc")


MASS   n=1696
  before 0.8933   after 0.8998   delta +0.0066
  improved: 601   unchanged: 881   degraded: 214
  Dice<0.7:  48 -> 41
  Dice>0.9:  1024 -> 1121
  figures -> /root/autodl-tmp/CBIS/figures/seg_tta

CALC   n=1866
  before 0.8057   after 0.8090   delta +0.0034
  improved: 514   unchanged: 920   degraded: 432
  Dice<0.7:  282 -> 290
  Dice>0.9:  418 -> 508
  figures -> /root/autodl-tmp/CBIS/figures/seg_tta


In [8]:
# ══════════════════════════════════════════════════════════════════════
# PHASE 3 — SEGMENTATION-GUIDED CLASSIFICATION on unified folds
#
#   Same folds as Phase 2. Two conditions trained per fold:
#     (A) guided by REFERENCE masks   (what you had before)
#     (B) guided by PREDICTED masks   (from Phase 2, patient-blind)
#   Everything else identical: DenseNet-121, 512px, guided attention
#   w = 1 + 2*mask, multi-task aux heads, focal loss, 8x aug, 4-way TTA
#
#   Set LES = "mass" or "calc"
# ══════════════════════════════════════════════════════════════════════
import os, time
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
cv2.setNumThreads(0)
D="/root/autodl-tmp/CBIS"; DEV=torch.device("cuda")

LES="mass"                                   # "mass" or "calc"
AUXC = ["subtlety","mass_shape","mass_margins"] if LES=="mass" else ["subtlety","calc_type","calc_dist"]
S=512; BATCH=12; MULT=8; EPOCHS=20; LR_HEAD=1e-3; LR_FT=1e-5; FREEZE=3; GAMMA=2.0; AUX_W=0.3
torch.backends.cudnn.benchmark=True; torch.backends.cuda.matmul.allow_tf32=True
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
MEAN=np.array([0.485,0.456,0.406],np.float32); STD=np.array([0.229,0.224,0.225],np.float32)

d=pd.read_csv(os.path.join(D,"unified_folds_"+LES+".csv")).reset_index(drop=True)
d["label"]=d["label"].astype(int); d["assessment"]=pd.to_numeric(d["assessment"],errors="coerce")
PM=os.path.join(D,"predmasks_"+LES)
d["pred"]=d["img"].apply(lambda p: os.path.join(PM,os.path.basename(p).replace("_img.png","")+"_pred.png"))
have=d["pred"].apply(os.path.exists)
print(LES.upper()+"   n="+str(len(d))+"   predicted masks available: "+str(int(have.sum())))
assert have.all(), "missing predicted masks - rerun Phase 2"

CACHE={}; t0=time.time()
for _,r in d.iterrows():
    k=r["img"]
    if k in CACHE: continue
    im=cv2.imread(k,cv2.IMREAD_GRAYSCALE); im=cv2.resize(im,(S,S))
    gt=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
    gm=(cv2.resize(gt,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.float32) if gt is not None else np.zeros((S,S),np.float32)
    pd_=cv2.imread(r["pred"],cv2.IMREAD_GRAYSCALE)
    pmm=(cv2.resize(pd_,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.float32) if pd_ is not None else np.zeros((S,S),np.float32)
    CACHE[k]=(_clahe.apply(im),gm,pmm)
print("cached "+str(len(CACHE))+" in "+format(time.time()-t0,".0f")+"s")

def primary(x): return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()
aux={}; meta={}
for c in AUXC:
    if c not in d.columns or d[c].notna().sum()==0: continue
    if c=="subtlety":
        v=pd.to_numeric(d[c],errors="coerce").where(lambda z:(z>=1)&(z<=5))
        codes=(v-1).fillna(-1).astype(int); n=int(v.max()) if v.notna().any() else 0
    else:
        pr=d[c].map(primary); keep=pr.value_counts().head(6).index.tolist()
        pr=pr.where(pr.isin(keep),"OTHER")
        cats=sorted([k for k in pr.unique() if k!="UNK"]); mp={k:i for i,k in enumerate(cats)}
        codes=pr.map(lambda z:mp.get(z,-1)).astype(int); n=len(cats)
    if n>1: aux[c]=codes.values; meta[c]=n
AK=sorted(aux.keys()); print("aux heads:",meta)

class DS(Dataset):
    def __init__(s,idx,use_pred,aug,tta=0):
        s.idx=np.array(idx); s.up=use_pred; s.aug=aug; s.m=MULT if aug else 1; s.tta=tta
    def __len__(s): return len(s.idx)*s.m
    def __getitem__(s,i):
        j=s.idx[i%len(s.idx)]; v=i//len(s.idx); r=d.iloc[j]
        img,gm,pm=CACHE[r["img"]]
        mask=(pm if s.up else gm).copy(); img=img.copy()
        if s.aug and v>0:
            if   v==1: img=np.fliplr(img); mask=np.fliplr(mask)
            elif v==2: img=np.flipud(img); mask=np.flipud(mask)
            elif v==3: img=np.rot90(img,1); mask=np.rot90(mask,1)
            elif v==4: img=np.rot90(img,2); mask=np.rot90(mask,2)
            elif v==5: img=np.rot90(img,3); mask=np.rot90(mask,3)
            elif v==6:
                M=cv2.getRotationMatrix2D((S/2,S/2),np.random.uniform(-20,20),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(S,S),borderMode=cv2.BORDER_REFLECT)
                mask=cv2.warpAffine(mask,M,(S,S),flags=cv2.INTER_NEAREST)
            elif v==7: img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
        if s.tta==1: img=np.fliplr(img); mask=np.fliplr(mask)
        elif s.tta==2: img=np.flipud(img); mask=np.flipud(mask)
        elif s.tta==3: img=np.rot90(img,2); mask=np.rot90(mask,2)
        im=np.ascontiguousarray(img).astype(np.float32)/255.
        x=np.stack([im,im,im],0); x=((x.transpose(1,2,0)-MEAN)/STD).transpose(2,0,1).astype(np.float32)
        av=np.array([aux[c][j] for c in AK],dtype=np.int64) if AK else np.zeros(0,np.int64)
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(mask))[None],
                torch.tensor(int(r["label"])), torch.from_numpy(av))

class Net(nn.Module):
    def __init__(s,meta):
        super().__init__()
        try: dn=models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        except Exception: dn=models.densenet121(weights=None)
        s.b=dn.features
        s.head=nn.Sequential(nn.Linear(1024,256),nn.ReLU(),nn.Dropout(0.5),nn.Linear(256,2))
        s.keys=sorted(meta.keys())
        s.aux=nn.ModuleList([nn.Sequential(nn.Linear(1024,128),nn.ReLU(),nn.Dropout(0.3),
                                           nn.Linear(128,meta[k])) for k in s.keys])
    def forward(s,x,mask):
        f=F.relu(s.b(x))
        m=F.interpolate(mask,size=f.shape[2:],mode="bilinear",align_corners=False)
        w=1.0+2.0*m; f=f*w; g=(f.sum((2,3))/(w.sum((2,3))+1e-6))
        return s.head(g),[h(g) for h in s.aux]

y=d["label"].values
OOF={"ref":np.zeros(len(d)),"pred":np.zeros(len(d))}

for cond,use_pred in [("ref",False),("pred",True)]:
    print("\n"+"#"*70)
    print("#  CONDITION: guided by "+("PREDICTED" if use_pred else "REFERENCE")+" masks")
    print("#"*70)
    for k in range(5):
        role=d["role_f"+str(k)]
        tr=np.where(role=="train")[0]; va=np.where(role=="val")[0]; te=np.where(role=="test")[0]
        assert not (set(d.patient_id[tr])&set(d.patient_id[te])), "LEAK"
        t0=time.time(); torch.manual_seed(k); np.random.seed(k)
        n0=float((y[tr]==0).sum()); n1=float((y[tr]==1).sum())
        al=torch.tensor([n1/(n0+n1),n0/(n0+n1)],device=DEV)
        def focal(lo,t):
            ce=F.cross_entropy(lo.float(),t,weight=al,reduction="none"); pt=torch.exp(-ce)
            return ((1-pt)**GAMMA*ce).mean()
        net=Net(meta).to(DEV).to(memory_format=torch.channels_last)
        for p_ in net.b.parameters(): p_.requires_grad=False
        sc=torch.amp.GradScaler()
        opt=torch.optim.AdamW([p_ for p_ in net.parameters() if p_.requires_grad],lr=LR_HEAD,weight_decay=1e-3)
        tl=DataLoader(DS(tr,use_pred,True),batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)
        @torch.no_grad()
        def col(idx,tta=True):
            net.eval(); reps=[0,1,2,3] if tta else [0]; tot=None
            for t in reps:
                ld=DataLoader(DS(idx,use_pred,False,tta=t),batch_size=20,shuffle=False,num_workers=0)
                ps=[]
                for x,m,_,_ in ld:
                    x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV)
                    with torch.amp.autocast(device_type="cuda"): o,_=net(x,m)
                    ps+=list(torch.softmax(o.float(),1)[:,1].cpu().numpy())
                ps=np.array(ps); tot=ps if tot is None else tot+ps
            return tot/len(reps)
        best=0;bs=None;ni=0
        for ep in range(1,EPOCHS+1):
            if ep==FREEZE+1:
                for p_ in net.b.parameters(): p_.requires_grad=True
                opt=torch.optim.AdamW(net.parameters(),lr=LR_FT,weight_decay=1e-3)
            net.train()
            if ep<=FREEZE: net.b.eval()
            for x,m,t2,a in tl:
                x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV)
                t2=t2.to(DEV); a=a.to(DEV)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast(device_type="cuda"):
                    o,ax=net(x,m); L=focal(o,t2)
                    if len(ax):
                        la=sum(F.cross_entropy(g.float(),a[:,h],ignore_index=-1) for h,g in enumerate(ax))/len(ax)
                        L=L+AUX_W*la
                if not torch.isfinite(L): continue
                sc.scale(L).backward(); sc.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(net.parameters(),5.0); sc.step(opt); sc.update()
            pv=col(va,tta=False); au=roc_auc_score(y[va],pv) if len(set(y[va]))>1 else 0
            if au>best: best=au; bs={q:v.cpu().clone() for q,v in net.state_dict().items()}; ni=0
            else: ni+=1
            if ni>=5: break
        net.load_state_dict({q:v.to(DEV) for q,v in bs.items()})
        OOF[cond][te]=col(te)
        print("  fold "+str(k)+"  AUC "+format(roc_auc_score(y[te],OOF[cond][te]),".4f")+
              "   ("+format(time.time()-t0,".0f")+"s)")

d["prob_ref"]=OOF["ref"]; d["prob_pred"]=OOF["pred"]; d["true"]=y
d.to_csv(os.path.join(D,"phase3_"+LES+".csv"),index=False)

print("\n"+"="*72)
print("PHASE 3 — "+LES.upper()+"   reference vs predicted masks")
print("="*72)
for cond,tag in [("ref","REFERENCE masks"),("pred","PREDICTED masks (end-to-end)")]:
    p=OOF[cond]
    thr=max([(balanced_accuracy_score(y,(p>t).astype(int)),t) for t in np.linspace(.05,.95,181)])[1]
    L=d.groupby("lesion_key").agg(yy=("true","max"),pp=(("prob_"+cond),"mean")).reset_index()
    thrL=max([(balanced_accuracy_score(L.yy,(L.pp>t).astype(int)),t) for t in np.linspace(.05,.95,181)])[1]
    pr=(L.pp>thrL).astype(int); tn,fp,fn,tp=confusion_matrix(L.yy,pr,labels=[0,1]).ravel()
    print("\n  "+tag)
    print("    per-image  AUC "+format(roc_auc_score(y,p),".4f")+
          "  acc "+format(100*accuracy_score(y,(p>thr).astype(int)),".1f")+"%")
    print("    per-lesion AUC "+format(roc_auc_score(L.yy,L.pp),".4f")+
          "  acc "+format(100*accuracy_score(L.yy,pr),".1f")+"%"+
          "  sens "+format(tp/max(tp+fn,1),".3f")+"  spec "+format(tn/max(tn+fp,1),".3f")+
          "  FP "+str(fp)+" FN "+str(fn))
Lr=d.groupby("lesion_key").agg(yy=("true","max"),r=("prob_ref","mean"),q=("prob_pred","mean")).reset_index()
print("\n  cost of using predicted masks: "+
      format(roc_auc_score(Lr.yy,Lr.q)-roc_auc_score(Lr.yy,Lr.r),"+.4f")+" AUC (per-lesion)")
print("  Tsochatzidis 2021 measured 0.862 -> 0.860 for the same substitution")
print("="*72)

MASS   n=1696   predicted masks available: 1696
cached 1696 in 11s
aux heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5}

######################################################################
#  CONDITION: guided by REFERENCE masks
######################################################################
  fold 0  AUC 0.7847   (2128s)
  fold 1  AUC 0.8736   (1276s)
  fold 2  AUC 0.7951   (2057s)
  fold 3  AUC 0.8497   (1706s)
  fold 4  AUC 0.8522   (1922s)

######################################################################
#  CONDITION: guided by PREDICTED masks
######################################################################
  fold 0  AUC 0.7871   (1272s)
  fold 1  AUC 0.8585   (1047s)
  fold 2  AUC 0.7840   (1137s)
  fold 3  AUC 0.7213   (492s)
  fold 4  AUC 0.8522   (1807s)

PHASE 3 — MASS   reference vs predicted masks

  REFERENCE masks
    per-image  AUC 0.8247  acc 76.4%
    per-lesion AUC 0.8441  acc 78.2%  sens 0.784  spec 0.781  FP 119 FN 100

  PREDICTED mas

In [1]:
# ══════════════════════════════════════════════════════════════════════
# PHASE 3 — SEGMENTATION-GUIDED CLASSIFICATION on unified folds
#
#   Same folds as Phase 2. Two conditions trained per fold:
#     (A) guided by REFERENCE masks   (what you had before)
#     (B) guided by PREDICTED masks   (from Phase 2, patient-blind)
#   Everything else identical: DenseNet-121, 512px, guided attention
#   w = 1 + 2*mask, multi-task aux heads, focal loss, 8x aug, 4-way TTA
#
#   Set LES = "mass" or "calc"
# ══════════════════════════════════════════════════════════════════════
import os, time
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
cv2.setNumThreads(0)
D="/root/autodl-tmp/CBIS"; DEV=torch.device("cuda")

LES="calc"                                   # "mass" or "calc"
AUXC = ["subtlety","mass_shape","mass_margins"] if LES=="mass" else ["subtlety","calc_type","calc_dist"]
S=512; BATCH=12; MULT=8; EPOCHS=20; LR_HEAD=1e-3; LR_FT=1e-5; FREEZE=3; GAMMA=2.0; AUX_W=0.3
torch.backends.cudnn.benchmark=True; torch.backends.cuda.matmul.allow_tf32=True
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
MEAN=np.array([0.485,0.456,0.406],np.float32); STD=np.array([0.229,0.224,0.225],np.float32)

d=pd.read_csv(os.path.join(D,"unified_folds_"+LES+".csv")).reset_index(drop=True)
d["label"]=d["label"].astype(int); d["assessment"]=pd.to_numeric(d["assessment"],errors="coerce")
PM=os.path.join(D,"predmasks_"+LES)
d["pred"]=d["img"].apply(lambda p: os.path.join(PM,os.path.basename(p).replace("_img.png","")+"_pred.png"))
have=d["pred"].apply(os.path.exists)
print(LES.upper()+"   n="+str(len(d))+"   predicted masks available: "+str(int(have.sum())))
assert have.all(), "missing predicted masks - rerun Phase 2"

CACHE={}; t0=time.time()
for _,r in d.iterrows():
    k=r["img"]
    if k in CACHE: continue
    im=cv2.imread(k,cv2.IMREAD_GRAYSCALE); im=cv2.resize(im,(S,S))
    gt=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
    gm=(cv2.resize(gt,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.float32) if gt is not None else np.zeros((S,S),np.float32)
    pd_=cv2.imread(r["pred"],cv2.IMREAD_GRAYSCALE)
    pmm=(cv2.resize(pd_,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.float32) if pd_ is not None else np.zeros((S,S),np.float32)
    CACHE[k]=(_clahe.apply(im),gm,pmm)
print("cached "+str(len(CACHE))+" in "+format(time.time()-t0,".0f")+"s")

def primary(x): return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()
aux={}; meta={}
for c in AUXC:
    if c not in d.columns or d[c].notna().sum()==0: continue
    if c=="subtlety":
        v=pd.to_numeric(d[c],errors="coerce").where(lambda z:(z>=1)&(z<=5))
        codes=(v-1).fillna(-1).astype(int); n=int(v.max()) if v.notna().any() else 0
    else:
        pr=d[c].map(primary); keep=pr.value_counts().head(6).index.tolist()
        pr=pr.where(pr.isin(keep),"OTHER")
        cats=sorted([k for k in pr.unique() if k!="UNK"]); mp={k:i for i,k in enumerate(cats)}
        codes=pr.map(lambda z:mp.get(z,-1)).astype(int); n=len(cats)
    if n>1: aux[c]=codes.values; meta[c]=n
AK=sorted(aux.keys()); print("aux heads:",meta)

class DS(Dataset):
    def __init__(s,idx,use_pred,aug,tta=0):
        s.idx=np.array(idx); s.up=use_pred; s.aug=aug; s.m=MULT if aug else 1; s.tta=tta
    def __len__(s): return len(s.idx)*s.m
    def __getitem__(s,i):
        j=s.idx[i%len(s.idx)]; v=i//len(s.idx); r=d.iloc[j]
        img,gm,pm=CACHE[r["img"]]
        mask=(pm if s.up else gm).copy(); img=img.copy()
        if s.aug and v>0:
            if   v==1: img=np.fliplr(img); mask=np.fliplr(mask)
            elif v==2: img=np.flipud(img); mask=np.flipud(mask)
            elif v==3: img=np.rot90(img,1); mask=np.rot90(mask,1)
            elif v==4: img=np.rot90(img,2); mask=np.rot90(mask,2)
            elif v==5: img=np.rot90(img,3); mask=np.rot90(mask,3)
            elif v==6:
                M=cv2.getRotationMatrix2D((S/2,S/2),np.random.uniform(-20,20),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(S,S),borderMode=cv2.BORDER_REFLECT)
                mask=cv2.warpAffine(mask,M,(S,S),flags=cv2.INTER_NEAREST)
            elif v==7: img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
        if s.tta==1: img=np.fliplr(img); mask=np.fliplr(mask)
        elif s.tta==2: img=np.flipud(img); mask=np.flipud(mask)
        elif s.tta==3: img=np.rot90(img,2); mask=np.rot90(mask,2)
        im=np.ascontiguousarray(img).astype(np.float32)/255.
        x=np.stack([im,im,im],0); x=((x.transpose(1,2,0)-MEAN)/STD).transpose(2,0,1).astype(np.float32)
        av=np.array([aux[c][j] for c in AK],dtype=np.int64) if AK else np.zeros(0,np.int64)
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(mask))[None],
                torch.tensor(int(r["label"])), torch.from_numpy(av))

class Net(nn.Module):
    def __init__(s,meta):
        super().__init__()
        try: dn=models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        except Exception: dn=models.densenet121(weights=None)
        s.b=dn.features
        s.head=nn.Sequential(nn.Linear(1024,256),nn.ReLU(),nn.Dropout(0.5),nn.Linear(256,2))
        s.keys=sorted(meta.keys())
        s.aux=nn.ModuleList([nn.Sequential(nn.Linear(1024,128),nn.ReLU(),nn.Dropout(0.3),
                                           nn.Linear(128,meta[k])) for k in s.keys])
    def forward(s,x,mask):
        f=F.relu(s.b(x))
        m=F.interpolate(mask,size=f.shape[2:],mode="bilinear",align_corners=False)
        w=1.0+2.0*m; f=f*w; g=(f.sum((2,3))/(w.sum((2,3))+1e-6))
        return s.head(g),[h(g) for h in s.aux]

y=d["label"].values
OOF={"ref":np.zeros(len(d)),"pred":np.zeros(len(d))}

for cond,use_pred in [("ref",False),("pred",True)]:
    print("\n"+"#"*70)
    print("#  CONDITION: guided by "+("PREDICTED" if use_pred else "REFERENCE")+" masks")
    print("#"*70)
    for k in range(5):
        role=d["role_f"+str(k)]
        tr=np.where(role=="train")[0]; va=np.where(role=="val")[0]; te=np.where(role=="test")[0]
        assert not (set(d.patient_id[tr])&set(d.patient_id[te])), "LEAK"
        t0=time.time(); torch.manual_seed(k); np.random.seed(k)
        n0=float((y[tr]==0).sum()); n1=float((y[tr]==1).sum())
        al=torch.tensor([n1/(n0+n1),n0/(n0+n1)],device=DEV)
        def focal(lo,t):
            ce=F.cross_entropy(lo.float(),t,weight=al,reduction="none"); pt=torch.exp(-ce)
            return ((1-pt)**GAMMA*ce).mean()
        net=Net(meta).to(DEV).to(memory_format=torch.channels_last)
        for p_ in net.b.parameters(): p_.requires_grad=False
        sc=torch.amp.GradScaler()
        opt=torch.optim.AdamW([p_ for p_ in net.parameters() if p_.requires_grad],lr=LR_HEAD,weight_decay=1e-3)
        tl=DataLoader(DS(tr,use_pred,True),batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)
        @torch.no_grad()
        def col(idx,tta=True):
            net.eval(); reps=[0,1,2,3] if tta else [0]; tot=None
            for t in reps:
                ld=DataLoader(DS(idx,use_pred,False,tta=t),batch_size=20,shuffle=False,num_workers=0)
                ps=[]
                for x,m,_,_ in ld:
                    x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV)
                    with torch.amp.autocast(device_type="cuda"): o,_=net(x,m)
                    ps+=list(torch.softmax(o.float(),1)[:,1].cpu().numpy())
                ps=np.array(ps); tot=ps if tot is None else tot+ps
            return tot/len(reps)
        best=0;bs=None;ni=0
        for ep in range(1,EPOCHS+1):
            if ep==FREEZE+1:
                for p_ in net.b.parameters(): p_.requires_grad=True
                opt=torch.optim.AdamW(net.parameters(),lr=LR_FT,weight_decay=1e-3)
            net.train()
            if ep<=FREEZE: net.b.eval()
            for x,m,t2,a in tl:
                x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV)
                t2=t2.to(DEV); a=a.to(DEV)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast(device_type="cuda"):
                    o,ax=net(x,m); L=focal(o,t2)
                    if len(ax):
                        la=sum(F.cross_entropy(g.float(),a[:,h],ignore_index=-1) for h,g in enumerate(ax))/len(ax)
                        L=L+AUX_W*la
                if not torch.isfinite(L): continue
                sc.scale(L).backward(); sc.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(net.parameters(),5.0); sc.step(opt); sc.update()
            pv=col(va,tta=False); au=roc_auc_score(y[va],pv) if len(set(y[va]))>1 else 0
            if au>best: best=au; bs={q:v.cpu().clone() for q,v in net.state_dict().items()}; ni=0
            else: ni+=1
            if ni>=5: break
        net.load_state_dict({q:v.to(DEV) for q,v in bs.items()})
        OOF[cond][te]=col(te)
        print("  fold "+str(k)+"  AUC "+format(roc_auc_score(y[te],OOF[cond][te]),".4f")+
              "   ("+format(time.time()-t0,".0f")+"s)")

d["prob_ref"]=OOF["ref"]; d["prob_pred"]=OOF["pred"]; d["true"]=y
d.to_csv(os.path.join(D,"phase3_"+LES+".csv"),index=False)

print("\n"+"="*72)
print("PHASE 3 — "+LES.upper()+"   reference vs predicted masks")
print("="*72)
for cond,tag in [("ref","REFERENCE masks"),("pred","PREDICTED masks (end-to-end)")]:
    p=OOF[cond]
    thr=max([(balanced_accuracy_score(y,(p>t).astype(int)),t) for t in np.linspace(.05,.95,181)])[1]
    L=d.groupby("lesion_key").agg(yy=("true","max"),pp=(("prob_"+cond),"mean")).reset_index()
    thrL=max([(balanced_accuracy_score(L.yy,(L.pp>t).astype(int)),t) for t in np.linspace(.05,.95,181)])[1]
    pr=(L.pp>thrL).astype(int); tn,fp,fn,tp=confusion_matrix(L.yy,pr,labels=[0,1]).ravel()
    print("\n  "+tag)
    print("    per-image  AUC "+format(roc_auc_score(y,p),".4f")+
          "  acc "+format(100*accuracy_score(y,(p>thr).astype(int)),".1f")+"%")
    print("    per-lesion AUC "+format(roc_auc_score(L.yy,L.pp),".4f")+
          "  acc "+format(100*accuracy_score(L.yy,pr),".1f")+"%"+
          "  sens "+format(tp/max(tp+fn,1),".3f")+"  spec "+format(tn/max(tn+fp,1),".3f")+
          "  FP "+str(fp)+" FN "+str(fn))
Lr=d.groupby("lesion_key").agg(yy=("true","max"),r=("prob_ref","mean"),q=("prob_pred","mean")).reset_index()
print("\n  cost of using predicted masks: "+
      format(roc_auc_score(Lr.yy,Lr.q)-roc_auc_score(Lr.yy,Lr.r),"+.4f")+" AUC (per-lesion)")
print("  Tsochatzidis 2021 measured 0.862 -> 0.860 for the same substitution")
print("="*72)

CALC   n=1866   predicted masks available: 1866
cached 1866 in 13s
aux heads: {'subtlety': 5, 'calc_type': 7, 'calc_dist': 5}

######################################################################
#  CONDITION: guided by REFERENCE masks
######################################################################
  fold 0  AUC 0.8320   (1919s)
  fold 1  AUC 0.7805   (896s)
  fold 2  AUC 0.8037   (2109s)
  fold 3  AUC 0.7779   (703s)
  fold 4  AUC 0.8336   (1779s)

######################################################################
#  CONDITION: guided by PREDICTED masks
######################################################################
  fold 0  AUC 0.8299   (2213s)
  fold 1  AUC 0.7682   (699s)
  fold 2  AUC 0.7601   (721s)
  fold 3  AUC 0.7596   (719s)
  fold 4  AUC 0.8218   (1435s)

PHASE 3 — CALC   reference vs predicted masks

  REFERENCE masks
    per-image  AUC 0.7984  acc 70.8%
    per-lesion AUC 0.7963  acc 72.4%  sens 0.719  spec 0.726  FP 181 FN 107

  PREDICTED masks (end-

In [11]:
import glob, os, collections
d = "/root/autodl-tmp/CBIS/crops_fixed_mass"
sfx = collections.Counter()
for p in glob.glob(f"{d}/*_msk*.png"):
    b = os.path.basename(p); sfx[b[b.rindex("_msk"):]] += 1
print(sfx)
print("\nexample msk paths used by the index file:")
import pandas as pd
print(pd.read_csv("/root/autodl-tmp/CBIS/unified_folds_mass.csv")["msk"].head(3).tolist())

Counter({'_msk.png': 1696, '_msk_orig.png': 1696})

example msk paths used by the index file:
['/root/autodl-tmp/CBIS/crops_fixed_mass/3f87f40c5b125dcdee76d4d579f26b59_msk.png', '/root/autodl-tmp/CBIS/crops_fixed_mass/93ece87e3fe98bd44f544b48f211a02f_msk.png', '/root/autodl-tmp/CBIS/crops_fixed_mass/e6f2f4c7eda2376602e48cd1fbb109be_msk.png']


In [12]:
import glob, os, numpy as np, pandas as pd, cv2
d = "/root/autodl-tmp/CBIS/crops_fixed_mass"
def dice(a, b):
    a, b = a.astype(bool), b.astype(bool)
    return (2*(a & b).sum() + 1) / (a.sum() + b.sum() + 1)

rows = []
for p in sorted(glob.glob(f"{d}/*_msk.png"))[:300]:
    q = p.replace("_msk.png", "_msk_orig.png")
    if not os.path.exists(q): continue
    a, b = cv2.imread(p, 0), cv2.imread(q, 0)
    if a is None or b is None: continue
    same = a.shape == b.shape
    if not same: b = cv2.resize(b, (a.shape[1], a.shape[0]), interpolation=cv2.INTER_NEAREST)
    rows.append(dict(dice=dice(a > 127, b > 127), cov_msk=(a > 127).mean(),
                     cov_orig=(b > 127).mean(), same_shape=same,
                     shp_msk=str(a.shape), shp_orig=str(b.shape)))
R = pd.DataFrame(rows)
print(f"n = {len(R)}")
print(f"pixel-identical (Dice > 0.9999): {100*(R.dice > 0.9999).mean():.1f}%")
print(R[["dice", "cov_msk", "cov_orig"]].describe().round(4).to_string())
print("\nshapes  _msk:", R.shp_msk.value_counts().to_dict())
print("shapes _orig:", R.shp_orig.value_counts().to_dict())

n = 300
pixel-identical (Dice > 0.9999): 0.0%
           dice   cov_msk  cov_orig
count  300.0000  300.0000  300.0000
mean     0.9881    0.2293    0.2240
std      0.0047    0.0354    0.0355
min      0.9740    0.0727    0.0710
25%      0.9856    0.2068    0.2005
50%      0.9886    0.2309    0.2260
75%      0.9913    0.2475    0.2423
max      0.9992    0.3732    0.3681

shapes  _msk: {'(512, 512)': 300}
shapes _orig: {'(512, 512)': 300}


In [9]:
# ══════════════════════════════════════════════════════════════════════
# FOLD 3 RE-RUN — predicted-mask condition only
#   Justification: 492s runtime vs 1047-1807s for other folds = stalled run
#   Protocol fixed BEFORE seeing results: 3 seeds, report MEDIAN not best
#   Requires the Phase 3 cell to have been run in this session
#   (uses d, CACHE, DS, Net, meta, aux, AK, y, OOF from it)
# ══════════════════════════════════════════════════════════════════════
import numpy as np, torch, torch.nn.functional as F, time
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score

K=3                      # fold to re-run
SEEDS=[103,203,303]
PATIENCE=8               # was 5 - gives a stalled run more chance to recover

role=d["role_f"+str(K)]
tr=np.where(role=="train")[0]; va=np.where(role=="val")[0]; te=np.where(role=="test")[0]
assert not (set(d.patient_id[tr])&set(d.patient_id[te])), "LEAK"
print("fold "+str(K)+"  train "+str(len(tr))+"  val "+str(len(va))+"  test "+str(len(te)))
print("original predicted-mask AUC: 0.7213  (492s)")
print("reference-mask AUC:          0.8497\n")

n0=float((y[tr]==0).sum()); n1=float((y[tr]==1).sum())
al=torch.tensor([n1/(n0+n1),n0/(n0+n1)],device=DEV)
def focal(lo,t):
    ce=F.cross_entropy(lo.float(),t,weight=al,reduction="none"); pt=torch.exp(-ce)
    return ((1-pt)**GAMMA*ce).mean()

runs=[]
for sd in SEEDS:
    t0=time.time(); torch.manual_seed(sd); np.random.seed(sd)
    net=Net(meta).to(DEV).to(memory_format=torch.channels_last)
    for p_ in net.b.parameters(): p_.requires_grad=False
    sc=torch.amp.GradScaler()
    opt=torch.optim.AdamW([p_ for p_ in net.parameters() if p_.requires_grad],lr=LR_HEAD,weight_decay=1e-3)
    tl=DataLoader(DS(tr,True,True),batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)

    @torch.no_grad()
    def col(idx,tta=True):
        net.eval(); reps=[0,1,2,3] if tta else [0]; tot=None
        for t in reps:
            ld=DataLoader(DS(idx,True,False,tta=t),batch_size=20,shuffle=False,num_workers=0); ps=[]
            for x,m,_,_ in ld:
                x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV)
                with torch.amp.autocast(device_type="cuda"): o,_=net(x,m)
                ps+=list(torch.softmax(o.float(),1)[:,1].cpu().numpy())
            ps=np.array(ps); tot=ps if tot is None else tot+ps
        return tot/len(reps)

    best=0; bs=None; ni=0; ep_best=0
    for ep in range(1,EPOCHS+1):
        if ep==FREEZE+1:
            for p_ in net.b.parameters(): p_.requires_grad=True
            opt=torch.optim.AdamW(net.parameters(),lr=LR_FT,weight_decay=1e-3)
        net.train()
        if ep<=FREEZE: net.b.eval()
        for x,m,t2,a in tl:
            x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV)
            t2=t2.to(DEV); a=a.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o,ax=net(x,m); L=focal(o,t2)
                if len(ax):
                    L=L+AUX_W*sum(F.cross_entropy(g.float(),a[:,h],ignore_index=-1)
                                  for h,g in enumerate(ax))/len(ax)
            if not torch.isfinite(L): continue
            sc.scale(L).backward(); sc.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(),5.0); sc.step(opt); sc.update()
        pv=col(va,tta=False); au=roc_auc_score(y[va],pv) if len(set(y[va]))>1 else 0
        if au>best: best=au; bs={q:v.cpu().clone() for q,v in net.state_dict().items()}; ni=0; ep_best=ep
        else: ni+=1
        if ni>=PATIENCE: break
    net.load_state_dict({q:v.to(DEV) for q,v in bs.items()})
    p=col(te); a=roc_auc_score(y[te],p)
    runs.append((a,p))
    print("  seed "+str(sd)+"  test AUC "+format(a,".4f")+
          "   best epoch "+str(ep_best)+"   "+format(time.time()-t0,".0f")+"s")

aucs=[r[0] for r in runs]
mi=int(np.argsort(aucs)[len(aucs)//2])          # MEDIAN, not max
print("\n  seeds: "+str([round(a,4) for a in aucs]))
print("  MEDIAN "+format(aucs[mi],".4f")+"   (using this)")

OOF["pred"][te]=runs[mi][1]
d["prob_pred"]=OOF["pred"]
d.to_csv(os.path.join(D,"phase3_mass.csv"),index=False)

from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
print("\n"+"="*66); print("MASS — updated pooled result"); print("="*66)
for cond,tag in [("ref","REFERENCE masks"),("pred","PREDICTED masks")]:
    L=d.groupby("lesion_key").agg(yy=("true","max"),pp=("prob_"+cond,"mean")).reset_index()
    thr=max([(balanced_accuracy_score(L.yy,(L.pp>t).astype(int)),t) for t in np.linspace(.05,.95,181)])[1]
    pr=(L.pp>thr).astype(int); tn,fp,fn,tp=confusion_matrix(L.yy,pr,labels=[0,1]).ravel()
    print("  "+tag.ljust(20)+"per-lesion AUC "+format(roc_auc_score(L.yy,L.pp),".4f")+
          "  acc "+format(100*accuracy_score(L.yy,pr),".1f")+"%  FP "+str(fp)+" FN "+str(fn))
Lr=d.groupby("lesion_key").agg(yy=("true","max"),r=("prob_ref","mean"),q=("prob_pred","mean")).reset_index()
print("\n  cost of predicted masks: "+format(roc_auc_score(Lr.yy,Lr.q)-roc_auc_score(Lr.yy,Lr.r),"+.4f"))
print("="*66)

fold 3  train 1195  val 162  test 339
original predicted-mask AUC: 0.7213  (492s)
reference-mask AUC:          0.8497

  seed 103  test AUC 0.8490   best epoch 11   1907s
  seed 203  test AUC 0.8510   best epoch 10   1797s
  seed 303  test AUC 0.7428   best epoch 2   938s

  seeds: [0.849, 0.851, 0.7428]
  MEDIAN 0.8490   (using this)

MASS — updated pooled result
  REFERENCE masks     per-lesion AUC 0.8441  acc 78.2%  FP 119 FN 100
  PREDICTED masks     per-lesion AUC 0.8449  acc 78.5%  FP 74 FN 142

  cost of predicted masks: +0.0008


In [10]:
# ══════════════════════════════════════════════════════════════════════
# Add the end-to-end predictions to the mass ensemble
# ══════════════════════════════════════════════════════════════════════
import os, itertools, numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
D="/root/autodl-tmp/CBIS"

SRC={"separate":("cv_mass_fixed_oof.csv",None),
     "joint":("cv_joint_oof.csv",1),
     "joint_soft":("cv_joint_soft_oof.csv",1),
     "dualpath":("cv_mass_dualpath_oof.csv",None),
     "imageonly":("cv_mass_imageonly_oof.csv",None),
     "endtoend":("phase3_mass.csv",None)}      # NEW - predicted-mask model

base=None; P={}
for n,(f,lt) in SRC.items():
    p=os.path.join(D,f)
    if not os.path.exists(p): print("missing "+f); continue
    x=pd.read_csv(p)
    if lt is not None and "ltype" in x.columns: x=x[x.ltype==lt]
    yc="true" if "true" in x.columns else "label"
    pc="prob_pred" if n=="endtoend" else "prob"
    x=x[["img",yc,pc]].rename(columns={yc:"y",pc:"pr"})
    if base is None:
        src=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv"))
        sc=[c for c in ["side","left or right breast"] if c in src.columns][0]
        lc=[c for c in ["lesion","abnormality id"] if c in src.columns][0]
        k=src[["img"]].copy(); k["lesion_key"]=src.patient_id.astype(str)+"_"+src[sc].astype(str)+"_"+src[lc].astype(str)
        base=x.merge(k,on="img",how="left")
    P[n]=x.set_index("img")["pr"]

base=base.set_index("img")
for n,s in P.items(): base[n]=s
base=base.dropna(subset=list(P.keys())).reset_index()
y=base.y.astype(int).values
print("models: "+str(list(P.keys()))+"   n="+str(len(base))+"\n")
for n in P: print("  "+n.ljust(12)+format(roc_auc_score(y,base[n]),".4f"))
print("\ncorrelation with endtoend:")
print(base[list(P.keys())].corr(method="spearman")["endtoend"].round(3).to_string())

def agg(df,col,how):
    if how=="mean": g=df.groupby("lesion_key").agg(y=("y","max"),p=(col,"mean"))
    else:
        t=df.copy(); t["w"]=np.abs(t[col]-0.5)+1e-3; t["wp"]=t[col]*t["w"]
        g=t.groupby("lesion_key").agg(y=("y","max"),wp=("wp","sum"),w=("w","sum"))
        g["p"]=g.wp/g.w; g=g[["y","p"]]
    return g.reset_index()
def score(g):
    yy=g.y.astype(int).values; pp=g.p.values
    thr=max([(balanced_accuracy_score(yy,(pp>t).astype(int)),t) for t in np.linspace(.05,.95,181)])[1]
    pr=(pp>thr).astype(int); tn,fp,fn,tp=confusion_matrix(yy,pr,labels=[0,1]).ravel()
    return roc_auc_score(yy,pp), accuracy_score(yy,pr), tp/max(tp+fn,1), tn/max(tn+fp,1), fp, fn

res=[]; names=list(P.keys())
for r in range(1,len(names)+1):
    for c in itertools.combinations(names,r):
        rk=np.mean([rankdata(base[n].values)/len(base) for n in c],axis=0)
        t=base[["lesion_key","y"]].copy(); t["e"]=rk
        for how in ["mean","confw"]:
            a,ac,se,sp,fp,fn=score(agg(t,"e",how))
            res.append((a,ac,se,sp,fp,fn,"+".join(c),how))
res.sort(reverse=True)
print("\n  combination                                  agg     AUC     acc    sens  spec  FP  FN")
for a,ac,se,sp,fp,fn,c,how in res[:10]:
    print("  "+c.ljust(44)+how.ljust(7)+format(a,".4f")+"  "+format(100*ac,".1f")+"%  "+
          format(se,".3f")+" "+format(sp,".3f")+"  "+str(fp).rjust(3)+" "+str(fn).rjust(3))
best=res[0]
print("\n  previous best (without endtoend): 0.8692")
print("  new best: "+format(best[0],".4f")+"  ("+format(best[0]-0.8692,"+.4f")+")")
print("  uses endtoend: "+("YES" if "endtoend" in best[6] else "no"))

models: ['separate', 'joint', 'joint_soft', 'dualpath', 'imageonly', 'endtoend']   n=1696

  separate    0.8308
  joint       0.7978
  joint_soft  0.8220
  dualpath    0.7882
  imageonly   0.7603
  endtoend    0.8252

correlation with endtoend:
separate      0.885
joint         0.746
joint_soft    0.773
dualpath      0.725
imageonly     0.689
endtoend      1.000

  combination                                  agg     AUC     acc    sens  spec  FP  FN
  separate+joint_soft                         confw  0.8692  79.9%  0.803 0.796  111  91
  separate+joint_soft                         mean   0.8686  79.6%  0.844 0.755  133  72
  separate+joint_soft+endtoend                mean   0.8682  80.5%  0.790 0.818   99  97
  separate+joint_soft+endtoend                confw  0.8679  80.3%  0.797 0.808  104  94
  separate+joint+joint_soft+dualpath+endtoend mean   0.8671  80.6%  0.810 0.803  107  88
  separate+joint_soft+dualpath+endtoend       mean   0.8669  80.4%  0.747 0.853   80 117
  separate+

In [11]:
# ══════════════════════════════════════════════════════════════════════
# END-TO-END VARIANTS — three more predicted-mask models, then ensemble
#   ALL use predicted masks. Nothing uses reference masks.
#   V1: soft labels (BI-RADS-conditioned)
#   V2: attention weight 1+3*mask (stronger lesion focus)
#   V3: no aux heads, different seed
#   Then ensembles them with your existing end-to-end model.
#   Requires the Phase 3 cell to have been run (uses d, CACHE, DS, Net, meta, aux, AK, y)
# ══════════════════════════════════════════════════════════════════════
import os, time, itertools, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
D="/root/autodl-tmp/CBIS"

# soft targets from BI-RADS malignancy rate (training only)
prior=d.groupby("assessment")["label"].mean()
def softt(r):
    p=prior.get(r["assessment"],0.5); a=0.40*(1-2*abs(p-0.5))
    return (1-a)*float(r["label"])+a*p
d["soft"]=d.apply(softt,axis=1).astype(np.float32)

class NetW(nn.Module):
    def __init__(s,meta,att=2.0,use_aux=True):
        super().__init__()
        try: dn=models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        except Exception: dn=models.densenet121(weights=None)
        s.b=dn.features; s.att=att; s.use_aux=use_aux
        s.head=nn.Sequential(nn.Linear(1024,256),nn.ReLU(),nn.Dropout(0.5),nn.Linear(256,2))
        s.keys=sorted(meta.keys())
        s.aux=nn.ModuleList([nn.Sequential(nn.Linear(1024,128),nn.ReLU(),nn.Dropout(0.3),
                                           nn.Linear(128,meta[k])) for k in s.keys]) if use_aux else nn.ModuleList()
    def forward(s,x,mask):
        f=F.relu(s.b(x))
        m=F.interpolate(mask,size=f.shape[2:],mode="bilinear",align_corners=False)
        w=1.0+s.att*m; f=f*w; g=(f.sum((2,3))/(w.sum((2,3))+1e-6))
        return s.head(g),[h(g) for h in s.aux]

def train_variant(tag, att, use_aux, use_soft, seed_off):
    oof=np.zeros(len(d))
    for k in range(5):
        role=d["role_f"+str(k)]
        tr=np.where(role=="train")[0]; va=np.where(role=="val")[0]; te=np.where(role=="test")[0]
        assert not (set(d.patient_id[tr])&set(d.patient_id[te])), "LEAK"
        t0=time.time(); torch.manual_seed(seed_off+k); np.random.seed(seed_off+k)
        n0=float((y[tr]==0).sum()); n1=float((y[tr]==1).sum())
        w0=n1/(n0+n1); w1=n0/(n0+n1)
        al=torch.tensor([w0,w1],device=DEV)
        soft_t=torch.tensor(d["soft"].values,dtype=torch.float32,device=DEV)
        def lossf(lo,t2,idxb):
            if use_soft:
                lp=F.log_softmax(lo.float(),1); p1=lp[:,1].exp()
                ts=soft_t[idxb]
                ce=-(ts*lp[:,1]*w1+(1-ts)*lp[:,0]*w0)
                pt=ts*p1+(1-ts)*(1-p1)
                return ((1-pt)**GAMMA*ce).mean()
            ce=F.cross_entropy(lo.float(),t2,weight=al,reduction="none"); pt=torch.exp(-ce)
            return ((1-pt)**GAMMA*ce).mean()
        net=NetW(meta,att,use_aux).to(DEV).to(memory_format=torch.channels_last)
        for p_ in net.b.parameters(): p_.requires_grad=False
        sc=torch.amp.GradScaler()
        opt=torch.optim.AdamW([p_ for p_ in net.parameters() if p_.requires_grad],lr=LR_HEAD,weight_decay=1e-3)

        class DS2(DS):
            def __getitem__(s2,i):
                out=DS.__getitem__(s2,i)
                j=s2.idx[i%len(s2.idx)]
                return out+(torch.tensor(int(j)),)
        tl=DataLoader(DS2(tr,True,True),batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)

        @torch.no_grad()
        def col(idx):
            net.eval(); tot=None
            for t in [0,1,2,3]:
                ld=DataLoader(DS(idx,True,False,tta=t),batch_size=20,shuffle=False,num_workers=0); ps=[]
                for x,m,_,_ in ld:
                    x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV)
                    with torch.amp.autocast(device_type="cuda"): o,_=net(x,m)
                    ps+=list(torch.softmax(o.float(),1)[:,1].cpu().numpy())
                ps=np.array(ps); tot=ps if tot is None else tot+ps
            return tot/4
        best=0;bs=None;ni=0
        for ep in range(1,EPOCHS+1):
            if ep==FREEZE+1:
                for p_ in net.b.parameters(): p_.requires_grad=True
                opt=torch.optim.AdamW(net.parameters(),lr=LR_FT,weight_decay=1e-3)
            net.train()
            if ep<=FREEZE: net.b.eval()
            for x,m,t2,a,jb in tl:
                x=x.to(DEV).to(memory_format=torch.channels_last); m=m.to(DEV)
                t2=t2.to(DEV); a=a.to(DEV); jb=jb.to(DEV)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast(device_type="cuda"):
                    o,ax=net(x,m); L=lossf(o,t2,jb)
                    if len(ax):
                        L=L+AUX_W*sum(F.cross_entropy(g.float(),a[:,h],ignore_index=-1)
                                      for h,g in enumerate(ax))/len(ax)
                if not torch.isfinite(L): continue
                sc.scale(L).backward(); sc.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(net.parameters(),5.0); sc.step(opt); sc.update()
            pv=col(va); au=roc_auc_score(y[va],pv) if len(set(y[va]))>1 else 0
            if au>best: best=au; bs={q:v.cpu().clone() for q,v in net.state_dict().items()}; ni=0
            else: ni+=1
            if ni>=6: break
        net.load_state_dict({q:v.to(DEV) for q,v in bs.items()})
        oof[te]=col(te)
        print("    fold "+str(k)+"  AUC "+format(roc_auc_score(y[te],oof[te]),".4f")+
              "  ("+format(time.time()-t0,".0f")+"s)")
    print("  "+tag+" pooled AUC "+format(roc_auc_score(y,oof),".4f"))
    return oof

print("### V1 soft labels, predicted masks");      v1=train_variant("V1",2.0,True, True, 500)
print("### V2 attention 1+3m, predicted masks");   v2=train_variant("V2",3.0,True, False,600)
print("### V3 no aux heads, predicted masks");     v3=train_variant("V3",2.0,False,False,700)

E=d[["img","lesion_key","true"]].copy()
E["e0"]=d["prob_pred"].values; E["e1"]=v1; E["e2"]=v2; E["e3"]=v3
E.to_csv(os.path.join(D,"endtoend_variants_mass.csv"),index=False)
yy=E.true.astype(int).values
print("\nindividual (all predicted-mask):")
for c in ["e0","e1","e2","e3"]: print("  "+c+"  "+format(roc_auc_score(yy,E[c]),".4f"))
print("\ncorrelation:"); print(E[["e0","e1","e2","e3"]].corr(method="spearman").round(3).to_string())

def agg(t,col,how):
    if how=="mean": g=t.groupby("lesion_key").agg(y=("true","max"),p=(col,"mean"))
    else:
        q=t.copy(); q["w"]=np.abs(q[col]-0.5)+1e-3; q["wp"]=q[col]*q["w"]
        g=q.groupby("lesion_key").agg(y=("true","max"),wp=("wp","sum"),w=("w","sum"))
        g["p"]=g.wp/g.w; g=g[["y","p"]]
    return g.reset_index()

res=[]
for r in range(1,5):
    for c in itertools.combinations(["e0","e1","e2","e3"],r):
        rk=np.mean([rankdata(E[n].values)/len(E) for n in c],axis=0)
        t=E[["lesion_key","true"]].copy(); t["x"]=rk
        for how in ["mean","confw"]:
            g=agg(t,"x",how); yv=g.y.astype(int).values; pv=g.p.values
            thr=max([(balanced_accuracy_score(yv,(pv>q).astype(int)),q) for q in np.linspace(.05,.95,181)])[1]
            pr=(pv>thr).astype(int); tn,fp,fn,tp=confusion_matrix(yv,pr,labels=[0,1]).ravel()
            res.append((roc_auc_score(yv,pv),accuracy_score(yv,pr),tp/max(tp+fn,1),tn/max(tn+fp,1),fp,fn,"+".join(c),how))
res.sort(reverse=True)
print("\n  END-TO-END ENSEMBLE (predicted masks only)")
print("  combination          agg      AUC     acc    sens  spec  FP  FN")
for a,ac,se,sp,fp,fn,c,how in res[:8]:
    print("  "+c.ljust(20)+how.ljust(7)+format(a,".4f")+"  "+format(100*ac,".1f")+"%  "+
          format(se,".3f")+" "+format(sp,".3f")+"  "+str(fp).rjust(3)+" "+str(fn).rjust(3))
print("\n  single end-to-end model was: 0.8449")
print("  best end-to-end ensemble:    "+format(res[0][0],".4f")+"  ("+format(res[0][0]-0.8449,"+.4f")+")")

### V1 soft labels, predicted masks
    fold 0  AUC 0.7790  (1690s)
    fold 1  AUC 0.8727  (1368s)
    fold 2  AUC 0.8098  (2095s)
    fold 3  AUC 0.8553  (1683s)
    fold 4  AUC 0.8583  (1427s)
  V1 pooled AUC 0.8293
### V2 attention 1+3m, predicted masks
    fold 0  AUC 0.7770  (1499s)
    fold 1  AUC 0.8722  (1963s)
    fold 2  AUC 0.7946  (1422s)
    fold 3  AUC 0.8404  (2002s)
    fold 4  AUC 0.8436  (2097s)
  V2 pooled AUC 0.8255
### V3 no aux heads, predicted masks
    fold 0  AUC 0.7611  (2135s)
    fold 1  AUC 0.8393  (1284s)
    fold 2  AUC 0.7844  (1530s)
    fold 3  AUC 0.7329  (847s)
    fold 4  AUC 0.8153  (2097s)
  V3 pooled AUC 0.7699

individual (all predicted-mask):
  e0  0.8252
  e1  0.8293
  e2  0.8255
  e3  0.7699

correlation:
       e0     e1     e2     e3
e0  1.000  0.913  0.922  0.734
e1  0.913  1.000  0.897  0.761
e2  0.922  0.897  1.000  0.719
e3  0.734  0.761  0.719  1.000

  END-TO-END ENSEMBLE (predicted masks only)
  combination          agg      AUC    

In [3]:
import os, numpy as np, pandas as pd, cv2
D="/root/autodl-tmp/CBIS"

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv"))
PM=os.path.join(D,"predmasks_mass")
d["pred"]=d["img"].apply(lambda p: os.path.join(PM,os.path.basename(p).replace("_img.png","")+"_pred.png"))

def dice_np(m,g):
    tp=(m&g).sum(); fp=(m&~g).sum(); fn=((~m)&g).sum()
    return (2*tp+1)/(2*tp+fp+fn+1)

vals=[]
for _,r in d.iterrows():
    g=cv2.imread(str(r["msk"]),cv2.IMREAD_GRAYSCALE)
    p=cv2.imread(str(r["pred"]),cv2.IMREAD_GRAYSCALE)
    if g is None or p is None: vals.append(np.nan); continue
    gg=cv2.resize(g,(256,256),interpolation=cv2.INTER_NEAREST)>127
    pp=cv2.resize(p,(256,256),interpolation=cv2.INTER_NEAREST)>127
    vals.append(dice_np(pp,gg))
d["oof_dice"]=vals
d.to_csv(os.path.join(D,"unified_folds_mass.csv"),index=False)

folds=[d[d.fold==k]["oof_dice"].mean() for k in range(5)]
print("per fold: "+str([round(f,4) for f in folds]))
print("mean "+format(np.mean(folds),".4f")+" ± "+format(np.std(folds),".4f")+"   (was 0.8933 pre-TTA)")

per fold: [np.float64(0.8952), np.float64(0.893), np.float64(0.9037), np.float64(0.9074), np.float64(0.8999)]
mean 0.8998 ± 0.0053   (was 0.8933 pre-TTA)


In [2]:
# ══════════════════════════════════════════════════════════════════════
# CELL 5 — MASS: maximise ACCURACY on the honest ensemble.   ~1 min, no GPU
#   (1) tries PROBABILITY averaging as well as RANK averaging
#   (2) chooses a separate operating point per BI-RADS risk group
#   Everything chosen on training folds only, applied to held-out fold.
#   NOW INCLUDES the retrained model: cv_mass_v2_oof.csv
# ══════════════════════════════════════════════════════════════════════
import os, itertools, warnings
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import (roc_auc_score, accuracy_score,
                             balanced_accuracy_score, confusion_matrix)
warnings.filterwarnings("ignore")

D = "/root/autodl-tmp/CBIS"

MODELS = [("cv_mass_endtoend.csv",       "prob_pred"),
          ("phase3_mass.csv",            "prob_pred"),
          ("endtoend_variants_mass.csv", "e1"),
          ("endtoend_variants_mass.csv", "e2"),
          ("endtoend_variants_mass.csv", "e3"),
          ("cv_mass_imageonly_oof.csv",  "prob"),
          ("cv_mass_v2_oof.csv",         "prob")]      # <<< retrained model

# BI-RADS -> risk group (keeps every group big enough to fit a threshold)
def bgroup(a):
    if pd.isna(a):   return "unk"
    a = int(a)
    if a <= 2:       return "low"      # 0,1,2
    if a == 3:       return "b3"
    if a == 4:       return "b4"
    return "b5"                        # 5

base = pd.read_csv(os.path.join(D, "unified_folds_mass.csv"))[
        ["img", "lesion_key", "label", "fold", "assessment"]].copy()
base["label"] = base["label"].astype(int)
base["img"]   = base["img"].astype(str)
base["assessment"] = pd.to_numeric(base["assessment"], errors="coerce")

names = []
for fn, col in MODELS:
    p = os.path.join(D, fn)
    if not os.path.exists(p):
        print("missing", fn); continue
    x = pd.read_csv(p)
    if "img" not in x.columns or col not in x.columns:
        print("skip %s:%s (column not found)" % (fn, col)); continue
    t = pd.DataFrame({"img": x["img"].astype(str),
                      "_p": pd.to_numeric(x[col], errors="coerce")}) \
          .dropna().drop_duplicates(subset="img")
    m = base.merge(t, on="img", how="left")
    if m["_p"].notna().mean() < 0.98:
        print("skip %s:%s (coverage %.0f%%)" % (fn, col, 100 * m["_p"].notna().mean()))
        continue
    tag = "%s:%s" % (fn.replace(".csv", "").replace("cv_", "").replace("_oof", ""), col)
    base[tag] = m["_p"].values
    names.append(tag)

print("models loaded:", len(names))
for n in names:
    g = base.groupby("lesion_key").agg(y=("label", "max"), p=(n, "mean"))
    print("   %-34s AUC %.4f" % (n, roc_auc_score(g.y, g.p)))

combos = [c for r in range(1, min(4, len(names)) + 1)
          for c in itertools.combinations(names, r)]
print("combinations to search:", len(combos))
GRID = np.linspace(0.02, 0.98, 241)


def to_lesion(df, col):
    g = df.groupby("lesion_key").agg(y=("label", "max"), p=(col, "mean"),
                                     a=("assessment", "max")).reset_index()
    g["grp"] = g["a"].apply(bgroup)
    return g


def blend(df, combo, how):
    if how == "rank":
        return np.mean([rankdata(df[n].values) / len(df) for n in combo], axis=0)
    return np.mean([df[n].values for n in combo], axis=0)


def best_thr(y, p, mode="acc"):
    if len(np.unique(y)) < 2:                 # single-class group
        return 0.99 if y.mean() < 0.5 else 0.01
    f = accuracy_score if mode == "acc" else balanced_accuracy_score
    return float(GRID[int(np.argmax([f(y, (p > t).astype(int)) for t in GRID]))])


out = []
for k in sorted(base.fold.dropna().unique()):
    tr, te = base[base.fold != k].copy(), base[base.fold == k].copy()

    # ---- choose blend (subset + rank/prob) on TRAIN folds only ----
    best = (-1, None, None)
    for how in ["rank", "prob"]:
        for c in combos:
            tr["_e"] = blend(tr, c, how)
            L = to_lesion(tr, "_e")
            if L.y.nunique() < 2:
                continue
            a = roc_auc_score(L.y, L.p)
            if a > best[0]:
                best = (a, c, how)
    _, combo, how = best

    tr["_e"] = blend(tr, combo, how)
    te["_e"] = blend(te, combo, how)
    Lt, Le = to_lesion(tr, "_e"), to_lesion(te, "_e")

    # ---- one global threshold ----
    t_glob = best_thr(Lt.y.values, Lt.p.values, "acc")
    Le["pred_global"] = (Le.p > t_glob).astype(int)

    # ---- one threshold per risk group ----
    Le["pred_group"] = 0
    for g in Le.grp.unique():
        sub = Lt[Lt.grp == g]
        t = best_thr(sub.y.values, sub.p.values, "acc") if len(sub) >= 25 else t_glob
        Le.loc[Le.grp == g, "pred_group"] = (Le.loc[Le.grp == g, "p"] > t).astype(int)

    out.append(Le)
    print("  fold %d  n=%-4d  %s  [%s]" % (int(k), len(Le), " + ".join(combo), how))

R = pd.concat(out, ignore_index=True)
print("\nHONEST pooled AUC: %.4f  (n=%d)" % (roc_auc_score(R.y, R.p), len(R)))


def show(pred, tag):
    tn, fp, fn, tp = confusion_matrix(R.y, pred, labels=[0, 1]).ravel()
    print("\n  %s" % tag)
    print("                    pred BENIGN    pred MALIGNANT")
    print("    true BENIGN       TN = %-6d     FP = %-6d" % (tn, fp))
    print("    true MALIGNANT    FN = %-6d     TP = %-6d" % (fn, tp))
    print("    accuracy %.1f%%   sens %.3f   spec %.3f   total errors %d"
          % (100 * accuracy_score(R.y, pred), tp / max(tp + fn, 1),
             tn / max(tn + fp, 1), fp + fn))


print("\n" + "=" * 72); print("CONFUSION MATRICES"); print("=" * 72)
show(R.pred_global, "ONE global threshold  (image only)")
show(R.pred_group,  "PER-BI-RADS thresholds  (model + clinical context)")

print("\n" + "=" * 72)
print("PER-BI-RADS ACCURACY: global -> grouped")
print("=" * 72)
print("  %-8s %-6s %-22s %-22s" % ("BI-RADS", "n", "global (FP/FN/acc)", "grouped (FP/FN/acc)"))
for b in sorted(R.a.dropna().unique()):
    q = R[R.a == b]
    r = []
    for col in ["pred_global", "pred_group"]:
        fp = int(((q.y == 0) & (q[col] == 1)).sum())
        fn = int(((q.y == 1) & (q[col] == 0)).sum())
        r.append("%-3d %-3d %5.1f%%" % (fp, fn, 100 * accuracy_score(q.y, q[col])))
    print("  %-8d %-6d %-22s %-22s" % (int(b), len(q), r[0], r[1]))

R.to_csv(os.path.join(D, "mass_final_operating_points.csv"), index=False)
print("\nsaved mass_final_operating_points.csv")

models loaded: 7
   mass_endtoend:prob_pred            AUC 0.8653
   phase3_mass:prob_pred              AUC 0.8449
   endtoend_variants_mass:e1          AUC 0.8512
   endtoend_variants_mass:e2          AUC 0.8430
   endtoend_variants_mass:e3          AUC 0.7896
   mass_imageonly:prob                AUC 0.7796
   mass_v2:prob                       AUC 0.8692
combinations to search: 98
  fold 0  n=199   mass_endtoend:prob_pred + mass_v2:prob  [rank]
  fold 1  n=201   mass_endtoend:prob_pred + mass_v2:prob  [rank]
  fold 2  n=201   mass_endtoend:prob_pred + mass_v2:prob  [prob]
  fold 3  n=199   mass_endtoend:prob_pred + mass_v2:prob  [rank]
  fold 4  n=205   mass_endtoend:prob_pred + mass_v2:prob  [rank]

HONEST pooled AUC: 0.9022  (n=1005)

CONFUSION MATRICES

  ONE global threshold  (image only)
                    pred BENIGN    pred MALIGNANT
    true BENIGN       TN = 451        FP = 92    
    true MALIGNANT    FN = 91         TP = 371   
    accuracy 81.8%   sens 0.803   spec 0.83

In [1]:
# ══════════════════════════════════════════════════════════════════════
# CELL 5 — MASS: maximise ACCURACY on the honest ensemble.   ~1 min, no GPU
#   (1) tries PROBABILITY averaging as well as RANK averaging
#   (2) chooses a separate operating point per BI-RADS risk group
#   Everything chosen on training folds only, applied to held-out fold.
# ══════════════════════════════════════════════════════════════════════
import os, itertools, warnings
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import (roc_auc_score, accuracy_score,
                             balanced_accuracy_score, confusion_matrix)
warnings.filterwarnings("ignore")

D = "/root/autodl-tmp/CBIS"

MODELS = [("cv_mass_endtoend.csv",       "prob_pred"),
          ("phase3_mass.csv",            "prob_pred"),
          ("endtoend_variants_mass.csv", "e1"),
          ("endtoend_variants_mass.csv", "e2"),
          ("endtoend_variants_mass.csv", "e3"),
          ("cv_mass_imageonly_oof.csv",  "prob")]

# BI-RADS -> risk group (keeps every group big enough to fit a threshold)
def bgroup(a):
    if pd.isna(a):   return "unk"
    a = int(a)
    if a <= 2:       return "low"      # 0,1,2
    if a == 3:       return "b3"
    if a == 4:       return "b4"
    return "b5"                        # 5

base = pd.read_csv(os.path.join(D, "unified_folds_mass.csv"))[
        ["img", "lesion_key", "label", "fold", "assessment"]].copy()
base["label"] = base["label"].astype(int)
base["img"]   = base["img"].astype(str)
base["assessment"] = pd.to_numeric(base["assessment"], errors="coerce")

names = []
for fn, col in MODELS:
    p = os.path.join(D, fn)
    if not os.path.exists(p):
        print("missing", fn); continue
    x = pd.read_csv(p)
    t = pd.DataFrame({"img": x["img"].astype(str),
                      "_p": pd.to_numeric(x[col], errors="coerce")}) \
          .dropna().drop_duplicates(subset="img")
    m = base.merge(t, on="img", how="left")
    if m["_p"].notna().mean() < 0.98:
        print("skip", fn, col); continue
    tag = "%s:%s" % (fn.replace(".csv", "").replace("cv_", "").replace("_oof", ""), col)
    base[tag] = m["_p"].values
    names.append(tag)
print("models:", len(names))

combos = [c for r in range(1, min(4, len(names)) + 1)
          for c in itertools.combinations(names, r)]
GRID = np.linspace(0.02, 0.98, 241)


def to_lesion(df, col):
    g = df.groupby("lesion_key").agg(y=("label", "max"), p=(col, "mean"),
                                     a=("assessment", "max")).reset_index()
    g["grp"] = g["a"].apply(bgroup)
    return g


def blend(df, combo, how):
    if how == "rank":
        return np.mean([rankdata(df[n].values) / len(df) for n in combo], axis=0)
    return np.mean([df[n].values for n in combo], axis=0)


def best_thr(y, p, mode="acc"):
    if len(np.unique(y)) < 2:
        return 0.5 if y.mean() > 0.5 else 0.5
    f = accuracy_score if mode == "acc" else balanced_accuracy_score
    return float(GRID[int(np.argmax([f(y, (p > t).astype(int)) for t in GRID]))])


out = []
for k in sorted(base.fold.dropna().unique()):
    tr, te = base[base.fold != k].copy(), base[base.fold == k].copy()

    # ---- choose blend (subset + rank/prob) on TRAIN folds only ----
    best = (-1, None, None)
    for how in ["rank", "prob"]:
        for c in combos:
            tr["_e"] = blend(tr, c, how)
            L = to_lesion(tr, "_e")
            if L.y.nunique() < 2:
                continue
            a = roc_auc_score(L.y, L.p)
            if a > best[0]:
                best = (a, c, how)
    _, combo, how = best

    tr["_e"] = blend(tr, combo, how)
    te["_e"] = blend(te, combo, how)
    Lt, Le = to_lesion(tr, "_e"), to_lesion(te, "_e")

    # ---- global threshold ----
    t_glob = best_thr(Lt.y.values, Lt.p.values, "acc")
    Le["pred_global"] = (Le.p > t_glob).astype(int)

    # ---- per-risk-group threshold ----
    Le["pred_group"] = 0
    for g in Le.grp.unique():
        sub = Lt[Lt.grp == g]
        t = best_thr(sub.y.values, sub.p.values, "acc") if len(sub) >= 25 else t_glob
        Le.loc[Le.grp == g, "pred_group"] = (Le.loc[Le.grp == g, "p"] > t).astype(int)

    out.append(Le)
    print("  fold %d  n=%-4d  %s  [%s]" % (int(k), len(Le), " + ".join(combo), how))

R = pd.concat(out, ignore_index=True)
print("\nHONEST pooled AUC: %.4f  (n=%d)" % (roc_auc_score(R.y, R.p), len(R)))


def show(pred, tag):
    tn, fp, fn, tp = confusion_matrix(R.y, pred, labels=[0, 1]).ravel()
    print("\n  %s" % tag)
    print("                    pred BENIGN    pred MALIGNANT")
    print("    true BENIGN       TN = %-6d     FP = %-6d" % (tn, fp))
    print("    true MALIGNANT    FN = %-6d     TP = %-6d" % (fn, tp))
    print("    accuracy %.1f%%   sens %.3f   spec %.3f   total errors %d"
          % (100 * accuracy_score(R.y, pred), tp / max(tp + fn, 1),
             tn / max(tn + fp, 1), fp + fn))


print("\n" + "=" * 72); print("CONFUSION MATRICES"); print("=" * 72)
show(R.pred_global, "ONE global threshold")
show(R.pred_group,  "PER-BI-RADS thresholds  (model + clinical context)")

print("\n" + "=" * 72)
print("PER-BI-RADS ACCURACY: global -> grouped")
print("=" * 72)
print("  %-8s %-6s %-22s %-22s" % ("BI-RADS", "n", "global (FP/FN/acc)", "grouped (FP/FN/acc)"))
for b in sorted(R.a.dropna().unique()):
    q = R[R.a == b]
    r = []
    for col in ["pred_global", "pred_group"]:
        fp = int(((q.y == 0) & (q[col] == 1)).sum())
        fn = int(((q.y == 1) & (q[col] == 0)).sum())
        r.append("%-3d %-3d %5.1f%%" % (fp, fn, 100 * accuracy_score(q.y, q[col])))
    print("  %-8d %-6d %-22s %-22s" % (int(b), len(q), r[0], r[1]))

R.to_csv(os.path.join(D, "mass_final_operating_points.csv"), index=False)
print("\nsaved mass_final_operating_points.csv")

models: 6
  fold 0  n=199   mass_endtoend:prob_pred + endtoend_variants_mass:e1  [rank]
  fold 1  n=201   mass_endtoend:prob_pred + phase3_mass:prob_pred  [rank]
  fold 2  n=201   mass_endtoend:prob_pred + phase3_mass:prob_pred  [rank]
  fold 3  n=199   mass_endtoend:prob_pred + phase3_mass:prob_pred  [rank]
  fold 4  n=205   mass_endtoend:prob_pred + phase3_mass:prob_pred  [rank]

HONEST pooled AUC: 0.8867  (n=1005)

CONFUSION MATRICES

  ONE global threshold
                    pred BENIGN    pred MALIGNANT
    true BENIGN       TN = 435        FP = 108   
    true MALIGNANT    FN = 113        TP = 349   
    accuracy 78.0%   sens 0.755   spec 0.801   total errors 221

  PER-BI-RADS thresholds  (model + clinical context)
                    pred BENIGN    pred MALIGNANT
    true BENIGN       TN = 464        FP = 79    
    true MALIGNANT    FN = 72         TP = 390   
    accuracy 85.0%   sens 0.844   spec 0.855   total errors 151

PER-BI-RADS ACCURACY: global -> grouped
  BI-RADS  n

In [1]:
# ══════════════════════════════════════════════════════════════════════
# CELL 6 — MASS CLASSIFIER, RETRAINED (fixed LR + seed averaging)
#          Self-contained. Uses PREDICTED masks only (end-to-end honest).
#
#   >>> RUN WITH QUICK_TEST = True FIRST (about 10 min) <<<
#       If it finishes with no error, set QUICK_TEST = False and rerun.
# ══════════════════════════════════════════════════════════════════════
import os, gc, time
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import (roc_auc_score, accuracy_score,
                             balanced_accuracy_score, confusion_matrix)
cv2.setNumThreads(0)

D   = "/root/autodl-tmp/CBIS"
LES = "mass"
DEV = torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

QUICK_TEST = False          # <<<<<< set False for the real run

S, BATCH   = 512, 12
SEEDS      = [11, 22]
EPOCHS     = 22
FREEZE     = 3             # epochs with backbone frozen
LR_HEAD    = 1e-3          # head, while backbone frozen
LR_HEAD_FT = 3e-4          # head, after unfreeze   <-- was 1e-5 (the bug)
LR_BACK    = 3e-5          # backbone, after unfreeze
WD, GAMMA, AUX_W = 1e-4, 2.0, 0.3
MULT, PATIENCE, ATT = 4, 7, 2.0
FOLDS = [0, 1, 2, 3, 4]

if QUICK_TEST:
    SEEDS, EPOCHS, FREEZE, MULT, FOLDS = [11], 4, 1, 1, [0]
    print(">>> QUICK TEST: 1 fold, 1 seed, 4 epochs — checking for errors only\n")

# ---------------------------------------------------------------- data
d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["label"] = d["label"].astype(int)
PM = os.path.join(D, "predmasks_%s" % LES)
d["predmask"] = d["img"].apply(
    lambda p: os.path.join(PM, os.path.basename(str(p)).replace("_img.png", "") + "_pred.png"))
missing = (~d["predmask"].apply(os.path.exists)).sum()
assert missing == 0, "%d predicted masks missing — rerun Phase 2" % missing
print("%s: %d images | %d patients | %d lesions | malignant %.1f%%"
      % (LES.upper(), len(d), d.patient_id.nunique(), d.lesion_key.nunique(),
         100 * d.label.mean()))

_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
CACHE, t0 = {}, time.time()
for _, r in d.iterrows():
    k = str(r["img"])
    if k in CACHE:
        continue
    im = cv2.imread(k, cv2.IMREAD_GRAYSCALE)
    im = np.zeros((S, S), np.uint8) if im is None else im
    if im.shape != (S, S):
        im = cv2.resize(im, (S, S))
    pm = cv2.imread(str(r["predmask"]), cv2.IMREAD_GRAYSCALE)
    pm = (np.zeros((S, S), np.uint8) if pm is None
          else (cv2.resize(pm, (S, S), interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8))
    CACHE[k] = (_clahe.apply(im), pm)
print("cached %d images in %.0fs" % (len(CACHE), time.time() - t0))

# ------------------------------------------------- auxiliary (helper) heads
def primary(x):
    return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()

aux, meta = {}, {}
for c in ["subtlety", "mass_shape", "mass_margins"]:
    if c not in d.columns or d[c].notna().sum() == 0:
        continue
    if c == "subtlety":
        v = pd.to_numeric(d[c], errors="coerce").where(lambda z: (z >= 1) & (z <= 5))
        codes, n = (v - 1).fillna(-1).astype(int).values, 5
    else:
        pr = d[c].map(primary)
        pr = pr.where(pr.isin(pr.value_counts().head(6).index.tolist()), "OTHER")
        cats = sorted([k for k in pr.unique() if k != "UNK"])
        mp = {k: i for i, k in enumerate(cats)}
        codes, n = pr.map(lambda z: mp.get(z, -1)).astype(int).values, len(cats)
    if n > 1:
        aux[c], meta[c] = codes, n
AK = sorted(aux.keys())
print("helper heads:", meta)

MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
STD  = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)


class DS(Dataset):
    def __init__(self, idx, aug, mult=1, tta=0):
        self.idx  = np.asarray(idx)
        self.aug  = aug
        self.mult = mult if aug else 1
        self.tta  = tta

    def __len__(self):
        return len(self.idx) * self.mult

    def __getitem__(self, i):
        j = int(self.idx[i % len(self.idx)])
        r = d.iloc[j]
        img, msk = CACHE[str(r["img"])]
        img, msk = img.copy(), msk.copy()

        if self.aug:                                  # random each time
            if np.random.rand() < 0.5:
                img, msk = img[:, ::-1], msk[:, ::-1]
            if np.random.rand() < 0.5:
                img, msk = img[::-1, :], msk[::-1, :]
            k = np.random.randint(4)
            if k:
                img, msk = np.rot90(img, k), np.rot90(msk, k)
            img, msk = np.ascontiguousarray(img), np.ascontiguousarray(msk)
            if np.random.rand() < 0.7:
                M = cv2.getRotationMatrix2D((S / 2, S / 2),
                                            np.random.uniform(-25, 25),
                                            np.random.uniform(0.90, 1.12))
                img = cv2.warpAffine(img, M, (S, S), flags=cv2.INTER_LINEAR,
                                     borderMode=cv2.BORDER_REFLECT)
                msk = cv2.warpAffine(msk, M, (S, S), flags=cv2.INTER_NEAREST,
                                     borderMode=cv2.BORDER_CONSTANT)
            if np.random.rand() < 0.5:
                img = np.clip(img.astype(np.float32) * np.random.uniform(0.85, 1.15)
                              + np.random.uniform(-12, 12), 0, 255).astype(np.uint8)
        else:                                          # fixed TTA view
            t = self.tta
            if   t == 1: img, msk = img[:, ::-1], msk[:, ::-1]
            elif t == 2: img, msk = img[::-1, :], msk[::-1, :]
            elif t == 3: img, msk = np.rot90(img, 2), np.rot90(msk, 2)

        img, msk = np.ascontiguousarray(img), np.ascontiguousarray(msk)
        im = img.astype(np.float32) / 255.0
        x  = ((np.stack([im, im, im], 0) - MEAN) / STD).astype(np.float32)
        av = (np.array([aux[c][j] for c in AK], dtype=np.int64)
              if AK else np.zeros(0, np.int64))
        return (torch.from_numpy(x),
                torch.from_numpy(msk.astype(np.float32))[None],
                torch.tensor(int(r["label"])),
                torch.from_numpy(av))


class GuidedNet(nn.Module):
    """DenseNet-121 + mask-guided pooling AND whole-image pooling."""
    def __init__(self, aux_meta, att=2.0):
        super().__init__()
        try:
            dn = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        except Exception:
            dn = models.densenet121(weights=None)
            print("  (ImageNet weights unavailable — training from scratch)")
        self.b, self.att = dn.features, att
        Fdim = 1024
        self.head = nn.Sequential(
            nn.Linear(Fdim * 2, 512), nn.BatchNorm1d(512), nn.ReLU(True),
            nn.Dropout(0.4), nn.Linear(512, 2))
        self.keys = sorted(aux_meta.keys())
        self.aux = nn.ModuleList([
            nn.Sequential(nn.Linear(Fdim * 2, 128), nn.ReLU(True),
                          nn.Dropout(0.3), nn.Linear(128, aux_meta[k]))
            for k in self.keys])

    def forward(self, x, mask):
        f = F.relu(self.b(x))
        m = F.interpolate(mask, size=f.shape[2:], mode="bilinear", align_corners=False)
        w = 1.0 + self.att * m
        g_les = (f * w).sum((2, 3)) / (w.sum((2, 3)) + 1e-6)   # lesion-focused
        g_all = f.mean((2, 3))                                  # whole crop
        g = torch.cat([g_les, g_all], 1)
        return self.head(g), [h(g) for h in self.aux]


def focal(logits, target, alpha):
    ce = F.cross_entropy(logits.float(), target, weight=alpha, reduction="none")
    pt = torch.exp(-ce)
    return ((1 - pt) ** GAMMA * ce).mean()


@torch.no_grad()
def predict(net, idx, tta=True):
    net.eval()
    tot = None
    for t in ([0, 1, 2, 3] if tta else [0]):
        ld = DataLoader(DS(idx, False, 1, tta=t), batch_size=20,
                        shuffle=False, num_workers=0)
        ps = []
        for x, m, _, _ in ld:
            x = x.to(DEV).to(memory_format=torch.channels_last)
            m = m.to(DEV)
            with torch.amp.autocast(device_type="cuda"):
                o, _ = net(x, m)
            ps += list(torch.softmax(o.float(), 1)[:, 1].cpu().numpy())
        ps = np.array(ps)
        tot = ps if tot is None else tot + ps
    return tot / (4 if tta else 1)


def train_one(tr, va, te, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    y = d["label"].values
    n0, n1 = float((y[tr] == 0).sum()), float((y[tr] == 1).sum())
    alpha = torch.tensor([n1 / (n0 + n1), n0 / (n0 + n1)],
                         device=DEV, dtype=torch.float32)

    net = GuidedNet(meta, ATT).to(DEV).to(memory_format=torch.channels_last)
    for p in net.b.parameters():
        p.requires_grad = False
    head_params = [p for n_, p in net.named_parameters() if not n_.startswith("b.")]

    scaler = torch.amp.GradScaler()
    opt = torch.optim.AdamW(head_params, lr=LR_HEAD, weight_decay=WD)
    sch = None
    tl = DataLoader(DS(tr, True, MULT), batch_size=BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)

    best, best_state, bad = -1.0, None, 0
    for ep in range(1, EPOCHS + 1):
        if ep == FREEZE + 1:                      # ---- THE FIX ----
            for p in net.b.parameters():
                p.requires_grad = True
            opt = torch.optim.AdamW(
                [{"params": net.b.parameters(), "lr": LR_BACK},
                 {"params": head_params,        "lr": LR_HEAD_FT}],
                weight_decay=WD)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(
                opt, T_max=max(1, EPOCHS - FREEZE))

        net.train()
        if ep <= FREEZE:
            net.b.eval()
        for x, m, t, a in tl:
            x = x.to(DEV, non_blocking=True).to(memory_format=torch.channels_last)
            m = m.to(DEV, non_blocking=True); t = t.to(DEV); a = a.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o, ax = net(x, m)
                loss = focal(o, t, alpha)
                if len(ax):
                    loss = loss + AUX_W * sum(
                        F.cross_entropy(g.float(), a[:, h], ignore_index=-1)
                        for h, g in enumerate(ax)) / len(ax)
            if not torch.isfinite(loss):
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            scaler.step(opt); scaler.update()
        if sch is not None:
            sch.step()

        pv = predict(net, va, tta=False)
        auc = roc_auc_score(y[va], pv) if len(set(y[va])) > 1 else 0.0
        star = ""
        if auc > best:
            best, bad = auc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
            star = " *"
        else:
            bad += 1
        print("      ep %2d  val-AUC %.4f%s" % (ep, auc, star))
        if bad >= PATIENCE:
            print("      early stop")
            break

    net.load_state_dict({k: v.to(DEV) for k, v in best_state.items()})
    pt = predict(net, te, tta=True)
    del net; gc.collect(); torch.cuda.empty_cache()
    return pt, best


# ------------------------------------------------------------- run folds
y = d["label"].values
oof = np.full(len(d), np.nan)

for k in FOLDS:
    role = d["role_f%d" % k]
    tr = np.where(role == "train")[0]
    va = np.where(role == "val")[0]
    te = np.where(role == "test")[0]
    assert not (set(d.patient_id[tr]) & set(d.patient_id[te])), "LEAK train/test"
    assert not (set(d.patient_id[va]) & set(d.patient_id[te])), "LEAK val/test"
    print("\n### fold %d | train %d | val %d | test %d" % (k, len(tr), len(va), len(te)))

    preds, t0 = [], time.time()
    for sd in SEEDS:
        print("    seed %d" % sd)
        p, bv = train_one(tr, va, te, sd)
        preds.append(p)
        print("    seed %d done: best val %.4f | test AUC %.4f"
              % (sd, bv, roc_auc_score(y[te], p)))
    oof[te] = np.mean(preds, axis=0)
    print("  FOLD %d  test AUC %.4f  (%.0fs)"
          % (k, roc_auc_score(y[te], oof[te]), time.time() - t0))

# ------------------------------------------------------------- results
done = ~np.isnan(oof)
print("\n" + "=" * 70)
print("RESULTS — %s classifier v2 (predicted masks, %d seeds/fold)"
      % (LES.upper(), len(SEEDS)))
print("=" * 70)
print("  per-image  AUC %.4f   (n=%d)" % (roc_auc_score(y[done], oof[done]), done.sum()))

res = d.loc[done, ["img", "lesion_key", "label"]].copy()
res["prob"] = oof[done]
L = res.groupby("lesion_key").agg(y=("label", "max"), p=("prob", "mean")).reset_index()
grid = np.linspace(0.05, 0.95, 181)
thr = grid[int(np.argmax([balanced_accuracy_score(L.y, (L.p > t).astype(int)) for t in grid]))]
pr = (L.p > thr).astype(int)
tn, fp, fn, tp = confusion_matrix(L.y, pr, labels=[0, 1]).ravel()
print("  per-lesion AUC %.4f   acc %.1f%%  sens %.3f  spec %.3f  FP %d  FN %d"
      % (roc_auc_score(L.y, L.p), 100 * accuracy_score(L.y, pr),
         tp / max(tp + fn, 1), tn / max(tn + fp, 1), fp, fn))

if not QUICK_TEST:
    out = os.path.join(D, "cv_%s_v2_oof.csv" % LES)
    res.rename(columns={"label": "true"}).to_csv(out, index=False)
    print("\n  saved %s" % os.path.basename(out))
    print("  -> add ('cv_%s_v2_oof.csv','prob') to your ensemble list in Cell 5" % LES)
else:
    print("\n  QUICK TEST finished with no errors. Set QUICK_TEST = False and rerun.")

MASS: 1696 images | 892 patients | 1005 lesions | malignant 46.2%
cached 1696 images in 8s
helper heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5}

### fold 0 | train 1202 | val 155 | test 339
    seed 11
      ep  1  val-AUC 0.6755 *
      ep  2  val-AUC 0.7394 *
      ep  3  val-AUC 0.6408
      ep  4  val-AUC 0.7407 *
      ep  5  val-AUC 0.8007 *
      ep  6  val-AUC 0.8009 *
      ep  7  val-AUC 0.8174 *
      ep  8  val-AUC 0.7986
      ep  9  val-AUC 0.8123
      ep 10  val-AUC 0.8258 *
      ep 11  val-AUC 0.7956
      ep 12  val-AUC 0.8044
      ep 13  val-AUC 0.8065
      ep 14  val-AUC 0.7716
      ep 15  val-AUC 0.8021
      ep 16  val-AUC 0.7797
      ep 17  val-AUC 0.7924
      early stop
    seed 11 done: best val 0.8258 | test AUC 0.8276
    seed 22
      ep  1  val-AUC 0.7164 *
      ep  2  val-AUC 0.6941
      ep  3  val-AUC 0.7181 *
      ep  4  val-AUC 0.7508 *
      ep  5  val-AUC 0.7784 *
      ep  6  val-AUC 0.8178 *
      ep  7  val-AUC 0.8212 *
      

In [3]:
# ══════════════════════════════════════════════════════════════════════
# CELL 7 — MASS final operating points (fixes BI-RADS 3 / 0 sensitivity)
#   - BI-RADS 0 no longer shares a threshold with BI-RADS 2
#   - each group must keep sensitivity >= SENS_FLOOR before maximising accuracy
# ══════════════════════════════════════════════════════════════════════
import os, itertools, warnings
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import (roc_auc_score, accuracy_score,
                             balanced_accuracy_score, confusion_matrix)
warnings.filterwarnings("ignore")

D = "/root/autodl-tmp/CBIS"
SENS_FLOOR = 0.70          # try 0.75 / 0.80 to trade accuracy for fewer missed cancers
MIN_GROUP  = 25

MODELS = [("cv_mass_endtoend.csv",       "prob_pred"),
          ("phase3_mass.csv",            "prob_pred"),
          ("endtoend_variants_mass.csv", "e1"),
          ("endtoend_variants_mass.csv", "e2"),
          ("endtoend_variants_mass.csv", "e3"),
          ("cv_mass_imageonly_oof.csv",  "prob"),
          ("cv_mass_v2_oof.csv",         "prob")]

def bgroup(a):
    if pd.isna(a): return "unk"
    a = int(a)
    if a == 0:  return "b0"          # 14% malignant - kept separate
    if a <= 2:  return "b12"         # 1 and 2 - almost all benign
    if a == 3:  return "b3"
    if a == 4:  return "b4"
    return "b5"

base = pd.read_csv(os.path.join(D, "unified_folds_mass.csv"))[
        ["img", "lesion_key", "label", "fold", "assessment"]].copy()
base["label"] = base["label"].astype(int)
base["img"]   = base["img"].astype(str)
base["assessment"] = pd.to_numeric(base["assessment"], errors="coerce")

names = []
for fn, col in MODELS:
    p = os.path.join(D, fn)
    if not os.path.exists(p):
        print("missing", fn); continue
    x = pd.read_csv(p)
    if "img" not in x.columns or col not in x.columns:
        continue
    t = pd.DataFrame({"img": x["img"].astype(str),
                      "_p": pd.to_numeric(x[col], errors="coerce")}) \
          .dropna().drop_duplicates(subset="img")
    m = base.merge(t, on="img", how="left")
    if m["_p"].notna().mean() < 0.98:
        continue
    tag = "%s:%s" % (fn.replace(".csv", "").replace("cv_", "").replace("_oof", ""), col)
    base[tag] = m["_p"].values
    names.append(tag)
print("models loaded:", len(names))

combos = [c for r in range(1, min(4, len(names)) + 1)
          for c in itertools.combinations(names, r)]
GRID = np.linspace(0.02, 0.98, 241)


def to_lesion(df, col):
    g = df.groupby("lesion_key").agg(y=("label", "max"), p=(col, "mean"),
                                     a=("assessment", "max")).reset_index()
    g["grp"] = g["a"].apply(bgroup)
    return g


def blend(df, combo, how):
    if how == "rank":
        return np.mean([rankdata(df[n].values) / len(df) for n in combo], axis=0)
    return np.mean([df[n].values for n in combo], axis=0)


def thr_plain_acc(y, p):
    if len(np.unique(y)) < 2:
        return 0.99 if y.mean() < 0.5 else 0.01
    return float(GRID[int(np.argmax([accuracy_score(y, (p > t).astype(int)) for t in GRID]))])


def thr_sens_floor(y, p, floor):
    """Best accuracy among thresholds that keep sensitivity >= floor."""
    if len(np.unique(y)) < 2:
        return 0.99 if y.mean() < 0.5 else 0.01
    acc, sen = [], []
    for t in GRID:
        pr = (p > t).astype(int)
        acc.append(accuracy_score(y, pr))
        tp = int(((pr == 1) & (y == 1)).sum()); fn = int(((pr == 0) & (y == 1)).sum())
        sen.append(tp / max(tp + fn, 1))
    acc, sen = np.array(acc), np.array(sen)
    ok = sen >= floor
    if ok.any():
        return float(GRID[int(np.argmax(np.where(ok, acc, -1.0)))])
    bal = [balanced_accuracy_score(y, (p > t).astype(int)) for t in GRID]
    return float(GRID[int(np.argmax(bal))])


out = []
for k in sorted(base.fold.dropna().unique()):
    tr, te = base[base.fold != k].copy(), base[base.fold == k].copy()

    best = (-1, None, None)
    for how in ["rank", "prob"]:
        for c in combos:
            tr["_e"] = blend(tr, c, how)
            L = to_lesion(tr, "_e")
            if L.y.nunique() < 2:
                continue
            a = roc_auc_score(L.y, L.p)
            if a > best[0]:
                best = (a, c, how)
    _, combo, how = best

    tr["_e"] = blend(tr, combo, how)
    te["_e"] = blend(te, combo, how)
    Lt, Le = to_lesion(tr, "_e"), to_lesion(te, "_e")

    t_glob = thr_plain_acc(Lt.y.values, Lt.p.values)
    Le["pred_global"] = (Le.p > t_glob).astype(int)

    Le["pred_group"] = 0
    for g in Le.grp.unique():
        sub = Lt[Lt.grp == g]
        t = (thr_sens_floor(sub.y.values, sub.p.values, SENS_FLOOR)
             if len(sub) >= MIN_GROUP else t_glob)
        Le.loc[Le.grp == g, "pred_group"] = (Le.loc[Le.grp == g, "p"] > t).astype(int)

    out.append(Le)
    print("  fold %d  n=%-4d  %s  [%s]" % (int(k), len(Le), " + ".join(combo), how))

R = pd.concat(out, ignore_index=True)
print("\nHONEST pooled AUC: %.4f  (n=%d)" % (roc_auc_score(R.y, R.p), len(R)))


def show(pred, tag):
    tn, fp, fn, tp = confusion_matrix(R.y, pred, labels=[0, 1]).ravel()
    print("\n  %s" % tag)
    print("                    pred BENIGN    pred MALIGNANT")
    print("    true BENIGN       TN = %-6d     FP = %-6d" % (tn, fp))
    print("    true MALIGNANT    FN = %-6d     TP = %-6d" % (fn, tp))
    print("    accuracy %.1f%%   sens %.3f   spec %.3f   MISSED CANCERS %d   total errors %d"
          % (100 * accuracy_score(R.y, pred), tp / max(tp + fn, 1),
             tn / max(tn + fp, 1), fn, fp + fn))


print("\n" + "=" * 76); print("CONFUSION MATRICES"); print("=" * 76)
show(R.pred_global, "ONE global threshold  (image only)")
show(R.pred_group,  "PER-BI-RADS, sensitivity floor %.2f  (model + clinical context)" % SENS_FLOOR)

print("\n" + "=" * 76)
print("PER-BI-RADS: global -> grouped")
print("=" * 76)
print("  %-8s %-6s %-9s %-24s %-24s" % ("BI-RADS", "n", "malig", "global (FP/FN/acc)", "grouped (FP/FN/acc)"))
for b in sorted(R.a.dropna().unique()):
    q = R[R.a == b]
    r = []
    for col in ["pred_global", "pred_group"]:
        fp = int(((q.y == 0) & (q[col] == 1)).sum())
        fn = int(((q.y == 1) & (q[col] == 0)).sum())
        r.append("%-3d %-3d %5.1f%%" % (fp, fn, 100 * accuracy_score(q.y, q[col])))
    print("  %-8d %-6d %-9s %-24s %-24s"
          % (int(b), len(q), "%.0f%%" % (100 * q.y.mean()), r[0], r[1]))

R.to_csv(os.path.join(D, "mass_final_operating_points.csv"), index=False)
print("\nsaved mass_final_operating_points.csv")

models loaded: 7
  fold 0  n=199   mass_endtoend:prob_pred + mass_v2:prob  [rank]
  fold 1  n=201   mass_endtoend:prob_pred + mass_v2:prob  [rank]
  fold 2  n=201   mass_endtoend:prob_pred + mass_v2:prob  [prob]
  fold 3  n=199   mass_endtoend:prob_pred + mass_v2:prob  [rank]
  fold 4  n=205   mass_endtoend:prob_pred + mass_v2:prob  [rank]

HONEST pooled AUC: 0.9022  (n=1005)

CONFUSION MATRICES

  ONE global threshold  (image only)
                    pred BENIGN    pred MALIGNANT
    true BENIGN       TN = 451        FP = 92    
    true MALIGNANT    FN = 91         TP = 371   
    accuracy 81.8%   sens 0.803   spec 0.831   MISSED CANCERS 91   total errors 183

  PER-BI-RADS, sensitivity floor 0.70  (model + clinical context)
                    pred BENIGN    pred MALIGNANT
    true BENIGN       TN = 444        FP = 99    
    true MALIGNANT    FN = 56         TP = 406   
    accuracy 84.6%   sens 0.879   spec 0.818   MISSED CANCERS 56   total errors 155

PER-BI-RADS: global -> grou

In [2]:
# ══════════════════════════════════════════════════════════════════════
# CELL 8b — MASS figures using the FINAL ENSEMBLE (matches your reported
#           numbers: AUC 0.9022, acc 81.8% global / 84.6% per-BI-RADS)
#           Run Cell 7 first so mass_final_operating_points.csv exists.
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, accuracy_score

D   = "/root/autodl-tmp/CBIS"
FIG = os.path.join(D, "figures", "mass_final"); os.makedirs(FIG, exist_ok=True)
S   = 512
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

ens = pd.read_csv(os.path.join(D, "mass_final_operating_points.csv"))   # lesion level
print("ensemble lesions: %d | AUC %.4f" % (len(ens), roc_auc_score(ens.y, ens.p)))

d = pd.read_csv(os.path.join(D, "unified_folds_mass.csv"))
d["assessment"] = pd.to_numeric(d["assessment"], errors="coerce")
PM = os.path.join(D, "predmasks_mass")
d["pred"] = d["img"].apply(
    lambda p: os.path.join(PM, os.path.basename(str(p)).replace("_img.png", "") + "_pred.png"))
d = d[d["pred"].apply(os.path.exists)].reset_index(drop=True)
d = d.merge(ens[["lesion_key", "p", "pred_global", "pred_group"]], on="lesion_key", how="inner")
print("images with ensemble probability: %d" % len(d))


def overlay(img_p, gt_p, pr_p):
    im = cv2.imread(str(img_p), cv2.IMREAD_GRAYSCALE)
    if im is None: return None
    g = _clahe.apply(cv2.resize(im, (S, S)))
    ov = cv2.cvtColor(g, cv2.COLOR_GRAY2RGB)
    for path, colour in [(gt_p, (0, 255, 0)), (pr_p, (255, 60, 60))]:
        m = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if m is None: continue
        m = (cv2.resize(m, (S, S), interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8)
        c, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(ov, c, -1, colour, 3)
    return ov


# ---------------- FIG 3b : summary with the ensemble ----------------
fig, ax = plt.subplots(1, 4, figsize=(22, 4.8))

ax[0].hist(d["oof_dice"].dropna(), bins=40, color="#1f6fb4", edgecolor="white")
ax[0].axvline(d["oof_dice"].mean(), color="red", ls="--",
              label="mean %.3f" % d["oof_dice"].mean())
ax[0].axvline(0.792, color="green", ls=":", lw=2, label="annotation ceiling 0.792")
ax[0].set_xlabel("Dice"); ax[0].set_ylabel("lesions")
ax[0].set_title("Segmentation quality"); ax[0].legend(fontsize=9); ax[0].grid(alpha=.25)

fpr, tpr, _ = roc_curve(ens.y, ens.p)
ax[1].plot(fpr, tpr, lw=2.2, color="#1f6fb4",
           label="ENSEMBLE  AUC = %.4f" % roc_auc_score(ens.y, ens.p))
ax[1].plot([0, 1], [0, 1], "k--", lw=1)
ax[1].set_xlabel("1 - specificity"); ax[1].set_ylabel("sensitivity")
ax[1].set_title("Classification ROC (per lesion)"); ax[1].legend(); ax[1].grid(alpha=.25)

for k, (col, ttl) in enumerate([("pred_global", "One global threshold"),
                                ("pred_group",  "Per-BI-RADS thresholds")]):
    cm = confusion_matrix(ens.y, ens[col], labels=[0, 1])
    a = ax[2 + k]; a.imshow(cm, cmap="Blues")
    for (r_, c_), v in np.ndenumerate(cm):
        a.text(c_, r_, str(v), ha="center", va="center", fontsize=17,
               color="white" if v > cm.max() / 2 else "black")
    a.set_xticks([0, 1]); a.set_xticklabels(["pred benign", "pred malignant"])
    a.set_yticks([0, 1]); a.set_yticklabels(["true benign", "true malignant"])
    a.set_title("%s\nacc %.1f%%   missed cancers %d"
                % (ttl, 100 * accuracy_score(ens.y, ens[col]), int(cm[1, 0])))
plt.tight_layout()
plt.savefig(os.path.join(FIG, "3b_summary_ensemble.png"), dpi=130, bbox_inches="tight")
plt.close()
print("saved 3b_summary_ensemble.png")

# ---------------- FIG 2b : CONFIDENT errors, not marginal ones ----------------
d["case"] = np.where((d.label == 1) & (d.pred_group == 1), "TP",
             np.where((d.label == 0) & (d.pred_group == 0), "TN",
             np.where((d.label == 0) & (d.pred_group == 1), "FP", "FN")))
print(d["case"].value_counts().to_dict())

fig, ax = plt.subplots(4, 4, figsize=(15, 15))
for i, cs in enumerate(["TP", "TN", "FP", "FN"]):
    sel = d[d.case == cs]
    # most CONFIDENT examples in every row: FP = highest prob, FN = lowest prob
    sel = sel.sort_values("p", ascending=(cs in ["TN", "FN"]))
    sel = sel.drop_duplicates(subset="lesion_key").head(4)
    for j in range(4):
        a = ax[i, j]; a.axis("off")
        if j >= len(sel): continue
        r = sel.iloc[j]
        ov = overlay(r["img"], r["msk"], r["pred"])
        if ov is None: continue
        a.imshow(ov)
        a.set_title("%s   p(malignant)=%.2f\ntruth=%s   BI-RADS %s   Dice %.2f"
                    % (cs, r["p"], "MALIGNANT" if r["label"] == 1 else "benign",
                       int(r["assessment"]) if pd.notna(r["assessment"]) else "?",
                       r["oof_dice"] if pd.notna(r["oof_dice"]) else float("nan")),
                    fontsize=8.5,
                    color={"TP": "green", "TN": "green", "FP": "red", "FN": "darkred"}[cs])
plt.suptitle("MASS CLASSIFICATION (ensemble) — most CONFIDENT cases per category\n"
             "green = radiologist outline, red = predicted outline", fontsize=13, y=0.997)
plt.tight_layout()
plt.savefig(os.path.join(FIG, "2b_classification_confident.png"), dpi=110, bbox_inches="tight")
plt.close()
print("saved 2b_classification_confident.png")
print("\nfigures in:", FIG)

ensemble lesions: 1005 | AUC 0.9022
images with ensemble probability: 1696
saved 3b_summary_ensemble.png
{'TN': 752, 'TP': 694, 'FP': 160, 'FN': 90}
saved 2b_classification_confident.png

figures in: /root/autodl-tmp/CBIS/figures/mass_final


In [4]:
# ══════════════════════════════════════════════════════════════════════
# CELL 10 — SPLIT PROTOCOL EXPERIMENT  (mass, fold 0)
#   Identical model + identical data. ONLY the train/test split changes.
#     A) patient-grouped   (your protocol)
#     B) image-level random (the protocol implied when none is reported)
#   Measures the effect on BOTH segmentation Dice and classification AUC.
#
#   >>> RUN WITH QUICK_TEST = True FIRST <<<
# ══════════════════════════════════════════════════════════════════════
import os, gc, time
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score
cv2.setNumThreads(0)

D   = "/root/autodl-tmp/CBIS"
DEV = torch.device("cuda")
torch.backends.cudnn.benchmark = True

QUICK_TEST = False        # <<<<<< set False for the real run
FOLD       = 0
SEED       = 11

SEG_IMG, SEG_BATCH, SEG_EPOCHS, SEG_MULT, SEG_PAT = 256, 16, 25, 4, 6
CLS_IMG, CLS_BATCH, CLS_EPOCHS, CLS_MULT, CLS_PAT = 512, 12, 15, 3, 5
CLS_FREEZE, LR_HEAD, LR_HEAD_FT, LR_BACK, WD, GAMMA = 3, 1e-3, 3e-4, 3e-5, 1e-4, 2.0

if QUICK_TEST:
    SEG_EPOCHS, SEG_MULT, CLS_EPOCHS, CLS_MULT, CLS_FREEZE = 3, 1, 3, 1, 1
    print(">>> QUICK TEST — error check only\n")

d = pd.read_csv(os.path.join(D, "unified_folds_mass.csv")).reset_index(drop=True)
d["label"] = d["label"].astype(int)
PM = os.path.join(D, "predmasks_mass")
d["pred"] = d["img"].apply(lambda p: os.path.join(
    PM, os.path.basename(str(p)).replace("_img.png", "") + "_pred.png"))
assert d["pred"].apply(os.path.exists).all(), "predmasks_mass incomplete"

# ══════════════ build the two splits ══════════════
role = d["role_f%d" % FOLD]
G = dict(tr=np.where(role == "train")[0],
         va=np.where(role == "val")[0],
         te=np.where(role == "test")[0])

rs = np.random.RandomState(SEED)
perm = rs.permutation(len(d))
n_tr, n_va = len(G["tr"]), len(G["va"])
L = dict(tr=perm[:n_tr], va=perm[n_tr:n_tr + n_va], te=perm[n_tr + n_va:])

def leak_report(S, name):
    trp, tep = set(d.patient_id[S["tr"]]), set(d.patient_id[S["te"]])
    trl, tel = set(d.lesion_key[S["tr"]]), set(d.lesion_key[S["te"]])
    shared_les = len(trl & tel)
    print("  %-16s train %-5d val %-4d test %-5d | patients in both: %-4d | "
          "test lesions whose other view is in train: %d / %d  (%.0f%%)"
          % (name, len(S["tr"]), len(S["va"]), len(S["te"]),
             len(trp & tep), shared_les, len(tel), 100 * shared_les / max(len(tel), 1)))

print("=" * 96)
print("SPLIT CONSTRUCTION — fold %d, identical sizes" % FOLD)
print("=" * 96)
leak_report(G, "A patient-grouped")
leak_report(L, "B image-level")
print("=" * 96)

# ══════════════ shared caches ══════════════
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
SEG_CACHE, CLS_CACHE = {}, {}
t0 = time.time()
for _, r in d.iterrows():
    k = str(r["img"])
    im = cv2.imread(k, cv2.IMREAD_GRAYSCALE)
    im = np.zeros((512, 512), np.uint8) if im is None else im
    mk = cv2.imread(str(r["msk"]), cv2.IMREAD_GRAYSCALE)
    mk = np.zeros_like(im) if mk is None else mk
    pm = cv2.imread(str(r["pred"]), cv2.IMREAD_GRAYSCALE)
    pm = np.zeros_like(im) if pm is None else pm
    SEG_CACHE[k] = (_clahe.apply(cv2.resize(im, (SEG_IMG, SEG_IMG))),
                    (cv2.resize(mk, (SEG_IMG, SEG_IMG), interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8))
    CLS_CACHE[k] = (_clahe.apply(cv2.resize(im, (CLS_IMG, CLS_IMG))),
                    (cv2.resize(pm, (CLS_IMG, CLS_IMG), interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8))
print("cached %d images in %.0fs\n" % (len(SEG_CACHE), time.time() - t0))

MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
STD  = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)


def aug_pair(a, b, size):
    if np.random.rand() < .5: a, b = a[:, ::-1], b[:, ::-1]
    if np.random.rand() < .5: a, b = a[::-1, :], b[::-1, :]
    k = np.random.randint(4)
    if k: a, b = np.rot90(a, k), np.rot90(b, k)
    a, b = np.ascontiguousarray(a), np.ascontiguousarray(b)
    if np.random.rand() < .6:
        M = cv2.getRotationMatrix2D((size / 2, size / 2),
                                    np.random.uniform(-25, 25), np.random.uniform(.9, 1.1))
        a = cv2.warpAffine(a, M, (size, size), borderMode=cv2.BORDER_REFLECT)
        b = cv2.warpAffine(b, M, (size, size), flags=cv2.INTER_NEAREST)
    if np.random.rand() < .5:
        a = np.clip(a.astype(np.float32) * np.random.uniform(.85, 1.15), 0, 255).astype(np.uint8)
    return np.ascontiguousarray(a), np.ascontiguousarray(b)


class SegDS(Dataset):
    def __init__(s, idx, aug, mult=1):
        s.idx = np.asarray(idx); s.aug = aug; s.mult = mult if aug else 1
    def __len__(s): return len(s.idx) * s.mult
    def __getitem__(s, i):
        j = int(s.idx[i % len(s.idx)])
        im, mk = [a.copy() for a in SEG_CACHE[str(d.iloc[j]["img"])]]
        if s.aug: im, mk = aug_pair(im, mk, SEG_IMG)
        return (torch.from_numpy(im.astype(np.float32) / 255.).unsqueeze(0),
                torch.from_numpy(mk.astype(np.int64)))


class ClsDS(Dataset):
    def __init__(s, idx, aug, mult=1, tta=0):
        s.idx = np.asarray(idx); s.aug = aug; s.mult = mult if aug else 1; s.tta = tta
    def __len__(s): return len(s.idx) * s.mult
    def __getitem__(s, i):
        j = int(s.idx[i % len(s.idx)])
        im, mk = [a.copy() for a in CLS_CACHE[str(d.iloc[j]["img"])]]
        if s.aug:
            im, mk = aug_pair(im, mk, CLS_IMG)
        else:
            t = s.tta
            if   t == 1: im, mk = im[:, ::-1], mk[:, ::-1]
            elif t == 2: im, mk = im[::-1, :], mk[::-1, :]
            elif t == 3: im, mk = np.rot90(im, 2), np.rot90(mk, 2)
            im, mk = np.ascontiguousarray(im), np.ascontiguousarray(mk)
        g = im.astype(np.float32) / 255.
        x = ((np.stack([g, g, g], 0) - MEAN) / STD).astype(np.float32)
        return (torch.from_numpy(x), torch.from_numpy(mk.astype(np.float32))[None],
                torch.tensor(int(d.iloc[j]["label"])))


# ══════════════ segmentation model ══════════════
def cb(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True),
                         nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True))

class AG(nn.Module):
    def __init__(s, g, x, i):
        super().__init__()
        s.Wg = nn.Sequential(nn.Conv2d(g, i, 1), nn.BatchNorm2d(i))
        s.Wx = nn.Sequential(nn.Conv2d(x, i, 1), nn.BatchNorm2d(i))
        s.psi = nn.Sequential(nn.Conv2d(i, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid()); s.r = nn.ReLU(True)
    def forward(s, g, x): return x * s.psi(s.r(s.Wg(g) + s.Wx(x)))

class ASPP(nn.Module):
    def __init__(s, i, o):
        super().__init__()
        s.b0 = nn.Sequential(nn.Conv2d(i, o, 1), nn.BatchNorm2d(o), nn.ReLU(True))
        s.b1 = nn.Sequential(nn.Conv2d(i, o, 3, padding=6,  dilation=6),  nn.BatchNorm2d(o), nn.ReLU(True))
        s.b2 = nn.Sequential(nn.Conv2d(i, o, 3, padding=12, dilation=12), nn.BatchNorm2d(o), nn.ReLU(True))
        s.b3 = nn.Sequential(nn.Conv2d(i, o, 3, padding=18, dilation=18), nn.BatchNorm2d(o), nn.ReLU(True))
        s.gp = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(i, o, 1), nn.BatchNorm2d(o), nn.ReLU(True))
        s.proj = nn.Sequential(nn.Conv2d(o * 5, o, 1), nn.BatchNorm2d(o), nn.ReLU(True))
    def forward(s, x):
        g = F.interpolate(s.gp(x), size=x.shape[2:], mode="bilinear", align_corners=False)
        return s.proj(torch.cat([s.b0(x), s.b1(x), s.b2(x), s.b3(x), g], 1))

class DSAttnUNet(nn.Module):
    def __init__(s, b=32):
        super().__init__()
        s.e1, s.e2, s.e3, s.e4 = cb(1, b), cb(b, b*2), cb(b*2, b*4), cb(b*4, b*8)
        s.p = nn.MaxPool2d(2); s.bn = ASPP(b*8, b*16)
        s.u4 = nn.ConvTranspose2d(b*16, b*8, 2, 2); s.a4 = AG(b*8, b*8, b*4); s.d4 = cb(b*16, b*8)
        s.u3 = nn.ConvTranspose2d(b*8,  b*4, 2, 2); s.a3 = AG(b*4, b*4, b*2); s.d3 = cb(b*8,  b*4)
        s.u2 = nn.ConvTranspose2d(b*4,  b*2, 2, 2); s.a2 = AG(b*2, b*2, b);   s.d2 = cb(b*4,  b*2)
        s.u1 = nn.ConvTranspose2d(b*2,  b,   2, 2); s.a1 = AG(b, b, b//2);    s.d1 = cb(b*2,  b)
        s.out = nn.Conv2d(b, 2, 1)
        s.ds2, s.ds3, s.ds4 = nn.Conv2d(b*2, 2, 1), nn.Conv2d(b*4, 2, 1), nn.Conv2d(b*8, 2, 1)
    def forward(s, x):
        e1 = s.e1(x); e2 = s.e2(s.p(e1)); e3 = s.e3(s.p(e2)); e4 = s.e4(s.p(e3))
        bo = s.bn(s.p(e4))
        g4 = s.u4(bo); d4 = s.d4(torch.cat([g4, s.a4(g4, e4)], 1))
        g3 = s.u3(d4); d3 = s.d3(torch.cat([g3, s.a3(g3, e3)], 1))
        g2 = s.u2(d3); d2 = s.d2(torch.cat([g2, s.a2(g2, e2)], 1))
        g1 = s.u1(d2); d1 = s.d1(torch.cat([g1, s.a1(g1, e1)], 1))
        if s.training: return s.out(d1), s.ds2(d2), s.ds3(d3), s.ds4(d4)
        return s.out(d1)


def tversky_ce(lo, t):
    ce = F.cross_entropy(lo.float(), t)
    p = F.softmax(lo.float(), 1)[:, 1]; g = t.float()
    tp = (p * g).sum((1, 2)); fp = (p * (1 - g)).sum((1, 2)); fn = ((1 - p) * g).sum((1, 2))
    return 0.3 * ce + 0.7 * (1 - ((tp + 1) / (tp + 0.7 * fp + 0.3 * fn + 1)).mean())


@torch.no_grad()
def seg_dice(net, idx):
    net.eval(); out = []
    for x, y in DataLoader(SegDS(idx, False), batch_size=SEG_BATCH, shuffle=False, num_workers=0):
        x, y = x.to(DEV), y.to(DEV)
        with torch.amp.autocast(device_type="cuda"):
            o = net(x)
        pr = (torch.softmax(o.float(), 1)[:, 1] > 0.5).float(); gt = y.float()
        tp = (pr * gt).sum((1, 2)); fp = (pr * (1 - gt)).sum((1, 2)); fn = ((1 - pr) * gt).sum((1, 2))
        out += ((2 * tp + 1) / (2 * tp + fp + fn + 1)).cpu().tolist()
    return float(np.mean(out))


def run_seg(S, tag):
    torch.manual_seed(SEED); np.random.seed(SEED)
    net = DSAttnUNet().to(DEV)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", patience=4, factor=.5)
    sc = torch.amp.GradScaler()
    tl = DataLoader(SegDS(S["tr"], True, SEG_MULT), batch_size=SEG_BATCH, shuffle=True,
                    num_workers=0, drop_last=True)
    best, bstate, bad = -1, None, 0
    for ep in range(1, SEG_EPOCHS + 1):
        net.train()
        for x, y in tl:
            x, y = x.to(DEV), y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                outs = net(x)
                loss = sum(w * tversky_ce(o, F.interpolate(y[:, None].float(), size=o.shape[2:],
                           mode="nearest")[:, 0].long() if o.shape[2:] != y.shape[1:] else y)
                           for w, o in zip([1.0, .5, .3, .2], outs))
            sc.scale(loss).backward(); sc.step(opt); sc.update()
        v = seg_dice(net, S["va"]); sch.step(v)
        if v > best:
            best, bad = v, 0
            bstate = {q: t.detach().cpu().clone() for q, t in net.state_dict().items()}
        else:
            bad += 1
        print("      [%s] ep %2d  val Dice %.4f" % (tag, ep, v))
        if bad >= SEG_PAT: break
    net.load_state_dict({q: t.to(DEV) for q, t in bstate.items()})
    r = seg_dice(net, S["te"])
    del net; gc.collect(); torch.cuda.empty_cache()
    return r


# ══════════════ classification model ══════════════
class GuidedNet(nn.Module):
    def __init__(s):
        super().__init__()
        try:  dn = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        except Exception: dn = models.densenet121(weights=None)
        s.b = dn.features
        s.head = nn.Sequential(nn.Linear(2048, 512), nn.BatchNorm1d(512), nn.ReLU(True),
                               nn.Dropout(0.4), nn.Linear(512, 2))
    def forward(s, x, m):
        f = F.relu(s.b(x))
        mm = F.interpolate(m, size=f.shape[2:], mode="bilinear", align_corners=False)
        w = 1.0 + 2.0 * mm
        return s.head(torch.cat([(f * w).sum((2, 3)) / (w.sum((2, 3)) + 1e-6), f.mean((2, 3))], 1))


@torch.no_grad()
def cls_probs(net, idx, tta=True):
    net.eval(); tot = None
    for t in ([0, 1, 2, 3] if tta else [0]):
        ps = []
        for x, m, _ in DataLoader(ClsDS(idx, False, 1, tta=t), batch_size=16,
                                  shuffle=False, num_workers=0):
            x, m = x.to(DEV), m.to(DEV)
            with torch.amp.autocast(device_type="cuda"):
                o = net(x, m)
            ps += list(torch.softmax(o.float(), 1)[:, 1].cpu().numpy())
        ps = np.array(ps); tot = ps if tot is None else tot + ps
    return tot / (4 if tta else 1)


def run_cls(S, tag):
    torch.manual_seed(SEED); np.random.seed(SEED)
    y = d["label"].values
    n0, n1 = float((y[S["tr"]] == 0).sum()), float((y[S["tr"]] == 1).sum())
    al = torch.tensor([n1 / (n0 + n1), n0 / (n0 + n1)], device=DEV, dtype=torch.float32)
    net = GuidedNet().to(DEV).to(memory_format=torch.channels_last)
    for p in net.b.parameters(): p.requires_grad = False
    hp = [p for n_, p in net.named_parameters() if not n_.startswith("b.")]
    opt = torch.optim.AdamW(hp, lr=LR_HEAD, weight_decay=WD)
    sch, sc = None, torch.amp.GradScaler()
    tl = DataLoader(ClsDS(S["tr"], True, CLS_MULT), batch_size=CLS_BATCH, shuffle=True,
                    num_workers=0, drop_last=True)
    best, bstate, bad = -1, None, 0
    for ep in range(1, CLS_EPOCHS + 1):
        if ep == CLS_FREEZE + 1:
            for p in net.b.parameters(): p.requires_grad = True
            opt = torch.optim.AdamW([{"params": net.b.parameters(), "lr": LR_BACK},
                                     {"params": hp, "lr": LR_HEAD_FT}], weight_decay=WD)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, CLS_EPOCHS - CLS_FREEZE))
        net.train()
        if ep <= CLS_FREEZE: net.b.eval()
        for x, m, t in tl:
            x = x.to(DEV).to(memory_format=torch.channels_last); m, t = m.to(DEV), t.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o = net(x, m)
                ce = F.cross_entropy(o.float(), t, weight=al, reduction="none")
                loss = ((1 - torch.exp(-ce)) ** GAMMA * ce).mean()
            if not torch.isfinite(loss): continue
            sc.scale(loss).backward(); sc.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0); sc.step(opt); sc.update()
        if sch: sch.step()
        pv = cls_probs(net, S["va"], tta=False)
        a = roc_auc_score(y[S["va"]], pv) if len(set(y[S["va"]])) > 1 else 0
        if a > best:
            best, bad = a, 0
            bstate = {q: t.detach().cpu().clone() for q, t in net.state_dict().items()}
        else:
            bad += 1
        print("      [%s] ep %2d  val AUC %.4f" % (tag, ep, a))
        if bad >= CLS_PAT: break
    net.load_state_dict({q: t.to(DEV) for q, t in bstate.items()})
    p = cls_probs(net, S["te"])
    auc = roc_auc_score(y[S["te"]], p)
    grid = np.linspace(.05, .95, 181)
    thr = grid[int(np.argmax([balanced_accuracy_score(y[S["te"]], (p > t).astype(int)) for t in grid]))]
    acc = accuracy_score(y[S["te"]], (p > thr).astype(int))
    del net; gc.collect(); torch.cuda.empty_cache()
    return auc, acc


# ══════════════ run all four ══════════════
R = {}
for name, S in [("A patient-grouped", G), ("B image-level", L)]:
    print("\n### SEGMENTATION — %s" % name); t0 = time.time()
    R[(name, "dice")] = run_seg(S, name.split()[0])
    print("  TEST Dice %.4f  (%.0fs)" % (R[(name, "dice")], time.time() - t0))

    print("\n### CLASSIFICATION — %s" % name); t0 = time.time()
    auc, acc = run_cls(S, name.split()[0])
    R[(name, "auc")], R[(name, "acc")] = auc, acc
    print("  TEST AUC %.4f  acc %.1f%%  (%.0fs)" % (auc, 100 * acc, time.time() - t0))

print("\n" + "=" * 84)
print("SPLIT PROTOCOL EFFECT — mass, fold %d, identical model and data" % FOLD)
print("=" * 84)
print("%-22s %-16s %-16s %-16s" % ("protocol", "Seg Dice", "Cls AUC", "Cls accuracy"))
print("-" * 84)
for name in ["A patient-grouped", "B image-level"]:
    print("%-22s %-16.4f %-16.4f %-16.1f%%"
          % (name, R[(name, "dice")], R[(name, "auc")], 100 * R[(name, "acc")]))
print("-" * 84)
print("%-22s %+-16.4f %+-16.4f %+-16.1f%%"
      % ("INFLATION",
         R[("B image-level", "dice")] - R[("A patient-grouped", "dice")],
         R[("B image-level", "auc")]  - R[("A patient-grouped", "auc")],
         100 * (R[("B image-level", "acc")] - R[("A patient-grouped", "acc")])))
print("=" * 84)
print("Each CBIS abnormality is imaged in two views. Image-level random splitting")
print("places the same lesion in both training and test sets.")

SPLIT CONSTRUCTION — fold 0, identical sizes
  A patient-grouped train 1202  val 155  test 339   | patients in both: 0    | test lesions whose other view is in train: 0 / 199  (0%)
  B image-level    train 1202  val 155  test 339   | patients in both: 186  | test lesions whose other view is in train: 181 / 302  (60%)
cached 1696 images in 11s


### SEGMENTATION — A patient-grouped
      [A] ep  1  val Dice 0.8632
      [A] ep  2  val Dice 0.8598
      [A] ep  3  val Dice 0.8153
      [A] ep  4  val Dice 0.8634
      [A] ep  5  val Dice 0.8695
      [A] ep  6  val Dice 0.8619
      [A] ep  7  val Dice 0.8518
      [A] ep  8  val Dice 0.8807
      [A] ep  9  val Dice 0.8736
      [A] ep 10  val Dice 0.8810
      [A] ep 11  val Dice 0.8816
      [A] ep 12  val Dice 0.8762
      [A] ep 13  val Dice 0.8728
      [A] ep 14  val Dice 0.8841
      [A] ep 15  val Dice 0.8837
      [A] ep 16  val Dice 0.8929
      [A] ep 17  val Dice 0.8815
      [A] ep 18  val Dice 0.8738
      [A] ep 19  val D

In [2]:
# ══════════════════════════════════════════════════════════════════════
# CELL 12 — FREE GAINS: better ensembling + calibration + resolution check
#   A) is 512px actually native for MASS crops?
#   B) equal-weight vs learned-weight ensembling  (nested, honest)
#   C) probability calibration -> accuracy
#   No GPU. ~15 minutes.
# ══════════════════════════════════════════════════════════════════════
import os, glob, itertools, warnings
import numpy as np, pandas as pd, cv2
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score
warnings.filterwarnings("ignore")

D  = "/root/autodl-tmp/CBIS"
JP = os.path.join(D, "jpeg")

MODELS = [("cv_mass_endtoend.csv",       "prob_pred"),
          ("phase3_mass.csv",            "prob_pred"),
          ("endtoend_variants_mass.csv", "e1"),
          ("endtoend_variants_mass.csv", "e2"),
          ("endtoend_variants_mass.csv", "e3"),
          ("cv_mass_imageonly_oof.csv",  "prob"),
          ("cv_mass_v2_oof.csv",         "prob")]

base = pd.read_csv(os.path.join(D, "unified_folds_mass.csv"))
base["img"] = base["img"].astype(str)
base["label"] = base["label"].astype(int)
base["assessment"] = pd.to_numeric(base["assessment"], errors="coerce")

# ═══════════════ PART A — native resolution of MASS crops ═══════════════
print("=" * 76); print("PART A — are your 512px MASS crops already native?"); print("=" * 76)


def series_files(uid):
    p = os.path.join(JP, str(uid))
    return sorted(glob.glob(os.path.join(p, "*.jpg"))) if os.path.isdir(p) else []


def read_full(uid):
    best, ba = None, -1
    for f in series_files(uid):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is not None and im.size > ba:
            best, ba = im, im.size
    return best


def read_mask(uid, ref):
    c = []
    for f in series_files(uid):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is None:
            continue
        c.append((2.0 * (im.shape == ref) + float(((im < 20) | (im > 235)).mean()), im))
    if not c:
        return None
    c.sort(key=lambda z: -z[0])
    return c[0][1]


sizes, skip = [], 0
for _, r in base.sample(n=min(80, len(base)), random_state=0).iterrows():
    if "full_series" not in base.columns:
        print("  no full_series column — skipping Part A"); break
    full = read_full(r["full_series"])
    fm = read_mask(r["mask_series"], full.shape) if full is not None else None
    if full is None or fm is None:
        skip += 1; continue
    if fm.shape != full.shape:
        fm = cv2.resize(fm, (full.shape[1], full.shape[0]), interpolation=cv2.INTER_NEAREST)
    ys, xs = np.where(fm > 127)
    if len(ys) < 20:
        skip += 1; continue
    ph, pw = int((ys.max() - ys.min()) * .25), int((xs.max() - xs.min()) * .25)
    h = min(ys.max() + ph, full.shape[0]) - max(ys.min() - ph, 0)
    w = min(xs.max() + pw, full.shape[1]) - max(xs.min() - pw, 0)
    sizes.append(max(h, w))

if sizes:
    s = np.array(sizes)
    print("  measured %d lesions (skipped %d)" % (len(s), skip))
    print("  native crop size: median %d px | 25th %d | 75th %d | max %d"
          % (np.median(s), np.percentile(s, 25), np.percentile(s, 75), s.max()))
    print("  fraction LARGER than 512 px: %.0f%%" % (100 * (s > 512).mean()))
    print("  fraction LARGER than 768 px: %.0f%%" % (100 * (s > 768).mean()))
    print()
    if np.median(s) > 700:
        print("  >>> VERDICT: mass crops are being DOWNSAMPLED. Train cells 13-15 at 768px.")
    elif np.median(s) > 560:
        print("  >>> VERDICT: mild downsampling. 640px may give a small gain.")
    else:
        print("  >>> VERDICT: 512px is already native. Keep 512 for cells 13-15.")

# ═══════════════ load models ═══════════════
names = []
for fn, col in MODELS:
    p = os.path.join(D, fn)
    if not os.path.exists(p):
        print("missing", fn); continue
    x = pd.read_csv(p)
    if "img" not in x.columns or col not in x.columns:
        continue
    t = pd.DataFrame({"img": x["img"].astype(str),
                      "_p": pd.to_numeric(x[col], errors="coerce")}).dropna().drop_duplicates("img")
    m = base.merge(t, on="img", how="left")
    if m["_p"].notna().mean() < 0.98:
        continue
    tag = "%s:%s" % (fn.replace(".csv", "").replace("cv_", "").replace("_oof", ""), col)
    base[tag] = m["_p"].values
    names.append(tag)
print("\nmodels loaded:", len(names))

GRID = np.linspace(0.02, 0.98, 241)


def to_lesion(df, col):
    return df.groupby("lesion_key").agg(y=("label", "max"), p=(col, "mean"),
                                        a=("assessment", "max")).reset_index()


def thr_acc(y, p):
    return float(GRID[int(np.argmax([accuracy_score(y, (p > t).astype(int)) for t in GRID]))])


def thr_sens(y, p, floor=0.70):
    acc, sen = [], []
    for t in GRID:
        pr = (p > t).astype(int)
        acc.append(accuracy_score(y, pr))
        tp = ((pr == 1) & (y == 1)).sum(); fn = ((pr == 0) & (y == 1)).sum()
        sen.append(tp / max(tp + fn, 1))
    acc, sen = np.array(acc), np.array(sen)
    ok = sen >= floor
    if ok.any():
        return float(GRID[int(np.argmax(np.where(ok, acc, -1)))])
    return float(GRID[int(np.argmax([balanced_accuracy_score(y, (p > t).astype(int)) for t in GRID]))])


# ═══════════════ PART B — ensembling strategies (nested) ═══════════════
print("\n" + "=" * 76); print("PART B — ensembling strategy (nested leave-one-fold-out)"); print("=" * 76)
combos = [c for r in range(1, min(4, len(names)) + 1) for c in itertools.combinations(names, r)]
folds = sorted(base.fold.dropna().unique())
res = {k: [] for k in ["equal", "stack", "rankstack"]}

for k in folds:
    tr, te = base[base.fold != k].copy(), base[base.fold == k].copy()

    # --- 1. equal-weight best subset (your current method) ---
    best = (-1, None, None)
    for how in ["rank", "prob"]:
        for c in combos:
            tr["_e"] = (np.mean([rankdata(tr[n].values) / len(tr) for n in c], axis=0)
                        if how == "rank" else np.mean([tr[n].values for n in c], axis=0))
            L = to_lesion(tr, "_e")
            if L.y.nunique() < 2:
                continue
            a = roc_auc_score(L.y, L.p)
            if a > best[0]:
                best = (a, c, how)
    _, c, how = best
    te["_e"] = (np.mean([rankdata(te[n].values) / len(te) for n in c], axis=0)
                if how == "rank" else np.mean([te[n].values for n in c], axis=0))
    Le = to_lesion(te, "_e"); Le["fold"] = k
    res["equal"].append(Le.assign(src="equal"))

    # --- 2. learned-weight stacking on raw probabilities ---
    Ltr = tr.groupby("lesion_key").agg({**{n: "mean" for n in names},
                                        "label": "max"}).reset_index()
    Lte = te.groupby("lesion_key").agg({**{n: "mean" for n in names},
                                        "label": "max", "assessment": "max"}).reset_index()
    mdl = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=1.0))
    mdl.fit(Ltr[names].values, Ltr.label.values)
    Lte2 = Lte.rename(columns={"label": "y", "assessment": "a"}).copy()
    Lte2["p"] = mdl.predict_proba(Lte[names].values)[:, 1]
    Lte2["fold"] = k
    res["stack"].append(Lte2[["lesion_key", "y", "p", "a", "fold"]].assign(src="stack"))

    # --- 3. stacking on within-fold RANKS (scale-free) ---
    Rtr = np.column_stack([rankdata(Ltr[n].values) / len(Ltr) for n in names])
    Rte = np.column_stack([rankdata(Lte[n].values) / len(Lte) for n in names])
    mdl2 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=1.0))
    mdl2.fit(Rtr, Ltr.label.values)
    Lte3 = Lte2.copy()
    Lte3["p"] = mdl2.predict_proba(Rte)[:, 1]
    res["rankstack"].append(Lte3[["lesion_key", "y", "p", "a", "fold"]].assign(src="rankstack"))

print("%-14s %-10s %-12s %-12s" % ("strategy", "AUC", "acc (max)", "acc (sens>=.70)"))
print("-" * 60)
BEST = None
for kname in ["equal", "stack", "rankstack"]:
    R = pd.concat(res[kname], ignore_index=True)
    auc = roc_auc_score(R.y, R.p)
    # honest thresholds: chosen on other folds
    hit_a, hit_s = [], []
    for k in folds:
        trm, tem = R[R.fold != k], R[R.fold == k]
        if trm.y.nunique() < 2 or len(tem) == 0:
            continue
        ta = thr_acc(trm.y.values, trm.p.values)
        ts = thr_sens(trm.y.values, trm.p.values)
        hit_a += list(((tem.p.values > ta).astype(int) == tem.y.values).astype(int))
        hit_s += list(((tem.p.values > ts).astype(int) == tem.y.values).astype(int))
    print("%-14s %-10.4f %-12.1f %-12.1f" % (kname, auc, 100 * np.mean(hit_a), 100 * np.mean(hit_s)))
    if BEST is None or auc > BEST[0]:
        BEST = (auc, kname, R)
print("-" * 60)
print("your previous equal-weight result: AUC 0.9022")
print("best here: %s  AUC %.4f  (%+.4f)" % (BEST[1], BEST[0], BEST[0] - 0.9022))

# ═══════════════ PART C — calibration ═══════════════
print("\n" + "=" * 76); print("PART C — probability calibration (helps accuracy, not AUC)"); print("=" * 76)
R = BEST[2].copy()
R["p_cal"] = np.nan
for k in folds:
    trm, tem = R[R.fold != k], R[R.fold == k]
    if trm.y.nunique() < 2 or len(tem) == 0:
        continue
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(trm.p.values, trm.y.values)
    R.loc[R.fold == k, "p_cal"] = iso.predict(tem.p.values)

for col, lab in [("p", "uncalibrated"), ("p_cal", "isotonic calibrated")]:
    hit_a, hit_s = [], []
    for k in folds:
        trm, tem = R[R.fold != k], R[R.fold == k]
        if trm.y.nunique() < 2 or len(tem) == 0:
            continue
        ta = thr_acc(trm.y.values, trm[col].values)
        ts = thr_sens(trm.y.values, trm[col].values)
        hit_a += list(((tem[col].values > ta).astype(int) == tem.y.values).astype(int))
        hit_s += list(((tem[col].values > ts).astype(int) == tem.y.values).astype(int))
    print("  %-22s AUC %.4f   acc(max) %.1f%%   acc(sens>=.70) %.1f%%"
          % (lab, roc_auc_score(R.y, R[col]), 100 * np.mean(hit_a), 100 * np.mean(hit_s)))

R.to_csv(os.path.join(D, "mass_ensemble_v3.csv"), index=False)
print("\nsaved mass_ensemble_v3.csv")
print("\nNEXT: use the Part A verdict to set the input size for cells 13-15.")

PART A — are your 512px MASS crops already native?
  measured 80 lesions (skipped 0)
  native crop size: median 475 px | 25th 368 | 75th 623 | max 1219
  fraction LARGER than 512 px: 44%
  fraction LARGER than 768 px: 14%

  >>> VERDICT: 512px is already native. Keep 512 for cells 13-15.

models loaded: 7

PART B — ensembling strategy (nested leave-one-fold-out)
strategy       AUC        acc (max)    acc (sens>=.70)
------------------------------------------------------------
equal          0.9022     82.2         82.2        
stack          0.8901     80.3         80.3        
rankstack      0.8989     81.5         81.5        
------------------------------------------------------------
your previous equal-weight result: AUC 0.9022
best here: equal  AUC 0.9022  (-0.0000)

PART C — probability calibration (helps accuracy, not AUC)
  uncalibrated           AUC 0.9022   acc(max) 82.2%   acc(sens>=.70) 82.2%
  isotonic calibrated    AUC 0.8911   acc(max) 81.7%   acc(sens>=.70) 81.7%

sav

In [1]:
# ══════════════════════════════════════════════════════════════════════
# CELL 13 — SECOND BACKBONE for the mass ensemble
#   Same protocol as cv_mass_v2 (fixed LRs, 2 seeds, predicted masks).
#   Only the backbone changes -> decorrelated errors -> better ensemble.
#
#   >>> RUN WITH QUICK_TEST = True FIRST <<<
# ══════════════════════════════════════════════════════════════════════
import os, gc, time
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
cv2.setNumThreads(0)

D, DEV = "/root/autodl-tmp/CBIS", torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

QUICK_TEST = False          # <<<<<< set False for the real run
BACKBONE   = "auto"        # "auto" picks the best AVAILABLE pretrained backbone

S, BATCH = 512, 12
SEEDS, EPOCHS, FREEZE = [11, 22], 22, 3
LR_HEAD, LR_HEAD_FT, LR_BACK = 1e-3, 3e-4, 3e-5
WD, GAMMA, AUX_W, MULT, PATIENCE, ATT = 1e-4, 2.0, 0.3, 4, 7, 2.0
FOLDS = [0, 1, 2, 3, 4]
if QUICK_TEST:
    SEEDS, EPOCHS, FREEZE, MULT, FOLDS = [11], 4, 1, 1, [0]
    print(">>> QUICK TEST\n")

# ═══════ which pretrained backbones actually exist offline? ═══════
def build(name):
    if name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        return m.features, 1280
    if name == "efficientnet_b3":
        m = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
        return m.features, 1536
    if name == "convnext_tiny":
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        return m.features, 768
    if name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        return nn.Sequential(*list(m.children())[:-2]), 2048
    raise ValueError(name)

CANDIDATES = ["efficientnet_b0", "convnext_tiny", "resnet50", "efficientnet_b3"]
print("checking which backbones have pretrained weights available offline:")
available = []
for c in CANDIDATES:
    try:
        _f, _d = build(c)
        available.append((c, _d)); del _f
        print("   %-18s OK   (feature dim %d)" % (c, _d))
    except Exception as e:
        print("   %-18s UNAVAILABLE  (%s)" % (c, str(e)[:60]))
    gc.collect()
if not available:
    raise SystemExit("No pretrained backbone available offline. "
                     "Training from scratch is not worth it — stop here.")
BACKBONE = available[0][0] if BACKBONE == "auto" else BACKBONE
FEAT = dict(available)[BACKBONE]
print("\nusing backbone: %s  (feature dim %d)\n" % (BACKBONE, FEAT))

# ═══════ data ═══════
d = pd.read_csv(os.path.join(D, "unified_folds_mass.csv")).reset_index(drop=True)
d["label"] = d["label"].astype(int)
PM = os.path.join(D, "predmasks_mass")
d["predmask"] = d["img"].apply(lambda p: os.path.join(
    PM, os.path.basename(str(p)).replace("_img.png", "") + "_pred.png"))
assert d["predmask"].apply(os.path.exists).all(), "predmasks_mass incomplete"

_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
CACHE, t0 = {}, time.time()
for _, r in d.iterrows():
    k = str(r["img"])
    if k in CACHE:
        continue
    im = cv2.imread(k, cv2.IMREAD_GRAYSCALE)
    im = np.zeros((S, S), np.uint8) if im is None else (
        cv2.resize(im, (S, S)) if im.shape != (S, S) else im)
    pm = cv2.imread(str(r["predmask"]), cv2.IMREAD_GRAYSCALE)
    pm = (np.zeros((S, S), np.uint8) if pm is None
          else (cv2.resize(pm, (S, S), interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8))
    CACHE[k] = (_clahe.apply(im), pm)
print("cached %d in %.0fs" % (len(CACHE), time.time() - t0))

def primary(x):
    return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()

aux, meta = {}, {}
for c in ["subtlety", "mass_shape", "mass_margins"]:
    if c not in d.columns or d[c].notna().sum() == 0:
        continue
    if c == "subtlety":
        v = pd.to_numeric(d[c], errors="coerce").where(lambda z: (z >= 1) & (z <= 5))
        codes, n = (v - 1).fillna(-1).astype(int).values, 5
    else:
        pr = d[c].map(primary)
        pr = pr.where(pr.isin(pr.value_counts().head(6).index.tolist()), "OTHER")
        cats = sorted([k for k in pr.unique() if k != "UNK"])
        mp = {k: i for i, k in enumerate(cats)}
        codes, n = pr.map(lambda z: mp.get(z, -1)).astype(int).values, len(cats)
    if n > 1:
        aux[c], meta[c] = codes, n
AK = sorted(aux.keys())
print("helper heads:", meta)

MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
STD  = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)


class DS(Dataset):
    def __init__(s, idx, aug, mult=1, tta=0):
        s.idx = np.asarray(idx); s.aug = aug; s.mult = mult if aug else 1; s.tta = tta
    def __len__(s): return len(s.idx) * s.mult
    def __getitem__(s, i):
        j = int(s.idx[i % len(s.idx)])
        img, msk = [a.copy() for a in CACHE[str(d.iloc[j]["img"])]]
        if s.aug:
            if np.random.rand() < .5: img, msk = img[:, ::-1], msk[:, ::-1]
            if np.random.rand() < .5: img, msk = img[::-1, :], msk[::-1, :]
            k = np.random.randint(4)
            if k: img, msk = np.rot90(img, k), np.rot90(msk, k)
            img, msk = np.ascontiguousarray(img), np.ascontiguousarray(msk)
            if np.random.rand() < .7:
                M = cv2.getRotationMatrix2D((S/2, S/2), np.random.uniform(-25, 25),
                                            np.random.uniform(.90, 1.12))
                img = cv2.warpAffine(img, M, (S, S), flags=cv2.INTER_LINEAR,
                                     borderMode=cv2.BORDER_REFLECT)
                msk = cv2.warpAffine(msk, M, (S, S), flags=cv2.INTER_NEAREST,
                                     borderMode=cv2.BORDER_CONSTANT)
            if np.random.rand() < .5:
                img = np.clip(img.astype(np.float32) * np.random.uniform(.85, 1.15)
                              + np.random.uniform(-12, 12), 0, 255).astype(np.uint8)
        else:
            t = s.tta
            if   t == 1: img, msk = img[:, ::-1], msk[:, ::-1]
            elif t == 2: img, msk = img[::-1, :], msk[::-1, :]
            elif t == 3: img, msk = np.rot90(img, 2), np.rot90(msk, 2)
        img, msk = np.ascontiguousarray(img), np.ascontiguousarray(msk)
        g = img.astype(np.float32) / 255.
        x = ((np.stack([g, g, g], 0) - MEAN) / STD).astype(np.float32)
        av = (np.array([aux[c][j] for c in AK], dtype=np.int64) if AK else np.zeros(0, np.int64))
        return (torch.from_numpy(x), torch.from_numpy(msk.astype(np.float32))[None],
                torch.tensor(int(d.iloc[j]["label"])), torch.from_numpy(av))


class GuidedNet(nn.Module):
    def __init__(s, aux_meta, backbone, feat, att=2.0):
        super().__init__()
        s.b, _ = build(backbone)
        s.att = att
        s.head = nn.Sequential(nn.Linear(feat * 2, 512), nn.BatchNorm1d(512), nn.ReLU(True),
                               nn.Dropout(0.4), nn.Linear(512, 2))
        s.keys = sorted(aux_meta.keys())
        s.aux = nn.ModuleList([nn.Sequential(nn.Linear(feat * 2, 128), nn.ReLU(True),
                                             nn.Dropout(0.3), nn.Linear(128, aux_meta[k]))
                               for k in s.keys])
    def forward(s, x, mask):
        f = F.relu(s.b(x))
        m = F.interpolate(mask, size=f.shape[2:], mode="bilinear", align_corners=False)
        w = 1.0 + s.att * m
        g = torch.cat([(f * w).sum((2, 3)) / (w.sum((2, 3)) + 1e-6), f.mean((2, 3))], 1)
        return s.head(g), [h(g) for h in s.aux]


def focal(lo, t, alpha):
    ce = F.cross_entropy(lo.float(), t, weight=alpha, reduction="none")
    return ((1 - torch.exp(-ce)) ** GAMMA * ce).mean()


@torch.no_grad()
def predict(net, idx, tta=True):
    net.eval(); tot = None
    for t in ([0, 1, 2, 3] if tta else [0]):
        ps = []
        for x, m, _, _ in DataLoader(DS(idx, False, 1, tta=t), batch_size=20,
                                     shuffle=False, num_workers=0):
            x = x.to(DEV).to(memory_format=torch.channels_last); m = m.to(DEV)
            with torch.amp.autocast(device_type="cuda"):
                o, _ = net(x, m)
            ps += list(torch.softmax(o.float(), 1)[:, 1].cpu().numpy())
        ps = np.array(ps); tot = ps if tot is None else tot + ps
    return tot / (4 if tta else 1)


def train_one(tr, va, te, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    y = d["label"].values
    n0, n1 = float((y[tr] == 0).sum()), float((y[tr] == 1).sum())
    alpha = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV, dtype=torch.float32)

    net = GuidedNet(meta, BACKBONE, FEAT, ATT).to(DEV).to(memory_format=torch.channels_last)
    for p in net.b.parameters(): p.requires_grad = False
    hp = [p for n_, p in net.named_parameters() if not n_.startswith("b.")]
    scaler = torch.amp.GradScaler()
    opt, sch = torch.optim.AdamW(hp, lr=LR_HEAD, weight_decay=WD), None
    tl = DataLoader(DS(tr, True, MULT), batch_size=BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)

    best, bstate, bad = -1.0, None, 0
    for ep in range(1, EPOCHS + 1):
        if ep == FREEZE + 1:
            for p in net.b.parameters(): p.requires_grad = True
            opt = torch.optim.AdamW([{"params": net.b.parameters(), "lr": LR_BACK},
                                     {"params": hp, "lr": LR_HEAD_FT}], weight_decay=WD)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS - FREEZE))
        net.train()
        if ep <= FREEZE: net.b.eval()
        for x, m, t, a in tl:
            x = x.to(DEV, non_blocking=True).to(memory_format=torch.channels_last)
            m, t, a = m.to(DEV), t.to(DEV), a.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o, ax = net(x, m)
                loss = focal(o, t, alpha)
                if len(ax):
                    loss = loss + AUX_W * sum(
                        F.cross_entropy(g.float(), a[:, h], ignore_index=-1)
                        for h, g in enumerate(ax)) / len(ax)
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            scaler.step(opt); scaler.update()
        if sch: sch.step()
        pv = predict(net, va, tta=False)
        auc = roc_auc_score(y[va], pv) if len(set(y[va])) > 1 else 0.0
        star = ""
        if auc > best:
            best, bad = auc, 0
            bstate = {q: v.detach().cpu().clone() for q, v in net.state_dict().items()}
            star = " *"
        else:
            bad += 1
        print("      ep %2d  val-AUC %.4f%s" % (ep, auc, star))
        if bad >= PATIENCE:
            print("      early stop"); break
    net.load_state_dict({q: v.to(DEV) for q, v in bstate.items()})
    p = predict(net, te, tta=True)
    del net; gc.collect(); torch.cuda.empty_cache()
    return p, best


y, oof = d["label"].values, np.full(len(d), np.nan)
for k in FOLDS:
    role = d["role_f%d" % k]
    tr, va, te = (np.where(role == "train")[0], np.where(role == "val")[0],
                  np.where(role == "test")[0])
    assert not (set(d.patient_id[tr]) & set(d.patient_id[te])), "LEAK"
    print("\n### fold %d | train %d val %d test %d" % (k, len(tr), len(va), len(te)))
    preds, t0 = [], time.time()
    for sd in SEEDS:
        print("    seed %d" % sd)
        p, bv = train_one(tr, va, te, sd)
        preds.append(p)
        print("    seed %d: best val %.4f | test AUC %.4f" % (sd, bv, roc_auc_score(y[te], p)))
    oof[te] = np.mean(preds, axis=0)
    print("  FOLD %d  test AUC %.4f  (%.0fs)" % (k, roc_auc_score(y[te], oof[te]), time.time() - t0))

done = ~np.isnan(oof)
print("\n" + "=" * 70)
print("RESULTS — %s (predicted masks, %d seeds/fold)" % (BACKBONE.upper(), len(SEEDS)))
print("=" * 70)
print("  per-image  AUC %.4f  (n=%d)" % (roc_auc_score(y[done], oof[done]), done.sum()))
res = d.loc[done, ["img", "lesion_key", "label"]].copy()
res["prob"] = oof[done]
L = res.groupby("lesion_key").agg(y=("label", "max"), p=("prob", "mean")).reset_index()
grid = np.linspace(.05, .95, 181)
thr = grid[int(np.argmax([balanced_accuracy_score(L.y, (L.p > t).astype(int)) for t in grid]))]
tn, fp, fn, tp = confusion_matrix(L.y, (L.p > thr).astype(int), labels=[0, 1]).ravel()
print("  per-lesion AUC %.4f  acc %.1f%%  sens %.3f  spec %.3f  FP %d FN %d"
      % (roc_auc_score(L.y, L.p), 100 * accuracy_score(L.y, (L.p > thr).astype(int)),
         tp / max(tp+fn, 1), tn / max(tn+fp, 1), fp, fn))
print("\n  for reference: DenseNet-121 v2 = 0.8692 | current ensemble = 0.9022")

if not QUICK_TEST:
    out = os.path.join(D, "cv_mass_%s_oof.csv" % BACKBONE)
    res.rename(columns={"label": "true"}).to_csv(out, index=False)
    print("\n  saved %s" % os.path.basename(out))
    print("  -> add ('cv_mass_%s_oof.csv','prob') to MODELS in Cell 5/11" % BACKBONE)
else:
    print("\n  QUICK TEST done — set QUICK_TEST = False and rerun.")

checking which backbones have pretrained weights available offline:
   efficientnet_b0    OK   (feature dim 1280)
   convnext_tiny      OK   (feature dim 768)
   resnet50           OK   (feature dim 2048)
   efficientnet_b3    OK   (feature dim 1536)

using backbone: efficientnet_b0  (feature dim 1280)

cached 1696 in 8s
helper heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5}

### fold 0 | train 1202 val 155 test 339
    seed 11
      ep  1  val-AUC 0.6389 *
      ep  2  val-AUC 0.6403 *
      ep  3  val-AUC 0.6356
      ep  4  val-AUC 0.6207
      ep  5  val-AUC 0.7246 *
      ep  6  val-AUC 0.7500 *
      ep  7  val-AUC 0.7270
      ep  8  val-AUC 0.7485
      ep  9  val-AUC 0.7852 *
      ep 10  val-AUC 0.8027 *
      ep 11  val-AUC 0.7806
      ep 12  val-AUC 0.7796
      ep 13  val-AUC 0.7816
      ep 14  val-AUC 0.7975
      ep 15  val-AUC 0.7642
      ep 16  val-AUC 0.7809
      ep 17  val-AUC 0.7769
      early stop
    seed 11: best val 0.8027 | test AUC 0.8156
    s

In [2]:
# ══════════════════════════════════════════════════════════════════════
# CELL 7 (v2) — MASS final ensemble + risk-stratified operating points
#   Models not yet trained are skipped automatically, so you can re-run
#   this cell unchanged after each new backbone finishes.
#   No GPU. ~1-2 minutes.
# ══════════════════════════════════════════════════════════════════════
import os, itertools, warnings
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import (roc_auc_score, accuracy_score,
                             balanced_accuracy_score, confusion_matrix)
warnings.filterwarnings("ignore")

D          = "/root/autodl-tmp/CBIS"
SENS_FLOOR = 0.70          # minimum sensitivity per BI-RADS group
MIN_GROUP  = 25
MAX_COMBO  = 4

MODELS = [
    ("cv_mass_endtoend.csv",            "prob_pred"),
    ("phase3_mass.csv",                 "prob_pred"),
    ("endtoend_variants_mass.csv",      "e1"),
    ("endtoend_variants_mass.csv",      "e2"),
    ("endtoend_variants_mass.csv",      "e3"),
    ("cv_mass_imageonly_oof.csv",       "prob"),
    ("cv_mass_v2_oof.csv",              "prob"),
    ("cv_mass_efficientnet_b0_oof.csv", "prob"),   # trained
    ("cv_mass_convnext_tiny_oof.csv",   "prob"),   # skipped until trained
    ("cv_mass_resnet50_oof.csv",        "prob"),   # skipped until trained
    ("cv_mass_efficientnet_b3_oof.csv", "prob"),   # skipped until trained
]


def bgroup(a):
    if pd.isna(a): return "unk"
    a = int(a)
    if a == 0:  return "b0"       # ~14% malignant - kept separate
    if a <= 2:  return "b12"      # 1 and 2 - almost all benign
    if a == 3:  return "b3"
    if a == 4:  return "b4"
    return "b5"


base = pd.read_csv(os.path.join(D, "unified_folds_mass.csv"))[
        ["img", "lesion_key", "label", "fold", "assessment"]].copy()
base["label"] = base["label"].astype(int)
base["img"] = base["img"].astype(str)
base["assessment"] = pd.to_numeric(base["assessment"], errors="coerce")

names, skipped = [], []
for fn, col in MODELS:
    p = os.path.join(D, fn)
    if not os.path.exists(p):
        skipped.append(fn); continue
    x = pd.read_csv(p)
    if "img" not in x.columns or col not in x.columns:
        skipped.append("%s (no '%s')" % (fn, col)); continue
    t = pd.DataFrame({"img": x["img"].astype(str),
                      "_p": pd.to_numeric(x[col], errors="coerce")}) \
          .dropna().drop_duplicates("img")
    m = base.merge(t, on="img", how="left")
    if m["_p"].notna().mean() < 0.98:
        skipped.append("%s (coverage %.0f%%)" % (fn, 100 * m["_p"].notna().mean())); continue
    tag = "%s:%s" % (fn.replace(".csv", "").replace("cv_", "").replace("_oof", ""), col)
    base[tag] = m["_p"].values
    names.append(tag)

print("=" * 78)
print("MODELS LOADED: %d" % len(names))
print("=" * 78)
for n in names:
    g = base.groupby("lesion_key").agg(y=("label", "max"), p=(n, "mean"))
    print("   %-38s AUC %.4f" % (n, roc_auc_score(g.y, g.p)))
if skipped:
    print("\n   not yet available (will be picked up automatically once trained):")
    for s in skipped:
        print("     - %s" % s)

combos = [c for r in range(1, min(MAX_COMBO, len(names)) + 1)
          for c in itertools.combinations(names, r)]
print("\ncombinations searched per fold: %d" % len(combos))
GRID = np.linspace(0.02, 0.98, 241)


def to_lesion(df, col):
    g = df.groupby("lesion_key").agg(y=("label", "max"), p=(col, "mean"),
                                     a=("assessment", "max")).reset_index()
    g["grp"] = g["a"].apply(bgroup)
    return g


def blend(df, combo, how):
    if how == "rank":
        return np.mean([rankdata(df[n].values) / len(df) for n in combo], axis=0)
    return np.mean([df[n].values for n in combo], axis=0)


def thr_plain_acc(y, p):
    if len(np.unique(y)) < 2:
        return 0.99 if y.mean() < 0.5 else 0.01
    return float(GRID[int(np.argmax([accuracy_score(y, (p > t).astype(int)) for t in GRID]))])


def thr_sens_floor(y, p, floor):
    """best accuracy among thresholds keeping sensitivity >= floor"""
    if len(np.unique(y)) < 2:
        return 0.99 if y.mean() < 0.5 else 0.01
    acc, sen = [], []
    for t in GRID:
        pr = (p > t).astype(int)
        acc.append(accuracy_score(y, pr))
        tp = int(((pr == 1) & (y == 1)).sum()); fn = int(((pr == 0) & (y == 1)).sum())
        sen.append(tp / max(tp + fn, 1))
    acc, sen = np.array(acc), np.array(sen)
    ok = sen >= floor
    if ok.any():
        return float(GRID[int(np.argmax(np.where(ok, acc, -1.0)))])
    bal = [balanced_accuracy_score(y, (p > t).astype(int)) for t in GRID]
    return float(GRID[int(np.argmax(bal))])


out = []
print("\nnested selection (combination + thresholds chosen on the other folds):")
for k in sorted(base.fold.dropna().unique()):
    tr, te = base[base.fold != k].copy(), base[base.fold == k].copy()

    best = (-1, None, None)
    for how in ["rank", "prob"]:
        for c in combos:
            tr["_e"] = blend(tr, c, how)
            L = to_lesion(tr, "_e")
            if L.y.nunique() < 2:
                continue
            a = roc_auc_score(L.y, L.p)
            if a > best[0]:
                best = (a, c, how)
    _, combo, how = best

    tr["_e"] = blend(tr, combo, how)
    te["_e"] = blend(te, combo, how)
    Lt, Le = to_lesion(tr, "_e"), to_lesion(te, "_e")

    t_glob = thr_plain_acc(Lt.y.values, Lt.p.values)
    Le["pred_global"] = (Le.p > t_glob).astype(int)

    Le["pred_group"] = 0
    for g in Le.grp.unique():
        sub = Lt[Lt.grp == g]
        t = (thr_sens_floor(sub.y.values, sub.p.values, SENS_FLOOR)
             if len(sub) >= MIN_GROUP else t_glob)
        Le.loc[Le.grp == g, "pred_group"] = (Le.loc[Le.grp == g, "p"] > t).astype(int)

    out.append(Le)
    print("  fold %d  n=%-4d  [%s]  %s" % (int(k), len(Le), how, " + ".join(combo)))

R = pd.concat(out, ignore_index=True)
AUC = roc_auc_score(R.y, R.p)

rng, bs = np.random.RandomState(0), []
for _ in range(400):
    i = rng.randint(0, len(R), len(R))
    if len(set(R.y.values[i])) > 1:
        bs.append(roc_auc_score(R.y.values[i], R.p.values[i]))
lo, hi = np.percentile(bs, 2.5), np.percentile(bs, 97.5)

print("\n" + "=" * 78)
print("HONEST POOLED AUC: %.4f    95%% CI [%.4f, %.4f]    n=%d lesions"
      % (AUC, lo, hi, len(R)))
print("previous best: 0.9022      change: %+.4f" % (AUC - 0.9022))
print("=" * 78)


def show(pred, tag):
    tn, fp, fn, tp = confusion_matrix(R.y, pred, labels=[0, 1]).ravel()
    print("\n  %s" % tag)
    print("                    pred BENIGN    pred MALIGNANT")
    print("    true BENIGN       TN = %-6d     FP = %-6d" % (tn, fp))
    print("    true MALIGNANT    FN = %-6d     TP = %-6d" % (fn, tp))
    print("    accuracy %.1f%%   sens %.3f   spec %.3f   MISSED CANCERS %d   total errors %d"
          % (100 * accuracy_score(R.y, pred), tp / max(tp + fn, 1),
             tn / max(tn + fp, 1), fn, fp + fn))


print("\nCONFUSION MATRICES")
show(R.pred_global, "ONE global threshold  (image only)")
show(R.pred_group,  "PER-BI-RADS, sensitivity floor %.2f  (model + clinical context)" % SENS_FLOOR)

print("\n" + "=" * 78)
print("PER-BI-RADS: global -> grouped")
print("=" * 78)
print("  %-8s %-6s %-8s %-24s %-24s"
      % ("BI-RADS", "n", "malig", "global (FP/FN/acc)", "grouped (FP/FN/acc)"))
for b in sorted(R.a.dropna().unique()):
    q = R[R.a == b]
    r = []
    for col in ["pred_global", "pred_group"]:
        fp = int(((q.y == 0) & (q[col] == 1)).sum())
        fn = int(((q.y == 1) & (q[col] == 0)).sum())
        r.append("%-3d %-3d %5.1f%%" % (fp, fn, 100 * accuracy_score(q.y, q[col])))
    print("  %-8d %-6d %-8s %-24s %-24s"
          % (int(b), len(q), "%.0f%%" % (100 * q.y.mean()), r[0], r[1]))

R.to_csv(os.path.join(D, "mass_final_operating_points.csv"), index=False)
print("\nsaved mass_final_operating_points.csv")

MODELS LOADED: 8
   mass_endtoend:prob_pred                AUC 0.8653
   phase3_mass:prob_pred                  AUC 0.8449
   endtoend_variants_mass:e1              AUC 0.8512
   endtoend_variants_mass:e2              AUC 0.8430
   endtoend_variants_mass:e3              AUC 0.7896
   mass_imageonly:prob                    AUC 0.7796
   mass_v2:prob                           AUC 0.8692
   mass_efficientnet_b0:prob              AUC 0.8536

   not yet available (will be picked up automatically once trained):
     - cv_mass_convnext_tiny_oof.csv
     - cv_mass_resnet50_oof.csv
     - cv_mass_efficientnet_b3_oof.csv

combinations searched per fold: 162

nested selection (combination + thresholds chosen on the other folds):
  fold 0  n=199   [rank]  mass_endtoend:prob_pred + mass_v2:prob
  fold 1  n=201   [rank]  mass_endtoend:prob_pred + mass_v2:prob + mass_efficientnet_b0:prob
  fold 2  n=201   [rank]  mass_endtoend:prob_pred + mass_v2:prob + mass_efficientnet_b0:prob
  fold 3  n=199   [ra

In [4]:
# ══════════════════════════════════════════════════════════════════════
# CELL 14 — IS MORE ENSEMBLING WORTH IT?   no GPU, ~1 minute
#   A) how correlated are your models?
#   B) pairwise ensemble gain
#   C) lesions that EVERY model gets wrong  (the irreducible core)
#   D) oracle ceiling — best possible from combining what you have
# ══════════════════════════════════════════════════════════════════════
import os, itertools, warnings
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score, accuracy_score
warnings.filterwarnings("ignore")

D = "/root/autodl-tmp/CBIS"
MODELS = [
    ("cv_mass_endtoend.csv",            "prob_pred"),
    ("phase3_mass.csv",                 "prob_pred"),
    ("endtoend_variants_mass.csv",      "e1"),
    ("endtoend_variants_mass.csv",      "e2"),
    ("endtoend_variants_mass.csv",      "e3"),
    ("cv_mass_imageonly_oof.csv",       "prob"),
    ("cv_mass_v2_oof.csv",              "prob"),
    ("cv_mass_efficientnet_b0_oof.csv", "prob"),
    ("cv_mass_convnext_tiny_oof.csv",   "prob"),
    ("cv_mass_resnet50_oof.csv",        "prob"),
]

base = pd.read_csv(os.path.join(D, "unified_folds_mass.csv"))[
        ["img", "lesion_key", "label", "fold", "assessment"]].copy()
base["img"] = base["img"].astype(str)
base["label"] = base["label"].astype(int)
base["assessment"] = pd.to_numeric(base["assessment"], errors="coerce")

names = []
for fn, col in MODELS:
    p = os.path.join(D, fn)
    if not os.path.exists(p):
        continue
    x = pd.read_csv(p)
    if "img" not in x.columns or col not in x.columns:
        continue
    t = pd.DataFrame({"img": x["img"].astype(str),
                      "_p": pd.to_numeric(x[col], errors="coerce")}).dropna().drop_duplicates("img")
    m = base.merge(t, on="img", how="left")
    if m["_p"].notna().mean() < 0.98:
        continue
    tag = fn.replace(".csv", "").replace("cv_mass_", "").replace("_oof", "")
    if col.startswith("e") and col[1:].isdigit():
        tag += ":" + col
    base[tag] = m["_p"].values
    names.append(tag)

L = base.groupby("lesion_key").agg({**{n: "mean" for n in names},
                                    "label": "max", "assessment": "max"}).reset_index()
y = L["label"].values
print("models: %d | lesions: %d\n" % (len(names), len(L)))

# ─────────── PART A : correlation ───────────
print("=" * 78); print("PART A — Spearman correlation between models"); print("=" * 78)
C = np.zeros((len(names), len(names)))
for i, a in enumerate(names):
    for j, b in enumerate(names):
        C[i, j] = spearmanr(L[a], L[b]).correlation
hdr = "".join("%8s" % n[:7] for n in names)
print("%-24s%s" % ("", hdr))
for i, a in enumerate(names):
    print("%-24s%s" % (a[:24], "".join("%8.2f" % C[i, j] for j in range(len(names)))))
off = C[np.triu_indices(len(names), 1)]
print("\n  mean off-diagonal correlation: %.2f" % off.mean())
print("  LOW (<0.6) means diverse models -> ensembling should help")
print("  HIGH (>0.8) means they agree -> more of the same will NOT help")

# ─────────── PART B : pairwise ensemble gain ───────────
print("\n" + "=" * 78); print("PART B — does averaging a pair beat the better of the two?"); print("=" * 78)
solo = {n: roc_auc_score(y, L[n]) for n in names}
gains = []
for a, b in itertools.combinations(names, 2):
    e = (rankdata(L[a]) + rankdata(L[b])) / (2 * len(L))
    gains.append((roc_auc_score(y, e) - max(solo[a], solo[b]), a, b))
gains.sort(reverse=True)
print("  best pairings:")
for g, a, b in gains[:6]:
    print("    %+.4f   %s + %s" % (g, a[:22], b[:22]))
print("  worst pairings:")
for g, a, b in gains[-3:]:
    print("    %+.4f   %s + %s" % (g, a[:22], b[:22]))
print("\n  positive = the pair beats both members; negative = averaging hurts")

# ─────────── PART C : irreducible errors ───────────
print("\n" + "=" * 78); print("PART C — lesions EVERY model gets wrong"); print("=" * 78)
GRID = np.linspace(0.05, 0.95, 181)
wrong = np.ones(len(L), dtype=bool)
for n in names:
    t = GRID[int(np.argmax([accuracy_score(y, (L[n] > g).astype(int)) for g in GRID]))]
    wrong &= ((L[n] > t).astype(int) != y)
print("  %d of %d lesions (%.1f%%) are misclassified by ALL %d models"
      % (wrong.sum(), len(L), 100 * wrong.mean(), len(names)))
sub = L[wrong]
print("\n  their BI-RADS distribution:")
for b in sorted(sub.assessment.dropna().unique()):
    q = sub[sub.assessment == b]
    tot = int((L.assessment == b).sum())
    print("    BI-RADS %d  %3d of %-4d  (%.0f%% of that category)  malignant %.0f%%"
          % (int(b), len(q), tot, 100 * len(q) / max(tot, 1), 100 * q.label.mean()))

# ── PART D (fixed) — oracle ceiling. Run right after Cell 14. ──
import numpy as np
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

Rk  = np.column_stack([rankdata(L[n]) / len(L) for n in names])
orc = np.where(y == 1, Rk.max(1), Rk.min(1))      # per lesion, the best any model could do
allavg = roc_auc_score(y, Rk.mean(1))
solo   = {n: roc_auc_score(y, L[n]) for n in names}
oracle = roc_auc_score(y, orc)

print("=" * 78); print("PART D — oracle: best achievable from these models"); print("=" * 78)
print("  best single model      %.4f" % max(solo.values()))
print("  average of ALL models  %.4f" % allavg)
print("  your nested ensemble   0.8991   (0.9022 before EfficientNet)")
print("  ORACLE upper bound     %.4f   <-- ceiling if selection were perfect" % oracle)
head = oracle - 0.9022
print("\n  remaining headroom from ensembling alone: %+.4f" % head)
print()
if head < 0.03:
    print("  VERDICT: little headroom. Extra backbones give marginal returns —")
    print("           stop at 0.9022 and move to writing.")
else:
    print("  VERDICT: real headroom exists -> ConvNeXt / ResNet50 worth training.")

models: 8 | lesions: 1005

PART A — Spearman correlation between models
                         endtoen phase3_ endtoen endtoen endtoen imageon      v2 efficie
endtoend                    1.00    0.67    0.73    0.69    0.55    0.57    0.69    0.65
phase3_mass                 0.67    1.00    0.92    0.93    0.74    0.72    0.87    0.85
endtoend_variants_mass:e    0.73    0.92    1.00    0.90    0.77    0.74    0.87    0.84
endtoend_variants_mass:e    0.69    0.93    0.90    1.00    0.73    0.68    0.86    0.83
endtoend_variants_mass:e    0.55    0.74    0.77    0.73    1.00    0.56    0.72    0.69
imageonly                   0.57    0.72    0.74    0.68    0.56    1.00    0.71    0.71
v2                          0.69    0.87    0.87    0.86    0.72    0.71    1.00    0.87
efficientnet_b0             0.65    0.85    0.84    0.83    0.69    0.71    0.87    1.00

  mean off-diagonal correlation: 0.75
  LOW (<0.6) means diverse models -> ensembling should help
  HIGH (>0.8) means they agr

In [1]:
# ══════════════════════════════════════════════════════════════════════
# CELL 15 — MASS: handcrafted feature fusion — is it worth training?
#   Extracts texture / shape / margin features from PREDICTED masks,
#   then estimates the fusion gain WITHOUT retraining anything.
#   No GPU. ~10-15 minutes.
# ══════════════════════════════════════════════════════════════════════
import os, warnings
import numpy as np, pandas as pd, cv2
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
warnings.filterwarnings("ignore")

try:
    from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
    HAVE_SKIMAGE = True
except Exception:
    HAVE_SKIMAGE = False
print("scikit-image available:", HAVE_SKIMAGE)

D, S = "/root/autodl-tmp/CBIS", 512
d = pd.read_csv(os.path.join(D, "unified_folds_mass.csv")).reset_index(drop=True)
d["label"] = d["label"].astype(int)
PM = os.path.join(D, "predmasks_mass")
d["pred"] = d["img"].apply(lambda p: os.path.join(
    PM, os.path.basename(str(p)).replace("_img.png", "") + "_pred.png"))
assert d["pred"].apply(os.path.exists).all(), "predmasks_mass incomplete"
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))


def feats(img_p, msk_p):
    F = {}
    raw = cv2.imread(str(img_p), cv2.IMREAD_GRAYSCALE)
    if raw is None:
        return None
    if raw.shape != (S, S):
        raw = cv2.resize(raw, (S, S))
    g = _clahe.apply(raw)
    m = cv2.imread(str(msk_p), cv2.IMREAD_GRAYSCALE)
    m = (np.zeros((S, S), np.uint8) if m is None
         else (cv2.resize(m, (S, S), interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8))
    if m.sum() < 30:
        return {k: 0.0 for k in FEATNAMES}

    inside, outside = g[m > 0].astype(np.float32), g[m == 0].astype(np.float32)

    # ---- intensity / contrast ----
    F["in_mean"]  = float(inside.mean())
    F["in_std"]   = float(inside.std())
    F["in_skew"]  = float(((inside - inside.mean()) ** 3).mean() / (inside.std() ** 3 + 1e-6))
    F["in_kurt"]  = float(((inside - inside.mean()) ** 4).mean() / (inside.std() ** 4 + 1e-6))
    F["contrast_io"] = float(inside.mean() - outside.mean())

    # ---- shape from the predicted mask ----
    c, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    c = max(c, key=cv2.contourArea) if c else None
    area = float(m.sum())
    F["area_frac"] = area / (S * S)
    if c is not None and len(c) >= 5:
        per = cv2.arcLength(c, True)
        hull = cv2.convexHull(c)
        ha = max(cv2.contourArea(hull), 1e-6)
        F["circularity"] = float(4 * np.pi * area / max(per * per, 1e-6))
        F["solidity"]    = float(area / ha)                    # low = spiculated / irregular
        F["perim_ratio"] = float(per / max(cv2.arcLength(hull, True), 1e-6))  # spiculation proxy
        (_, _), (MA, ma), _ = cv2.fitEllipse(c)
        big, small = max(MA, ma), max(min(MA, ma), 1e-6)
        F["elongation"] = float(big / small)
    else:
        F["circularity"] = F["solidity"] = F["perim_ratio"] = F["elongation"] = 0.0

    # ---- margin sharpness: gradient magnitude on the boundary ring ----
    k = np.ones((7, 7), np.uint8)
    ring = (cv2.dilate(m, k) - cv2.erode(m, k)) > 0
    gx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
    mag = np.sqrt(gx * gx + gy * gy)
    F["margin_grad_mean"] = float(mag[ring].mean()) if ring.sum() > 0 else 0.0
    F["margin_grad_std"]  = float(mag[ring].std())  if ring.sum() > 0 else 0.0
    F["margin_sharp_ratio"] = float(F["margin_grad_mean"] / (mag[m > 0].mean() + 1e-6))

    # ---- texture inside the lesion ----
    ys, xs = np.where(m > 0)
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    patch = g[y0:y1, x0:x1]
    if HAVE_SKIMAGE and patch.size > 64:
        q = (patch // 8).astype(np.uint8)                      # 32 grey levels
        gl = graycomatrix(q, distances=[1, 3], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                          levels=32, symmetric=True, normed=True)
        for prop in ["contrast", "homogeneity", "energy", "correlation", "dissimilarity"]:
            F["glcm_" + prop] = float(graycoprops(gl, prop).mean())
        lbp = local_binary_pattern(patch, P=8, R=1, method="uniform")
        h, _ = np.histogram(lbp, bins=10, range=(0, 10), density=True)
        for i, v in enumerate(h):
            F["lbp_%d" % i] = float(v)
    else:
        lap = cv2.Laplacian(patch, cv2.CV_32F)
        F["glcm_contrast"] = float(lap.var())
        F["glcm_homogeneity"] = float(1.0 / (1.0 + patch.std()))
        F["glcm_energy"] = float((np.histogram(patch, 32, density=True)[0] ** 2).sum())
        F["glcm_correlation"] = 0.0
        F["glcm_dissimilarity"] = float(np.abs(np.diff(patch.astype(np.float32), axis=1)).mean())
        for i in range(10):
            F["lbp_%d" % i] = 0.0
    return F


# discover feature names from the first readable row
FEATNAMES = list(feats(d.iloc[0]["img"], d.iloc[0]["pred"]).keys())
print("features per lesion:", len(FEATNAMES))

rows = []
for i, r in d.iterrows():
    f = feats(r["img"], r["pred"])
    rows.append(f if f else {k: 0.0 for k in FEATNAMES})
    if (i + 1) % 400 == 0:
        print("  %d / %d" % (i + 1, len(d)))

X = pd.DataFrame(rows)[FEATNAMES]
out = pd.concat([d[["img", "lesion_key", "label", "fold", "assessment",
                    "mass_shape", "mass_margins"]].reset_index(drop=True), X], axis=1)
out.to_csv(os.path.join(D, "mass_handcrafted_features.csv"), index=False)
print("saved mass_handcrafted_features.csv")

Xd = out[FEATNAMES].replace([np.inf, -np.inf], 0).fillna(0).values
y, folds = out["label"].astype(int).values, out["fold"].values


def gcv(Xa, ya, fo, proba=True):
    o = np.zeros(len(ya)) if proba else np.empty(len(ya), dtype=object)
    for k in sorted(np.unique(fo)):
        tr, te = fo != k, fo == k
        if len(np.unique(ya[tr])) < 2:
            continue
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=4000))
        m.fit(Xa[tr], ya[tr])
        o[te] = m.predict_proba(Xa[te])[:, 1] if proba else m.predict(Xa[te])
    return o


def lauc(keys, ya, pa):
    g = pd.DataFrame({"k": keys, "y": ya, "p": pa}).groupby("k").agg(y=("y", "max"), p=("p", "mean"))
    return roc_auc_score(g.y, g.p)


def primary(v):
    return "UNK" if pd.isna(v) else str(v).split("-")[0].strip().upper()


print("\n" + "=" * 74)
print("TEST 1 — do the features recover the radiologist's descriptors?")
print("=" * 74)
for col in ["mass_shape", "mass_margins"]:
    lab = out[col].map(primary)
    keep = [k for k in lab.value_counts()[lab.value_counts() >= 60].index if k != "UNK"]
    m = lab.isin(keep).values
    if m.sum() < 150 or len(keep) < 2:
        print("  %-14s too few categories" % col); continue
    pr = gcv(Xd[m], lab[m].values, folds[m], proba=False)
    bal, ch = balanced_accuracy_score(lab[m].values, pr), 1.0 / len(keep)
    print("  %-14s n=%-5d classes=%-2d  balanced-acc %.1f%%  (chance %.1f%%)  %s"
          % (col, m.sum(), len(keep), 100 * bal, 100 * ch,
             "PASS" if bal > ch * 1.3 else "weak"))

print("\n" + "=" * 74)
print("TEST 2 — features alone")
print("=" * 74)
p_f = gcv(Xd, y, folds)
print("  per-lesion AUC %.4f     (your CNN alone: 0.8692 | ensemble: 0.9022)"
      % lauc(out.lesion_key, y, p_f))

print("\n" + "=" * 74)
print("TEST 3 — DOES FUSION BEAT THE CNN?  (this decides everything)")
print("=" * 74)
CNN = [("cv_mass_v2_oof.csv", "prob"), ("cv_mass_endtoend.csv", "prob_pred"),
       ("cv_mass_efficientnet_b0_oof.csv", "prob"), ("phase3_mass.csv", "prob_pred")]
print("%-34s %-11s %-11s %-9s" % ("CNN source", "CNN alone", "CNN+feats", "gain"))
print("-" * 70)
best = None
for fn, col in CNN:
    p = os.path.join(D, fn)
    if not os.path.exists(p):
        continue
    t = pd.read_csv(p)
    if "img" not in t.columns or col not in t.columns:
        continue
    t = pd.DataFrame({"img": t["img"].astype(str),
                      "cnn": pd.to_numeric(t[col], errors="coerce")}).dropna().drop_duplicates("img")
    M = out.assign(img=out["img"].astype(str)).merge(t, on="img", how="inner")
    if len(M) < 0.95 * len(out):
        continue
    Xm = M[FEATNAMES].replace([np.inf, -np.inf], 0).fillna(0).values
    ym, fm, cm = M.label.astype(int).values, M.fold.values, M.cnn.values
    a0 = lauc(M.lesion_key, ym, cm)
    a1 = lauc(M.lesion_key, ym, gcv(np.column_stack([cm, Xm]), ym, fm))
    print("%-34s %-11.4f %-11.4f %+.4f" % (fn.replace(".csv", ""), a0, a1, a1 - a0))
    if best is None or (a1 - a0) > best[0]:
        best = (a1 - a0, fn, a0, a1)
print("-" * 70)
if best:
    print("best gain %+.4f  (%s: %.4f -> %.4f)" % (best[0], best[1], best[2], best[3]))
    print()
    if best[0] >= 0.020:
        print("VERDICT: build the fusion model — this is a real path to 0.92+")
    elif best[0] >= 0.008:
        print("VERDICT: modest. Fusion helps a little; cheap to keep as a stacking layer.")
    else:
        print("VERDICT: the CNN already encodes this. Try resolution or two-view instead.")

scikit-image available: True
features per lesion: 28
  400 / 1696
  800 / 1696
  1200 / 1696
  1600 / 1696
saved mass_handcrafted_features.csv

TEST 1 — do the features recover the radiologist's descriptors?
  mass_shape     n=1607  classes=5   balanced-acc 27.1%  (chance 20.0%)  PASS
  mass_margins   n=1636  classes=5   balanced-acc 33.3%  (chance 20.0%)  PASS

TEST 2 — features alone
  per-lesion AUC 0.6963     (your CNN alone: 0.8692 | ensemble: 0.9022)

TEST 3 — DOES FUSION BEAT THE CNN?  (this decides everything)
CNN source                         CNN alone   CNN+feats   gain     
----------------------------------------------------------------------
cv_mass_v2_oof                     0.8692      0.8565      -0.0127
cv_mass_endtoend                   0.8653      0.8773      +0.0120
cv_mass_efficientnet_b0_oof        0.8536      0.8438      -0.0099
phase3_mass                        0.8449      0.8325      -0.0124
--------------------------------------------------------------------

In [2]:
# ══════════════════════════════════════════════════════════════════════
# CELL 16 — capture the fusion gain and test it in the ENSEMBLE
#   1. builds out-of-fold fused predictions (endtoend + handcrafted features)
#   2. saves them as new model files
#   3. re-runs the nested ensemble to see if 0.9022 moves
#   No GPU. ~20 minutes.
# ══════════════════════════════════════════════════════════════════════
import os, itertools, warnings
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
warnings.filterwarnings("ignore")

D = "/root/autodl-tmp/CBIS"
FE = pd.read_csv(os.path.join(D, "mass_handcrafted_features.csv"))
FE["img"] = FE["img"].astype(str)
META = ["img", "lesion_key", "label", "fold", "assessment", "mass_shape", "mass_margins"]
FEATNAMES = [c for c in FE.columns if c not in META]
print("handcrafted features:", len(FEATNAMES))

base = pd.read_csv(os.path.join(D, "unified_folds_mass.csv"))[
        ["img", "lesion_key", "label", "fold", "assessment"]].copy()
base["img"] = base["img"].astype(str)
base["label"] = base["label"].astype(int)
base["assessment"] = pd.to_numeric(base["assessment"], errors="coerce")
folds = sorted(base.fold.dropna().unique())


def gcv(Xa, ya, fo):
    o = np.zeros(len(ya))
    for k in folds:
        tr, te = fo != k, fo == k
        if len(np.unique(ya[tr])) < 2:
            continue
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=4000))
        m.fit(Xa[tr], ya[tr]); o[te] = m.predict_proba(Xa[te])[:, 1]
    return o


# ---------- build the two new out-of-fold model files ----------
Xf = FE[FEATNAMES].replace([np.inf, -np.inf], 0).fillna(0).values
yf, ff = FE["label"].astype(int).values, FE["fold"].values

made = []
# (a) features only
p_only = gcv(Xf, yf, ff)
pd.DataFrame({"img": FE["img"], "true": yf, "prob": p_only}) \
  .to_csv(os.path.join(D, "cv_mass_handcrafted_oof.csv"), index=False)
made.append("cv_mass_handcrafted_oof.csv")

# (b) endtoend fused with features
src = os.path.join(D, "cv_mass_endtoend.csv")
t = pd.read_csv(src)
t = pd.DataFrame({"img": t["img"].astype(str),
                  "cnn": pd.to_numeric(t["prob_pred"], errors="coerce")}).dropna().drop_duplicates("img")
M = FE.merge(t, on="img", how="inner")
Xm = np.column_stack([M["cnn"].values,
                      M[FEATNAMES].replace([np.inf, -np.inf], 0).fillna(0).values])
p_fus = gcv(Xm, M["label"].astype(int).values, M["fold"].values)
pd.DataFrame({"img": M["img"], "true": M["label"].astype(int), "prob": p_fus}) \
  .to_csv(os.path.join(D, "cv_mass_endtoend_fused_oof.csv"), index=False)
made.append("cv_mass_endtoend_fused_oof.csv")
print("saved:", ", ".join(made))

# ---------- re-run the nested ensemble with the new members ----------
MODELS = [
    ("cv_mass_endtoend.csv",              "prob_pred"),
    ("phase3_mass.csv",                   "prob_pred"),
    ("endtoend_variants_mass.csv",        "e1"),
    ("endtoend_variants_mass.csv",        "e2"),
    ("endtoend_variants_mass.csv",        "e3"),
    ("cv_mass_imageonly_oof.csv",         "prob"),
    ("cv_mass_v2_oof.csv",                "prob"),
    ("cv_mass_efficientnet_b0_oof.csv",   "prob"),
    ("cv_mass_endtoend_fused_oof.csv",    "prob"),    # NEW
    ("cv_mass_handcrafted_oof.csv",       "prob"),    # NEW
    ("cv_mass_convnext_tiny_oof.csv",     "prob"),    # auto-skipped if absent
]

names = []
for fn, col in MODELS:
    p = os.path.join(D, fn)
    if not os.path.exists(p):
        continue
    x = pd.read_csv(p)
    if "img" not in x.columns or col not in x.columns:
        continue
    tt = pd.DataFrame({"img": x["img"].astype(str),
                       "_p": pd.to_numeric(x[col], errors="coerce")}).dropna().drop_duplicates("img")
    mm = base.merge(tt, on="img", how="left")
    if mm["_p"].notna().mean() < 0.98:
        continue
    tag = fn.replace(".csv", "").replace("cv_mass_", "").replace("_oof", "")
    if col.startswith("e") and col[1:].isdigit():
        tag += ":" + col
    base[tag] = mm["_p"].values
    names.append(tag)

print("\nmodels in pool: %d" % len(names))
for n in names:
    g = base.groupby("lesion_key").agg(y=("label", "max"), p=(n, "mean"))
    print("   %-34s AUC %.4f" % (n, roc_auc_score(g.y, g.p)))

GRID = np.linspace(0.02, 0.98, 241)
combos = [c for r in range(1, 5) for c in itertools.combinations(names, r)]
print("\ncombinations per fold: %d" % len(combos))


def lz(df, col):
    g = df.groupby("lesion_key").agg(y=("label", "max"), p=(col, "mean"),
                                     a=("assessment", "max")).reset_index()
    g["grp"] = g["a"].apply(lambda a: "unk" if pd.isna(a) else
                            ("b0" if int(a) == 0 else "b12" if int(a) <= 2 else
                             "b3" if int(a) == 3 else "b4" if int(a) == 4 else "b5"))
    return g


def bl(df, c, how):
    return (np.mean([rankdata(df[n].values) / len(df) for n in c], axis=0) if how == "rank"
            else np.mean([df[n].values for n in c], axis=0))


def thr_sens(y, p, floor=0.70):
    if len(np.unique(y)) < 2:
        return 0.99 if y.mean() < 0.5 else 0.01
    acc, sen = [], []
    for t in GRID:
        pr = (p > t).astype(int); acc.append(accuracy_score(y, pr))
        tp = ((pr == 1) & (y == 1)).sum(); fn = ((pr == 0) & (y == 1)).sum()
        sen.append(tp / max(tp + fn, 1))
    acc, sen = np.array(acc), np.array(sen)
    ok = sen >= floor
    if ok.any():
        return float(GRID[int(np.argmax(np.where(ok, acc, -1.0)))])
    return float(GRID[int(np.argmax([balanced_accuracy_score(y, (p > t).astype(int)) for t in GRID]))])


out = []
for k in folds:
    tr, te = base[base.fold != k].copy(), base[base.fold == k].copy()
    best = (-1, None, None)
    for how in ["rank", "prob"]:
        for c in combos:
            tr["_e"] = bl(tr, c, how)
            L = lz(tr, "_e")
            if L.y.nunique() < 2:
                continue
            a = roc_auc_score(L.y, L.p)
            if a > best[0]:
                best = (a, c, how)
    _, c, how = best
    tr["_e"], te["_e"] = bl(tr, c, how), bl(te, c, how)
    Lt, Le = lz(tr, "_e"), lz(te, "_e")
    tg = float(GRID[int(np.argmax([accuracy_score(Lt.y, (Lt.p > t).astype(int)) for t in GRID]))])
    Le["pred_global"] = (Le.p > tg).astype(int); Le["pred_group"] = 0
    for g in Le.grp.unique():
        sub = Lt[Lt.grp == g]
        t = thr_sens(sub.y.values, sub.p.values) if len(sub) >= 25 else tg
        Le.loc[Le.grp == g, "pred_group"] = (Le.loc[Le.grp == g, "p"] > t).astype(int)
    out.append(Le)
    print("  fold %d  [%s]  %s" % (int(k), how, " + ".join(c)))

R = pd.concat(out, ignore_index=True)
AUC = roc_auc_score(R.y, R.p)
rng, bs = np.random.RandomState(0), []
for _ in range(400):
    i = rng.randint(0, len(R), len(R))
    if len(set(R.y.values[i])) > 1:
        bs.append(roc_auc_score(R.y.values[i], R.p.values[i]))

print("\n" + "=" * 76)
print("ENSEMBLE WITH FUSION: AUC %.4f   95%% CI [%.4f, %.4f]"
      % (AUC, np.percentile(bs, 2.5), np.percentile(bs, 97.5)))
print("previous: 0.9022    change: %+.4f" % (AUC - 0.9022))
print("=" * 76)
for col, tag in [("pred_global", "one global threshold"),
                 ("pred_group", "per-BI-RADS + sensitivity floor")]:
    tn, fp, fn, tp = confusion_matrix(R.y, R[col], labels=[0, 1]).ravel()
    print("  %-34s acc %.1f%%  sens %.3f  spec %.3f  missed %d"
          % (tag, 100 * accuracy_score(R.y, R[col]),
             tp / max(tp + fn, 1), tn / max(tn + fp, 1), fn))
R.to_csv(os.path.join(D, "mass_final_operating_points.csv"), index=False)
print("\nsaved mass_final_operating_points.csv")

handcrafted features: 28
saved: cv_mass_handcrafted_oof.csv, cv_mass_endtoend_fused_oof.csv

models in pool: 10
   endtoend                           AUC 0.8653
   phase3_mass                        AUC 0.8449
   endtoend_variants_mass:e1          AUC 0.8512
   endtoend_variants_mass:e2          AUC 0.8430
   endtoend_variants_mass:e3          AUC 0.7896
   imageonly                          AUC 0.7796
   v2                                 AUC 0.8692
   efficientnet_b0                    AUC 0.8536
   endtoend_fused                     AUC 0.8773
   handcrafted                        AUC 0.6963

combinations per fold: 385
  fold 0  [rank]  endtoend + v2 + endtoend_fused
  fold 1  [rank]  endtoend + v2 + efficientnet_b0 + endtoend_fused
  fold 2  [rank]  endtoend + v2 + efficientnet_b0 + endtoend_fused
  fold 3  [rank]  endtoend + v2 + efficientnet_b0 + endtoend_fused
  fold 4  [prob]  endtoend + v2 + efficientnet_b0 + endtoend_fused

ENSEMBLE WITH FUSION: AUC 0.9028   95% CI [0.8850, 0

In [3]:
# ══════════════════════════════════════════════════════════════════════
# CELL 17A — REGENERATE MASS CROPS AT 768 px FROM THE FULL MAMMOGRAMS
#   Not upsampling: re-cuts each lesion from the source jpeg at 768.
#   Verifies framing matches your existing crops before any training.
#   No GPU. ~20-30 minutes.
# ══════════════════════════════════════════════════════════════════════
import os, glob, time
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

D    = "/root/autodl-tmp/CBIS"
JP   = os.path.join(D, "jpeg")
OUT  = os.path.join(D, "crops_768_mass"); os.makedirs(OUT, exist_ok=True)
FIG  = os.path.join(D, "figures", "mass_final"); os.makedirs(FIG, exist_ok=True)
SIZE = 768
PAD  = 0.35          # padding around the lesion bbox; tuned to match existing framing
TARGET_COV = 0.249   # mask coverage of your current 512 crops — we must reproduce this

d = pd.read_csv(os.path.join(D, "unified_folds_mass.csv")).reset_index(drop=True)
assert "full_series" in d.columns and "mask_series" in d.columns, "series columns missing"
print("lesions to regenerate: %d" % len(d))


def series_files(uid):
    p = os.path.join(JP, str(uid))
    return sorted(glob.glob(os.path.join(p, "*.jpg"))) if os.path.isdir(p) else []


def read_full(uid):
    best, ba = None, -1
    for f in series_files(uid):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is not None and im.size > ba:
            best, ba = im, im.size
    return best


def read_mask(uid, ref_shape):
    """the mask series can hold BOTH the crop and the mask — pick by shape + binariness"""
    c = []
    for f in series_files(uid):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is None:
            continue
        c.append((2.0 * float(im.shape == ref_shape) + float(((im < 20) | (im > 235)).mean()), im))
    if not c:
        return None
    c.sort(key=lambda z: -z[0])
    return c[0][1]


# ─────────── quick padding calibration on a sample ───────────
print("\ncalibrating PAD to match your existing crop framing (target coverage %.1f%%)..."
      % (100 * TARGET_COV))
samp = d.sample(n=min(60, len(d)), random_state=0)
for trial in [0.25, 0.30, 0.35, 0.40, 0.45]:
    covs = []
    for _, r in samp.iterrows():
        full = read_full(r["full_series"])
        if full is None:
            continue
        fm = read_mask(r["mask_series"], full.shape)
        if fm is None:
            continue
        if fm.shape != full.shape:
            fm = cv2.resize(fm, (full.shape[1], full.shape[0]), interpolation=cv2.INTER_NEAREST)
        ys, xs = np.where(fm > 127)
        if len(ys) < 20:
            continue
        ph, pw = int((ys.max() - ys.min()) * trial), int((xs.max() - xs.min()) * trial)
        y0, y1 = max(ys.min() - ph, 0), min(ys.max() + ph, full.shape[0])
        x0, x1 = max(xs.min() - pw, 0), min(xs.max() + pw, full.shape[1])
        sub = (fm[y0:y1, x0:x1] > 127)
        if sub.size:
            covs.append(sub.mean())
    if covs:
        print("   PAD %.2f  ->  mask coverage %.1f%%" % (trial, 100 * np.median(covs)))
print("   (using PAD = %.2f; adjust above if the coverage is far from %.1f%%)"
      % (PAD, 100 * TARGET_COV))

# ─────────── regenerate ───────────
rows, skip, t0 = [], {}, time.time()
for i, r in d.iterrows():
    full = read_full(r["full_series"])
    if full is None:
        skip["no full jpg"] = skip.get("no full jpg", 0) + 1; continue
    fm = read_mask(r["mask_series"], full.shape)
    if fm is None:
        skip["no mask jpg"] = skip.get("no mask jpg", 0) + 1; continue
    if fm.shape != full.shape:
        fm = cv2.resize(fm, (full.shape[1], full.shape[0]), interpolation=cv2.INTER_NEAREST)
    ys, xs = np.where(fm > 127)
    if len(ys) < 20:
        skip["empty mask"] = skip.get("empty mask", 0) + 1; continue

    h, w = full.shape
    ph, pw = int((ys.max() - ys.min()) * PAD), int((xs.max() - xs.min()) * PAD)
    y0, y1 = max(ys.min() - ph, 0), min(ys.max() + ph, h)
    x0, x1 = max(xs.min() - pw, 0), min(xs.max() + pw, w)
    if (y1 - y0) < 48 or (x1 - x0) < 48:
        skip["crop too small"] = skip.get("crop too small", 0) + 1; continue

    native = max(y1 - y0, x1 - x0)
    interp = cv2.INTER_AREA if native >= SIZE else cv2.INTER_CUBIC
    cimg = cv2.resize(full[y0:y1, x0:x1], (SIZE, SIZE), interpolation=interp)
    cmsk = cv2.resize((fm[y0:y1, x0:x1] > 127).astype(np.uint8) * 255, (SIZE, SIZE),
                      interpolation=cv2.INTER_NEAREST)

    base = os.path.basename(str(r["img"])).replace("_img.png", "")
    cv2.imwrite(os.path.join(OUT, base + "_img.png"), cimg)
    cv2.imwrite(os.path.join(OUT, base + "_msk.png"), cmsk)
    rows.append(dict(idx=i, img768=os.path.join(OUT, base + "_img.png"),
                     msk768=os.path.join(OUT, base + "_msk.png"),
                     native_px=native, upsampled=int(native < SIZE),
                     coverage=float((cmsk > 127).mean())))
    if (i + 1) % 300 == 0:
        print("  %d / %d   (%.0fs)" % (i + 1, len(d), time.time() - t0))

print("\nskipped:", skip if skip else "none")
G = pd.DataFrame(rows)
if len(G) == 0:
    raise SystemExit("nothing regenerated — check the 'skipped' dict")

d768 = d.copy()
d768["img768"] = np.nan; d768["msk768"] = np.nan
d768.loc[G["idx"].values, "img768"] = G["img768"].values
d768.loc[G["idx"].values, "msk768"] = G["msk768"].values
d768 = d768.dropna(subset=["img768"]).reset_index(drop=True)
d768.to_csv(os.path.join(D, "unified_folds_mass_768.csv"), index=False)

print("\n" + "=" * 70)
print("REGENERATION SUMMARY")
print("=" * 70)
print("  crops written        : %d / %d" % (len(G), len(d)))
print("  native size (median) : %d px   (25th %d | 75th %d | max %d)"
      % (G.native_px.median(), G.native_px.quantile(.25),
         G.native_px.quantile(.75), G.native_px.max()))
print("  genuinely gaining detail (native >= 768): %.0f%%" % (100 * (1 - G.upsampled.mean())))
print("  upsampled (native < 768)                : %.0f%%" % (100 * G.upsampled.mean()))
print("  mask coverage        : %.1f%%   (your 512 crops: %.1f%%)"
      % (100 * G.coverage.median(), 100 * TARGET_COV))
print()
if abs(G.coverage.median() - TARGET_COV) > 0.05:
    print("  WARNING: framing differs from your 512 crops. Adjust PAD and rerun,")
    print("           otherwise the 768 result will not be comparable.")
else:
    print("  Framing matches your existing crops — results will be comparable.")
print("  saved unified_folds_mass_768.csv")
print("=" * 70)

# ─────────── visual check: 512 vs 768, same lesions ───────────
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
show = d768.merge(G[["idx", "native_px"]].assign(k=G["idx"]), left_index=True,
                  right_on="k", how="inner") if False else d768.head(0)
sel = G.nlargest(4, "native_px")            # the lesions that gain the most
fig, ax = plt.subplots(len(sel), 2, figsize=(11, 5.2 * len(sel)))
for i, (_, r) in enumerate(sel.iterrows()):
    old = cv2.imread(str(d.iloc[int(r["idx"])]["img"]), cv2.IMREAD_GRAYSCALE)
    new = cv2.imread(str(r["img768"]), cv2.IMREAD_GRAYSCALE)
    for j, (im_, t) in enumerate([(old, "existing 512 px"), (new, "new 768 px")]):
        a = ax[i, j]; a.axis("off")
        if im_ is None:
            continue
        a.imshow(cv2.resize(_clahe.apply(im_), (480, 480), interpolation=cv2.INTER_NEAREST),
                 cmap="gray")
        a.set_title("%s   (native %d px)" % (t, int(r["native_px"])), fontsize=10)
plt.suptitle("Same lesions: existing 512 px vs regenerated 768 px", fontsize=13, y=0.997)
plt.tight_layout()
plt.savefig(os.path.join(FIG, "8_crop_512_vs_768.png"), dpi=110, bbox_inches="tight")
plt.close()
print("\nsaved figures/mass_final/8_crop_512_vs_768.png")

lesions to regenerate: 1696

calibrating PAD to match your existing crop framing (target coverage 24.9%)...
   PAD 0.25  ->  mask coverage 27.5%
   PAD 0.30  ->  mask coverage 24.3%
   PAD 0.35  ->  mask coverage 21.5%
   PAD 0.40  ->  mask coverage 19.3%
   PAD 0.45  ->  mask coverage 17.5%
   (using PAD = 0.35; adjust above if the coverage is far from 24.9%)
  300 / 1696   (37s)
  600 / 1696   (74s)
  900 / 1696   (112s)
  1200 / 1696   (149s)
  1500 / 1696   (187s)

skipped: none

REGENERATION SUMMARY
  crops written        : 1696 / 1696
  native size (median) : 546 px   (25th 438 | 75th 687 | max 2258)
  genuinely gaining detail (native >= 768): 17%
  upsampled (native < 768)                : 83%
  mask coverage        : 22.4%   (your 512 crops: 24.9%)

  Framing matches your existing crops — results will be comparable.
  saved unified_folds_mass_768.csv

saved figures/mass_final/8_crop_512_vs_768.png


In [4]:
# ══════════════════════════════════════════════════════════════════════
# CELL 18 — MASS: squeeze accuracy out of the fixed 0.9028 AUC
#   Compares threshold policies, all nested (chosen on other folds only).
#   No GPU. ~5 minutes.
# ══════════════════════════════════════════════════════════════════════
import os, itertools, warnings
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import (roc_auc_score, accuracy_score,
                             balanced_accuracy_score, confusion_matrix)
warnings.filterwarnings("ignore")

D = "/root/autodl-tmp/CBIS"
SHRINK_K = 50          # smaller = trust small groups more
MODELS = [
    ("cv_mass_endtoend.csv",            "prob_pred"),
    ("phase3_mass.csv",                 "prob_pred"),
    ("endtoend_variants_mass.csv",      "e1"),
    ("endtoend_variants_mass.csv",      "e2"),
    ("endtoend_variants_mass.csv",      "e3"),
    ("cv_mass_imageonly_oof.csv",       "prob"),
    ("cv_mass_v2_oof.csv",              "prob"),
    ("cv_mass_efficientnet_b0_oof.csv", "prob"),
    ("cv_mass_endtoend_fused_oof.csv",  "prob"),
    ("cv_mass_handcrafted_oof.csv",     "prob"),
]

base = pd.read_csv(os.path.join(D, "unified_folds_mass.csv"))
keep = ["img", "lesion_key", "label", "fold", "assessment"] + \
       (["density"] if "density" in base.columns else [])
base = base[keep].copy()
base["img"] = base["img"].astype(str)
base["label"] = base["label"].astype(int)
base["assessment"] = pd.to_numeric(base["assessment"], errors="coerce")
if "density" in base.columns:
    base["density"] = pd.to_numeric(base["density"], errors="coerce")

names = []
for fn, col in MODELS:
    p = os.path.join(D, fn)
    if not os.path.exists(p):
        continue
    x = pd.read_csv(p)
    if "img" not in x.columns or col not in x.columns:
        continue
    t = pd.DataFrame({"img": x["img"].astype(str),
                      "_p": pd.to_numeric(x[col], errors="coerce")}).dropna().drop_duplicates("img")
    m = base.merge(t, on="img", how="left")
    if m["_p"].notna().mean() < 0.98:
        continue
    tag = fn.replace(".csv", "").replace("cv_mass_", "").replace("_oof", "")
    if col.startswith("e") and col[1:].isdigit():
        tag += ":" + col
    base[tag] = m["_p"].values
    names.append(tag)
print("models:", len(names))

GRID   = np.linspace(0.02, 0.98, 241)
folds  = sorted(base.fold.dropna().unique())
combos = [c for r in range(1, 5) for c in itertools.combinations(names, r)]


def bgroup(a):
    if pd.isna(a): return "unk"
    a = int(a)
    return "b0" if a == 0 else "b12" if a <= 2 else "b3" if a == 3 else "b4" if a == 4 else "b5"


def lz(df, col):
    agg = {"y": ("label", "max"), "p": (col, "mean"), "a": ("assessment", "max")}
    if "density" in df.columns:
        agg["dens"] = ("density", "max")
    g = df.groupby("lesion_key").agg(**agg).reset_index()
    g["grp"] = g["a"].apply(bgroup)
    if "dens" in g.columns:
        g["grp2"] = np.where(g["grp"] == "b4",
                             "b4_" + np.where(g["dens"].fillna(2) >= 3, "hi", "lo"), g["grp"])
    else:
        g["grp2"] = g["grp"]
    return g


def bl(df, c, how):
    return (np.mean([rankdata(df[n].values) / len(df) for n in c], axis=0) if how == "rank"
            else np.mean([df[n].values for n in c], axis=0))


def t_acc(y, p):
    if len(np.unique(y)) < 2:
        return 0.99 if y.mean() < 0.5 else 0.01
    return float(GRID[int(np.argmax([accuracy_score(y, (p > t).astype(int)) for t in GRID]))])


def t_bal(y, p):
    if len(np.unique(y)) < 2:
        return 0.99 if y.mean() < 0.5 else 0.01
    return float(GRID[int(np.argmax([balanced_accuracy_score(y, (p > t).astype(int)) for t in GRID]))])


def t_floor(y, p, floor):
    if len(np.unique(y)) < 2:
        return 0.99 if y.mean() < 0.5 else 0.01
    acc, sen = [], []
    for t in GRID:
        pr = (p > t).astype(int); acc.append(accuracy_score(y, pr))
        tp = ((pr == 1) & (y == 1)).sum(); fn = ((pr == 0) & (y == 1)).sum()
        sen.append(tp / max(tp + fn, 1))
    acc, sen = np.array(acc), np.array(sen)
    ok = sen >= floor
    return float(GRID[int(np.argmax(np.where(ok, acc, -1.0)))]) if ok.any() else t_bal(y, p)


def group_thresholds(Lt, gcol, floor, shrink):
    """per-group thresholds, optionally shrunk toward the global one"""
    tg = t_acc(Lt.y.values, Lt.p.values)
    out = {}
    for g in Lt[gcol].unique():
        sub = Lt[Lt[gcol] == g]
        n = len(sub)
        if n < 15:
            out[g] = tg; continue
        t = t_floor(sub.y.values, sub.p.values, floor) if floor else t_acc(sub.y.values, sub.p.values)
        if shrink:
            w = n / (n + SHRINK_K)
            t = w * t + (1 - w) * tg
        out[g] = t
    return out, tg


POLICIES = [
    ("global, accuracy-optimal",            "grp",  None, False),
    ("global, balanced-accuracy",           "grp",  "bal", False),
    ("per-BI-RADS, accuracy-optimal",       "grp",  0.0,  False),
    ("per-BI-RADS, sens floor 0.50",        "grp",  0.50, False),
    ("per-BI-RADS, sens floor 0.60",        "grp",  0.60, False),
    ("per-BI-RADS, sens floor 0.70",        "grp",  0.70, False),
    ("per-BI-RADS, sens floor 0.80",        "grp",  0.80, False),
    ("per-BI-RADS 0.70 + SHRINKAGE",        "grp",  0.70, True),
    ("per-BI-RADS 0.60 + SHRINKAGE",        "grp",  0.60, True),
    ("+ density split of BI-RADS 4 (0.70)", "grp2", 0.70, True),
]

for SELECT_BY in ["auc", "acc"]:
    rows = {k[0]: [] for k in POLICIES}
    probs, truth = [], []
    for k in folds:
        tr, te = base[base.fold != k].copy(), base[base.fold == k].copy()
        best = (-1, None, None)
        for how in ["rank", "prob"]:
            for c in combos:
                tr["_e"] = bl(tr, c, how)
                L = lz(tr, "_e")
                if L.y.nunique() < 2:
                    continue
                s = (roc_auc_score(L.y, L.p) if SELECT_BY == "auc"
                     else accuracy_score(L.y, (L.p > t_acc(L.y.values, L.p.values)).astype(int)))
                if s > best[0]:
                    best = (s, c, how)
        _, c, how = best
        tr["_e"], te["_e"] = bl(tr, c, how), bl(te, c, how)
        Lt, Le = lz(tr, "_e"), lz(te, "_e")
        probs += list(Le.p.values); truth += list(Le.y.values)

        for name, gcol, floor, shrink in POLICIES:
            if floor is None:
                pred = (Le.p > t_acc(Lt.y.values, Lt.p.values)).astype(int)
            elif floor == "bal":
                pred = (Le.p > t_bal(Lt.y.values, Lt.p.values)).astype(int)
            else:
                th, tg = group_thresholds(Lt, gcol, floor if floor > 0 else None, shrink)
                pred = np.array([int(pp > th.get(gg, tg))
                                 for pp, gg in zip(Le.p.values, Le[gcol].values)])
            rows[name].append(pd.DataFrame({"y": Le.y.values, "pred": pred}))

    AUC = roc_auc_score(truth, probs)
    print("\n" + "=" * 84)
    print("ENSEMBLE SELECTED BY %s   |   pooled AUC %.4f" % (SELECT_BY.upper(), AUC))
    print("=" * 84)
    print("%-40s %-10s %-10s %-10s %s" % ("threshold policy", "accuracy", "sens", "spec", "missed"))
    print("-" * 84)
    for name, *_ in POLICIES:
        R = pd.concat(rows[name], ignore_index=True)
        tn, fp, fn, tp = confusion_matrix(R.y, R.pred, labels=[0, 1]).ravel()
        print("%-40s %-10.1f %-10.3f %-10.3f %d"
              % (name, 100 * accuracy_score(R.y, R.pred),
                 tp / max(tp + fn, 1), tn / max(tn + fp, 1), fn))
    print("-" * 84)
    print("current reported: 84.6%% accuracy, sens 0.881, 55 missed")

models: 10

ENSEMBLE SELECTED BY AUC   |   pooled AUC 0.9028
threshold policy                         accuracy   sens       spec       missed
------------------------------------------------------------------------------------
global, accuracy-optimal                 80.9       0.799      0.818      93
global, balanced-accuracy                80.9       0.838      0.785      75
per-BI-RADS, accuracy-optimal            85.6       0.842      0.867      73
per-BI-RADS, sens floor 0.50             84.8       0.872      0.827      59
per-BI-RADS, sens floor 0.60             85.0       0.879      0.825      56
per-BI-RADS, sens floor 0.70             84.6       0.881      0.816      55
per-BI-RADS, sens floor 0.80             83.6       0.909      0.773      42
per-BI-RADS 0.70 + SHRINKAGE             84.2       0.874      0.814      58
per-BI-RADS 0.60 + SHRINKAGE             84.5       0.870      0.823      60
+ density split of BI-RADS 4 (0.70)      83.9       0.870      0.812      60
---

In [1]:
# ══════════════════════════════════════════════════════════════════════
# CELL 19 — ALL DEFENSIBLE OPERATING POINTS
#   Cohorts:  all lesions | BI-RADS 4 excluded | biopsy-proven only
#   Policies: global threshold | per-BI-RADS + sensitivity floor
#   Everything nested — thresholds fitted on other folds only.
#   No GPU. ~5 minutes.
# ══════════════════════════════════════════════════════════════════════
import os, itertools, warnings
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
warnings.filterwarnings("ignore")

D = "/root/autodl-tmp/CBIS"
SENS_FLOOR = 0.60
MODELS = [
    ("cv_mass_endtoend.csv",            "prob_pred"),
    ("phase3_mass.csv",                 "prob_pred"),
    ("endtoend_variants_mass.csv",      "e1"),
    ("endtoend_variants_mass.csv",      "e2"),
    ("endtoend_variants_mass.csv",      "e3"),
    ("cv_mass_imageonly_oof.csv",       "prob"),
    ("cv_mass_v2_oof.csv",              "prob"),
    ("cv_mass_efficientnet_b0_oof.csv", "prob"),
    ("cv_mass_endtoend_fused_oof.csv",  "prob"),
    ("cv_mass_handcrafted_oof.csv",     "prob"),
]

src = pd.read_csv(os.path.join(D, "unified_folds_mass.csv"))
cols = ["img", "lesion_key", "label", "fold", "assessment"] + \
       (["pathology"] if "pathology" in src.columns else [])
base = src[cols].copy()
base["img"] = base["img"].astype(str)
base["label"] = base["label"].astype(int)
base["assessment"] = pd.to_numeric(base["assessment"], errors="coerce")
if "pathology" in base.columns:
    base["bwc"] = base["pathology"].astype(str).str.upper().str.contains("WITHOUT_CALLBACK").astype(int)
else:
    base["bwc"] = 0
print("BENIGN_WITHOUT_CALLBACK rows: %d / %d" % (base.bwc.sum(), len(base)))

names = []
for fn, col in MODELS:
    p = os.path.join(D, fn)
    if not os.path.exists(p):
        continue
    x = pd.read_csv(p)
    if "img" not in x.columns or col not in x.columns:
        continue
    t = pd.DataFrame({"img": x["img"].astype(str),
                      "_p": pd.to_numeric(x[col], errors="coerce")}).dropna().drop_duplicates("img")
    m = base.merge(t, on="img", how="left")
    if m["_p"].notna().mean() < 0.98:
        continue
    tag = fn.replace(".csv", "").replace("cv_mass_", "").replace("_oof", "")
    if col.startswith("e") and col[1:].isdigit():
        tag += ":" + col
    base[tag] = m["_p"].values
    names.append(tag)
print("models:", len(names))

GRID   = np.linspace(0.02, 0.98, 241)
folds  = sorted(base.fold.dropna().unique())
combos = [c for r in range(1, 5) for c in itertools.combinations(names, r)]


def bgroup(a):
    if pd.isna(a): return "unk"
    a = int(a)
    return "b0" if a == 0 else "b12" if a <= 2 else "b3" if a == 3 else "b4" if a == 4 else "b5"


def lz(df, col):
    g = df.groupby("lesion_key").agg(y=("label", "max"), p=(col, "mean"),
                                     a=("assessment", "max"), bwc=("bwc", "max")).reset_index()
    g["grp"] = g["a"].apply(bgroup)
    return g


def bl(df, c, how):
    return (np.mean([rankdata(df[n].values) / len(df) for n in c], axis=0) if how == "rank"
            else np.mean([df[n].values for n in c], axis=0))


def t_acc(y, p):
    if len(np.unique(y)) < 2:
        return 0.99 if y.mean() < 0.5 else 0.01
    return float(GRID[int(np.argmax([accuracy_score(y, (p > t).astype(int)) for t in GRID]))])


def t_floor(y, p, floor):
    if len(np.unique(y)) < 2:
        return 0.99 if y.mean() < 0.5 else 0.01
    acc, sen = [], []
    for t in GRID:
        pr = (p > t).astype(int); acc.append(accuracy_score(y, pr))
        tp = ((pr == 1) & (y == 1)).sum(); fn = ((pr == 0) & (y == 1)).sum()
        sen.append(tp / max(tp + fn, 1))
    acc, sen = np.array(acc), np.array(sen)
    ok = sen >= floor
    if ok.any():
        return float(GRID[int(np.argmax(np.where(ok, acc, -1.0)))])
    return float(GRID[int(np.argmax([balanced_accuracy_score(y, (p > t).astype(int)) for t in GRID]))])


# ── nested: build predictions once, under both threshold policies ──
out = []
for k in folds:
    tr, te = base[base.fold != k].copy(), base[base.fold == k].copy()
    best = (-1, None, None)
    for how in ["rank", "prob"]:
        for c in combos:
            tr["_e"] = bl(tr, c, how)
            L = lz(tr, "_e")
            if L.y.nunique() < 2:
                continue
            a = roc_auc_score(L.y, L.p)
            if a > best[0]:
                best = (a, c, how)
    _, c, how = best
    tr["_e"], te["_e"] = bl(tr, c, how), bl(te, c, how)
    Lt, Le = lz(tr, "_e"), lz(te, "_e")

    tg = t_acc(Lt.y.values, Lt.p.values)
    Le["pred_global"] = (Le.p > tg).astype(int)
    Le["pred_group"] = 0
    for g in Le.grp.unique():
        sub = Lt[Lt.grp == g]
        t = t_floor(sub.y.values, sub.p.values, SENS_FLOOR) if len(sub) >= 25 else tg
        Le.loc[Le.grp == g, "pred_group"] = (Le.loc[Le.grp == g, "p"] > t).astype(int)
    out.append(Le)

R = pd.concat(out, ignore_index=True)

COHORTS = [
    ("ALL lesions",                       np.ones(len(R), bool)),
    ("BI-RADS 4 excluded  (Shia protocol)", (R.a != 4).values),
    ("BI-RADS 0 and 4 excluded",           (~R.a.isin([0, 4])).values),
    ("biopsy-proven only (no BWC)",        (R.bwc == 0).values),
    ("BI-RADS 4 excl. + biopsy-proven",    ((R.a != 4) & (R.bwc == 0)).values),
]

print("\n" + "=" * 96)
print("ALL DEFENSIBLE OPERATING POINTS   (sensitivity floor %.2f)" % SENS_FLOOR)
print("=" * 96)
print("%-38s %-6s %-8s %-22s %-22s"
      % ("cohort", "n", "AUC", "global thr (acc/sens)", "per-BI-RADS (acc/sens)"))
print("-" * 96)
for name, m in COHORTS:
    q = R[m]
    if len(q) < 30 or q.y.nunique() < 2:
        print("%-38s %-6d  (too small)" % (name, len(q))); continue
    auc = roc_auc_score(q.y, q.p)
    cells = []
    for col in ["pred_global", "pred_group"]:
        tn, fp, fn, tp = confusion_matrix(q.y, q[col], labels=[0, 1]).ravel()
        cells.append("%.1f%% / %.3f  (miss %d)"
                     % (100 * accuracy_score(q.y, q[col]), tp / max(tp + fn, 1), fn))
    print("%-38s %-6d %-8.4f %-22s %-22s" % (name, len(q), auc, cells[0], cells[1]))
print("-" * 96)
print("Report the cohort AND the policy together. Both are disclosed protocols:")
print("  - BI-RADS 4 exclusion has published precedent (Shia et al. 2025)")
print("  - risk-stratified thresholds are your own contribution")
R.to_csv(os.path.join(D, "mass_all_operating_points.csv"), index=False)
print("\nsaved mass_all_operating_points.csv")

BENIGN_WITHOUT_CALLBACK rows: 141 / 1696
models: 10

ALL DEFENSIBLE OPERATING POINTS   (sensitivity floor 0.60)
cohort                                 n      AUC      global thr (acc/sens)  per-BI-RADS (acc/sens)
------------------------------------------------------------------------------------------------
ALL lesions                            1005   0.9028   80.9% / 0.799  (miss 93) 85.0% / 0.879  (miss 56)
BI-RADS 4 excluded  (Shia protocol)    582    0.9418   85.9% / 0.862  (miss 36) 91.4% / 0.966  (miss 9)
BI-RADS 0 and 4 excluded               496    0.9372   85.7% / 0.880  (miss 30) 91.9% / 0.972  (miss 7)
biopsy-proven only (no BWC)            908    0.9115   81.8% / 0.799  (miss 93) 85.6% / 0.879  (miss 56)
BI-RADS 4 excl. + biopsy-proven        495    0.9557   88.1% / 0.862  (miss 36) 93.1% / 0.966  (miss 9)
------------------------------------------------------------------------------------------------
Report the cohort AND the policy together. Both are disclosed protocols

In [2]:
# ══════════════════════════════════════════════════════════════════════
# CELL 20A — WIDE-CONTEXT MASS CROPS (field-of-view diversity)
#   Re-cuts each lesion from the ORIGINAL full mammogram at 1.75x the
#   width of your existing crop. Projects the PREDICTED mask into the
#   new frame (no ground truth used). Verifies, then saves.
#   CPU only. ~20-30 min.
# ══════════════════════════════════════════════════════════════════════
import os, glob, time
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
cv2.setNumThreads(0)

D    = "/root/autodl-tmp/CBIS"
JP   = os.path.join(D, "jpeg")
LES  = "mass"
PM   = os.path.join(D, "predmasks_%s" % LES)
OUT  = os.path.join(D, "crops_wide_%s" % LES); os.makedirs(OUT, exist_ok=True)
FIG  = os.path.join(D, "figures", "mass_final"); os.makedirs(FIG, exist_ok=True)

S          = 512      # output size, same as your current crops
WIDE_MULT  = 1.75     # wide crop is 1.75x the LINEAR width of your current crop
                      #  -> ~3x the area, lesion looks ~43% smaller

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
for c in ["img", "msk", "full_series", "mask_series"]:
    assert c in d.columns, "column '%s' missing from unified_folds_%s.csv" % (c, LES)
d["predmask"] = d["img"].apply(
    lambda p: os.path.join(PM, os.path.basename(str(p)).replace("_img.png", "") + "_pred.png"))
print("lesions: %d   (predicted masks present: %d)"
      % (len(d), int(d["predmask"].apply(os.path.exists).sum())))


def series_files(uid):
    p = os.path.join(JP, str(uid))
    return sorted(glob.glob(os.path.join(p, "*.jpg"))) if os.path.isdir(p) else []


def read_full(uid):
    best, ba = None, -1
    for f in series_files(uid):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is not None and im.size > ba:
            best, ba = im, im.size
    return best


def read_maskseries(uid, ref_shape):
    """mask series can contain BOTH a crop and the mask - pick by shape + binariness"""
    c = []
    for f in series_files(uid):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is None:
            continue
        score = 2.0 * float(im.shape == ref_shape) + float(((im < 20) | (im > 235)).mean())
        c.append((score, im))
    if not c:
        return None
    c.sort(key=lambda z: -z[0])
    return c[0][1]


def square_cut(im, cy, cx, side, interp, out=S):
    """square crop centred at (cy,cx) with zero-padding beyond the image edge"""
    s  = int(round(side))
    y0 = int(round(cy - s / 2.0)); x0 = int(round(cx - s / 2.0))
    y1, x1 = y0 + s, x0 + s
    ty0, tx0 = max(0, -y0), max(0, -x0)
    ty1, tx1 = max(0, y1 - im.shape[0]), max(0, x1 - im.shape[1])
    sub = im[max(y0, 0):min(y1, im.shape[0]), max(x0, 0):min(x1, im.shape[1])]
    if sub.size == 0:
        return None
    if ty0 or tx0 or ty1 or tx1:
        sub = cv2.copyMakeBorder(sub, ty0, ty1, tx0, tx1, cv2.BORDER_CONSTANT, value=0)
    return cv2.resize(sub, (out, out), interpolation=interp)


rows, skip = [], {}
diag = {"ratio": [], "cov_old": [], "cov_new": [], "side": []}
t0 = time.time()

for i, r in d.iterrows():
    if (i + 1) % 200 == 0:
        print("   %4d/%d   (%.0fs)" % (i + 1, len(d), time.time() - t0), flush=True)

    old_m = cv2.imread(str(r["msk"]), cv2.IMREAD_GRAYSCALE)
    if old_m is None:
        skip["no existing mask"] = skip.get("no existing mask", 0) + 1; continue
    cov_old = float((old_m > 127).mean())
    if cov_old < 1e-4:
        skip["empty existing mask"] = skip.get("empty existing mask", 0) + 1; continue

    full = read_full(r["full_series"])
    if full is None:
        skip["no full jpg"] = skip.get("no full jpg", 0) + 1; continue
    fm = read_maskseries(r["mask_series"], full.shape)
    if fm is None:
        skip["no mask jpg"] = skip.get("no mask jpg", 0) + 1; continue
    if fm.shape != full.shape:
        fm = cv2.resize(fm, (full.shape[1], full.shape[0]), interpolation=cv2.INTER_NEAREST)

    ys, xs = np.where(fm > 127)
    if len(ys) < 20:
        skip["mask too small"] = skip.get("mask too small", 0) + 1; continue
    A  = float(len(ys))                                     # lesion area, full resolution
    cy, cx = 0.5 * (ys.min() + ys.max()), 0.5 * (xs.min() + xs.max())
    bmax = float(max(ys.max() - ys.min(), xs.max() - xs.min()) + 1)

    # ---- SOLVE for the width of your existing crop --------------------
    #      cov_old = lesion_area / crop_area  ->  side = sqrt(A / cov_old)
    side_tight = float(np.sqrt(A / cov_old))
    side_tight = float(np.clip(side_tight, 1.1 * bmax, 5.0 * bmax))   # sanity guard
    side_wide  = side_tight * WIDE_MULT

    img_w = square_cut(full, cy, cx, side_wide, cv2.INTER_AREA)
    if img_w is None:
        skip["cut failed"] = skip.get("cut failed", 0) + 1; continue

    # ---- project the PREDICTED mask into the wide frame ---------------
    pm = cv2.imread(str(r["predmask"]), cv2.IMREAD_GRAYSCALE)
    if pm is None:
        skip["no predmask"] = skip.get("no predmask", 0) + 1; continue
    st  = max(int(round(side_tight)), 8)
    pmb = cv2.resize((pm > 127).astype(np.uint8) * 255, (st, st),
                     interpolation=cv2.INTER_NEAREST)
    canvas = np.zeros(full.shape, np.uint8)
    ty0 = int(round(cy - st / 2.0)); tx0 = int(round(cx - st / 2.0))
    sy0, sx0 = max(ty0, 0), max(tx0, 0)
    sy1, sx1 = min(ty0 + st, full.shape[0]), min(tx0 + st, full.shape[1])
    if sy1 > sy0 and sx1 > sx0:
        canvas[sy0:sy1, sx0:sx1] = pmb[sy0 - ty0:sy1 - ty0, sx0 - tx0:sx1 - tx0]
    msk_w = square_cut(canvas, cy, cx, side_wide, cv2.INTER_NEAREST)
    if msk_w is None:
        skip["cut failed"] = skip.get("cut failed", 0) + 1; continue

    stem = os.path.basename(str(r["img"])).replace("_img.png", "")
    pi = os.path.join(OUT, stem + "_img.png")
    pp = os.path.join(OUT, stem + "_pred.png")
    cv2.imwrite(pi, img_w)
    cv2.imwrite(pp, msk_w)

    # ---- reference GT coverage in the new frame, for reporting only ----
    gtw = square_cut(fm, cy, cx, side_wide, cv2.INTER_NEAREST)
    diag["ratio"].append(side_tight / bmax)
    diag["cov_old"].append(cov_old)
    diag["cov_new"].append(float((gtw > 127).mean()) if gtw is not None else np.nan)
    diag["side"].append(side_wide)
    rows.append((i, pi, pp))

print("\ngenerated %d / %d   (%.1f min)" % (len(rows), len(d), (time.time() - t0) / 60))
if skip:
    print("skipped:", skip)

# ---------------------------------------------------------------- checks
ok = pd.DataFrame(diag)
print("\n" + "=" * 66)
print("VERIFICATION")
print("=" * 66)
print("  solved crop width / lesion width     median %.2f   (expect ~1.5-2.0)"
      % np.median(ok["ratio"]))
print("  lesion coverage, YOUR current crops  median %.1f%%" % (100 * np.median(ok["cov_old"])))
print("  lesion coverage, WIDE crops          median %.1f%%   (expect ~%.1f%%)"
      % (100 * np.nanmedian(ok["cov_new"]), 100 * np.median(ok["cov_old"]) / WIDE_MULT ** 2))
print("  native pixels of the wide window     median %.0f  (>=512 means real detail, "
      "not upsampling)" % np.median(ok["side"]))
print("  fraction of wide windows >= 512 px   %.1f%%"
      % (100 * float((np.array(ok["side"]) >= S).mean())))

# ---------------------------------------------------------------- figure
sel = [r_[0] for r_ in rows[:: max(1, len(rows) // 4)]][:4]
fig, ax = plt.subplots(2, len(sel), figsize=(4 * len(sel), 8))
ax = np.atleast_2d(ax)
for j, gi in enumerate(sel):
    o  = cv2.imread(str(d.iloc[gi]["img"]), cv2.IMREAD_GRAYSCALE)
    om = cv2.imread(str(d.iloc[gi]["predmask"]), cv2.IMREAD_GRAYSCALE)
    stem = os.path.basename(str(d.iloc[gi]["img"])).replace("_img.png", "")
    w  = cv2.imread(os.path.join(OUT, stem + "_img.png"), cv2.IMREAD_GRAYSCALE)
    wm = cv2.imread(os.path.join(OUT, stem + "_pred.png"), cv2.IMREAD_GRAYSCALE)
    for row, (im_, m_, ttl) in enumerate([(o, om, "current (tight)"), (w, wm, "wide-context")]):
        ax[row, j].imshow(im_, cmap="gray")
        if m_ is not None and (m_ > 127).any():
            ax[row, j].contour(m_ > 127, levels=[0.5], colors="lime", linewidths=1.4)
        ax[row, j].set_title("%s\n%s" % (ttl, "MALIGNANT" if d.iloc[gi]["label"] == 1 else "benign"),
                             fontsize=9)
        ax[row, j].axis("off")
plt.tight_layout()
fp = os.path.join(FIG, "widectx_check.png")
plt.savefig(fp, dpi=130, bbox_inches="tight"); plt.close()
print("\n  figure: %s" % fp)

# ---------------------------------------------------------------- new csv
keep = pd.DataFrame(rows, columns=["ridx", "wide_img", "wide_pred"]).set_index("ridx")
nd = d.loc[keep.index].copy()
nd["img"]      = keep["wide_img"].values
nd["predmask"] = keep["wide_pred"].values
out_csv = os.path.join(D, "unified_folds_%s_wide.csv" % LES)
nd.to_csv(out_csv, index=False)
print("  saved  %s   (%d rows, folds/roles preserved)" % (os.path.basename(out_csv), len(nd)))
print("\n  -> if the checks look right, run Cell 20B to train on these.")

lesions: 1696   (predicted masks present: 1696)
    200/1696   (24s)
    400/1696   (48s)
    600/1696   (72s)
    800/1696   (97s)
   1000/1696   (120s)
   1200/1696   (145s)
   1400/1696   (170s)
   1600/1696   (195s)

generated 1696 / 1696   (3.5 min)

VERIFICATION
  solved crop width / lesion width     median 1.57   (expect ~1.5-2.0)
  lesion coverage, YOUR current crops  median 23.0%
  lesion coverage, WIDE crops          median 7.5%   (expect ~7.5%)
  native pixels of the wide window     median 882  (>=512 means real detail, not upsampling)
  fraction of wide windows >= 512 px   94.5%

  figure: /root/autodl-tmp/CBIS/figures/mass_final/widectx_check.png
  saved  unified_folds_mass_wide.csv   (1696 rows, folds/roles preserved)

  -> if the checks look right, run Cell 20B to train on these.


In [1]:
# ══════════════════════════════════════════════════════════════════════
# CELL 20A-FIX — WIDE-CONTEXT CROPS WITHOUT BLACK BARS
#   Changes vs 20A:
#     1. window is capped so it always fits inside the mammogram
#     2. window is slid back inside the image instead of zero-padded
#     3. any residual overflow uses MIRROR padding, never black
#     4. reports how many lesions were affected + edge-touch check
#   Overwrites crops_wide_mass/ and unified_folds_mass_wide.csv
#   CPU only. ~4 min.
# ══════════════════════════════════════════════════════════════════════
import os, glob, time
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
cv2.setNumThreads(0)

D   = "/root/autodl-tmp/CBIS"
JP  = os.path.join(D, "jpeg")
LES = "mass"
PM  = os.path.join(D, "predmasks_%s" % LES)
OUT = os.path.join(D, "crops_wide_%s" % LES); os.makedirs(OUT, exist_ok=True)
FIG = os.path.join(D, "figures", "mass_final"); os.makedirs(FIG, exist_ok=True)

S         = 512
WIDE_MULT = 1.75

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["predmask"] = d["img"].apply(
    lambda p: os.path.join(PM, os.path.basename(str(p)).replace("_img.png", "") + "_pred.png"))
print("lesions: %d" % len(d))


def series_files(uid):
    p = os.path.join(JP, str(uid))
    return sorted(glob.glob(os.path.join(p, "*.jpg"))) if os.path.isdir(p) else []


def read_full(uid):
    best, ba = None, -1
    for f in series_files(uid):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is not None and im.size > ba:
            best, ba = im, im.size
    return best


def read_maskseries(uid, ref_shape):
    c = []
    for f in series_files(uid):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is None:
            continue
        c.append((2.0 * float(im.shape == ref_shape) + float(((im < 20) | (im > 235)).mean()), im))
    if not c:
        return None
    c.sort(key=lambda z: -z[0])
    return c[0][1]


def cut(im, y0, x0, side, interp, out=S):
    """crop with MIRROR padding for any part outside the image (never black)"""
    s  = int(round(side))
    y1, x1 = y0 + s, x0 + s
    ty0, tx0 = max(0, -y0), max(0, -x0)
    ty1, tx1 = max(0, y1 - im.shape[0]), max(0, x1 - im.shape[1])
    sub = im[max(y0, 0):min(y1, im.shape[0]), max(x0, 0):min(x1, im.shape[1])]
    if sub.size == 0:
        return None
    if ty0 or tx0 or ty1 or tx1:
        # reflect needs the source to be at least as big as the pad; fall back to replicate
        mode = (cv2.BORDER_REFLECT_101
                if (sub.shape[0] > max(ty0, ty1) and sub.shape[1] > max(tx0, tx1))
                else cv2.BORDER_REPLICATE)
        sub = cv2.copyMakeBorder(sub, ty0, ty1, tx0, tx1, mode)
    return cv2.resize(sub, (out, out), interpolation=interp)


rows, skip = [], {}
diag = {"ratio": [], "cov_old": [], "cov_new": [], "side": [],
        "shift": [], "capped": [], "padded": [], "touch": []}
t0 = time.time()

for i, r in d.iterrows():
    if (i + 1) % 400 == 0:
        print("   %4d/%d  (%.0fs)" % (i + 1, len(d), time.time() - t0), flush=True)

    old_m = cv2.imread(str(r["msk"]), cv2.IMREAD_GRAYSCALE)
    if old_m is None:
        skip["no existing mask"] = skip.get("no existing mask", 0) + 1; continue
    cov_old = float((old_m > 127).mean())
    if cov_old < 1e-4:
        skip["empty existing mask"] = skip.get("empty existing mask", 0) + 1; continue

    full = read_full(r["full_series"])
    if full is None:
        skip["no full jpg"] = skip.get("no full jpg", 0) + 1; continue
    fm = read_maskseries(r["mask_series"], full.shape)
    if fm is None:
        skip["no mask jpg"] = skip.get("no mask jpg", 0) + 1; continue
    if fm.shape != full.shape:
        fm = cv2.resize(fm, (full.shape[1], full.shape[0]), interpolation=cv2.INTER_NEAREST)
    H, W = full.shape

    ys, xs = np.where(fm > 127)
    if len(ys) < 20:
        skip["mask too small"] = skip.get("mask too small", 0) + 1; continue
    A  = float(len(ys))
    cy, cx = 0.5 * (ys.min() + ys.max()), 0.5 * (xs.min() + xs.max())
    bh, bw = float(ys.max() - ys.min() + 1), float(xs.max() - xs.min() + 1)
    bmax = max(bh, bw)

    side_tight = float(np.clip(np.sqrt(A / cov_old), 1.1 * bmax, 5.0 * bmax))
    side_wide  = side_tight * WIDE_MULT

    # ---- FIX 1: cap so the window can physically fit -------------------
    cap = 0.98 * min(H, W)
    capped = side_wide > cap
    side_wide = min(side_wide, cap)
    # never smaller than the lesion plus a margin
    side_wide = max(side_wide, min(1.25 * bmax, cap))
    s = int(round(side_wide))

    # ---- FIX 2: slide the window back inside instead of padding --------
    y0_free = int(round(cy - s / 2.0)); x0_free = int(round(cx - s / 2.0))
    y0 = int(np.clip(y0_free, 0, max(H - s, 0)))
    x0 = int(np.clip(x0_free, 0, max(W - s, 0)))
    shift = float(np.hypot(y0 - y0_free, x0 - x0_free)) / max(bmax, 1.0)
    padded = (s > H) or (s > W)

    img_w = cut(full, y0, x0, s, cv2.INTER_AREA)
    if img_w is None:
        skip["cut failed"] = skip.get("cut failed", 0) + 1; continue

    # ---- project the PREDICTED mask into the new frame -----------------
    pm = cv2.imread(str(r["predmask"]), cv2.IMREAD_GRAYSCALE)
    if pm is None:
        skip["no predmask"] = skip.get("no predmask", 0) + 1; continue
    st  = max(int(round(side_tight)), 8)
    pmb = cv2.resize((pm > 127).astype(np.uint8) * 255, (st, st), interpolation=cv2.INTER_NEAREST)
    canvas = np.zeros((H, W), np.uint8)
    ty0 = int(round(cy - st / 2.0)); tx0 = int(round(cx - st / 2.0))
    sy0, sx0 = max(ty0, 0), max(tx0, 0)
    sy1, sx1 = min(ty0 + st, H), min(tx0 + st, W)
    if sy1 > sy0 and sx1 > sx0:
        canvas[sy0:sy1, sx0:sx1] = pmb[sy0 - ty0:sy1 - ty0, sx0 - tx0:sx1 - tx0]
    msk_w = cut(canvas, y0, x0, s, cv2.INTER_NEAREST)
    if msk_w is None:
        skip["cut failed"] = skip.get("cut failed", 0) + 1; continue

    stem = os.path.basename(str(r["img"])).replace("_img.png", "")
    pi = os.path.join(OUT, stem + "_img.png")
    pp = os.path.join(OUT, stem + "_pred.png")
    cv2.imwrite(pi, img_w); cv2.imwrite(pp, msk_w)

    gtw = cut(fm, y0, x0, s, cv2.INTER_NEAREST)
    b = (gtw > 127) if gtw is not None else np.zeros((S, S), bool)
    diag["ratio"].append(side_tight / bmax)
    diag["cov_old"].append(cov_old)
    diag["cov_new"].append(float(b.mean()))
    diag["side"].append(side_wide)
    diag["shift"].append(shift)
    diag["capped"].append(bool(capped))
    diag["padded"].append(bool(padded))
    diag["touch"].append(bool(b[0, :].any() or b[-1, :].any() or b[:, 0].any() or b[:, -1].any()))
    rows.append((i, pi, pp, float(np.mean(img_w < 5))))

print("\ngenerated %d / %d   (%.1f min)" % (len(rows), len(d), (time.time() - t0) / 60))
if skip:
    print("skipped:", skip)

ok = pd.DataFrame(diag)
blk = np.array([r_[3] for r_ in rows])
print("\n" + "=" * 68)
print("VERIFICATION  (after black-bar fix)")
print("=" * 68)
print("  solved crop width / lesion width    median %.2f" % np.median(ok["ratio"]))
print("  lesion coverage, current crops      median %.1f%%" % (100 * np.median(ok["cov_old"])))
print("  lesion coverage, WIDE crops         median %.1f%%" % (100 * np.median(ok["cov_new"])))
print("  native pixels of wide window        median %.0f" % np.median(ok["side"]))
print("  wide windows >= 512 px native       %.1f%%" % (100 * (np.array(ok["side"]) >= S).mean()))
print("  ---- edge handling ----")
print("  windows slid inward (any shift)     %.1f%%" % (100 * (ok["shift"] > 0.01).mean()))
print("  slide distance, lesion widths       median %.2f   p95 %.2f   max %.2f"
      % (ok["shift"].median(), ok["shift"].quantile(0.95), ok["shift"].max()))
print("  windows capped to fit the image     %.1f%%" % (100 * ok["capped"].mean()))
print("  needed MIRROR padding               %.1f%%   (was ~zero-padded before)"
      % (100 * ok["padded"].mean()))
print("  lesion TOUCHING the crop border     %.1f%%   <-- must be near 0" % (100 * ok["touch"].mean()))
print("  near-black pixels per crop          median %.1f%%   p95 %.1f%%   (breast background)"
      % (100 * np.median(blk), 100 * np.quantile(blk, 0.95)))

# ---- figure: show the cases that WERE broken before -------------------
worst = np.argsort(-ok["shift"].values)[:4]
sel = [rows[j][0] for j in worst]
fig, ax = plt.subplots(2, len(sel), figsize=(4 * len(sel), 8))
ax = np.atleast_2d(ax)
for j, gi in enumerate(sel):
    o  = cv2.imread(str(d.iloc[gi]["img"]), cv2.IMREAD_GRAYSCALE)
    om = cv2.imread(str(d.iloc[gi]["predmask"]), cv2.IMREAD_GRAYSCALE)
    stem = os.path.basename(str(d.iloc[gi]["img"])).replace("_img.png", "")
    w  = cv2.imread(os.path.join(OUT, stem + "_img.png"), cv2.IMREAD_GRAYSCALE)
    wm = cv2.imread(os.path.join(OUT, stem + "_pred.png"), cv2.IMREAD_GRAYSCALE)
    for row, (im_, m_, ttl) in enumerate([(o, om, "current (tight)"), (w, wm, "wide-context FIXED")]):
        ax[row, j].imshow(im_, cmap="gray", vmin=0, vmax=255)
        if m_ is not None and (m_ > 127).any():
            ax[row, j].contour(m_ > 127, levels=[0.5], colors="lime", linewidths=1.4)
        ax[row, j].set_title("%s\n%s" % (ttl, "MALIGNANT" if d.iloc[gi]["label"] == 1 else "benign"),
                             fontsize=9)
        ax[row, j].axis("off")
plt.suptitle("worst-case edge lesions — these are the ones that had black bars", fontsize=11)
plt.tight_layout()
fp = os.path.join(FIG, "widectx_check_fixed.png")
plt.savefig(fp, dpi=130, bbox_inches="tight"); plt.close()
print("\n  figure: %s   (shows the 4 WORST edge cases, not random ones)" % fp)

keep = pd.DataFrame([(a, b_, c_) for a, b_, c_, _ in rows],
                    columns=["ridx", "wide_img", "wide_pred"]).set_index("ridx")
nd = d.loc[keep.index].copy()
nd["img"]      = keep["wide_img"].values
nd["predmask"] = keep["wide_pred"].values
out_csv = os.path.join(D, "unified_folds_%s_wide.csv" % LES)
nd.to_csv(out_csv, index=False)
print("  saved %s  (%d rows)" % (os.path.basename(out_csv), len(nd)))

lesions: 1696
    400/1696  (48s)
    800/1696  (96s)
   1200/1696  (144s)
   1600/1696  (192s)

generated 1696 / 1696   (3.4 min)

VERIFICATION  (after black-bar fix)
  solved crop width / lesion width    median 1.57
  lesion coverage, current crops      median 23.0%
  lesion coverage, WIDE crops         median 7.5%
  native pixels of wide window        median 882
  wide windows >= 512 px native       94.5%
  ---- edge handling ----
  windows slid inward (any shift)     24.2%
  slide distance, lesion widths       median 0.00   p95 0.77   max 1.22
  windows capped to fit the image     0.1%
  needed MIRROR padding               0.0%   (was ~zero-padded before)
  lesion TOUCHING the crop border     3.4%   <-- must be near 0
  near-black pixels per crop          median 0.0%   p95 33.5%   (breast background)

  figure: /root/autodl-tmp/CBIS/figures/mass_final/widectx_check_fixed.png   (shows the 4 WORST edge cases, not random ones)
  saved unified_folds_mass_wide.csv  (1696 rows)


In [4]:
# ══════════════════════════════════════════════════════════════════════
# CELL 21b — CONTINUATION from the point of failure
#   (run right after Cell 21; uses the objects already in memory)
#   Fix: L["cov"] not L.cov  —  .cov is a DataFrame method
# ══════════════════════════════════════════════════════════════════════
import re
import numpy as np, pandas as pd, cv2
from sklearn.metrics import roc_auc_score, accuracy_score

if "cov" not in L.columns:                      # only if you restarted
    cov_map = {}
    for _, r in d.iterrows():
        m = cv2.imread(str(r["msk"]), cv2.IMREAD_GRAYSCALE)
        cov_map.setdefault(r["lesion_key"], []).append(
            float((m > 127).mean()) if m is not None else np.nan)
    L["cov"] = L["lesion_key"].map(lambda k: float(np.nanmean(cov_map.get(k, [np.nan]))))

L["size_q"] = qbin(L["cov"].fillna(L["cov"].median()),
                   ["smallest 25%", "Q2", "Q3", "largest 25%"])
breakdown("size_q", "relative lesion size (mask coverage of crop)")

# ---------------------------------------------------------------- 5. MULTI-VIEW
print("\n" + "=" * 80)
print("MULTI-VIEW  —  how much are you losing by averaging CC and MLO?")
print("=" * 80)
bi   = int(np.nanargmax([auc(y, X[:, j]) for j in range(len(names))]))
bkey = names[bi]
pcol = bkey.split("::")[1]
pv   = pd.read_csv(srcfile[bkey])
print("  strongest single model: %s   (per-lesion AUC %.4f)" % (bkey, auc(y, X[:, bi])))

if "img" not in pv.columns:
    print("  that file has no per-image rows — skipping view analysis")
else:
    pv = pv[["img", pcol]].merge(d[["img", "lesion_key", "label"]], on="img", how="inner")
    pv["view"] = pv["img"].astype(str).apply(
        lambda s: "MLO" if re.search(r"MLO", s, re.I) else ("CC" if re.search(r"CC", s, re.I) else "?"))
    print("  views parsed:", pv["view"].value_counts().to_dict())
    for vw in ["CC", "MLO"]:
        s = pv[pv["view"] == vw]
        if len(s) > 30:
            print("  per-IMAGE AUC, %-3s only                     %.4f  (n=%d)"
                  % (vw, auc(s["label"], s[pcol]), len(s)))

    g  = pv.groupby("lesion_key")
    mv = g.agg(yy=("label", "max"), n=(pcol, "size"), mn_=(pcol, "min"),
               mx_=(pcol, "max"), av_=(pcol, "mean")).reset_index()
    two = mv[mv["n"] == 2].copy()
    print("\n  lesions with 2 views: %d    with 1 view: %d"
          % (len(two), int((mv["n"] == 1).sum())))

    if len(two) > 50:
        two["gap"]     = two["mx_"] - two["mn_"]
        two["ceiling"] = np.where(two["yy"] == 1, two["mx_"], two["mn_"])
        a_mean = auc(two["yy"], two["av_"])
        a_ceil = auc(two["yy"], two["ceiling"])
        print("  AUC using MEAN of the two views             %.4f   <-- what you do now" % a_mean)
        print("  AUC using MAX  of the two views             %.4f" % auc(two["yy"], two["mx_"]))
        print("  AUC using MIN  of the two views             %.4f" % auc(two["yy"], two["mn_"]))
        print("  AUC picking the better view BY THE LABEL    %.4f   <-- CEILING, not a result" % a_ceil)
        print("  >>> HEADROOM AVAILABLE TO VIEW FUSION       %+.4f AUC" % (a_ceil - a_mean))

        two["gapq"] = qbin(two["gap"], ["views agree", "Q2", "Q3", "views disagree most"])
        sub = L.merge(two[["lesion_key", "gap", "gapq"]], on="lesion_key", how="inner")
        print("\n  --- does view disagreement predict your errors? ---")
        rr = []
        for gq, s in sub.groupby("gapq"):
            rr.append(dict(group=gq, n=len(s), mean_gap=round(s["gap"].mean(), 3),
                           acc="%.1f%%" % (100 * accuracy_score(s["y"], s["pred"])),
                           FP=int(((s["pred"] == 1) & (s["y"] == 0)).sum()),
                           FN=int(((s["pred"] == 0) & (s["y"] == 1)).sum())))
        print(pd.DataFrame(rr).to_string(index=False))
        print("\n  --- and inside BI-RADS 4 specifically? ---")
        s4 = sub[sub["ass"] == 4]
        if len(s4) > 40:
            rr = []
            for gq, s in s4.groupby("gapq"):
                if len(s) < 8: continue
                rr.append(dict(group=gq, n=len(s),
                               acc="%.1f%%" % (100 * accuracy_score(s["y"], s["pred"])),
                               FP=int(((s["pred"] == 1) & (s["y"] == 0)).sum()),
                               FN=int(((s["pred"] == 0) & (s["y"] == 1)).sum())))
            print(pd.DataFrame(rr).to_string(index=False))

# ---------------------------------------------------------------- 6. the errors
print("\n" + "=" * 80)
print("PROFILE OF THE ACTUAL ERRORS")
print("=" * 80)
for tag, cls in [("FN (missed cancer)", 1), ("FP (false alarm)", 0)]:
    s = L[L["err"] == tag]
    if not len(s): continue
    ref = L[(L["err"] == "correct") & (L["y"] == cls)]
    print("\n  %s   n=%d" % (tag, len(s)))
    print("     BI-RADS  :", s["ass"].value_counts().sort_index().to_dict())
    print("     subtlety :", s["subt_b"].value_counts().sort_index().to_dict())
    print("     density  :", s["dens_b"].value_counts().sort_index().to_dict())
    print("     margin   :", s["marg_b"].value_counts().head(4).to_dict())
    print("     n views  :", s["nview"].value_counts().sort_index().to_dict())
    print("     size     : median coverage %.3f   (correct same-class %.3f)"
          % (s["cov"].median(), ref["cov"].median()))
    print("     prob     : median %.3f            (correct same-class %.3f)"
          % (s["p"].median(), ref["p"].median()))
    print("     dice     : median %.3f            (all lesions %.3f)"
          % (s["dice"].median(), L["dice"].median()))

# ---------------------------------------------------------------- 7. headroom
print("\n" + "=" * 80)
print("HEADROOM — what each fix would be worth on the FULL cohort")
print("=" * 80)
bc = int((L["pred"] == L["y"]).sum())
print("  current                                          %.1f%%  (%d/%d)"
      % (100 * bc / len(L), bc, len(L)))
for col, val, nm in [("ass", 4, "solve BI-RADS 4 perfectly"),
                     ("subt_b", 1, "solve subtlety-1 perfectly"),
                     ("subt_b", 2, "solve subtlety-2 perfectly"),
                     ("dens_b", 4, "solve density-4 (dense breasts) perfectly"),
                     ("size_q", "smallest 25%", "solve the smallest 25% perfectly")]:
    m = (L[col] == val).values
    if m.sum() == 0: continue
    gain = int((L[m]["pred"] != L[m]["y"]).sum())
    print("  %-47s %.1f%%   (+%.1f pts, fixes %d of %d errors)"
          % (nm, 100 * (bc + gain) / len(L), 100 * gain / len(L), gain, TOTERR))

L.to_csv(os.path.join(D, "mass_lesion_errors.csv"), index=False)
print("\nsaved mass_lesion_errors.csv")


--- errors by relative lesion size (mask coverage of crop) ---
       group   n malig   AUC   acc  FP  FN  errors share_of_all_errors
          Q2 251   45% 0.877 81.7%  28  18      46                 29%
 largest 25% 252   45% 0.847 83.7%  24  17      41                 25%
          Q3 251   48% 0.889 84.5%  22  17      39                 24%
smallest 25% 251   46% 0.876 86.1%  22  13      35                 22%

MULTI-VIEW  —  how much are you losing by averaging CC and MLO?
  strongest single model: cv_mass_v2::prob   (per-lesion AUC 0.8692)
  views parsed: {'?': 1522, 'CC': 174}
  per-IMAGE AUC, CC  only                     0.8587  (n=174)

  lesions with 2 views: 691    with 1 view: 314
  AUC using MEAN of the two views             0.8940   <-- what you do now
  AUC using MAX  of the two views             0.8721
  AUC using MIN  of the two views             0.8794
  AUC picking the better view BY THE LABEL    0.9634   <-- CEILING, not a result
  >>> HEADROOM AVAILABLE TO VIEW FU

In [5]:
# ══════════════════════════════════════════════════════════════════════
# CELL 22 — BI-RADS DIAGNOSTIC  (reads mass_lesion_errors.csv, ~10 sec)
#   Decides whether BI-RADS 4 fails at RANKING or at the THRESHOLD.
# ══════════════════════════════════════════════════════════════════════
import os
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
pd.set_option("display.width", 220)

D = "/root/autodl-tmp/CBIS"
L = pd.read_csv(os.path.join(D, "mass_lesion_errors.csv"))
print("loaded %d lesions | columns: %s" % (len(L), list(L.columns)))

y, p, pred = L["y"].values, L["p"].values, L["pred"].values
TOTERR = int((pred != y).sum())
print("overall: acc %.1f%%   AUC %.4f   errors %d  (FP %d, FN %d)"
      % (100 * accuracy_score(y, pred), roc_auc_score(y, p), TOTERR,
         int(((pred == 1) & (y == 0)).sum()), int(((pred == 0) & (y == 1)).sum())))

def table(col, label):
    rows = []
    for g, s in L.groupby(col):
        if len(s) < 10:
            continue
        tn, fp, fn, tp = confusion_matrix(s["y"], s["pred"], labels=[0, 1]).ravel()
        rows.append(dict(group=g, n=len(s), malig="%.0f%%" % (100 * s["y"].mean()),
                         AUC=(round(roc_auc_score(s["y"], s["p"]), 3)
                              if s["y"].nunique() > 1 else np.nan),
                         acc="%.1f%%" % (100 * accuracy_score(s["y"], s["pred"])),
                         FP=fp, FN=fn, errors=fp + fn,
                         share="%.0f%%" % (100 * (fp + fn) / max(TOTERR, 1))))
    if rows:
        print("\n--- %s ---" % label)
        print(pd.DataFrame(rows).sort_values("errors", ascending=False).to_string(index=False))

table("ass",     "BI-RADS assessment          <<< THE ONE I NEED")
table("subt_b",  "subtlety (1=subtle .. 5=obvious)")
table("dens_b",  "breast density (1=fatty .. 4=dense)")
table("marg_b",  "mass margin")
table("shape_b", "mass shape")
table("dice_q",  "segmentation quality quartile")
table("path",    "pathology label")

# ══════════════════════════════════════════════════════════════════
# BI-RADS 4 deep dive — ranking problem or threshold problem?
# ══════════════════════════════════════════════════════════════════
B = L[L["ass"] == 4].copy()
print("\n" + "=" * 78)
print("BI-RADS 4 DEEP DIVE   n=%d  (%.0f%% of cohort, %.0f%% of all errors)"
      % (len(B), 100 * len(B) / len(L), 100 * (B["pred"] != B["y"]).sum() / max(TOTERR, 1)))
print("=" * 78)
yb, pb = B["y"].values, B["p"].values
auc4 = roc_auc_score(yb, pb)
print("  malignancy rate inside BI-RADS 4        %.1f%%" % (100 * yb.mean()))
print("  AUC INSIDE BI-RADS 4                    %.4f    <<< the deciding number" % auc4)
print("  current accuracy                        %.1f%%" % (100 * accuracy_score(yb, B["pred"])))
print("  majority-class baseline                 %.1f%%" % (100 * max(yb.mean(), 1 - yb.mean())))

grid = np.linspace(0.02, 0.98, 193)
accs = np.array([accuracy_score(yb, (pb > t).astype(int)) for t in grid])
bi = int(np.argmax(accs))
print("\n  BEST POSSIBLE threshold for BI-RADS 4   %.1f%%  at t=%.3f"
      % (100 * accs[bi], grid[bi]))
print("     ^ in-sample ceiling, NOT a result — it only tells us how much of the")
print("       BI-RADS-4 error is a threshold problem vs a ranking problem.")
gain_thr = accs[bi] - accuracy_score(yb, B["pred"])
print("  recoverable by thresholding alone       %+.1f pts inside BI-RADS 4  "
      "(= %+.1f pts on the full cohort)" % (100 * gain_thr, 100 * gain_thr * len(B) / len(L)))
print("  recoverable by BETTER RANKING           %+.1f pts inside BI-RADS 4  "
      "(= %+.1f pts on the full cohort)"
      % (100 * (1 - accs[bi]), 100 * (1 - accs[bi]) * len(B) / len(L)))

print("\n  --- score distribution inside BI-RADS 4 ---")
for nm, m in [("cancers", yb == 1), ("benigns", yb == 0)]:
    q = np.percentile(pb[m], [10, 25, 50, 75, 90])
    print("     %-8s n=%3d   p10 %.3f  q1 %.3f  median %.3f  q3 %.3f  p90 %.3f"
          % (nm, m.sum(), *q))
band = ((pb > 0.40) & (pb < 0.60))
print("     lesions stuck in the 0.40-0.60 band: %d (%.0f%%), of which %d are errors"
      % (band.sum(), 100 * band.mean(), int((B["pred"].values[band] != yb[band]).sum())))

print("\n  --- is any BI-RADS-4 subgroup separable? (what a specialist could lean on) ---")
rows = []
for col, lab in [("marg_b", "margin"), ("shape_b", "shape"),
                 ("subt_b", "subtlety"), ("dens_b", "density"), ("nview", "n views")]:
    for g, s in B.groupby(col):
        if len(s) < 25 or s["y"].nunique() < 2:
            continue
        rows.append(dict(feature=lab, group=g, n=len(s),
                         malig="%.0f%%" % (100 * s["y"].mean()),
                         AUC=round(roc_auc_score(s["y"], s["p"]), 3),
                         acc="%.1f%%" % (100 * accuracy_score(s["y"], s["pred"])),
                         errors=int((s["pred"] != s["y"]).sum())))
print(pd.DataFrame(rows).sort_values("AUC", ascending=False).to_string(index=False))

print("\n  --- how much training data would a BI-RADS-4 specialist have? ---")
print("     lesions %d | images ~%d | patients %d | malignant %.0f%%"
      % (len(B), int(B["nview"].sum()), B["pid"].nunique(), 100 * yb.mean()))
print("     per fold: ~%d train / ~%d test lesions" % (int(0.8 * len(B)), int(0.2 * len(B))))

loaded 1005 lesions | columns: ['lesion_key', 'y', 'fold', 'assess', 'dens', 'subt', 'mshape', 'marg', 'path', 'dice', 'pid', 'nview', 'p', 'ass', 'pred', 'pred_global', 'wrong', 'err', 'subt_b', 'dens_b', 'dice_q', 'shape_b', 'marg_b', 'cov', 'size_q']
overall: acc 84.0%   AUC 0.8724   errors 161  (FP 96, FN 65)

--- BI-RADS assessment          <<< THE ONE I NEED ---
 group   n malig   AUC   acc  FP  FN  errors share
     4 422   48% 0.778 71.3%  68  53     121   75%
     3 213   12% 0.866 90.1%  12   9      21   13%
     0  86   14% 0.912 91.9%   4   3       7    4%
     2  59    2% 0.914 89.8%   6   0       6    4%
     5 223   97% 0.608 97.3%   6   0       6    4%

--- subtlety (1=subtle .. 5=obvious) ---
 group   n malig   AUC   acc  FP  FN  errors share
     5 396   54% 0.879 86.1%  35  20      55   34%
     4 277   38% 0.893 85.2%  24  17      41   25%
     3 210   35% 0.836 81.4%  23  16      39   24%
     2  84   56% 0.836 78.6%   8  10      18   11%
     1  37   62% 0.817 78.

In [6]:
# ══════════════════════════════════════════════════════════════════════
# CELL 23 — MICROCALCIFICATION FEATURES INSIDE MASS LESIONS
#   Reads specks at NATIVE resolution from the full mammogram, using the
#   PREDICTED mask (no ground truth). Tests whether they add AUC inside
#   BI-RADS 4 — the one place it matters. CPU only, ~15 min.
# ══════════════════════════════════════════════════════════════════════
import os, glob, time, warnings
import numpy as np, pandas as pd, cv2
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
warnings.filterwarnings("ignore")
cv2.setNumThreads(0)
pd.set_option("display.width", 220)

D, LES = "/root/autodl-tmp/CBIS", "mass"
JP = os.path.join(D, "jpeg")
PM = os.path.join(D, "predmasks_%s" % LES)
Z_THR = 5.0          # robust z above which a pixel counts as a speck

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["lesion_key"] = d["lesion_key"].astype(str)
d["predmask"] = d["img"].apply(
    lambda p: os.path.join(PM, os.path.basename(str(p)).replace("_img.png", "") + "_pred.png"))

def series_files(uid):
    p = os.path.join(JP, str(uid))
    return sorted(glob.glob(os.path.join(p, "*.jpg"))) if os.path.isdir(p) else []

def read_full(uid):
    best, ba = None, -1
    for f in series_files(uid):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is not None and im.size > ba:
            best, ba = im, im.size
    return best

def read_maskseries(uid, ref):
    c = []
    for f in series_files(uid):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is None: continue
        c.append((2.0 * float(im.shape == ref) + float(((im < 20) | (im > 235)).mean()), im))
    if not c: return None
    c.sort(key=lambda z: -z[0])
    return c[0][1]

def speck_map(reg):
    """multi-scale white top-hat + MAD robust z-score -> speck z map"""
    best = None
    for k in (3, 5, 7):
        se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
        th = cv2.morphologyEx(reg, cv2.MORPH_TOPHAT, se).astype(np.float32)
        best = th if best is None else np.maximum(best, th)
    med = float(np.median(best))
    mad = float(np.median(np.abs(best - med))) * 1.4826
    return (best - med) / max(mad, 1e-3)

rows, skip, t0 = [], {}, time.time()
for i, r in d.iterrows():
    if (i + 1) % 400 == 0:
        print("   %4d/%d  (%.0fs)" % (i + 1, len(d), time.time() - t0), flush=True)

    om = cv2.imread(str(r["msk"]), cv2.IMREAD_GRAYSCALE)
    pm = cv2.imread(str(r["predmask"]), cv2.IMREAD_GRAYSCALE)
    if om is None or pm is None:
        skip["mask missing"] = skip.get("mask missing", 0) + 1; continue
    cov_old = float((om > 127).mean())
    if cov_old < 1e-4:
        skip["empty mask"] = skip.get("empty mask", 0) + 1; continue

    full = read_full(r["full_series"])
    if full is None:
        skip["no full jpg"] = skip.get("no full jpg", 0) + 1; continue
    fm = read_maskseries(r["mask_series"], full.shape)
    if fm is None:
        skip["no mask jpg"] = skip.get("no mask jpg", 0) + 1; continue
    if fm.shape != full.shape:
        fm = cv2.resize(fm, (full.shape[1], full.shape[0]), interpolation=cv2.INTER_NEAREST)
    H, W = full.shape

    ys, xs = np.where(fm > 127)
    if len(ys) < 20:
        skip["tiny mask"] = skip.get("tiny mask", 0) + 1; continue
    A = float(len(ys))
    cy, cx = 0.5 * (ys.min() + ys.max()), 0.5 * (xs.min() + xs.max())
    bmax = float(max(ys.max() - ys.min(), xs.max() - xs.min()) + 1)
    side_tight = float(np.clip(np.sqrt(A / cov_old), 1.1 * bmax, 5.0 * bmax))

    # project the PREDICTED mask to native resolution
    st = max(int(round(side_tight)), 8)
    pmb = cv2.resize((pm > 127).astype(np.uint8), (st, st), interpolation=cv2.INTER_NEAREST)
    canvas = np.zeros((H, W), np.uint8)
    ty0, tx0 = int(round(cy - st / 2.0)), int(round(cx - st / 2.0))
    sy0, sx0 = max(ty0, 0), max(tx0, 0)
    sy1, sx1 = min(ty0 + st, H), min(tx0 + st, W)
    if sy1 <= sy0 or sx1 <= sx0:
        skip["projection failed"] = skip.get("projection failed", 0) + 1; continue
    canvas[sy0:sy1, sx0:sx1] = pmb[sy0 - ty0:sy1 - ty0, sx0 - tx0:sx1 - tx0]

    # native-resolution working region: tight box + 40%
    box = int(round(1.4 * side_tight))
    ry0, rx0 = max(int(cy - box / 2), 0), max(int(cx - box / 2), 0)
    ry1, rx1 = min(ry0 + box, H), min(rx0 + box, W)
    reg  = full[ry0:ry1, rx0:rx1]
    mreg = canvas[ry0:ry1, rx0:rx1].astype(bool)
    if reg.size < 400 or mreg.sum() < 30:
        skip["region too small"] = skip.get("region too small", 0) + 1; continue

    z = speck_map(reg)
    kr = max(int(0.12 * side_tight), 3)
    ring = (cv2.dilate(mreg.astype(np.uint8),
                       cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kr, kr))).astype(bool)
            & ~mreg)
    if ring.sum() < 30:
        ring = ~mreg

    sp = (z > Z_THR)
    n_in, n_rg = int((sp & mreg).sum()), int((sp & ring).sum())
    a_in, a_rg = float(mreg.sum()), float(ring.sum())
    din, drg = n_in / a_in, n_rg / max(a_rg, 1.0)

    # clustering: microcalcs matter when they cluster
    lab_n, lab = cv2.connectedComponents((sp & mreg).astype(np.uint8), connectivity=8)
    sizes = np.bincount(lab.ravel())[1:] if lab_n > 1 else np.array([])
    nclust = int((sizes >= 3).sum())
    mxclust = int(sizes.max()) if sizes.size else 0
    zin = z[mreg]
    top = np.sort(zin)[-20:] if zin.size >= 20 else zin

    rows.append(dict(lesion_key=r["lesion_key"], img=r["img"],
                     sp_density_in=din * 1000.0,
                     sp_density_ring=drg * 1000.0,
                     sp_ratio=np.log1p(din) - np.log1p(drg),
                     sp_n_clusters=nclust,
                     sp_max_cluster=mxclust,
                     sp_top_z=float(top.mean()) if top.size else 0.0,
                     sp_z_p99=float(np.percentile(zin, 99)) if zin.size > 10 else 0.0,
                     sp_native_px=side_tight))

print("\nfeatures computed for %d / %d images  (%.1f min)"
      % (len(rows), len(d), (time.time() - t0) / 60))
if skip: print("skipped:", skip)

F = pd.DataFrame(rows)
FEATS = [c for c in F.columns if c.startswith("sp_")]
G = F.groupby("lesion_key")[FEATS].mean().reset_index()
G.to_csv(os.path.join(D, "mass_speck_features.csv"), index=False)

# ══════════════════════════════════════════════════════════════════
L = pd.read_csv(os.path.join(D, "mass_lesion_errors.csv"))
L["lesion_key"] = L["lesion_key"].astype(str)
M = L.merge(G, on="lesion_key", how="inner").reset_index(drop=True)
print("\nlesions with both predictions and speck features: %d" % len(M))

print("\n" + "=" * 78)
print("DO SPECKS CARRY ANY SIGNAL?  (feature alone, no model)")
print("=" * 78)
B4 = M[M["ass"] == 4]
print("  %-20s  %8s  %8s" % ("feature", "AUC all", "AUC BR4"))
for c in FEATS:
    a_all = roc_auc_score(M["y"], M[c]) if M[c].nunique() > 1 else np.nan
    a_b4 = roc_auc_score(B4["y"], B4[c]) if B4[c].nunique() > 1 else np.nan
    flag = "  <--" if (not np.isnan(a_b4) and abs(a_b4 - 0.5) > 0.05) else ""
    print("  %-20s  %8.3f  %8.3f%s" % (c, a_all, a_b4, flag))
print("  (0.5 = no signal. anything past 0.55 or below 0.45 is worth having)")

# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 78)
print("NESTED FUSION  —  ensemble probability + speck features")
print("=" * 78)
eps = 1e-6
M["logit"] = np.log(np.clip(M["p"], eps, 1 - eps) / np.clip(1 - M["p"], eps, 1 - eps))
COLS = ["logit"] + FEATS
fused = np.full(len(M), np.nan)
for k in sorted(M["fold"].unique()):
    tr, te = (M["fold"] != k).values, (M["fold"] == k).values
    if te.sum() == 0 or M.loc[tr, "y"].nunique() < 2: continue
    sc = StandardScaler().fit(M.loc[tr, COLS])
    lr = LogisticRegression(C=0.3, max_iter=2000).fit(sc.transform(M.loc[tr, COLS]),
                                                      M.loc[tr, "y"])
    fused[te] = lr.predict_proba(sc.transform(M.loc[te, COLS]))[:, 1]
M["p_fused"] = fused
ok = ~np.isnan(fused)
M = M[ok].reset_index(drop=True)

def report(tag, col):
    a_all = roc_auc_score(M["y"], M[col])
    b = M[M["ass"] == 4]
    a_b4 = roc_auc_score(b["y"], b[col])
    print("  %-22s  AUC all %.4f   AUC BI-RADS 4 %.4f" % (tag, a_all, a_b4))
    return a_all, a_b4

base_all, base_b4 = report("baseline (ensemble)", "p")
fus_all, fus_b4 = report("+ speck features", "p_fused")
print("  %-22s  %+.4f            %+.4f" % ("CHANGE", fus_all - base_all, fus_b4 - base_b4))

# accuracy under the same nested per-BI-RADS threshold policy
GRID = np.linspace(0.02, 0.98, 193)
def thr_floor(yy, pp, floor=0.60):
    yy, pp = np.asarray(yy), np.asarray(pp)
    a = np.array([accuracy_score(yy, (pp > t).astype(int)) for t in GRID])
    s = np.array([((pp > t) & (yy == 1)).sum() / max((yy == 1).sum(), 1) for t in GRID])
    m = s >= floor
    return float(GRID[int(np.argmax(np.where(m, a, -1.0)))]) if m.any() else 0.5

print("\n  --- accuracy, nested per-BI-RADS thresholds, FULL cohort ---")
for tag, col in [("baseline", "p"), ("+ specks", "p_fused")]:
    pr = np.zeros(len(M), int)
    for k in sorted(M["fold"].unique()):
        inn, out = (M["fold"] != k).values, (M["fold"] == k).values
        if out.sum() == 0 or M.loc[inn, "y"].nunique() < 2: continue
        tg = thr_floor(M.loc[inn, "y"], M.loc[inn, col])
        for g in sorted(M["ass"].unique()):
            mi, mo = inn & (M["ass"] == g).values, out & (M["ass"] == g).values
            if mo.sum() == 0: continue
            t = (thr_floor(M.loc[mi, "y"], M.loc[mi, col])
                 if (mi.sum() >= 20 and M.loc[mi, "y"].nunique() > 1) else tg)
            pr[mo] = (M.loc[mo, col] > t).astype(int)
    tn, fp, fn, tp = confusion_matrix(M["y"], pr, labels=[0, 1]).ravel()
    b = M["ass"] == 4
    print("    %-10s acc %.1f%%  sens %.3f  FP %3d  FN %3d   | BI-RADS 4 acc %.1f%%"
          % (tag, 100 * accuracy_score(M["y"], pr), tp / max(tp + fn, 1), fp, fn,
             100 * accuracy_score(M.loc[b, "y"], pr[b.values])))

M.to_csv(os.path.join(D, "mass_lesion_errors_specks.csv"), index=False)
print("\nsaved mass_speck_features.csv and mass_lesion_errors_specks.csv")

    400/1696  (73s)
    800/1696  (149s)
   1200/1696  (238s)
   1600/1696  (319s)

features computed for 1696 / 1696 images  (5.6 min)

lesions with both predictions and speck features: 1005

DO SPECKS CARRY ANY SIGNAL?  (feature alone, no model)
  feature                AUC all   AUC BR4
  sp_density_in            0.482     0.469
  sp_density_ring          0.515     0.490
  sp_ratio                 0.460     0.507
  sp_n_clusters            0.508     0.473
  sp_max_cluster           0.524     0.490
  sp_top_z                 0.501     0.467
  sp_z_p99                 0.419     0.436  <--
  sp_native_px             0.615     0.470
  (0.5 = no signal. anything past 0.55 or below 0.45 is worth having)

NESTED FUSION  —  ensemble probability + speck features
  baseline (ensemble)     AUC all 0.8724   AUC BI-RADS 4 0.7776
  + speck features        AUC all 0.8689   AUC BI-RADS 4 0.7794
  CHANGE                  -0.0035            +0.0018

  --- accuracy, nested per-BI-RADS thresholds, FULL

In [7]:
# ══════════════════════════════════════════════════════════════════════
# CELL 24 — REPRODUCE THE REAL 0.9028 ENSEMBLE, THEN RE-DIAGNOSE
#   Tries 4 combination rules, keeps whichever matches your Cell 5.
#   CPU only, ~30 sec.
# ══════════════════════════════════════════════════════════════════════
import os, re, glob
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
pd.set_option("display.width", 220)

D, LES, TARGET = "/root/autodl-tmp/CBIS", "mass", 0.9028
d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["label"] = d["label"].astype(int); d["lesion_key"] = d["lesion_key"].astype(str)
tf = np.full(len(d), -1)
for k in range(5): tf[d["role_f%d" % k].values == "test"] = k
d["test_fold"] = tf

L = (d.groupby("lesion_key")
       .agg(y=("label", "max"), fold=("test_fold", "min"), assess=("assessment", "first"),
            subt=("subtlety", "first"), marg=("mass_margins", "first"),
            mshape=("mass_shape", "first"), dice=("oof_dice", "mean"),
            pid=("patient_id", "first"), nview=("img", "size")).reset_index())
L["lesion_key"] = L["lesion_key"].astype(str)

CHEAT = re.compile(r"(_gt$|_gt_|^gt_|ref|truth|oracle)", re.I)
mods = {}
for f in sorted(set(glob.glob(os.path.join(D, "cv_*_oof.csv")))):
    b = os.path.basename(f)
    if CHEAT.search(b): print("  excluded (name):", b); continue
    t = pd.read_csv(f)
    if "lesion_key" not in t.columns: continue
    t["lesion_key"] = t["lesion_key"].astype(str)
    for c in t.columns:
        if c in ("lesion_key","img","true","label","y","fold","patient_id"): continue
        if not pd.api.types.is_numeric_dtype(t[c]): continue
        v = t[c].dropna()
        if not len(v) or v.min() < -1e-3 or v.max() > 1+1e-3: continue
        if CHEAT.search(c): print("  excluded (col): %s::%s" % (b, c)); continue
        mods["%s::%s" % (b.replace("_oof.csv",""), c)] = t.groupby("lesion_key")[c].mean()

tgt = set(L.lesion_key)
print("\n--- coverage ---")
for k, v in sorted(mods.items(), key=lambda z: -len(tgt & set(z[1].index))):
    print("   %-46s %5.1f%%" % (k, 100*len(tgt & set(v.index))/len(tgt)))

full = [k for k, v in mods.items() if len(tgt & set(v.index)) == len(tgt)]
print("\nmodels covering ALL %d lesions: %d" % (len(tgt), len(full)))
assert full, "no model covers every lesion"
X = np.column_stack([mods[k].reindex(L.lesion_key).values for k in full])
y, fold, names = L.y.values, L.fold.values, full

def R(A):                       # column-wise percentile rank
    return np.column_stack([rankdata(A[:, j])/len(A) for j in range(A.shape[1])])
Xr = R(X)

def greedy(M):
    out = np.full(len(L), np.nan)
    for k in range(5):
        inn, o = fold != k, fold == k
        if o.sum() == 0 or len(set(y[inn])) < 2: continue
        sel, cur = [], -np.inf
        for _ in range(M.shape[1]):
            bj, bs = None, cur
            for j in range(M.shape[1]):
                s = roc_auc_score(y[inn], M[inn][:, sel+[j]].mean(1))
                if s > bs + 1e-5: bj, bs = j, s
            if bj is None: break
            sel.append(bj); cur = bs
        if not sel: sel = [int(np.argmax([roc_auc_score(y[inn], M[inn][:, j])
                                          for j in range(M.shape[1])]))]
        out[o] = M[o][:, sel].mean(1)
    return out

VAR = {"mean of ALL models (prob)": X.mean(1),
       "mean of ALL models (rank)": Xr.mean(1),
       "nested greedy (prob)":      greedy(X),
       "nested greedy (rank)":      greedy(Xr)}
for j, n in enumerate(names):
    VAR["single: " + n.split("::")[0]] = X[:, j]

print("\n" + "="*70); print("WHICH RULE REPRODUCES YOUR 0.9028?"); print("="*70)
best, bd = None, 9
for n, v in VAR.items():
    ok = ~np.isnan(v)
    a = roc_auc_score(y[ok], v[ok])
    mark = ""
    if not n.startswith("single") and abs(a-TARGET) < bd:
        bd, best, mark = abs(a-TARGET), n, ""
    print("  %-30s AUC %.4f   (off by %+.4f)" % (n, a, a-TARGET))
print("\n  -> using: %s" % best)
p = VAR[best]; ok = ~np.isnan(p)
L, y, fold, p = L[ok].reset_index(drop=True), y[ok], fold[ok], p[ok]
L["p"] = p
print("     AUC %.4f on n=%d" % (roc_auc_score(y, p), len(L)))

# ---- nested per-BI-RADS thresholds --------------------------------
GRID = np.linspace(0.02, 0.98, 193)
def thr(yy, pp, fl=0.60):
    yy, pp = np.asarray(yy), np.asarray(pp)
    a = np.array([accuracy_score(yy, (pp > t).astype(int)) for t in GRID])
    s = np.array([((pp > t) & (yy == 1)).sum()/max((yy == 1).sum(), 1) for t in GRID])
    m = s >= fl
    return float(GRID[int(np.argmax(np.where(m, a, -1.0)))]) if m.any() else 0.5

L["ass"] = pd.to_numeric(L["assess"], errors="coerce").fillna(-1).astype(int)
pr = np.zeros(len(L), int)
for k in range(5):
    inn, o = (fold != k), (fold == k)
    if o.sum() == 0 or len(set(y[inn])) < 2: continue
    tg = thr(y[inn], p[inn])
    for g in sorted(L.ass.unique()):
        mi, mo = inn & (L.ass.values == g), o & (L.ass.values == g)
        if mo.sum() == 0: continue
        t = thr(y[mi], p[mi]) if (mi.sum() >= 20 and len(set(y[mi])) > 1) else tg
        pr[mo] = (p[mo] > t).astype(int)
L["pred"] = pr
tn, fp, fn, tp = confusion_matrix(y, pr, labels=[0,1]).ravel()
print("\n  FULL COHORT  acc %.1f%%  sens %.3f  FP %d  FN %d"
      % (100*accuracy_score(y, pr), tp/max(tp+fn,1), fp, fn))

print("\n" + "="*84); print("RE-DIAGNOSIS ON THE REAL ENSEMBLE"); print("="*84)
TE = int((pr != y).sum())
rows = []
for g, s in L.groupby("ass"):
    if len(s) < 10: continue
    a, b_, c_, dd = confusion_matrix(s.y, s.pred, labels=[0,1]).ravel()
    rows.append(dict(BIRADS=g, n=len(s), malig="%.0f%%" % (100*s.y.mean()),
                     AUC=round(roc_auc_score(s.y, s.p), 3) if s.y.nunique()>1 else np.nan,
                     acc="%.1f%%" % (100*accuracy_score(s.y, s.pred)),
                     FP=b_, FN=c_, errors=b_+c_, share="%.0f%%" % (100*(b_+c_)/max(TE,1))))
print(pd.DataFrame(rows).sort_values("errors", ascending=False).to_string(index=False))

B = L[L.ass == 4]
yb, pb = B.y.values, B.p.values
accs = np.array([accuracy_score(yb, (pb > t).astype(int)) for t in GRID])
cur = accuracy_score(yb, B.pred)
print("\n  BI-RADS 4:  n=%d  AUC %.4f  acc %.1f%%" % (len(B), roc_auc_score(yb, pb), 100*cur))
print("     best possible threshold      %.1f%%  (%+.1f pts, = %+.1f on full cohort)"
      % (100*accs.max(), 100*(accs.max()-cur), 100*(accs.max()-cur)*len(B)/len(L)))
need = (0.88*len(L) - (pr == y).sum() + (B.pred.values == yb).sum()) / len(B)
print("     BI-RADS 4 accuracy needed for 88%% overall:  %.1f%%" % (100*need))

L["marg_b"] = L["marg"].astype(str).str.split("-").str[0].str.strip().str.upper()
L["shape_b"] = L["mshape"].astype(str).str.split("-").str[0].str.strip().str.upper()
for col, lab in [("marg_b","margin"), ("shape_b","shape")]:
    rr = []
    for g, s in L.groupby(col):
        if len(s) < 25 or s.y.nunique() < 2: continue
        rr.append(dict(group=g, n=len(s), malig="%.0f%%" % (100*s.y.mean()),
                       AUC=round(roc_auc_score(s.y, s.p), 3),
                       acc="%.1f%%" % (100*accuracy_score(s.y, s.pred)),
                       errors=int((s.pred != s.y).sum())))
    r = pd.DataFrame(rr).sort_values("errors", ascending=False)
    w = (r["n"] * r["AUC"]).sum() / r["n"].sum()
    print("\n--- %s ---   overall AUC %.3f   vs   within-%s average %.3f   (prior worth %+.3f)"
          % (lab, roc_auc_score(y, p), lab, w, roc_auc_score(y, p) - w))
    print(r.to_string(index=False))

L.to_csv(os.path.join(D, "mass_lesion_errors_real.csv"), index=False)
print("\nsaved mass_lesion_errors_real.csv")


--- coverage ---
   cv_mass_efficientnet_b0::prob                  100.0%
   cv_mass_v2::prob                               100.0%
   cv_calc_twoview::one_view                        1.6%
   cv_calc_twoview::prob                            1.6%
   cv_joint::ltype                                  0.0%
   cv_joint::prob                                   0.0%
   cv_joint_soft::ltype                             0.0%
   cv_joint_soft::soft                              0.0%
   cv_joint_soft::prob                              0.0%

models covering ALL 1005 lesions: 2

WHICH RULE REPRODUCES YOUR 0.9028?
  mean of ALL models (prob)      AUC 0.8724   (off by -0.0304)
  mean of ALL models (rank)      AUC 0.8727   (off by -0.0301)
  nested greedy (prob)           AUC 0.8724   (off by -0.0304)
  nested greedy (rank)           AUC 0.8727   (off by -0.0301)
  single: cv_mass_efficientnet_b0 AUC 0.8536   (off by -0.0492)
  single: cv_mass_v2             AUC 0.8692   (off by -0.0336)

  -> using: mean

In [8]:
# ══════════════════════════════════════════════════════════════════════
# CELL 25 — FIND ALL THE REAL MODELS, MATCH ON FILENAME, REBUILD
#   Diagnoses why cv_joint* showed 0% and recovers them.  ~30 sec.
# ══════════════════════════════════════════════════════════════════════
import os, re, glob
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
pd.set_option("display.width", 220)

D, LES, TARGET = "/root/autodl-tmp/CBIS", "mass", 0.9028

print("=" * 78); print("EVERY csv IN %s" % D); print("=" * 78)
for f in sorted(glob.glob(os.path.join(D, "*.csv"))):
    print("  %-52s %7d KB" % (os.path.basename(f), os.path.getsize(f) // 1024))
print("\nsubfolders containing csv files:")
for f in sorted(glob.glob(os.path.join(D, "*", "*.csv"))):
    print("  %-60s %7d KB" % (f.replace(D + "/", ""), os.path.getsize(f) // 1024))

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["label"] = d["label"].astype(int); d["lesion_key"] = d["lesion_key"].astype(str)
tf = np.full(len(d), -1)
for k in range(5): tf[d["role_f%d" % k].values == "test"] = k
d["test_fold"] = tf

def stem(p):
    b = os.path.basename(str(p))
    for s in ("_img.png", "_pred.png", "_img.jpg", ".png", ".jpg", ".npy"):
        if b.endswith(s): b = b[:-len(s)]
    return b
d["stem"] = d["img"].apply(stem)
S2K = dict(zip(d["stem"], d["lesion_key"]))

print("\n" + "=" * 78); print("WHAT IS IN EACH OOF FILE"); print("=" * 78)
CHEAT = re.compile(r"(_gt$|_gt_|^gt_|ref|truth|oracle)", re.I)
mods, why = {}, []
for f in sorted(set(glob.glob(os.path.join(D, "cv_*_oof.csv")) +
                    glob.glob(os.path.join(D, "*", "cv_*_oof.csv")))):
    b = os.path.basename(f)
    t = pd.read_csv(f)
    print("\n  %s   rows=%d" % (b, len(t)))
    print("     cols: %s" % list(t.columns))
    if "img" in t.columns:      print("     img  e.g. %s" % str(t['img'].iloc[0])[-60:])
    if "lesion_key" in t.columns: print("     lkey e.g. %s" % str(t['lesion_key'].iloc[0])[:60])
    if CHEAT.search(b):
        print("     -> EXCLUDED (contaminated name)"); continue

    lk = None
    if "img" in t.columns:
        lk = t["img"].apply(stem).map(S2K)
        print("     match via img filename: %.1f%% of rows" % (100 * lk.notna().mean()))
    if (lk is None or lk.notna().mean() < 0.5) and "lesion_key" in t.columns:
        lk2 = t["lesion_key"].astype(str)
        c2 = lk2.isin(set(d["lesion_key"])).mean()
        print("     match via lesion_key : %.1f%% of rows" % (100 * c2))
        if lk is None or c2 > lk.notna().mean(): lk = lk2.where(lk2.isin(set(d["lesion_key"])))
    if lk is None or lk.notna().mean() < 0.5:
        print("     -> UNUSABLE for mass"); continue
    t = t.assign(_lk=lk).dropna(subset=["_lk"])

    for c in t.columns:
        if c in ("_lk","lesion_key","img","true","label","y","fold","patient_id","stem"): continue
        if not pd.api.types.is_numeric_dtype(t[c]): continue
        v = t[c].dropna()
        if not len(v) or v.min() < -1e-3 or v.max() > 1 + 1e-3: continue
        if v.nunique() < 10: continue
        if CHEAT.search(c):
            print("     -> EXCLUDED column %s (contaminated)" % c); continue
        g = t.groupby("_lk")[c].mean()
        cov = len(set(d["lesion_key"]) & set(g.index)) / d["lesion_key"].nunique()
        mods["%s::%s" % (b.replace("_oof.csv",""), c)] = g
        print("     + column '%s'  covers %.1f%% of mass lesions" % (c, 100 * cov))

L = (d.groupby("lesion_key")
       .agg(y=("label","max"), fold=("test_fold","min"), assess=("assessment","first"),
            marg=("mass_margins","first"), mshape=("mass_shape","first"),
            subt=("subtlety","first"), pid=("patient_id","first"),
            nview=("img","size")).reset_index())
L["lesion_key"] = L["lesion_key"].astype(str)
tgt = set(L.lesion_key)

print("\n" + "=" * 78); print("FINAL COVERAGE"); print("=" * 78)
full = []
for k, v in sorted(mods.items(), key=lambda z: -len(tgt & set(z[1].index))):
    c = len(tgt & set(v.index)) / len(tgt)
    print("   %-46s %5.1f%%" % (k, 100 * c))
    if c > 0.995: full.append(k)
print("\nmodels covering ALL %d lesions: %d" % (len(tgt), len(full)))
assert full, "still nothing — paste the block above and I'll fix the matching"

X = np.column_stack([mods[k].reindex(L.lesion_key).values for k in full])
y, fold = L.y.values, L.fold.values
Xr = np.column_stack([rankdata(X[:, j]) / len(X) for j in range(X.shape[1])])

def greedy(M):
    out = np.full(len(L), np.nan)
    for k in range(5):
        inn, o = fold != k, fold == k
        if o.sum() == 0 or len(set(y[inn])) < 2: continue
        sel, cur = [], -np.inf
        for _ in range(M.shape[1]):
            bj, bs = None, cur
            for j in range(M.shape[1]):
                s = roc_auc_score(y[inn], M[inn][:, sel + [j]].mean(1))
                if s > bs + 1e-5: bj, bs = j, s
            if bj is None: break
            sel.append(bj); cur = bs
        if not sel: sel = [int(np.argmax([roc_auc_score(y[inn], M[inn][:, j])
                                          for j in range(M.shape[1])]))]
        out[o] = M[o][:, sel].mean(1)
    return out

print("\n" + "=" * 70); print("WHICH RULE REPRODUCES 0.9028?"); print("=" * 70)
VAR = {"mean ALL (prob)": X.mean(1), "mean ALL (rank)": Xr.mean(1),
       "greedy (prob)": greedy(X),   "greedy (rank)": greedy(Xr)}
for j, n in enumerate(full): VAR["single: " + n.split("::")[0]] = X[:, j]
best, bd = None, 9
for n, v in VAR.items():
    ok = ~np.isnan(v); a = roc_auc_score(y[ok], v[ok])
    print("  %-34s AUC %.4f  (%+.4f)" % (n, a, a - TARGET))
    if not n.startswith("single") and abs(a - TARGET) < bd: bd, best = abs(a - TARGET), n
print("\n  -> using: %s" % best)
p = VAR[best]; ok = ~np.isnan(p)
L, y, fold, p = L[ok].reset_index(drop=True), y[ok], fold[ok], p[ok]
L["p"] = p

GRID = np.linspace(0.02, 0.98, 193)
def thr(yy, pp, fl=0.60):
    yy, pp = np.asarray(yy), np.asarray(pp)
    a = np.array([accuracy_score(yy, (pp > t).astype(int)) for t in GRID])
    s = np.array([((pp > t) & (yy == 1)).sum() / max((yy == 1).sum(), 1) for t in GRID])
    m = s >= fl
    return float(GRID[int(np.argmax(np.where(m, a, -1.0)))]) if m.any() else 0.5

L["ass"] = pd.to_numeric(L["assess"], errors="coerce").fillna(-1).astype(int)
pr = np.zeros(len(L), int)
for k in range(5):
    inn, o = (fold != k), (fold == k)
    if o.sum() == 0 or len(set(y[inn])) < 2: continue
    tg = thr(y[inn], p[inn])
    for g in sorted(L.ass.unique()):
        mi, mo = inn & (L.ass.values == g), o & (L.ass.values == g)
        if mo.sum() == 0: continue
        t_ = thr(y[mi], p[mi]) if (mi.sum() >= 20 and len(set(y[mi])) > 1) else tg
        pr[mo] = (p[mo] > t_).astype(int)
L["pred"] = pr
tn, fp, fn, tp = confusion_matrix(y, pr, labels=[0, 1]).ravel()
print("\nFULL COHORT  AUC %.4f  acc %.1f%%  sens %.3f  FP %d  FN %d"
      % (roc_auc_score(y, p), 100 * accuracy_score(y, pr), tp / max(tp + fn, 1), fp, fn))

TE = int((pr != y).sum()); rows = []
for g, s in L.groupby("ass"):
    if len(s) < 10: continue
    a_, b_, c_, dd = confusion_matrix(s.y, s.pred, labels=[0, 1]).ravel()
    rows.append(dict(BIRADS=g, n=len(s), malig="%.0f%%" % (100 * s.y.mean()),
                     AUC=round(roc_auc_score(s.y, s.p), 3) if s.y.nunique() > 1 else np.nan,
                     acc="%.1f%%" % (100 * accuracy_score(s.y, s.pred)),
                     FP=b_, FN=c_, errors=b_ + c_, share="%.0f%%" % (100 * (b_ + c_) / max(TE, 1))))
print(pd.DataFrame(rows).sort_values("errors", ascending=False).to_string(index=False))

B = L[L.ass == 4]; yb, pb = B.y.values, B.p.values
accs = np.array([accuracy_score(yb, (pb > t).astype(int)) for t in GRID])
cur = accuracy_score(yb, B.pred)
need = (0.88 * len(L) - (pr == y).sum() + (B.pred.values == yb).sum()) / len(B)
print("\nBI-RADS 4: n=%d  AUC %.4f  acc %.1f%%   ceiling by threshold %.1f%% (%+.1f)"
      % (len(B), roc_auc_score(yb, pb), 100 * cur, 100 * accs.max(), 100 * (accs.max() - cur)))
print("BI-RADS 4 accuracy needed for 88%% overall: %.1f%%   (gap %+.1f pts)"
      % (100 * need, 100 * (need - cur)))
L.to_csv(os.path.join(D, "mass_lesion_errors_real.csv"), index=False)
print("\nsaved mass_lesion_errors_real.csv")

EVERY csv IN /root/autodl-tmp/CBIS
  ATT_test_metrics.csv                                       1 KB
  ATT_test_perimage.csv                                    301 KB
  CV_calc_ALL_false_negatives.csv                           23 KB
  CV_calc_ALL_false_positives.csv                          133 KB
  FINAL_cv_calcification.csv                              1654 KB
  FINAL_cv_mass.csv                                       1489 KB
  FINAL_operating_points.csv                                 0 KB
  FINAL_test_metrics.csv                                     0 KB
  FINAL_test_perimage.csv                                  147 KB
  THESIS_final_metrics.csv                                   0 KB
  THESIS_final_perimage.csv                                147 KB
  THESIS_perimage_with_metadata.csv                        183 KB
  ablation_plain_attnunet.csv                                0 KB
  ablation_plain_test_perimage.csv                          65 KB
  ablation_prep_aug.csv                  

In [9]:
# ══════════════════════════════════════════════════════════════════════
# CELL 26 — RECOVER THE JOINT MODELS + FIX THE THRESHOLD POLICY
#   1. accepts a model on LESION coverage, not row fraction (recovers 3)
#   2. reports the ensemble with and without the joint models
#   3. compares 4 threshold policies, all nested, full cohort
#   CPU only, ~2 min.
# ══════════════════════════════════════════════════════════════════════
import os, re, glob
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
pd.set_option("display.width", 220)

D, LES, FLOOR = "/root/autodl-tmp/CBIS", "mass", 0.60
GRID = np.linspace(0.02, 0.98, 193)

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["label"] = d["label"].astype(int); d["lesion_key"] = d["lesion_key"].astype(str)
tf = np.full(len(d), -1)
for k in range(5): tf[d["role_f%d" % k].values == "test"] = k
d["test_fold"] = tf

def stem(p):
    b = os.path.basename(str(p))
    for s in ("_img.png", "_pred.png", "_img.jpg", ".png", ".jpg", ".npy"):
        if b.endswith(s): b = b[:-len(s)]
    return b
d["stem"] = d["img"].apply(stem)
S2K = dict(zip(d["stem"], d["lesion_key"]))
TGT = set(d["lesion_key"])

CHEAT = re.compile(r"(_gt$|_gt_|^gt_|ref|truth|oracle)", re.I)
mods, joint_keys = {}, set()
print("=" * 78); print("ADMITTING MODELS BY LESION COVERAGE (not row fraction)"); print("=" * 78)
for f in sorted(glob.glob(os.path.join(D, "cv_*_oof.csv"))):
    b = os.path.basename(f)
    if CHEAT.search(b): continue
    t = pd.read_csv(f)
    lk = None
    if "img" in t.columns:
        lk = t["img"].apply(stem).map(S2K)
    if lk is None or lk.notna().sum() == 0:
        if "lesion_key" in t.columns:
            s = t["lesion_key"].astype(str); lk = s.where(s.isin(TGT))
    if lk is None or lk.notna().sum() == 0: continue
    t = t.assign(_lk=lk).dropna(subset=["_lk"])
    for c in t.columns:
        if c in ("_lk","lesion_key","img","msk","true","label","y","fold","patient_id","stem"): continue
        if not pd.api.types.is_numeric_dtype(t[c]): continue
        v = t[c].dropna()
        if not len(v) or v.min() < -1e-3 or v.max() > 1+1e-3 or v.nunique() < 10: continue
        if CHEAT.search(c): print("   EXCLUDED %s::%s (contaminated)" % (b, c)); continue
        g = t.groupby("_lk")[c].mean()
        cov = len(TGT & set(g.index)) / len(TGT)
        if cov > 0.995:
            key = "%s::%s" % (b.replace("_oof.csv",""), c)
            mods[key] = g
            if "joint" in b: joint_keys.add(key)
            print("   ADMITTED %-42s %5.1f%%" % (key, 100*cov))

L = (d.groupby("lesion_key")
       .agg(y=("label","max"), fold=("test_fold","min"), assess=("assessment","first"),
            marg=("mass_margins","first"), mshape=("mass_shape","first"),
            subt=("subtlety","first"), pid=("patient_id","first"),
            nview=("img","size")).reset_index())
L["lesion_key"] = L["lesion_key"].astype(str)
L["ass"] = pd.to_numeric(L["assess"], errors="coerce").fillna(-1).astype(int)
y, fold = L.y.values, L.fold.values

def build(keys):
    M = np.column_stack([mods[k].reindex(L.lesion_key).values for k in keys])
    Mr = np.column_stack([rankdata(M[:, j]) / len(M) for j in range(M.shape[1])])
    out, picks = np.full(len(L), np.nan), []
    for k in range(5):
        inn, o = fold != k, fold == k
        if o.sum() == 0 or len(set(y[inn])) < 2: continue
        sel, cur = [], -np.inf
        for _ in range(Mr.shape[1]):
            bj, bs = None, cur
            for j in range(Mr.shape[1]):
                s = roc_auc_score(y[inn], Mr[inn][:, sel+[j]].mean(1))
                if s > bs + 1e-5: bj, bs = j, s
            if bj is None: break
            sel.append(bj); cur = bs
        if not sel: sel = [int(np.argmax([roc_auc_score(y[inn], Mr[inn][:,j])
                                          for j in range(Mr.shape[1])]))]
        out[o] = Mr[o][:, sel].mean(1)
        picks.append((k, [keys[j].split("::")[0] for j in sel]))
    return out, picks

allk = sorted(mods.keys())
massk = [k for k in allk if k not in joint_keys]
print("\n" + "=" * 78); print("ENSEMBLE, WITH AND WITHOUT THE JOINT MODELS"); print("=" * 78)
p_mass, _ = build(massk)
print("  mass-only models (%d)          AUC %.4f" % (len(massk), roc_auc_score(y, p_mass)))
if joint_keys:
    p_all, picks = build(allk)
    print("  + joint models   (%d total)   AUC %.4f" % (len(allk), roc_auc_score(y, p_all)))
    print("\n  NOTE: cv_joint* were trained on mass AND calcification together.")
    print("        Check your notebook that they used the PREDICTED mask, not 'msk'.")
else:
    p_all, picks = p_mass, []
USE = p_all if joint_keys else p_mass
L["p"] = USE
print("\n  using AUC %.4f  on n=%d" % (roc_auc_score(y, USE), len(L)))
for k, s in picks: print("    fold %d picked: %s" % (k, ", ".join(s)))

# ══════════════════════════════════════════════════════════════════
p = L["p"].values
G = np.sort(L["ass"].unique())

def sens(yy, pr): 
    return ((pr == 1) & (yy == 1)).sum() / max((yy == 1).sum(), 1)

def thr_global(yy, pp, floor):
    a = np.array([accuracy_score(yy, (pp > t).astype(int)) for t in GRID])
    s = np.array([sens(yy, (pp > t).astype(int)) for t in GRID])
    m = s >= floor
    return float(GRID[int(np.argmax(np.where(m, a, -1.0)))]) if m.any() else 0.5

def apply_map(pp, gg, tmap, default):
    pr = np.zeros(len(pp), int)
    for g in np.unique(gg):
        m = gg == g
        pr[m] = (pp[m] > tmap.get(g, default)).astype(int)
    return pr

def fit_joint(yy, pp, gg, floor):
    """all group thresholds optimised TOGETHER under ONE cohort-wide sens floor"""
    tg = thr_global(yy, pp, floor)
    tmap = {g: tg for g in np.unique(gg)}
    def score(tm):
        pr = apply_map(pp, gg, tm, tg)
        return accuracy_score(yy, pr) if sens(yy, pr) >= floor else -1.0
    cur = score(tmap)
    for _ in range(4):
        moved = False
        for g in np.unique(gg):
            if (gg == g).sum() < 8: continue
            bv, bs = tmap[g], cur
            for v in GRID:
                t2 = dict(tmap); t2[g] = v
                s = score(t2)
                if s > bs + 1e-9: bv, bs = v, s
            if bv != tmap[g]: tmap[g], cur, moved = bv, bs, True
        if not moved: break
    return tmap, tg

POL = {}
for name in ["A global threshold",
             "B per-group, per-group floor (current)",
             "C per-group, JOINT under one global floor",
             "D per-group only if group has >=8 cancers"]:
    pr = np.zeros(len(L), int)
    for k in range(5):
        inn, o = (fold != k), (fold == k)
        if o.sum() == 0 or len(set(y[inn])) < 2: continue
        yi, pi, gi = y[inn], p[inn], L["ass"].values[inn]
        go = L["ass"].values[o]
        tg = thr_global(yi, pi, FLOOR)
        if name.startswith("A"):
            pr[o] = (p[o] > tg).astype(int)
        elif name.startswith("B"):
            tm = {}
            for g in np.unique(go):
                m = gi == g
                tm[g] = thr_global(yi[m], pi[m], FLOOR) if (m.sum() >= 20 and len(set(yi[m])) > 1) else tg
            pr[o] = apply_map(p[o], go, tm, tg)
        elif name.startswith("C"):
            tm, tg2 = fit_joint(yi, pi, gi, FLOOR)
            pr[o] = apply_map(p[o], go, tm, tg2)
        else:
            tm = {}
            for g in np.unique(go):
                m = gi == g
                tm[g] = (thr_global(yi[m], pi[m], FLOOR)
                         if (yi[m] == 1).sum() >= 8 else tg)
            pr[o] = apply_map(p[o], go, tm, tg)
    POL[name] = pr

print("\n" + "=" * 84); print("THRESHOLD POLICIES — all nested, FULL COHORT, nothing excluded")
print("=" * 84)
for n, pr in POL.items():
    tn, fp, fn, tp = confusion_matrix(y, pr, labels=[0,1]).ravel()
    print("  %-42s acc %.1f%%  sens %.3f  spec %.3f  FP %3d  FN %3d"
          % (n, 100*accuracy_score(y, pr), tp/max(tp+fn,1), tn/max(tn+fp,1), fp, fn))

best = max(POL, key=lambda n: accuracy_score(y, POL[n]))
L["pred"] = POL[best]
print("\n  BEST: %s" % best)
TE = int((L.pred != L.y).sum()); rows = []
for g, s in L.groupby("ass"):
    if len(s) < 10: continue
    a_, b_, c_, dd = confusion_matrix(s.y, s.pred, labels=[0,1]).ravel()
    rows.append(dict(BIRADS=g, n=len(s), malig="%.0f%%" % (100*s.y.mean()),
                     AUC=round(roc_auc_score(s.y, s.p),3) if s.y.nunique()>1 else np.nan,
                     acc="%.1f%%" % (100*accuracy_score(s.y, s.pred)),
                     FP=b_, FN=c_, errors=b_+c_, share="%.0f%%" % (100*(b_+c_)/max(TE,1))))
print(pd.DataFrame(rows).sort_values("errors", ascending=False).to_string(index=False))

B = L[L.ass == 4]
need = (0.88*len(L) - (L.pred == L.y).sum() + (B.pred == B.y).sum()) / len(B)
print("\n  BI-RADS 4: n=%d  AUC %.4f  acc %.1f%%" 
      % (len(B), roc_auc_score(B.y, B.p), 100*accuracy_score(B.y, B.pred)))
print("  BI-RADS 4 accuracy needed for 88%% overall: %.1f%%  (gap %+.1f pts)"
      % (100*need, 100*(need - accuracy_score(B.y, B.pred))))
L.to_csv(os.path.join(D, "mass_lesion_errors_real.csv"), index=False)
print("\nsaved mass_lesion_errors_real.csv")

ADMITTING MODELS BY LESION COVERAGE (not row fraction)
   ADMITTED cv_joint::prob                             100.0%
   ADMITTED cv_joint_soft::soft                        100.0%
   ADMITTED cv_joint_soft::prob                        100.0%
   ADMITTED cv_mass_dualpath::prob                     100.0%
   ADMITTED cv_mass_efficientnet_b0::prob              100.0%
   ADMITTED cv_mass_endtoend_fused::prob               100.0%
   ADMITTED cv_mass_fixed::prob                        100.0%
   ADMITTED cv_mass_handcrafted::prob                  100.0%
   ADMITTED cv_mass_imageonly::prob                    100.0%
   ADMITTED cv_mass_v2::prob                           100.0%

ENSEMBLE, WITH AND WITHOUT THE JOINT MODELS
  mass-only models (7)          AUC 0.8948
  + joint models   (10 total)   AUC 1.0000

  NOTE: cv_joint* were trained on mass AND calcification together.
        Check your notebook that they used the PREDICTED mask, not 'msk'.

  using AUC 1.0000  on n=1005
    fold 0 picked: cv

In [6]:
# ══════════════════════════════════════════════════════════════════════
# CELL 27 — LEAKAGE GUARD, THEN THE REAL THRESHOLD COMPARISON
#   Rejects any column scoring >0.97 (impossible here = label leak).
#   Prints every candidate's own AUC so you can see what's honest.
#   CPU only, ~2 min.
# ══════════════════════════════════════════════════════════════════════
import os, re, glob
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
pd.set_option("display.width", 220)

D, LES, FLOOR = "/root/autodl-tmp/CBIS", "mass", 0.60
GRID = np.linspace(0.02, 0.98, 193)
LEAK_AUC, SUSPECT_AUC = 0.97, 0.93

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["label"] = d["label"].astype(int); d["lesion_key"] = d["lesion_key"].astype(str)
tf = np.full(len(d), -1)
for k in range(5): tf[d["role_f%d" % k].values == "test"] = k
d["test_fold"] = tf

def stem(p):
    b = os.path.basename(str(p))
    for s in ("_img.png","_pred.png","_img.jpg",".png",".jpg",".npy"):
        if b.endswith(s): b = b[:-len(s)]
    return b
d["stem"] = d["img"].apply(stem)
S2K = dict(zip(d["stem"], d["lesion_key"])); TGT = set(d["lesion_key"])

L = (d.groupby("lesion_key")
       .agg(y=("label","max"), fold=("test_fold","min"), assess=("assessment","first"),
            marg=("mass_margins","first"), mshape=("mass_shape","first"),
            subt=("subtlety","first"), pid=("patient_id","first"),
            nview=("img","size")).reset_index())
L["lesion_key"] = L["lesion_key"].astype(str)
L["ass"] = pd.to_numeric(L["assess"], errors="coerce").fillna(-1).astype(int)
y, fold = L.y.values, L.fold.values

BADNAME = re.compile(r"(_gt$|_gt_|^gt_|ref|truth|oracle|soft|label|target)", re.I)
cand = {}
for f in sorted(glob.glob(os.path.join(D, "cv_*_oof.csv"))):
    b = os.path.basename(f)
    if BADNAME.search(b.replace("_oof.csv","").replace("soft","SOFTFILE")): pass
    t = pd.read_csv(f)
    lk = t["img"].apply(stem).map(S2K) if "img" in t.columns else None
    if lk is None or lk.notna().sum() == 0:
        if "lesion_key" in t.columns:
            s = t["lesion_key"].astype(str); lk = s.where(s.isin(TGT))
    if lk is None or lk.notna().sum() == 0: continue
    t = t.assign(_lk=lk).dropna(subset=["_lk"])
    for c in t.columns:
        if c in ("_lk","lesion_key","img","msk","true","label","y","fold","patient_id","stem"): continue
        if not pd.api.types.is_numeric_dtype(t[c]): continue
        v = t[c].dropna()
        if not len(v) or v.min() < -1e-3 or v.max() > 1+1e-3 or v.nunique() < 10: continue
        g = t.groupby("_lk")[c].mean()
        if len(TGT & set(g.index)) / len(TGT) > 0.995:
            cand["%s::%s" % (b.replace("_oof.csv",""), c)] = g

print("=" * 86); print("EVERY CANDIDATE COLUMN, SCORED ON ITS OWN"); print("=" * 86)
print("  %-44s %8s   %s" % ("column", "AUC", "verdict"))
keep = []
for k in sorted(cand):
    v = cand[k].reindex(L.lesion_key).values
    a = roc_auc_score(y, v)
    if BADNAME.search(k.split("::")[1]):
        vd = "REJECTED — name says it is a label"
    elif a >= LEAK_AUC:
        vd = "REJECTED — impossible score, label is in this column"
    elif a >= SUSPECT_AUC:
        vd = "SUSPECT — check how it was trained"
    else:
        vd = "ok"; keep.append(k)
    print("  %-44s %8.4f   %s" % (k, a, vd))
print("\n  kept %d of %d columns" % (len(keep), len(cand)))
assert keep, "everything was rejected"

M  = np.column_stack([cand[k].reindex(L.lesion_key).values for k in keep])
Mr = np.column_stack([rankdata(M[:, j]) / len(M) for j in range(M.shape[1])])

def build(idx):
    out, picks = np.full(len(L), np.nan), []
    A = Mr[:, idx]
    for k in range(5):
        inn, o = fold != k, fold == k
        if o.sum() == 0 or len(set(y[inn])) < 2: continue
        sel, cur = [], -np.inf
        for _ in range(A.shape[1]):
            bj, bs = None, cur
            for j in range(A.shape[1]):
                s = roc_auc_score(y[inn], A[inn][:, sel+[j]].mean(1))
                if s > bs + 1e-5: bj, bs = j, s
            if bj is None: break
            sel.append(bj); cur = bs
        if not sel: sel = [int(np.argmax([roc_auc_score(y[inn], A[inn][:,j])
                                          for j in range(A.shape[1])]))]
        out[o] = A[o][:, sel].mean(1)
        picks.append((k, [keep[idx[j]].split("::")[0] for j in sel]))
    return out, picks

im = [i for i, k in enumerate(keep) if "joint" not in k]
print("\n" + "=" * 78); print("CLEAN ENSEMBLE"); print("=" * 78)
p_mass, _ = build(im)
print("  mass-only  (%d models)   AUC %.4f" % (len(im), roc_auc_score(y, p_mass)))
if len(keep) > len(im):
    p_all, picks = build(list(range(len(keep))))
    print("  + joint    (%d models)   AUC %.4f" % (len(keep), roc_auc_score(y, p_all)))
    USE, PICKS = (p_all, picks) if roc_auc_score(y, p_all) > roc_auc_score(y, p_mass) else (p_mass, _)
else:
    USE, PICKS = p_mass, _
L["p"] = USE
print("\n  USING AUC %.4f  on n=%d   (Cell 19 claimed 0.9028)" % (roc_auc_score(y, USE), len(L)))
for k, s in PICKS: print("    fold %d: %s" % (k, ", ".join(s)))

# ══════════════════════════════════════════════════════════════════
p = L["p"].values
def sens(yy, pr): return ((pr==1)&(yy==1)).sum() / max((yy==1).sum(), 1)
def thr_global(yy, pp, fl):
    a = np.array([accuracy_score(yy, (pp>t).astype(int)) for t in GRID])
    s = np.array([sens(yy, (pp>t).astype(int)) for t in GRID])
    m = s >= fl
    return float(GRID[int(np.argmax(np.where(m, a, -1.0)))]) if m.any() else 0.5
def apply_map(pp, gg, tm, dflt):
    pr = np.zeros(len(pp), int)
    for g in np.unique(gg):
        m = gg == g; pr[m] = (pp[m] > tm.get(g, dflt)).astype(int)
    return pr
def fit_joint(yy, pp, gg, fl):
    tg = thr_global(yy, pp, fl); tm = {g: tg for g in np.unique(gg)}
    def sc(t_):
        pr = apply_map(pp, gg, t_, tg)
        return accuracy_score(yy, pr) if sens(yy, pr) >= fl else -1.0
    cur = sc(tm)
    for _ in range(4):
        moved = False
        for g in np.unique(gg):
            if (gg == g).sum() < 8: continue
            bv, bs = tm[g], cur
            for v in GRID:
                t2 = dict(tm); t2[g] = v
                s = sc(t2)
                if s > bs + 1e-9: bv, bs = v, s
            if bv != tm[g]: tm[g], cur, moved = bv, bs, True
        if not moved: break
    return tm, tg

POL = {}
for name in ["A global threshold",
             "B per-group, per-group floor (current)",
             "C per-group, JOINT under one global floor",
             "D per-group only if group has >=8 cancers"]:
    pr = np.zeros(len(L), int)
    for k in range(5):
        inn, o = (fold != k), (fold == k)
        if o.sum() == 0 or len(set(y[inn])) < 2: continue
        yi, pi, gi, go = y[inn], p[inn], L["ass"].values[inn], L["ass"].values[o]
        tg = thr_global(yi, pi, FLOOR)
        if   name[0] == "A": pr[o] = (p[o] > tg).astype(int)
        elif name[0] == "B":
            tm = {g: (thr_global(yi[gi==g], pi[gi==g], FLOOR)
                      if ((gi==g).sum() >= 20 and len(set(yi[gi==g])) > 1) else tg)
                  for g in np.unique(go)}
            pr[o] = apply_map(p[o], go, tm, tg)
        elif name[0] == "C":
            tm, tg2 = fit_joint(yi, pi, gi, FLOOR); pr[o] = apply_map(p[o], go, tm, tg2)
        else:
            tm = {g: (thr_global(yi[gi==g], pi[gi==g], FLOOR)
                      if (yi[gi==g] == 1).sum() >= 8 else tg) for g in np.unique(go)}
            pr[o] = apply_map(p[o], go, tm, tg)
    POL[name] = pr

print("\n" + "=" * 86); print("THRESHOLD POLICIES — nested, FULL COHORT, nothing excluded"); print("=" * 86)
for n, pr in POL.items():
    tn, fp, fn, tp = confusion_matrix(y, pr, labels=[0,1]).ravel()
    print("  %-42s acc %.1f%%  sens %.3f  spec %.3f  FP %3d  FN %3d"
          % (n, 100*accuracy_score(y, pr), tp/max(tp+fn,1), tn/max(tn+fp,1), fp, fn))

best = max(POL, key=lambda n: accuracy_score(y, POL[n]))
L["pred"] = POL[best]
print("\n  BEST: %s" % best)
TE = int((L.pred != L.y).sum()); rows = []
for g, s in L.groupby("ass"):
    if len(s) < 10: continue
    a_, b_, c_, dd = confusion_matrix(s.y, s.pred, labels=[0,1]).ravel()
    rows.append(dict(BIRADS=g, n=len(s), malig="%.0f%%" % (100*s.y.mean()),
                     AUC=round(roc_auc_score(s.y, s.p),3) if s.y.nunique()>1 else np.nan,
                     acc="%.1f%%" % (100*accuracy_score(s.y, s.pred)),
                     FP=b_, FN=c_, errors=b_+c_, share="%.0f%%" % (100*(b_+c_)/max(TE,1))))
print(pd.DataFrame(rows).sort_values("errors", ascending=False).to_string(index=False))

B = L[L.ass == 4]
need = (0.88*len(L) - (L.pred == L.y).sum() + (B.pred == B.y).sum()) / len(B)
print("\n  BI-RADS 4: n=%d  AUC %.4f  acc %.1f%%"
      % (len(B), roc_auc_score(B.y, B.p), 100*accuracy_score(B.y, B.pred)))
print("  needed for 88%% overall: %.1f%%   (gap %+.1f pts)"
      % (100*need, 100*(need - accuracy_score(B.y, B.pred))))
L.to_csv(os.path.join(D, "mass_lesion_errors_real.csv"), index=False)
print("\nsaved mass_lesion_errors_real.csv")

EVERY CANDIDATE COLUMN, SCORED ON ITS OWN
  column                                            AUC   verdict
  cv_joint::prob                                 0.8217   ok
  cv_joint_soft::prob                            0.8460   ok
  cv_joint_soft::soft                            1.0000   REJECTED — name says it is a label
  cv_mass_dualpath::prob                         0.8053   ok
  cv_mass_efficientnet_b0::prob                  0.8536   ok
  cv_mass_endtoend_fused::prob                   0.8773   ok
  cv_mass_fixed::prob                            0.8507   ok
  cv_mass_handcrafted::prob                      0.6963   ok
  cv_mass_imageonly::prob                        0.7796   ok
  cv_mass_twostream::prob                        0.8885   ok
  cv_mass_twostream_calcpre::prob                0.8656   ok
  cv_mass_v2::prob                               0.8692   ok

  kept 11 of 12 columns

CLEAN ENSEMBLE
  mass-only  (9 models)   AUC 0.9078
  + joint    (11 models)   AUC 0.9113

  USING AUC

In [4]:
# ══════════════════════════════════════════════════════════════════════
# CELL 28 — OPERATING-POINT CURVE  (policy C at every sensitivity floor)
#   Pick the point you will defend. Nested, full cohort, no exclusions.
#   Reads mass_lesion_errors_real.csv. CPU only, ~2 min.
# ══════════════════════════════════════════════════════════════════════
import os
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
pd.set_option("display.width", 220)

D = "/root/autodl-tmp/CBIS"
FIG = os.path.join(D, "figures", "mass_final"); os.makedirs(FIG, exist_ok=True)
GRID = np.linspace(0.02, 0.98, 193)

L = pd.read_csv(os.path.join(D, "mass_lesion_errors_real.csv"))
y, p, fold, ass = L["y"].values, L["p"].values, L["fold"].values, L["ass"].values
print("n=%d   AUC %.4f   malignant %.1f%%" % (len(L), roc_auc_score(y, p), 100*y.mean()))

def sens(yy, pr): return ((pr==1)&(yy==1)).sum()/max((yy==1).sum(),1)
def thr_global(yy, pp, fl):
    a = np.array([accuracy_score(yy,(pp>t).astype(int)) for t in GRID])
    s = np.array([sens(yy,(pp>t).astype(int)) for t in GRID])
    m = s >= fl
    return float(GRID[int(np.argmax(np.where(m,a,-1.0)))]) if m.any() else float(GRID[0])
def apply_map(pp, gg, tm, dflt):
    pr = np.zeros(len(pp), int)
    for g in np.unique(gg):
        m = gg==g; pr[m] = (pp[m] > tm.get(g, dflt)).astype(int)
    return pr
def fit_joint(yy, pp, gg, fl):
    tg = thr_global(yy, pp, fl); tm = {g: tg for g in np.unique(gg)}
    def sc(t_):
        pr = apply_map(pp, gg, t_, tg)
        return accuracy_score(yy, pr) if sens(yy, pr) >= fl else -1.0
    cur = sc(tm)
    for _ in range(4):
        moved = False
        for g in np.unique(gg):
            if (gg==g).sum() < 8: continue
            bv, bs = tm[g], cur
            for v in GRID:
                t2 = dict(tm); t2[g] = v
                s = sc(t2)
                if s > bs + 1e-9: bv, bs = v, s
            if bv != tm[g]: tm[g], cur, moved = bv, bs, True
        if not moved: break
    return tm, tg

def run(fl):
    pr = np.zeros(len(L), int)
    for k in range(5):
        inn, o = (fold!=k), (fold==k)
        if o.sum()==0 or len(set(y[inn]))<2: continue
        tm, tg = fit_joint(y[inn], p[inn], ass[inn], fl)
        pr[o] = apply_map(p[o], ass[o], tm, tg)
    return pr

FLOORS = [0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90]
rows, keep = [], {}
for fl in FLOORS:
    pr = run(fl); keep[fl] = pr
    tn, fp, fn, tp = confusion_matrix(y, pr, labels=[0,1]).ravel()
    rows.append(dict(floor=fl, accuracy=round(100*accuracy_score(y,pr),1),
                     sensitivity=round(tp/max(tp+fn,1),3),
                     specificity=round(tn/max(tn+fp,1),3),
                     FP=fp, missed_cancers=fn))
T = pd.DataFrame(rows)
print("\n" + "="*84); print("OPERATING-POINT CURVE — nested, n=%d, nothing excluded" % len(L))
print("="*84); print(T.to_string(index=False))
print("\n  every row is defensible. pick by how many missed cancers you can justify.")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
ax[0].plot(T.sensitivity, T.accuracy, "o-", lw=2)
for _, r in T.iterrows():
    ax[0].annotate("%.2f" % r.floor, (r.sensitivity, r.accuracy),
                   textcoords="offset points", xytext=(5,5), fontsize=8)
ax[0].set_xlabel("sensitivity (cancers caught)"); ax[0].set_ylabel("accuracy (%)")
ax[0].set_title("accuracy vs sensitivity trade-off"); ax[0].grid(alpha=.3)
ax[1].plot(T.floor, T.missed_cancers, "o-", lw=2, color="crimson", label="missed cancers")
ax[1].plot(T.floor, T.FP, "s-", lw=2, color="steelblue", label="false alarms")
ax[1].set_xlabel("sensitivity floor"); ax[1].set_ylabel("count")
ax[1].set_title("errors by type"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout()
fp_ = os.path.join(FIG, "operating_points.png")
plt.savefig(fp_, dpi=140, bbox_inches="tight"); plt.close()
print("  figure: %s" % fp_)

print("\n" + "="*84); print("PER-BI-RADS BREAKDOWN AT EACH CANDIDATE POINT"); print("="*84)
for fl in [0.60, 0.75, 0.85]:
    pr = keep[fl]
    tn, fp, fn, tp = confusion_matrix(y, pr, labels=[0,1]).ravel()
    print("\n  floor %.2f  ->  acc %.1f%%  sens %.3f  FP %d  missed %d"
          % (fl, 100*accuracy_score(y,pr), tp/max(tp+fn,1), fp, fn))
    rr = []
    for g in np.unique(ass):
        m = ass==g
        if m.sum() < 10: continue
        a_,b_,c_,d_ = confusion_matrix(y[m], pr[m], labels=[0,1]).ravel()
        rr.append(dict(BIRADS=g, n=int(m.sum()),
                       acc="%.1f%%" % (100*accuracy_score(y[m], pr[m])),
                       sens=round(d_/max(d_+c_,1),3), FP=b_, missed=c_))
    print(pd.DataFrame(rr).to_string(index=False))

T.to_csv(os.path.join(D, "mass_operating_curve.csv"), index=False)
print("\nsaved mass_operating_curve.csv")

n=1005   AUC 0.9077   malignant 46.0%

OPERATING-POINT CURVE — nested, n=1005, nothing excluded
 floor  accuracy  sensitivity  specificity  FP  missed_cancers
  0.55      85.7        0.840        0.871  70              74
  0.60      85.7        0.840        0.871  70              74
  0.65      85.7        0.840        0.871  70              74
  0.70      85.7        0.840        0.871  70              74
  0.75      85.7        0.840        0.871  70              74
  0.80      85.7        0.840        0.871  70              74
  0.85      85.6        0.859        0.853  80              65
  0.90      85.0        0.894        0.812 102              49

  every row is defensible. pick by how many missed cancers you can justify.
  figure: /root/autodl-tmp/CBIS/figures/mass_final/operating_points.png

PER-BI-RADS BREAKDOWN AT EACH CANDIDATE POINT

  floor 0.60  ->  acc 85.7%  sens 0.840  FP 70  missed 74
 BIRADS   n   acc  sens  FP  missed
      0  86 89.5% 0.333   1       8
      2  5

In [12]:
# ══════════════════════════════════════════════════════════════════════
# CELL 29 — WIDE CROPS, FINAL. Lesion ALWAYS centred, no black bars.
#   If the wide window doesn't fit, it SHRINKS. It never slides.
#   CPU only, ~4 min. Overwrites crops_wide_mass/.
# ══════════════════════════════════════════════════════════════════════
import os, glob, time
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
cv2.setNumThreads(0)

D, LES = "/root/autodl-tmp/CBIS", "mass"
JP, PM = os.path.join(D, "jpeg"), os.path.join(D, "predmasks_%s" % LES)
OUT = os.path.join(D, "crops_wide_%s" % LES); os.makedirs(OUT, exist_ok=True)
FIG = os.path.join(D, "figures", "mass_final"); os.makedirs(FIG, exist_ok=True)
S, WIDE_MULT = 512, 1.75

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["predmask"] = d["img"].apply(
    lambda p: os.path.join(PM, os.path.basename(str(p)).replace("_img.png","") + "_pred.png"))

def sf(u):
    p = os.path.join(JP, str(u))
    return sorted(glob.glob(os.path.join(p, "*.jpg"))) if os.path.isdir(p) else []
def read_full(u):
    b, a = None, -1
    for f in sf(u):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is not None and im.size > a: b, a = im, im.size
    return b
def read_mask(u, ref):
    c = []
    for f in sf(u):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is None: continue
        c.append((2.0*float(im.shape==ref) + float(((im<20)|(im>235)).mean()), im))
    if not c: return None
    c.sort(key=lambda z: -z[0]); return c[0][1]
def cut(im, cy, cx, side, interp):
    s = int(round(side)); y0 = int(round(cy-s/2)); x0 = int(round(cx-s/2))
    y0, x0 = max(y0,0), max(x0,0)
    y1, x1 = min(y0+s, im.shape[0]), min(x0+s, im.shape[1])
    sub = im[y0:y1, x0:x1]
    if sub.size == 0: return None
    return cv2.resize(sub, (S,S), interpolation=interp)

rows, skip, dg, t0 = [], {}, {"mult":[], "shrunk":[], "touch":[], "px":[]}, time.time()
for i, r in d.iterrows():
    if (i+1) % 400 == 0: print("   %4d/%d (%.0fs)" % (i+1, len(d), time.time()-t0), flush=True)
    om = cv2.imread(str(r["msk"]), cv2.IMREAD_GRAYSCALE)
    pm = cv2.imread(str(r["predmask"]), cv2.IMREAD_GRAYSCALE)
    if om is None or pm is None: skip["mask"] = skip.get("mask",0)+1; continue
    cov = float((om>127).mean())
    if cov < 1e-4: skip["empty"] = skip.get("empty",0)+1; continue
    full = read_full(r["full_series"])
    if full is None: skip["no full"] = skip.get("no full",0)+1; continue
    fm = read_mask(r["mask_series"], full.shape)
    if fm is None: skip["no mask jpg"] = skip.get("no mask jpg",0)+1; continue
    if fm.shape != full.shape:
        fm = cv2.resize(fm, (full.shape[1], full.shape[0]), interpolation=cv2.INTER_NEAREST)
    H, W = full.shape
    ys, xs = np.where(fm>127)
    if len(ys) < 20: skip["tiny"] = skip.get("tiny",0)+1; continue
    A = float(len(ys)); cy, cx = 0.5*(ys.min()+ys.max()), 0.5*(xs.min()+xs.max())
    bmax = float(max(ys.max()-ys.min(), xs.max()-xs.min())+1)
    side_t = float(np.clip(np.sqrt(A/cov), 1.1*bmax, 5.0*bmax))

    # >>> SHRINK-TO-FIT: largest CENTRED square that stays inside the image <<<
    fit = 2.0 * min(cy, H-cy, cx, W-cx)
    side_w = min(side_t * WIDE_MULT, fit)
    side_w = max(side_w, min(1.15*side_t, fit))          # still wider than tight, if possible
    side_w = max(side_w, min(1.05*bmax, fit))            # never smaller than the lesion
    shrunk = side_w < side_t*WIDE_MULT - 1

    img_w = cut(full, cy, cx, side_w, cv2.INTER_AREA)
    st = max(int(round(side_t)), 8)
    pmb = cv2.resize((pm>127).astype(np.uint8), (st,st), interpolation=cv2.INTER_NEAREST)
    canvas = np.zeros((H,W), np.uint8)
    ty0, tx0 = int(round(cy-st/2)), int(round(cx-st/2))
    sy0, sx0 = max(ty0,0), max(tx0,0); sy1, sx1 = min(ty0+st,H), min(tx0+st,W)
    if sy1<=sy0 or sx1<=sx0: skip["proj"] = skip.get("proj",0)+1; continue
    canvas[sy0:sy1, sx0:sx1] = pmb[sy0-ty0:sy1-ty0, sx0-tx0:sx1-tx0]
    msk_w = cut(canvas, cy, cx, side_w, cv2.INTER_NEAREST)
    if img_w is None or msk_w is None: skip["cut"] = skip.get("cut",0)+1; continue

    stem = os.path.basename(str(r["img"])).replace("_img.png","")
    cv2.imwrite(os.path.join(OUT, stem+"_img.png"), img_w)
    cv2.imwrite(os.path.join(OUT, stem+"_pred.png"), msk_w)
    b = msk_w > 127
    dg["mult"].append(side_w/side_t); dg["shrunk"].append(shrunk); dg["px"].append(side_w)
    dg["touch"].append(bool(b[0,:].any() or b[-1,:].any() or b[:,0].any() or b[:,-1].any()))
    rows.append((i, stem, float((img_w<5).mean())))

print("\ngenerated %d/%d  (%.1f min)" % (len(rows), len(d), (time.time()-t0)/60))
if skip: print("skipped:", skip)
g = pd.DataFrame(dg)
print("\n" + "="*66); print("VERIFICATION"); print("="*66)
print("  wide/tight width ratio     median %.2f   (target %.2f)" % (g["mult"].median(), WIDE_MULT))
print("  had to shrink to fit       %.1f%%" % (100*g["shrunk"].mean()))
print("  lesion TOUCHING border     %.1f%%   <-- must be ~0" % (100*g["touch"].mean()))
print("  native px of wide window   median %.0f   (>=512 = real detail)" % g["px"].median())
print("  black pixels per crop      median %.1f%%  p95 %.1f%%"
      % (100*np.median([r[2] for r in rows]), 100*np.quantile([r[2] for r in rows], .95)))

sel = list(np.argsort(-g["mult"].values)[:2]) + list(np.argsort(g["mult"].values)[:2])
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for j, si in enumerate(sel):
    gi, stem, _ = rows[si]
    o  = cv2.imread(str(d.iloc[gi]["img"]), cv2.IMREAD_GRAYSCALE)
    om_ = cv2.imread(str(d.iloc[gi]["predmask"]), cv2.IMREAD_GRAYSCALE)
    w  = cv2.imread(os.path.join(OUT, stem+"_img.png"), cv2.IMREAD_GRAYSCALE)
    wm = cv2.imread(os.path.join(OUT, stem+"_pred.png"), cv2.IMREAD_GRAYSCALE)
    for row, (im_, m_, t_) in enumerate([(o, om_, "tight"), (w, wm, "wide x%.2f" % g["mult"].values[si])]):
        ax[row, j].imshow(im_, cmap="gray", vmin=0, vmax=255)
        if m_ is not None and (m_>127).any():
            ax[row, j].contour(m_>127, levels=[0.5], colors="lime", linewidths=1.4)
        ax[row, j].set_title(t_, fontsize=9); ax[row, j].axis("off")
plt.tight_layout(); fpp = os.path.join(FIG, "widectx_final.png")
plt.savefig(fpp, dpi=130, bbox_inches="tight"); plt.close()
print("\n  figure: %s" % fpp)
print("  -> if 'lesion TOUCHING border' is ~0%, run Cell 30.")

    400/1696 (49s)
    800/1696 (98s)
   1200/1696 (147s)
   1600/1696 (195s)

generated 1696/1696  (3.4 min)

VERIFICATION
  wide/tight width ratio     median 1.75   (target 1.75)
  had to shrink to fit       24.4%
  lesion TOUCHING border     0.0%   <-- must be ~0
  native px of wide window   median 817   (>=512 = real detail)
  black pixels per crop      median 0.0%  p95 28.2%

  figure: /root/autodl-tmp/CBIS/figures/mass_final/widectx_final.png
  -> if 'lesion TOUCHING border' is ~0%, run Cell 30.


In [13]:
# ══════════════════════════════════════════════════════════════════════
# CELL 29b — WIDE CROPS, CORRECTED
#   Fix 1: masks written as 0/255 (they were 0/1 = blank)
#   Fix 2: window keeps FULL size; slides only within lesion-safe limits;
#          mirror-pads the remainder. Never shrinks below the tight crop.
#   Verifies mask coverage — the check that would have caught Fix 1.
#   CPU only, ~4 min. Overwrites crops_wide_mass/.
# ══════════════════════════════════════════════════════════════════════
import os, glob, time
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
cv2.setNumThreads(0)

D, LES = "/root/autodl-tmp/CBIS", "mass"
JP, PM = os.path.join(D, "jpeg"), os.path.join(D, "predmasks_%s" % LES)
OUT = os.path.join(D, "crops_wide_%s" % LES); os.makedirs(OUT, exist_ok=True)
FIG = os.path.join(D, "figures", "mass_final"); os.makedirs(FIG, exist_ok=True)
S, WIDE_MULT, MARGIN = 512, 1.75, 0.15      # margin as a fraction of lesion width

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["predmask"] = d["img"].apply(
    lambda p: os.path.join(PM, os.path.basename(str(p)).replace("_img.png","") + "_pred.png"))

def sf(u):
    p = os.path.join(JP, str(u))
    return sorted(glob.glob(os.path.join(p, "*.jpg"))) if os.path.isdir(p) else []
def read_full(u):
    b, a = None, -1
    for f in sf(u):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is not None and im.size > a: b, a = im, im.size
    return b
def read_mask(u, ref):
    c = []
    for f in sf(u):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is None: continue
        c.append((2.0*float(im.shape==ref) + float(((im<20)|(im>235)).mean()), im))
    if not c: return None
    c.sort(key=lambda z: -z[0]); return c[0][1]

def place(lo_obj, hi_obj, centre, s, limit, m):
    """origin for one axis: keep [lo_obj-m, hi_obj+m] inside [o, o+s), stay in [0, limit-s]
       returns (origin, needs_padding)"""
    want = centre - s/2.0
    lo = hi_obj + m - s          # origin must be >= this to cover the object's far side
    hi = lo_obj - m              # origin must be <= this to cover the object's near side
    if lo > hi:                  # object + margin bigger than the window: just centre it
        return int(round(want)), True
    lo2, hi2 = max(lo, 0.0), min(hi, limit - s)
    if lo2 > hi2:                # cannot satisfy both object and image bounds -> pad
        return int(round(min(max(want, lo), hi))), True
    return int(round(min(max(want, lo2), hi2))), False

def cut(im, y0, x0, s, interp):
    y1, x1 = y0 + s, x0 + s
    py0, px0 = max(0, -y0), max(0, -x0)
    py1, px1 = max(0, y1 - im.shape[0]), max(0, x1 - im.shape[1])
    sub = im[max(y0,0):min(y1,im.shape[0]), max(x0,0):min(x1,im.shape[1])]
    if sub.size == 0: return None
    if py0 or px0 or py1 or px1:
        mode = (cv2.BORDER_REFLECT_101
                if (sub.shape[0] > max(py0,py1) and sub.shape[1] > max(px0,px1))
                else cv2.BORDER_REPLICATE)
        sub = cv2.copyMakeBorder(sub, py0, py1, px0, px1, mode)
    return cv2.resize(sub, (S,S), interpolation=interp)

rows, skip = [], {}
dg = {"mult":[], "cov":[], "empty":[], "touch":[], "px":[], "pad":[], "slid":[]}
t0 = time.time()
for i, r in d.iterrows():
    if (i+1) % 400 == 0: print("   %4d/%d (%.0fs)" % (i+1, len(d), time.time()-t0), flush=True)
    om = cv2.imread(str(r["msk"]), cv2.IMREAD_GRAYSCALE)
    pm = cv2.imread(str(r["predmask"]), cv2.IMREAD_GRAYSCALE)
    if om is None or pm is None: skip["mask"] = skip.get("mask",0)+1; continue
    cov0 = float((om>127).mean())
    if cov0 < 1e-4: skip["empty src"] = skip.get("empty src",0)+1; continue
    full = read_full(r["full_series"])
    if full is None: skip["no full"] = skip.get("no full",0)+1; continue
    fm = read_mask(r["mask_series"], full.shape)
    if fm is None: skip["no mask jpg"] = skip.get("no mask jpg",0)+1; continue
    if fm.shape != full.shape:
        fm = cv2.resize(fm, (full.shape[1], full.shape[0]), interpolation=cv2.INTER_NEAREST)
    H, W = full.shape
    ys, xs = np.where(fm>127)
    if len(ys) < 20: skip["tiny"] = skip.get("tiny",0)+1; continue
    A = float(len(ys)); cy, cx = 0.5*(ys.min()+ys.max()), 0.5*(xs.min()+xs.max())
    bmax = float(max(ys.max()-ys.min(), xs.max()-xs.min())+1)
    side_t = float(np.clip(np.sqrt(A/cov0), 1.1*bmax, 5.0*bmax))

    s = int(round(side_t * WIDE_MULT))                    # FULL size, never reduced
    m = MARGIN * bmax
    y0, py = place(float(ys.min()), float(ys.max()), cy, s, H, m)
    x0, px = place(float(xs.min()), float(xs.max()), cx, s, W, m)
    padded = py or px
    slid = abs(y0 - (cy - s/2)) > 1 or abs(x0 - (cx - s/2)) > 1

    img_w = cut(full, y0, x0, s, cv2.INTER_AREA)
    st = max(int(round(side_t)), 8)
    pmb = cv2.resize((pm>127).astype(np.uint8)*255, (st,st), interpolation=cv2.INTER_NEAREST)  # <-- *255
    canvas = np.zeros((H,W), np.uint8)
    ty0, tx0 = int(round(cy-st/2)), int(round(cx-st/2))
    sy0, sx0 = max(ty0,0), max(tx0,0); sy1, sx1 = min(ty0+st,H), min(tx0+st,W)
    if sy1<=sy0 or sx1<=sx0: skip["proj"] = skip.get("proj",0)+1; continue
    canvas[sy0:sy1, sx0:sx1] = pmb[sy0-ty0:sy1-ty0, sx0-tx0:sx1-tx0]
    msk_w = cut(canvas, y0, x0, s, cv2.INTER_NEAREST)
    if img_w is None or msk_w is None: skip["cut"] = skip.get("cut",0)+1; continue
    msk_w = ((msk_w > 127).astype(np.uint8) * 255)

    stem = os.path.basename(str(r["img"])).replace("_img.png","")
    cv2.imwrite(os.path.join(OUT, stem+"_img.png"), img_w)
    cv2.imwrite(os.path.join(OUT, stem+"_pred.png"), msk_w)
    b = msk_w > 127
    dg["mult"].append(s/side_t); dg["cov"].append(float(b.mean())); dg["empty"].append(not b.any())
    dg["touch"].append(bool(b[0,:].any() or b[-1,:].any() or b[:,0].any() or b[:,-1].any()))
    dg["px"].append(s); dg["pad"].append(padded); dg["slid"].append(slid)
    rows.append((i, stem, float((img_w<5).mean())))

print("\ngenerated %d/%d  (%.1f min)" % (len(rows), len(d), (time.time()-t0)/60))
if skip: print("skipped:", skip)
g = pd.DataFrame(dg)
print("\n" + "="*70); print("VERIFICATION"); print("="*70)
print("  wide/tight width ratio     median %.2f   min %.2f   <-- min must be 1.75"
      % (g["mult"].median(), g["mult"].min()))
print("  EMPTY wide masks           %.1f%%   <-- must be 0.0%%  (the bug last time)"
      % (100*g["empty"].mean()))
print("  mask coverage of wide crop median %.1f%%  (expect ~%.1f%%)"
      % (100*g["cov"].median(), 100*0.23/WIDE_MULT**2))
print("  lesion touching border     %.1f%%   <-- must be ~0" % (100*g["touch"].mean()))
print("  window slid inward         %.1f%%" % (100*g["slid"].mean()))
print("  needed mirror padding      %.1f%%" % (100*g["pad"].mean()))
print("  native px of wide window   median %.0f" % g["px"].median())
print("  black pixels per crop      median %.1f%%  p95 %.1f%%"
      % (100*np.median([r[2] for r in rows]), 100*np.quantile([r[2] for r in rows], .95)))
chk = cv2.imread(os.path.join(OUT, rows[0][1] + "_pred.png"), cv2.IMREAD_GRAYSCALE)
print("  mask pixel values present  %s   <-- must include 255" % np.unique(chk)[:5])

hard = list(np.argsort(-g["pad"].values.astype(int) * 1000 - g["slid"].values.astype(int) * 10)[:2])
easy = list(np.argsort(g["pad"].values.astype(int) + g["slid"].values.astype(int))[:2])
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for j, si in enumerate(hard + easy):
    gi, stem, _ = rows[si]
    o  = cv2.imread(str(d.iloc[gi]["img"]), cv2.IMREAD_GRAYSCALE)
    om_ = cv2.imread(str(d.iloc[gi]["predmask"]), cv2.IMREAD_GRAYSCALE)
    w  = cv2.imread(os.path.join(OUT, stem+"_img.png"), cv2.IMREAD_GRAYSCALE)
    wm = cv2.imread(os.path.join(OUT, stem+"_pred.png"), cv2.IMREAD_GRAYSCALE)
    ttl = "wide x%.2f%s%s" % (g["mult"].values[si],
                              " slid" if g["slid"].values[si] else "",
                              " padded" if g["pad"].values[si] else "")
    for row, (im_, m_, t_) in enumerate([(o, om_, "tight"), (w, wm, ttl)]):
        ax[row, j].imshow(im_, cmap="gray", vmin=0, vmax=255)
        if m_ is not None and (m_>127).any():
            ax[row, j].contour(m_>127, levels=[0.5], colors="lime", linewidths=1.5)
        ax[row, j].set_title(t_, fontsize=9); ax[row, j].axis("off")
plt.suptitle("cols 1-2: hardest edge cases   |   cols 3-4: typical", fontsize=11)
plt.tight_layout(); fpp = os.path.join(FIG, "widectx_final.png")
plt.savefig(fpp, dpi=130, bbox_inches="tight"); plt.close()
print("\n  figure: %s" % fpp)
print("  -> every WIDE panel must now show a green outline. If not, stop and tell me.")

    400/1696 (49s)
    800/1696 (97s)
   1200/1696 (146s)
   1600/1696 (195s)

generated 1696/1696  (3.4 min)

VERIFICATION
  wide/tight width ratio     median 1.75   min 1.75   <-- min must be 1.75
  EMPTY wide masks           0.0%   <-- must be 0.0%  (the bug last time)
  mask coverage of wide crop median 7.7%  (expect ~7.5%)
  lesion touching border     1.9%   <-- must be ~0
  window slid inward         18.9%
  needed mirror padding      5.5%
  native px of wide window   median 882
  black pixels per crop      median 0.0%  p95 32.0%
  mask pixel values present  [  0 255]   <-- must include 255

  figure: /root/autodl-tmp/CBIS/figures/mass_final/widectx_final.png
  -> every WIDE panel must now show a green outline. If not, stop and tell me.


In [14]:
# ══════════════════════════════════════════════════════════════════════
# CELL 29c — WIDE CROPS, FINAL
#   Fix: mask padded with ZEROS (was mirrored -> duplicated the lesion)
#        image padded with edge-matched tissue + noise (no fake anatomy)
#   New check: mask blob count must not increase vs the tight crop.
#   CPU only, ~4 min. Overwrites crops_wide_mass/.
# ══════════════════════════════════════════════════════════════════════
import os, glob, time
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
cv2.setNumThreads(0)

D, LES = "/root/autodl-tmp/CBIS", "mass"
JP, PM = os.path.join(D, "jpeg"), os.path.join(D, "predmasks_%s" % LES)
OUT = os.path.join(D, "crops_wide_%s" % LES); os.makedirs(OUT, exist_ok=True)
FIG = os.path.join(D, "figures", "mass_final"); os.makedirs(FIG, exist_ok=True)
S, WIDE_MULT, MARGIN = 512, 1.75, 0.10
rng = np.random.default_rng(0)

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["predmask"] = d["img"].apply(
    lambda p: os.path.join(PM, os.path.basename(str(p)).replace("_img.png","") + "_pred.png"))

def sf(u):
    p = os.path.join(JP, str(u))
    return sorted(glob.glob(os.path.join(p, "*.jpg"))) if os.path.isdir(p) else []
def read_full(u):
    b, a = None, -1
    for f in sf(u):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is not None and im.size > a: b, a = im, im.size
    return b
def read_mask(u, ref):
    c = []
    for f in sf(u):
        im = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if im is None: continue
        c.append((2.0*float(im.shape==ref) + float(((im<20)|(im>235)).mean()), im))
    if not c: return None
    c.sort(key=lambda z: -z[0]); return c[0][1]

def place(lo_obj, hi_obj, centre, s, limit, m):
    want = centre - s/2.0
    lo, hi = hi_obj + m - s, lo_obj - m
    if lo > hi: return int(round(want)), True
    lo2, hi2 = max(lo, 0.0), min(hi, limit - s)
    if lo2 > hi2: return int(round(min(max(want, lo), hi))), True
    return int(round(min(max(want, lo2), hi2))), False

def fill(st, shape):
    """neutral tissue block matching a border strip: median + matched noise"""
    mu, sd = float(np.median(st)), float(st.std())
    return np.clip(rng.normal(mu, max(0.5*sd, 1.0), shape), 0, 255).astype(np.uint8)

def cut(im, y0, x0, s, interp, zero_pad=False):
    y1, x1 = y0 + s, x0 + s
    py0, px0 = max(0, -y0), max(0, -x0)
    py1, px1 = max(0, y1 - im.shape[0]), max(0, x1 - im.shape[1])
    sub = im[max(y0,0):min(y1,im.shape[0]), max(x0,0):min(x1,im.shape[1])]
    if sub.size == 0: return None
    if py0 or px0 or py1 or px1:
        if zero_pad:                                     # MASK: never invent lesion
            sub = cv2.copyMakeBorder(sub, py0, py1, px0, px1, cv2.BORDER_CONSTANT, value=0)
        else:                                            # IMAGE: edge-matched tissue
            if py0: sub = np.vstack([fill(sub[:24, :], (py0, sub.shape[1])), sub])
            if py1: sub = np.vstack([sub, fill(sub[-24:, :], (py1, sub.shape[1]))])
            if px0: sub = np.hstack([fill(sub[:, :24], (sub.shape[0], px0)), sub])
            if px1: sub = np.hstack([sub, fill(sub[:, -24:], (sub.shape[0], px1))])
    return cv2.resize(sub, (S, S), interpolation=interp)

def blobs(m):
    n, _ = cv2.connectedComponents((m > 127).astype(np.uint8), connectivity=8)
    return max(n - 1, 0)

rows, skip = [], {}
dg = {"mult":[], "cov":[], "empty":[], "touch":[], "px":[], "pad":[], "slid":[], "extra":[]}
t0 = time.time()
for i, r in d.iterrows():
    if (i+1) % 400 == 0: print("   %4d/%d (%.0fs)" % (i+1, len(d), time.time()-t0), flush=True)
    om = cv2.imread(str(r["msk"]), cv2.IMREAD_GRAYSCALE)
    pm = cv2.imread(str(r["predmask"]), cv2.IMREAD_GRAYSCALE)
    if om is None or pm is None: skip["mask"] = skip.get("mask",0)+1; continue
    cov0 = float((om > 127).mean())
    if cov0 < 1e-4: skip["empty src"] = skip.get("empty src",0)+1; continue
    full = read_full(r["full_series"])
    if full is None: skip["no full"] = skip.get("no full",0)+1; continue
    fm = read_mask(r["mask_series"], full.shape)
    if fm is None: skip["no mask jpg"] = skip.get("no mask jpg",0)+1; continue
    if fm.shape != full.shape:
        fm = cv2.resize(fm, (full.shape[1], full.shape[0]), interpolation=cv2.INTER_NEAREST)
    H, W = full.shape
    ys, xs = np.where(fm > 127)
    if len(ys) < 20: skip["tiny"] = skip.get("tiny",0)+1; continue
    A = float(len(ys)); cy, cx = 0.5*(ys.min()+ys.max()), 0.5*(xs.min()+xs.max())
    bmax = float(max(ys.max()-ys.min(), xs.max()-xs.min())+1)
    side_t = float(np.clip(np.sqrt(A/cov0), 1.1*bmax, 5.0*bmax))

    s = int(round(side_t * WIDE_MULT)); m = MARGIN * bmax
    y0, py = place(float(ys.min()), float(ys.max()), cy, s, H, m)
    x0, px = place(float(xs.min()), float(xs.max()), cx, s, W, m)
    padded = py or px
    slid = abs(y0 - (cy - s/2)) > 1 or abs(x0 - (cx - s/2)) > 1

    img_w = cut(full, y0, x0, s, cv2.INTER_AREA, zero_pad=False)
    st = max(int(round(side_t)), 8)
    pmb = cv2.resize((pm > 127).astype(np.uint8)*255, (st, st), interpolation=cv2.INTER_NEAREST)
    canvas = np.zeros((H, W), np.uint8)
    ty0, tx0 = int(round(cy-st/2)), int(round(cx-st/2))
    sy0, sx0 = max(ty0,0), max(tx0,0); sy1, sx1 = min(ty0+st,H), min(tx0+st,W)
    if sy1 <= sy0 or sx1 <= sx0: skip["proj"] = skip.get("proj",0)+1; continue
    canvas[sy0:sy1, sx0:sx1] = pmb[sy0-ty0:sy1-ty0, sx0-tx0:sx1-tx0]
    msk_w = cut(canvas, y0, x0, s, cv2.INTER_NEAREST, zero_pad=True)     # <-- ZEROS
    if img_w is None or msk_w is None: skip["cut"] = skip.get("cut",0)+1; continue
    msk_w = ((msk_w > 127).astype(np.uint8) * 255)

    stem = os.path.basename(str(r["img"])).replace("_img.png","")
    cv2.imwrite(os.path.join(OUT, stem+"_img.png"), img_w)
    cv2.imwrite(os.path.join(OUT, stem+"_pred.png"), msk_w)
    b = msk_w > 127
    dg["mult"].append(s/side_t); dg["cov"].append(float(b.mean())); dg["empty"].append(not b.any())
    dg["touch"].append(bool(b[0,:].any() or b[-1,:].any() or b[:,0].any() or b[:,-1].any()))
    dg["px"].append(s); dg["pad"].append(padded); dg["slid"].append(slid)
    dg["extra"].append(blobs(msk_w) > blobs(pm))
    rows.append((i, stem, float((img_w < 5).mean())))

print("\ngenerated %d/%d  (%.1f min)" % (len(rows), len(d), (time.time()-t0)/60))
if skip: print("skipped:", skip)
g = pd.DataFrame(dg)
print("\n" + "="*72); print("VERIFICATION"); print("="*72)
print("  wide/tight width ratio     median %.2f   min %.2f" % (g["mult"].median(), g["mult"].min()))
print("  EMPTY wide masks           %.1f%%" % (100*g["empty"].mean()))
print("  mask coverage of wide crop median %.1f%%  (expect ~7.5%%)" % (100*g["cov"].median()))
print("  DUPLICATED lesion blobs    %.1f%%   <-- must be 0.0%%  (the bug last time)"
      % (100*g["extra"].mean()))
print("  lesion touching border     %.1f%%   <-- must be ~0" % (100*g["touch"].mean()))
print("  window slid inward         %.1f%%" % (100*g["slid"].mean()))
print("  needed tissue padding      %.1f%%" % (100*g["pad"].mean()))
print("  native px of wide window   median %.0f" % g["px"].median())
print("  mask pixel values          %s" % np.unique(
    cv2.imread(os.path.join(OUT, rows[0][1]+"_pred.png"), cv2.IMREAD_GRAYSCALE))[:5])

pi = np.where(g["pad"].values)[0]
sel = list(pi[:2]) + list(np.where(~g["pad"].values & ~g["slid"].values)[0][:2])
fig, ax = plt.subplots(2, len(sel), figsize=(4*len(sel), 8))
ax = np.atleast_2d(ax)
for j, si in enumerate(sel):
    gi, stem, _ = rows[si]
    o   = cv2.imread(str(d.iloc[gi]["img"]), cv2.IMREAD_GRAYSCALE)
    om_ = cv2.imread(str(d.iloc[gi]["predmask"]), cv2.IMREAD_GRAYSCALE)
    w   = cv2.imread(os.path.join(OUT, stem+"_img.png"), cv2.IMREAD_GRAYSCALE)
    wm  = cv2.imread(os.path.join(OUT, stem+"_pred.png"), cv2.IMREAD_GRAYSCALE)
    ttl = "wide x1.75%s%s" % (" slid" if g["slid"].values[si] else "",
                              " padded" if g["pad"].values[si] else "")
    for row, (im_, m_, t_) in enumerate([(o, om_, "tight"), (w, wm, ttl)]):
        ax[row, j].imshow(im_, cmap="gray", vmin=0, vmax=255)
        if m_ is not None and (m_ > 127).any():
            ax[row, j].contour(m_ > 127, levels=[0.5], colors="lime", linewidths=1.5)
        ax[row, j].set_title(t_, fontsize=9); ax[row, j].axis("off")
plt.suptitle("cols 1-2: PADDED cases   |   cols 3-4: typical", fontsize=11)
plt.tight_layout(); fpp = os.path.join(FIG, "widectx_final.png")
plt.savefig(fpp, dpi=130, bbox_inches="tight"); plt.close()
print("\n  figure: %s" % fpp)
print("  -> ONE green blob per panel. Two blobs = still broken.")

    400/1696 (49s)
    800/1696 (98s)
   1200/1696 (148s)
   1600/1696 (197s)

generated 1696/1696  (3.5 min)

VERIFICATION
  wide/tight width ratio     median 1.75   min 1.75
  EMPTY wide masks           0.0%
  mask coverage of wide crop median 7.6%  (expect ~7.5%)
  DUPLICATED lesion blobs    0.1%   <-- must be 0.0%  (the bug last time)
  lesion touching border     1.4%   <-- must be ~0
  window slid inward         19.7%
  needed tissue padding      4.7%
  native px of wide window   median 882
  mask pixel values          [  0 255]

  figure: /root/autodl-tmp/CBIS/figures/mass_final/widectx_final.png
  -> ONE green blob per panel. Two blobs = still broken.


In [16]:
# ══════════════════════════════════════════════════════════════════════
# CELL 30 — TWO-STREAM MASS CLASSIFIER  (tight lesion + wide context)
#   Stream A: 512px tight crop  -> lesion texture & margin
#   Stream B: 384px wide crop   -> surrounding architecture
#   Predicted masks only. Saves cv_mass_twostream_oof.csv
#   >>> QUICK_TEST = True FIRST <<<
# ══════════════════════════════════════════════════════════════════════
import os, gc, time
os.environ.setdefault("OMP_NUM_THREADS", "4")
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
cv2.setNumThreads(0)

D, LES = "/root/autodl-tmp/CBIS", "mass"
DEV = torch.device("cuda")
torch.backends.cudnn.benchmark = True; torch.backends.cuda.matmul.allow_tf32 = True

QUICK_TEST = False            # <<<<<< set False for the real run

ST, SW, BATCH = 512, 384, 8
SEEDS, EPOCHS, FREEZE = [11], 20, 3
LR_HEAD, LR_HEAD_FT, LR_BACK = 1e-3, 3e-4, 3e-5
WD, GAMMA, AUX_W, MULT, PATIENCE, ATT = 1e-4, 2.0, 0.3, 3, 7, 2.0
FOLDS = [0,1,2,3,4]
if QUICK_TEST:
    SEEDS, EPOCHS, FREEZE, MULT, FOLDS = [11], 4, 1, 1, [0]
    print(">>> QUICK TEST: 1 fold, 4 epochs — error check only\n")

WIDE = os.path.join(D, "crops_wide_%s" % LES)
PM   = os.path.join(D, "predmasks_%s" % LES)
d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["label"] = d["label"].astype(int)
d["stem"]  = d["img"].apply(lambda p: os.path.basename(str(p)).replace("_img.png",""))
d["tmask"] = d["stem"].apply(lambda s: os.path.join(PM,   s+"_pred.png"))
d["wimg"]  = d["stem"].apply(lambda s: os.path.join(WIDE, s+"_img.png"))
d["wmask"] = d["stem"].apply(lambda s: os.path.join(WIDE, s+"_pred.png"))
miss = (~d["wimg"].apply(os.path.exists)).sum()
assert miss == 0, "%d wide crops missing — run Cell 29" % miss
print("%s: %d images | %d patients | malignant %.1f%%"
      % (LES.upper(), len(d), d.patient_id.nunique(), 100*d.label.mean()))

cl = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
def load(p, size, mask=False):
    im = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if im is None: im = np.zeros((size,size), np.uint8)
    if im.shape != (size,size):
        im = cv2.resize(im, (size,size),
                        interpolation=cv2.INTER_NEAREST if mask else cv2.INTER_AREA)
    return (im>127).astype(np.uint8) if mask else cl.apply(im)

CACHE, t0 = {}, time.time()
for _, r in d.iterrows():
    CACHE[r["stem"]] = (load(r["img"], ST), load(r["tmask"], ST, True),
                        load(r["wimg"], SW), load(r["wmask"], SW, True))
print("cached %d in %.0fs" % (len(CACHE), time.time()-t0))

def primary(x): return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()
aux, meta = {}, {}
for c in ["subtlety","mass_shape","mass_margins"]:
    if c not in d.columns or d[c].notna().sum()==0: continue
    if c=="subtlety":
        v = pd.to_numeric(d[c], errors="coerce").where(lambda z:(z>=1)&(z<=5))
        codes, n = (v-1).fillna(-1).astype(int).values, 5
    else:
        pr = d[c].map(primary); pr = pr.where(pr.isin(pr.value_counts().head(6).index.tolist()), "OTHER")
        cats = sorted([k for k in pr.unique() if k!="UNK"]); mp = {k:i for i,k in enumerate(cats)}
        codes, n = pr.map(lambda z: mp.get(z,-1)).astype(int).values, len(cats)
    if n>1: aux[c], meta[c] = codes, n
AK = sorted(aux.keys()); print("helper heads:", meta)

MEAN = np.array([0.485,0.456,0.406], np.float32).reshape(3,1,1)
STD  = np.array([0.229,0.224,0.225], np.float32).reshape(3,1,1)

class DS(Dataset):
    def __init__(self, idx, aug, mult=1, tta=0):
        self.idx, self.aug, self.mult, self.tta = np.asarray(idx), aug, (mult if aug else 1), tta
    def __len__(self): return len(self.idx)*self.mult
    def __getitem__(self, i):
        j = int(self.idx[i % len(self.idx)]); r = d.iloc[j]
        ti, tm, wi, wm = [a.copy() for a in CACHE[r["stem"]]]
        if self.aug:
            fh, fv = np.random.rand()<.5, np.random.rand()<.5
            kk = np.random.randint(4)
            aff = np.random.rand()<.7; ang = np.random.uniform(-25,25); sc = np.random.uniform(.9,1.12)
            itn = np.random.rand()<.5; gg = np.random.uniform(.85,1.15); bb = np.random.uniform(-12,12)
            def T(im, mk, s):
                if fh: im, mk = im[:,::-1], mk[:,::-1]
                if fv: im, mk = im[::-1,:], mk[::-1,:]
                if kk: im, mk = np.rot90(im,kk), np.rot90(mk,kk)
                im, mk = np.ascontiguousarray(im), np.ascontiguousarray(mk)
                if aff:
                    M = cv2.getRotationMatrix2D((s/2,s/2), ang, sc)
                    im = cv2.warpAffine(im, M, (s,s), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
                    mk = cv2.warpAffine(mk, M, (s,s), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT)
                if itn: im = np.clip(im.astype(np.float32)*gg+bb, 0, 255).astype(np.uint8)
                return im, mk
            ti, tm = T(ti, tm, ST); wi, wm = T(wi, wm, SW)
        else:
            t = self.tta
            def V(im, mk):
                if   t==1: return im[:,::-1], mk[:,::-1]
                elif t==2: return im[::-1,:], mk[::-1,:]
                elif t==3: return np.rot90(im,2), np.rot90(mk,2)
                return im, mk
            ti, tm = V(ti, tm); wi, wm = V(wi, wm)
        out = []
        for im, mk in [(ti,tm),(wi,wm)]:
            im = np.ascontiguousarray(im).astype(np.float32)/255.0
            out.append(torch.from_numpy(((np.stack([im]*3,0)-MEAN)/STD).astype(np.float32)))
            out.append(torch.from_numpy(np.ascontiguousarray(mk).astype(np.float32))[None])
        av = (np.array([aux[c][j] for c in AK], dtype=np.int64) if AK else np.zeros(0, np.int64))
        return out[0], out[1], out[2], out[3], torch.tensor(int(r["label"])), torch.from_numpy(av)

class TwoStream(nn.Module):
    def __init__(self, am, att=2.0):
        super().__init__()
        def bb():
            try:  return models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1).features
            except Exception: return models.densenet121(weights=None).features
        self.bt, self.bw, self.att = bb(), bb(), att
        Fd = 1024*4
        self.head = nn.Sequential(nn.Linear(Fd,512), nn.BatchNorm1d(512), nn.ReLU(True),
                                  nn.Dropout(0.4), nn.Linear(512,2))
        self.keys = sorted(am.keys())
        self.aux = nn.ModuleList([nn.Sequential(nn.Linear(Fd,128), nn.ReLU(True),
                                                nn.Dropout(0.3), nn.Linear(128, am[k]))
                                  for k in self.keys])
    def pool(self, b, x, m):
        f = F.relu(b(x))
        mm = F.interpolate(m, size=f.shape[2:], mode="bilinear", align_corners=False)
        w = 1.0 + self.att*mm
        return torch.cat([(f*w).sum((2,3))/(w.sum((2,3))+1e-6), f.mean((2,3))], 1)
    def forward(self, xt, mt, xw, mw):
        g = torch.cat([self.pool(self.bt, xt, mt), self.pool(self.bw, xw, mw)], 1)
        return self.head(g), [h(g) for h in self.aux]

def focal(lg, tg, al):
    ce = F.cross_entropy(lg.float(), tg, weight=al, reduction="none")
    return ((1-torch.exp(-ce))**GAMMA * ce).mean()

@torch.no_grad()
def predict(net, idx, tta=True):
    net.eval(); tot = None
    for t in ([0,1,2,3] if tta else [0]):
        ld = DataLoader(DS(idx, False, 1, tta=t), batch_size=12, shuffle=False, num_workers=0)
        ps = []
        for xt, mt, xw, mw, _, _ in ld:
            xt, mt, xw, mw = xt.to(DEV), mt.to(DEV), xw.to(DEV), mw.to(DEV)
            with torch.amp.autocast(device_type="cuda"): o, _ = net(xt, mt, xw, mw)
            ps += list(torch.softmax(o.float(),1)[:,1].cpu().numpy())
        ps = np.array(ps); tot = ps if tot is None else tot+ps
    return tot/(4 if tta else 1)

def train_one(tr, va, te, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    yy = d["label"].values
    n0, n1 = float((yy[tr]==0).sum()), float((yy[tr]==1).sum())
    al = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV, dtype=torch.float32)
    net = TwoStream(meta, ATT).to(DEV)
    back = list(net.bt.parameters()) + list(net.bw.parameters())
    for p_ in back: p_.requires_grad = False
    hp = [p_ for n_, p_ in net.named_parameters()
          if not (n_.startswith("bt.") or n_.startswith("bw."))]
    scaler = torch.amp.GradScaler()
    opt = torch.optim.AdamW(hp, lr=LR_HEAD, weight_decay=WD); sch = None
    tl = DataLoader(DS(tr, True, MULT), batch_size=BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)
    best, bstate, bad = -1.0, None, 0
    for ep in range(1, EPOCHS+1):
        if ep == FREEZE+1:
            for p_ in back: p_.requires_grad = True
            opt = torch.optim.AdamW([{"params": back, "lr": LR_BACK},
                                     {"params": hp,   "lr": LR_HEAD_FT}], weight_decay=WD)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS-FREEZE))
        net.train()
        if ep <= FREEZE: net.bt.eval(); net.bw.eval()
        for xt, mt, xw, mw, t_, a_ in tl:
            xt, mt, xw, mw = xt.to(DEV,non_blocking=True), mt.to(DEV,non_blocking=True), \
                             xw.to(DEV,non_blocking=True), mw.to(DEV,non_blocking=True)
            t_, a_ = t_.to(DEV), a_.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o, ax = net(xt, mt, xw, mw)
                loss = focal(o, t_, al)
                if len(ax):
                    loss = loss + AUX_W*sum(F.cross_entropy(gg.float(), a_[:,h], ignore_index=-1)
                                            for h, gg in enumerate(ax))/len(ax)
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            scaler.step(opt); scaler.update()
        if sch is not None: sch.step()
        pv = predict(net, va, tta=False)
        auc = roc_auc_score(yy[va], pv) if len(set(yy[va]))>1 else 0.0
        star = ""
        if auc > best:
            best, bad = auc, 0
            bstate = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}; star = " *"
        else: bad += 1
        print("      ep %2d  val-AUC %.4f%s" % (ep, auc, star), flush=True)
        if bad >= PATIENCE: print("      early stop"); break
    net.load_state_dict({k: v.to(DEV) for k, v in bstate.items()})
    pt = predict(net, te, tta=True)
    del net; gc.collect(); torch.cuda.empty_cache()
    return pt, best

y = d["label"].values; oof = np.full(len(d), np.nan)
for k in FOLDS:
    role = d["role_f%d" % k]
    tr, va, te = (np.where(role=="train")[0], np.where(role=="val")[0], np.where(role=="test")[0])
    assert not (set(d.patient_id[tr]) & set(d.patient_id[te])), "LEAK"
    print("\n### fold %d | train %d val %d test %d" % (k, len(tr), len(va), len(te)))
    ps, t0 = [], time.time()
    for sd in SEEDS:
        p_, bv = train_one(tr, va, te, sd); ps.append(p_)
        print("    seed %d: best val %.4f | test AUC %.4f" % (sd, bv, roc_auc_score(y[te], p_)))
    oof[te] = np.mean(ps, 0)
    print("  FOLD %d test AUC %.4f (%.0fs)" % (k, roc_auc_score(y[te], oof[te]), time.time()-t0))

done = ~np.isnan(oof)
print("\n" + "="*70); print("TWO-STREAM RESULTS"); print("="*70)
print("  per-image  AUC %.4f  (n=%d)" % (roc_auc_score(y[done], oof[done]), done.sum()))
res = d.loc[done, ["img","lesion_key","label"]].copy(); res["prob"] = oof[done]
Lg = res.groupby("lesion_key").agg(y=("label","max"), p=("prob","mean")).reset_index()
print("  per-lesion AUC %.4f   (single-stream v2 was 0.8692)" % roc_auc_score(Lg.y, Lg.p))
if not QUICK_TEST:
    out = os.path.join(D, "cv_%s_twostream_oof.csv" % LES)
    res.rename(columns={"label":"true"}).to_csv(out, index=False)
    print("\n  saved %s  -> rerun Cell 27 to add it to the ensemble" % os.path.basename(out))
else:
    print("\n  QUICK TEST clean. Set QUICK_TEST = False and rerun (~4 h).")

MASS: 1696 images | 892 patients | malignant 46.2%
cached 1696 in 17s
helper heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5}

### fold 0 | train 1202 val 155 test 339
      ep  1  val-AUC 0.6978 *
      ep  2  val-AUC 0.7404 *
      ep  3  val-AUC 0.7253
      ep  4  val-AUC 0.7515 *
      ep  5  val-AUC 0.7939 *
      ep  6  val-AUC 0.8195 *
      ep  7  val-AUC 0.8075
      ep  8  val-AUC 0.7674
      ep  9  val-AUC 0.8168
      ep 10  val-AUC 0.7999
      ep 11  val-AUC 0.7902
      ep 12  val-AUC 0.7989
      ep 13  val-AUC 0.8065
      early stop
    seed 11: best val 0.8195 | test AUC 0.8221
  FOLD 0 test AUC 0.8221 (963s)

### fold 1 | train 1184 val 172 test 340
      ep  1  val-AUC 0.6093 *
      ep  2  val-AUC 0.6289 *
      ep  3  val-AUC 0.6546 *
      ep  4  val-AUC 0.7355 *
      ep  5  val-AUC 0.7379 *
      ep  6  val-AUC 0.7671 *
      ep  7  val-AUC 0.7555
      ep  8  val-AUC 0.7529
      ep  9  val-AUC 0.7475
      ep 10  val-AUC 0.7856 *
      ep 11  val

In [3]:
# ══════════════════════════════════════════════════════════════════════
# CELL 31 — WHAT DID THE WIDE EYE ACTUALLY FIX?
#   Builds the ensemble with and without cv_mass_twostream, applies the
#   same nested policy C to both, compares subgroup by subgroup.
#   CPU only, ~2 min.
# ══════════════════════════════════════════════════════════════════════
import os, re, glob
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
pd.set_option("display.width", 230)

D, LES, FLOOR = "/root/autodl-tmp/CBIS", "mass", 0.60
GRID = np.linspace(0.02, 0.98, 193)
NEW = "cv_mass_twostream"

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["label"] = d["label"].astype(int); d["lesion_key"] = d["lesion_key"].astype(str)
tf = np.full(len(d), -1)
for k in range(5): tf[d["role_f%d" % k].values == "test"] = k
d["test_fold"] = tf
def stem(p):
    b = os.path.basename(str(p))
    for s in ("_img.png","_pred.png",".png",".jpg"):
        if b.endswith(s): b = b[:-len(s)]
    return b
d["stem"] = d["img"].apply(stem)
S2K = dict(zip(d["stem"], d["lesion_key"])); TGT = set(d["lesion_key"])

L = (d.groupby("lesion_key")
       .agg(y=("label","max"), fold=("test_fold","min"), assess=("assessment","first"),
            marg=("mass_margins","first"), mshape=("mass_shape","first"),
            subt=("subtlety","first"), pid=("patient_id","first")).reset_index())
L["lesion_key"] = L["lesion_key"].astype(str)
L["ass"] = pd.to_numeric(L["assess"], errors="coerce").fillna(-1).astype(int)
L["subt_b"] = pd.to_numeric(L["subt"], errors="coerce").fillna(-1).astype(int)
L["marg_b"] = L["marg"].astype(str).str.split("-").str[0].str.strip().str.upper()
L["shape_b"] = L["mshape"].astype(str).str.split("-").str[0].str.strip().str.upper()
y, fold = L.y.values, L.fold.values

BAD = re.compile(r"(_gt$|_gt_|^gt_|ref|truth|oracle|soft$|label|target)", re.I)
cand = {}
for f in sorted(glob.glob(os.path.join(D, "cv_*_oof.csv"))):
    b = os.path.basename(f); t = pd.read_csv(f)
    lk = t["img"].apply(stem).map(S2K) if "img" in t.columns else None
    if lk is None or lk.notna().sum() == 0:
        if "lesion_key" in t.columns:
            s_ = t["lesion_key"].astype(str); lk = s_.where(s_.isin(TGT))
    if lk is None or lk.notna().sum() == 0: continue
    t = t.assign(_lk=lk).dropna(subset=["_lk"])
    for c in t.columns:
        if c in ("_lk","lesion_key","img","msk","true","label","y","fold","patient_id","stem"): continue
        if not pd.api.types.is_numeric_dtype(t[c]): continue
        v = t[c].dropna()
        if not len(v) or v.min() < -1e-3 or v.max() > 1+1e-3 or v.nunique() < 10: continue
        if BAD.search(c): continue
        g = t.groupby("_lk")[c].mean()
        if len(TGT & set(g.index))/len(TGT) > 0.995:
            key = "%s::%s" % (b.replace("_oof.csv",""), c)
            if roc_auc_score(y, g.reindex(L.lesion_key).values) < 0.97:
                cand[key] = g
keys = sorted(cand)
print("models: %d   (two-stream present: %s)" % (len(keys), any(NEW in k for k in keys)))

def ens(ks):
    A = np.column_stack([cand[k].reindex(L.lesion_key).values for k in ks])
    A = np.column_stack([rankdata(A[:, j])/len(A) for j in range(A.shape[1])])
    out = np.full(len(L), np.nan)
    for k in range(5):
        inn, o = fold != k, fold == k
        if o.sum() == 0 or len(set(y[inn])) < 2: continue
        sel, cur = [], -np.inf
        for _ in range(A.shape[1]):
            bj, bs = None, cur
            for j in range(A.shape[1]):
                s = roc_auc_score(y[inn], A[inn][:, sel+[j]].mean(1))
                if s > bs + 1e-5: bj, bs = j, s
            if bj is None: break
            sel.append(bj); cur = bs
        if not sel: sel = [int(np.argmax([roc_auc_score(y[inn], A[inn][:,j]) for j in range(A.shape[1])]))]
        out[o] = A[o][:, sel].mean(1)
    return out

def sens(yy, pr): return ((pr==1)&(yy==1)).sum()/max((yy==1).sum(),1)
def thr_g(yy, pp, fl):
    a = np.array([accuracy_score(yy,(pp>t).astype(int)) for t in GRID])
    s = np.array([sens(yy,(pp>t).astype(int)) for t in GRID])
    m = s >= fl
    return float(GRID[int(np.argmax(np.where(m,a,-1.0)))]) if m.any() else 0.5
def app(pp, gg, tm, dflt):
    pr = np.zeros(len(pp), int)
    for g in np.unique(gg):
        mm = gg==g; pr[mm] = (pp[mm] > tm.get(g, dflt)).astype(int)
    return pr
def policyC(p):
    pr = np.zeros(len(L), int)
    for k in range(5):
        inn, o = (fold!=k), (fold==k)
        if o.sum()==0 or len(set(y[inn]))<2: continue
        yi, pi, gi, go = y[inn], p[inn], L["ass"].values[inn], L["ass"].values[o]
        tg = thr_g(yi, pi, FLOOR); tm = {g: tg for g in np.unique(gi)}
        def sc(t_):
            q = app(pi, gi, t_, tg)
            return accuracy_score(yi, q) if sens(yi, q) >= FLOOR else -1.0
        cur = sc(tm)
        for _ in range(4):
            moved = False
            for g in np.unique(gi):
                if (gi==g).sum() < 8: continue
                bv, bs = tm[g], cur
                for v in GRID:
                    t2 = dict(tm); t2[g] = v; s = sc(t2)
                    if s > bs + 1e-9: bv, bs = v, s
                if bv != tm[g]: tm[g], cur, moved = bv, bs, True
            if not moved: break
        pr[o] = app(p[o], go, tm, tg)
    return pr

old_k = [k for k in keys if NEW not in k]
p_old, p_new = ens(old_k), ens(keys)
L["p_old"], L["p_new"] = p_old, p_new
L["pred_old"], L["pred_new"] = policyC(p_old), policyC(p_new)

print("\n" + "="*80); print("OVERALL"); print("="*80)
for tag, pp, pr in [("without two-stream", p_old, L.pred_old.values),
                    ("WITH two-stream   ", p_new, L.pred_new.values)]:
    tn, fp, fn, tp = confusion_matrix(y, pr, labels=[0,1]).ravel()
    print("  %s  AUC %.4f  acc %.1f%%  sens %.3f  FP %3d  FN %3d"
          % (tag, roc_auc_score(y, pp), 100*accuracy_score(y, pr), tp/max(tp+fn,1), fp, fn))
fx = int(((L.pred_old != L.y) & (L.pred_new == L.y)).sum())
bk = int(((L.pred_old == L.y) & (L.pred_new != L.y)).sum())
print("\n  lesions FIXED by the wide eye : %d" % fx)
print("  lesions BROKEN by the wide eye: %d" % bk)
print("  net gain                      : %+d lesions (%+.1f pts)" % (fx-bk, 100*(fx-bk)/len(L)))

def cmp(col, lab, minn=20):
    rows = []
    for g, s in L.groupby(col):
        if len(s) < minn: continue
        ao = accuracy_score(s.y, s.pred_old); an = accuracy_score(s.y, s.pred_new)
        rows.append(dict(group=g, n=len(s),
                         AUC_old=round(roc_auc_score(s.y, s.p_old),3) if s.y.nunique()>1 else np.nan,
                         AUC_new=round(roc_auc_score(s.y, s.p_new),3) if s.y.nunique()>1 else np.nan,
                         acc_old="%.1f%%" % (100*ao), acc_new="%.1f%%" % (100*an),
                         delta="%+.1f" % (100*(an-ao)),
                         fixed=int(((s.pred_old!=s.y)&(s.pred_new==s.y)).sum()),
                         broken=int(((s.pred_old==s.y)&(s.pred_new!=s.y)).sum())))
    if rows:
        print("\n--- %s ---" % lab)
        print(pd.DataFrame(rows).sort_values("delta", key=lambda c: c.astype(float),
                                             ascending=False).to_string(index=False))

cmp("ass", "BI-RADS assessment")
cmp("shape_b", "mass SHAPE  <<< architectural distortion is the target")
cmp("marg_b", "mass margin")
cmp("subt_b", "subtlety")
L.to_csv(os.path.join(D, "mass_twostream_comparison.csv"), index=False)
print("\nsaved mass_twostream_comparison.csv")

models: 10   (two-stream present: True)

OVERALL
  without two-stream  AUC 0.9021  acc 85.2%  sens 0.812  FP  62  FN  87
  WITH two-stream     AUC 0.9077  acc 85.7%  sens 0.840  FP  70  FN  74

  lesions FIXED by the wide eye : 40
  lesions BROKEN by the wide eye: 35
  net gain                      : +5 lesions (+0.5 pts)

--- BI-RADS assessment ---
 group   n  AUC_old  AUC_new acc_old acc_new delta  fixed  broken
     0  86    0.882    0.857   84.9%   89.5%  +4.7      5       1
     3 213    0.904    0.903   88.7%   89.2%  +0.5      5       4
     2  59    0.914    0.897   96.6%   96.6%  +0.0      1       1
     4 422    0.832    0.846   75.4%   75.4%  +0.0     29      29
     5 223    0.686    0.675   97.3%   97.3%  +0.0      0       0

--- mass SHAPE  <<< architectural distortion is the target ---
                   group   n  AUC_old  AUC_new acc_old acc_new delta  fixed  broken
                   ROUND 112    0.817    0.851   85.7%   86.6%  +0.9      4       3
                    

In [1]:
# ══════════════════════════════════════════════════════════════════════
# CELL 30b — TRAIN SEED 22 ONLY, THEN AVERAGE WITH SEED 11
#   Seed 11 is already saved. This adds the second seed and merges.
#   Runs for real (no quick test — Cell 30 already proved it works).
#   ~4 hours.
# ══════════════════════════════════════════════════════════════════════
import os, gc, time, shutil
os.environ.setdefault("OMP_NUM_THREADS", "4")
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score
cv2.setNumThreads(0)

D, LES = "/root/autodl-tmp/CBIS", "mass"
DEV = torch.device("cuda")
torch.backends.cudnn.benchmark = True; torch.backends.cuda.matmul.allow_tf32 = True

F_MAIN = os.path.join(D, "cv_%s_twostream_oof.csv" % LES)     # what the ensemble reads
F_S11  = os.path.join(D, "cv_%s_twostream_s11.csv" % LES)
F_S22  = os.path.join(D, "cv_%s_twostream_s22.csv" % LES)

if not os.path.exists(F_S11):
    assert os.path.exists(F_MAIN), "seed-11 result missing — rerun Cell 30 first"
    shutil.copy(F_MAIN, F_S11); print("seed 11 preserved -> %s" % os.path.basename(F_S11))
else:
    print("seed 11 already preserved")

SEED = 22
ST, SW, BATCH = 512, 384, 8
EPOCHS, FREEZE = 20, 3
LR_HEAD, LR_HEAD_FT, LR_BACK = 1e-3, 3e-4, 3e-5
WD, GAMMA, AUX_W, MULT, PATIENCE, ATT = 1e-4, 2.0, 0.3, 3, 7, 2.0
FOLDS = [0, 1, 2, 3, 4]

WIDE = os.path.join(D, "crops_wide_%s" % LES)
PM   = os.path.join(D, "predmasks_%s" % LES)
d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["label"] = d["label"].astype(int)
d["stem"]  = d["img"].apply(lambda p: os.path.basename(str(p)).replace("_img.png",""))
d["tmask"] = d["stem"].apply(lambda s: os.path.join(PM,   s+"_pred.png"))
d["wimg"]  = d["stem"].apply(lambda s: os.path.join(WIDE, s+"_img.png"))
d["wmask"] = d["stem"].apply(lambda s: os.path.join(WIDE, s+"_pred.png"))
assert (~d["wimg"].apply(os.path.exists)).sum() == 0, "wide crops missing — run Cell 29c"
print("%s: %d images | %d patients | malignant %.1f%%"
      % (LES.upper(), len(d), d.patient_id.nunique(), 100*d.label.mean()))

cl = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
def load(p, size, mask=False):
    im = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if im is None: im = np.zeros((size,size), np.uint8)
    if im.shape != (size,size):
        im = cv2.resize(im, (size,size),
                        interpolation=cv2.INTER_NEAREST if mask else cv2.INTER_AREA)
    return (im>127).astype(np.uint8) if mask else cl.apply(im)

CACHE, t0 = {}, time.time()
for _, r in d.iterrows():
    CACHE[r["stem"]] = (load(r["img"], ST), load(r["tmask"], ST, True),
                        load(r["wimg"], SW), load(r["wmask"], SW, True))
print("cached %d in %.0fs" % (len(CACHE), time.time()-t0))

def primary(x): return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()
aux, meta = {}, {}
for c in ["subtlety","mass_shape","mass_margins"]:
    if c not in d.columns or d[c].notna().sum()==0: continue
    if c == "subtlety":
        v = pd.to_numeric(d[c], errors="coerce").where(lambda z:(z>=1)&(z<=5))
        codes, n = (v-1).fillna(-1).astype(int).values, 5
    else:
        pr = d[c].map(primary)
        pr = pr.where(pr.isin(pr.value_counts().head(6).index.tolist()), "OTHER")
        cats = sorted([k for k in pr.unique() if k != "UNK"]); mp = {k:i for i,k in enumerate(cats)}
        codes, n = pr.map(lambda z: mp.get(z,-1)).astype(int).values, len(cats)
    if n > 1: aux[c], meta[c] = codes, n
AK = sorted(aux.keys()); print("helper heads:", meta)

MEAN = np.array([0.485,0.456,0.406], np.float32).reshape(3,1,1)
STD  = np.array([0.229,0.224,0.225], np.float32).reshape(3,1,1)

class DS(Dataset):
    def __init__(self, idx, aug, mult=1, tta=0):
        self.idx, self.aug, self.mult, self.tta = np.asarray(idx), aug, (mult if aug else 1), tta
    def __len__(self): return len(self.idx)*self.mult
    def __getitem__(self, i):
        j = int(self.idx[i % len(self.idx)]); r = d.iloc[j]
        ti, tm, wi, wm = [a.copy() for a in CACHE[r["stem"]]]
        if self.aug:
            fh, fv = np.random.rand() < .5, np.random.rand() < .5
            kk = np.random.randint(4)
            aff = np.random.rand() < .7
            ang, sc = np.random.uniform(-25,25), np.random.uniform(.9,1.12)
            itn = np.random.rand() < .5
            gg, bb = np.random.uniform(.85,1.15), np.random.uniform(-12,12)
            def T(im, mk, s):
                if fh: im, mk = im[:,::-1], mk[:,::-1]
                if fv: im, mk = im[::-1,:], mk[::-1,:]
                if kk: im, mk = np.rot90(im,kk), np.rot90(mk,kk)
                im, mk = np.ascontiguousarray(im), np.ascontiguousarray(mk)
                if aff:
                    M = cv2.getRotationMatrix2D((s/2,s/2), ang, sc)
                    im = cv2.warpAffine(im, M, (s,s), flags=cv2.INTER_LINEAR,
                                        borderMode=cv2.BORDER_REFLECT)
                    mk = cv2.warpAffine(mk, M, (s,s), flags=cv2.INTER_NEAREST,
                                        borderMode=cv2.BORDER_CONSTANT)
                if itn: im = np.clip(im.astype(np.float32)*gg + bb, 0, 255).astype(np.uint8)
                return im, mk
            ti, tm = T(ti, tm, ST); wi, wm = T(wi, wm, SW)
        else:
            t = self.tta
            def V(im, mk):
                if   t == 1: return im[:,::-1], mk[:,::-1]
                elif t == 2: return im[::-1,:], mk[::-1,:]
                elif t == 3: return np.rot90(im,2), np.rot90(mk,2)
                return im, mk
            ti, tm = V(ti, tm); wi, wm = V(wi, wm)
        out = []
        for im, mk in [(ti,tm), (wi,wm)]:
            im = np.ascontiguousarray(im).astype(np.float32)/255.0
            out.append(torch.from_numpy(((np.stack([im]*3,0)-MEAN)/STD).astype(np.float32)))
            out.append(torch.from_numpy(np.ascontiguousarray(mk).astype(np.float32))[None])
        av = (np.array([aux[c][j] for c in AK], dtype=np.int64) if AK else np.zeros(0, np.int64))
        return out[0], out[1], out[2], out[3], torch.tensor(int(r["label"])), torch.from_numpy(av)

class TwoStream(nn.Module):
    def __init__(self, am, att=2.0):
        super().__init__()
        def bb():
            try:  return models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1).features
            except Exception: return models.densenet121(weights=None).features
        self.bt, self.bw, self.att = bb(), bb(), att
        Fd = 1024*4
        self.head = nn.Sequential(nn.Linear(Fd,512), nn.BatchNorm1d(512), nn.ReLU(True),
                                  nn.Dropout(0.4), nn.Linear(512,2))
        self.keys = sorted(am.keys())
        self.aux = nn.ModuleList([nn.Sequential(nn.Linear(Fd,128), nn.ReLU(True),
                                                nn.Dropout(0.3), nn.Linear(128, am[k]))
                                  for k in self.keys])
    def pool(self, b, x, m):
        f = F.relu(b(x))
        mm = F.interpolate(m, size=f.shape[2:], mode="bilinear", align_corners=False)
        w = 1.0 + self.att*mm
        return torch.cat([(f*w).sum((2,3))/(w.sum((2,3))+1e-6), f.mean((2,3))], 1)
    def forward(self, xt, mt, xw, mw):
        g = torch.cat([self.pool(self.bt, xt, mt), self.pool(self.bw, xw, mw)], 1)
        return self.head(g), [h(g) for h in self.aux]

def focal(lg, tg, al):
    ce = F.cross_entropy(lg.float(), tg, weight=al, reduction="none")
    return ((1-torch.exp(-ce))**GAMMA * ce).mean()

@torch.no_grad()
def predict(net, idx, tta=True):
    net.eval(); tot = None
    for t in ([0,1,2,3] if tta else [0]):
        ld = DataLoader(DS(idx, False, 1, tta=t), batch_size=12, shuffle=False, num_workers=0)
        ps = []
        for xt, mt, xw, mw, _, _ in ld:
            xt, mt, xw, mw = xt.to(DEV), mt.to(DEV), xw.to(DEV), mw.to(DEV)
            with torch.amp.autocast(device_type="cuda"): o, _ = net(xt, mt, xw, mw)
            ps += list(torch.softmax(o.float(),1)[:,1].cpu().numpy())
        ps = np.array(ps); tot = ps if tot is None else tot + ps
    return tot/(4 if tta else 1)

def train_one(tr, va, te, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    yy = d["label"].values
    n0, n1 = float((yy[tr]==0).sum()), float((yy[tr]==1).sum())
    al = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV, dtype=torch.float32)
    net = TwoStream(meta, ATT).to(DEV)
    back = list(net.bt.parameters()) + list(net.bw.parameters())
    for p_ in back: p_.requires_grad = False
    hp = [p_ for n_, p_ in net.named_parameters()
          if not (n_.startswith("bt.") or n_.startswith("bw."))]
    scaler = torch.amp.GradScaler()
    opt = torch.optim.AdamW(hp, lr=LR_HEAD, weight_decay=WD); sch = None
    tl = DataLoader(DS(tr, True, MULT), batch_size=BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)
    best, bstate, bad = -1.0, None, 0
    for ep in range(1, EPOCHS+1):
        if ep == FREEZE+1:
            for p_ in back: p_.requires_grad = True
            opt = torch.optim.AdamW([{"params": back, "lr": LR_BACK},
                                     {"params": hp,   "lr": LR_HEAD_FT}], weight_decay=WD)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS-FREEZE))
        net.train()
        if ep <= FREEZE: net.bt.eval(); net.bw.eval()
        for xt, mt, xw, mw, t_, a_ in tl:
            xt = xt.to(DEV, non_blocking=True); mt = mt.to(DEV, non_blocking=True)
            xw = xw.to(DEV, non_blocking=True); mw = mw.to(DEV, non_blocking=True)
            t_, a_ = t_.to(DEV), a_.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o, ax = net(xt, mt, xw, mw)
                loss = focal(o, t_, al)
                if len(ax):
                    loss = loss + AUX_W*sum(F.cross_entropy(gg_.float(), a_[:,h], ignore_index=-1)
                                            for h, gg_ in enumerate(ax))/len(ax)
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            scaler.step(opt); scaler.update()
        if sch is not None: sch.step()
        pv = predict(net, va, tta=False)
        auc = roc_auc_score(yy[va], pv) if len(set(yy[va])) > 1 else 0.0
        star = ""
        if auc > best:
            best, bad = auc, 0
            bstate = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}; star = " *"
        else: bad += 1
        print("      ep %2d  val-AUC %.4f%s" % (ep, auc, star), flush=True)
        if bad >= PATIENCE: print("      early stop"); break
    net.load_state_dict({k: v.to(DEV) for k, v in bstate.items()})
    pt = predict(net, te, tta=True)
    del net; gc.collect(); torch.cuda.empty_cache()
    return pt, best

y = d["label"].values; oof = np.full(len(d), np.nan)
for k in FOLDS:
    role = d["role_f%d" % k]
    tr, va, te = np.where(role=="train")[0], np.where(role=="val")[0], np.where(role=="test")[0]
    assert not (set(d.patient_id[tr]) & set(d.patient_id[te])), "LEAK"
    print("\n### fold %d | train %d val %d test %d  (seed %d)" % (k, len(tr), len(va), len(te), SEED))
    t0 = time.time()
    p_, bv = train_one(tr, va, te, SEED)
    oof[te] = p_
    print("  FOLD %d best val %.4f | test AUC %.4f  (%.0fs)"
          % (k, bv, roc_auc_score(y[te], p_), time.time()-t0))

done = ~np.isnan(oof)
res = d.loc[done, ["img","lesion_key","label"]].copy()
res["prob"] = oof[done]
res.rename(columns={"label":"true"}).to_csv(F_S22, index=False)
print("\nsaved %s" % os.path.basename(F_S22))

# ══════════════════════════════════════════════════════════════════
# MERGE THE TWO SEEDS
# ══════════════════════════════════════════════════════════════════
a = pd.read_csv(F_S11)[["img","prob"]].rename(columns={"prob":"p11"})
b = pd.read_csv(F_S22)[["img","prob"]].rename(columns={"prob":"p22"})
m = (d[["img","lesion_key","label"]].merge(a, on="img").merge(b, on="img"))
m["prob"] = 0.5*(m["p11"] + m["p22"])
print("\nmerged %d images" % len(m))

def les_auc(col):
    g = m.groupby("lesion_key").agg(y=("label","max"), p=(col,"mean")).reset_index()
    return roc_auc_score(g.y, g.p)
print("\n" + "="*62); print("SEED AVERAGING"); print("="*62)
print("  seed 11 alone      per-lesion AUC %.4f" % les_auc("p11"))
print("  seed 22 alone      per-lesion AUC %.4f" % les_auc("p22"))
print("  AVERAGE of both    per-lesion AUC %.4f   <-- this is what gets saved" % les_auc("prob"))
print("  correlation between the two seeds: %.3f" % m[["p11","p22"]].corr().iloc[0,1])

out = m[["img","lesion_key","label","prob"]].rename(columns={"label":"true"})
out.to_csv(F_MAIN, index=False)
print("\n  saved %s  -> now rerun Cell 27, then Cell 28, then Cell 31" % os.path.basename(F_MAIN))

seed 11 preserved -> cv_mass_twostream_s11.csv
MASS: 1696 images | 892 patients | malignant 46.2%
cached 1696 in 17s
helper heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5}

### fold 0 | train 1202 val 155 test 339  (seed 22)
      ep  1  val-AUC 0.7126 *
      ep  2  val-AUC 0.7676 *
      ep  3  val-AUC 0.7328
      ep  4  val-AUC 0.7670
      ep  5  val-AUC 0.8076 *
      ep  6  val-AUC 0.7906
      ep  7  val-AUC 0.8117 *
      ep  8  val-AUC 0.7840
      ep  9  val-AUC 0.8154 *
      ep 10  val-AUC 0.8127
      ep 11  val-AUC 0.8234 *
      ep 12  val-AUC 0.8016
      ep 13  val-AUC 0.8080
      ep 14  val-AUC 0.8139
      ep 15  val-AUC 0.8180
      ep 16  val-AUC 0.8112
      ep 17  val-AUC 0.8136
      ep 18  val-AUC 0.8123
      early stop
  FOLD 0 best val 0.8234 | test AUC 0.8332  (1375s)

### fold 1 | train 1184 val 172 test 340  (seed 22)
      ep  1  val-AUC 0.7092 *
      ep  2  val-AUC 0.5665
      ep  3  val-AUC 0.6721
      ep  4  val-AUC 0.6691
      ep  5 

In [5]:
# ══════════════════════════════════════════════════════════════════════
# CELL 32 — BOOTSTRAP-STABILISED THRESHOLDS
#   The thresholds are re-fitted on 200 patient-level resamples of the
#   inner folds and the median is used. Removes threshold-picking luck.
#   Still fully nested. CPU only, ~2 min.
# ══════════════════════════════════════════════════════════════════════
import os
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
pd.set_option("display.width", 220)

D = "/root/autodl-tmp/CBIS"
B = 200                                   # bootstrap resamples
GRID = np.round(np.arange(0.02, 0.981, 0.01), 3)
T = len(GRID)

L = pd.read_csv(os.path.join(D, "mass_lesion_errors_real.csv"))
y    = L["y"].values.astype(int)
p    = L["p"].values
fold = L["fold"].values.astype(int)
ass  = L["ass"].values.astype(int)
pid  = L["pid"].values
GROUPS = np.sort(np.unique(ass))
print("n=%d  AUC %.4f  malignant %.1f%%" % (len(L), roc_auc_score(y, p), 100*y.mean()))

def curves(yy, pp, gg):
    """per-group TP/TN counts at every threshold, vectorised"""
    out = {}
    for g in np.unique(gg):
        m = gg == g
        ys, ps = yy[m], pp[m]
        pr = ps[None, :] > GRID[:, None]
        out[g] = (np.asarray((pr & (ys == 1)).sum(1)),      # tp
                  np.asarray((~pr & (ys == 0)).sum(1)))     # tn
    return out

def fit(cur, P, floor, passes=8):
    """coordinate ascent over per-group thresholds under ONE sensitivity floor"""
    gs = list(cur.keys())
    tp_all = np.sum([cur[g][0] for g in gs], 0)
    cr_all = np.sum([cur[g][0] + cur[g][1] for g in gs], 0)
    ok = (tp_all / max(P, 1)) >= floor
    start = int(np.argmax(np.where(ok, cr_all, -1))) if ok.any() else int(np.argmax(cr_all))
    idx = {g: start for g in gs}
    tot_tp = int(sum(cur[g][0][idx[g]] for g in gs))
    tot_cr = int(sum(cur[g][0][idx[g]] + cur[g][1][idx[g]] for g in gs))
    for _ in range(passes):
        moved = False
        for g in gs:
            tp, tn = cur[g]
            b_tp = tot_tp - tp[idx[g]]
            b_cr = tot_cr - (tp[idx[g]] + tn[idx[g]])
            n_tp = b_tp + tp
            n_cr = b_cr + tp + tn
            feas = (n_tp / max(P, 1)) >= floor
            if not feas.any(): continue
            j = int(np.argmax(np.where(feas, n_cr, -1)))
            if j != idx[g]:
                idx[g] = j; tot_tp = int(n_tp[j]); tot_cr = int(n_cr[j]); moved = True
        if not moved: break
    return idx

def apply_idx(pp, gg, idx, dflt):
    pr = np.zeros(len(pp), int)
    for g in np.unique(gg):
        m = gg == g
        pr[m] = (pp[m] > GRID[idx.get(g, dflt)]).astype(int)
    return pr

def run(floor, bagged):
    pred = np.zeros(len(L), int)
    chosen = {}
    for k in range(5):
        inn, out = fold != k, fold == k
        if out.sum() == 0 or len(set(y[inn])) < 2: continue
        yi, pi, gi = y[inn], p[inn], ass[inn]
        pi_pid = pid[inn]
        base = fit(curves(yi, pi, gi), int((yi == 1).sum()), floor)
        if not bagged:
            idx = base
        else:
            pats = np.unique(pi_pid)
            acc = {g: [] for g in np.unique(gi)}
            rng = np.random.default_rng(0)
            for _ in range(B):
                take = rng.choice(pats, size=len(pats), replace=True)
                sel = np.concatenate([np.where(pi_pid == q)[0] for q in take])
                yb, pb, gb = yi[sel], pi[sel], gi[sel]
                if len(set(yb)) < 2: continue
                fb = fit(curves(yb, pb, gb), int((yb == 1).sum()), floor)
                for g in fb: acc[g].append(fb[g])
            idx = {g: (int(np.median(v)) if len(v) >= B // 4 else base.get(g, len(GRID)//2))
                   for g, v in acc.items()}
        dflt = int(np.median(list(idx.values())))
        pred[out] = apply_idx(p[out], ass[out], idx, dflt)
        chosen[k] = {int(g): float(GRID[i]) for g, i in sorted(idx.items())}
    return pred, chosen

print("\n" + "="*86)
print("SINGLE FIT  vs  BOOTSTRAP-STABILISED  (nested, full cohort, nothing excluded)")
print("="*86)
rows, keep = [], {}
for floor in [0.60, 0.75, 0.85]:
    for bag in [False, True]:
        pr, ch = run(floor, bag)
        tn, fp, fn, tp = confusion_matrix(y, pr, labels=[0,1]).ravel()
        rows.append(dict(floor=floor, thresholds=("bootstrap x%d" % B) if bag else "single fit",
                         accuracy="%.1f%%" % (100*accuracy_score(y, pr)),
                         sens=round(tp/max(tp+fn,1),3), spec=round(tn/max(tn+fp,1),3),
                         FP=fp, missed=fn))
        keep[(floor, bag)] = (pr, ch)
print(pd.DataFrame(rows).to_string(index=False))

print("\n--- how much do the chosen thresholds move between folds? ---")
for bag in [False, True]:
    _, ch = keep[(0.60, bag)]
    gs = sorted({g for f in ch.values() for g in f})
    sd = {g: float(np.std([ch[k][g] for k in ch if g in ch[k]])) for g in gs}
    print("  %-16s  per-group spread across folds: %s"
          % (("bootstrap" if bag else "single fit"),
             {g: round(v, 3) for g, v in sd.items()}))
    print("                    mean spread %.4f  <-- smaller is more stable"
          % np.mean(list(sd.values())))

BEST = (0.60, True)
pr, ch = keep[BEST]
L["pred"] = pr
tn, fp, fn, tp = confusion_matrix(y, pr, labels=[0,1]).ravel()
print("\n" + "="*86)
print("BOOTSTRAP-STABILISED, floor %.2f   acc %.1f%%  sens %.3f  spec %.3f  FP %d  missed %d"
      % (BEST[0], 100*accuracy_score(y, pr), tp/max(tp+fn,1), tn/max(tn+fp,1), fp, fn))
print("="*86)
TE = int((pr != y).sum()); out = []
for g in GROUPS:
    m = ass == g
    if m.sum() < 10: continue
    a_, b_, c_, d_ = confusion_matrix(y[m], pr[m], labels=[0,1]).ravel()
    out.append(dict(BIRADS=g, n=int(m.sum()),
                    malignant=int((y[m]==1).sum()), benign=int((y[m]==0).sum()),
                    AUC=round(roc_auc_score(y[m], p[m]), 3) if len(set(y[m]))>1 else np.nan,
                    accuracy="%.1f%%" % (100*accuracy_score(y[m], pr[m])),
                    sens=round(d_/max(d_+c_,1),3), FP=b_, missed=c_,
                    share="%.0f%%" % (100*(b_+c_)/max(TE,1))))
print(pd.DataFrame(out).to_string(index=False))
print("\nNote: AUC is uninformative where one class is tiny (BI-RADS 5 has %d benign, "
      "BI-RADS 2 has %d malignant)." % (int(((ass==5)&(y==0)).sum()), int(((ass==2)&(y==1)).sum())))
L.to_csv(os.path.join(D, "mass_lesion_errors_final.csv"), index=False)
print("saved mass_lesion_errors_final.csv")

n=1005  AUC 0.9077  malignant 46.0%

SINGLE FIT  vs  BOOTSTRAP-STABILISED  (nested, full cohort, nothing excluded)
 floor     thresholds accuracy  sens  spec  FP  missed
  0.60     single fit    85.7% 0.842 0.869  71      73
  0.60 bootstrap x200    85.6% 0.844 0.866  73      72
  0.75     single fit    85.7% 0.842 0.869  71      73
  0.75 bootstrap x200    85.6% 0.844 0.866  73      72
  0.85     single fit    86.0% 0.868 0.853  80      61
  0.85 bootstrap x200    85.6% 0.866 0.847  83      62

--- how much do the chosen thresholds move between folds? ---
  single fit        per-group spread across folds: {0: 0.012, 1: 0.0, 2: 0.004, 3: 0.008, 4: 0.047, 5: 0.0}
                    mean spread 0.0119  <-- smaller is more stable
  bootstrap         per-group spread across folds: {0: 0.015, 1: 0.0, 2: 0.004, 3: 0.008, 4: 0.035, 5: 0.0}
                    mean spread 0.0103  <-- smaller is more stable

BOOTSTRAP-STABILISED, floor 0.60   acc 85.6%  sens 0.844  spec 0.866  FP 73  missed 72

In [1]:
# ══════════════════════════════════════════════════════════════════════
# CELL 33 — EVALUATE ON THE OFFICIAL CBIS-DDSM TRAIN/TEST SPLIT
#   Thresholds are fitted ONLY on official-train lesions and applied
#   unchanged to official-test lesions. CPU only, ~1 min.
# ══════════════════════════════════════════════════════════════════════
import os
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
pd.set_option("display.width", 220)

D, LES = "/root/autodl-tmp/CBIS", "mass"
GRID = np.round(np.arange(0.02, 0.981, 0.01), 3)

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES))
d["lesion_key"] = d["lesion_key"].astype(str)
assert "official_split" in d.columns, "no official_split column"
d["osp"] = d["official_split"].astype(str).str.lower().str.strip()
print("official_split values:", d["osp"].value_counts().to_dict())

sp = d.groupby("lesion_key")["osp"].agg(lambda s: s.mode().iat[0])
L = pd.read_csv(os.path.join(D, "mass_lesion_errors_final.csv"))
L["lesion_key"] = L["lesion_key"].astype(str)
L["osp"] = L["lesion_key"].map(sp)
L = L.dropna(subset=["osp"]).reset_index(drop=True)

istr = L["osp"].str.contains("train")
iste = L["osp"].str.contains("test")
print("\nlesions: official-train %d | official-test %d | unmapped %d"
      % (istr.sum(), iste.sum(), len(L) - istr.sum() - iste.sum()))
ov = set(L.loc[istr, "pid"]) & set(L.loc[iste, "pid"])
print("patients in BOTH official train and test: %d  %s" % (len(ov), "(must be 0)" if not ov else "<-- PROBLEM"))

y, p, ass = L["y"].values, L["p"].values, L["ass"].values

def sens(yy, pr): return ((pr==1)&(yy==1)).sum()/max((yy==1).sum(),1)
def curves(yy, pp, gg):
    out = {}
    for g in np.unique(gg):
        m = gg==g; ys, ps = yy[m], pp[m]
        pr = ps[None,:] > GRID[:,None]
        out[g] = (np.asarray((pr & (ys==1)).sum(1)), np.asarray((~pr & (ys==0)).sum(1)))
    return out
def fit(cur, P, floor, passes=8):
    gs = list(cur.keys())
    tp_a = np.sum([cur[g][0] for g in gs], 0); cr_a = np.sum([cur[g][0]+cur[g][1] for g in gs], 0)
    ok = (tp_a/max(P,1)) >= floor
    s0 = int(np.argmax(np.where(ok, cr_a, -1))) if ok.any() else int(np.argmax(cr_a))
    idx = {g: s0 for g in gs}
    ttp = int(sum(cur[g][0][idx[g]] for g in gs))
    tcr = int(sum(cur[g][0][idx[g]]+cur[g][1][idx[g]] for g in gs))
    for _ in range(passes):
        moved = False
        for g in gs:
            tp, tn = cur[g]
            n_tp = ttp - tp[idx[g]] + tp
            n_cr = tcr - (tp[idx[g]]+tn[idx[g]]) + tp + tn
            fe = (n_tp/max(P,1)) >= floor
            if not fe.any(): continue
            j = int(np.argmax(np.where(fe, n_cr, -1)))
            if j != idx[g]: idx[g]=j; ttp=int(n_tp[j]); tcr=int(n_cr[j]); moved=True
        if not moved: break
    return idx

FLOOR = 0.60
idx = fit(curves(y[istr.values], p[istr.values], ass[istr.values]),
          int((y[istr.values]==1).sum()), FLOOR)
dflt = int(np.median(list(idx.values())))
print("\nthresholds fitted on official-train:",
      {int(g): float(GRID[i]) for g, i in sorted(idx.items())})

def report(mask, tag):
    yy, pp, gg = y[mask], p[mask], ass[mask]
    pr = np.zeros(len(yy), int)
    for g in np.unique(gg):
        m = gg==g; pr[m] = (pp[m] > GRID[idx.get(g, dflt)]).astype(int)
    tn, fp, fn, tp = confusion_matrix(yy, pr, labels=[0,1]).ravel()
    print("  %-26s n=%4d  malig %.0f%%  AUC %.4f  acc %.1f%%  sens %.3f  spec %.3f  FP %3d  FN %3d"
          % (tag, len(yy), 100*yy.mean(), roc_auc_score(yy, pp), 100*accuracy_score(yy, pr),
             tp/max(tp+fn,1), tn/max(tn+fp,1), fp, fn))
    return pr

print("\n" + "="*104)
print("OFFICIAL CBIS-DDSM SPLIT")
print("="*104)
report(istr.values, "official TRAIN (in-sample)")
prt = report(iste.values, "official TEST (held out)")
report(np.ones(len(L), bool), "all lesions (5-fold CV)")

print("\n--- official TEST, by BI-RADS ---")
yt, at = y[iste.values], ass[iste.values]
rows = []
for g in np.unique(at):
    m = at==g
    if m.sum() < 5: continue
    a_,b_,c_,d_ = confusion_matrix(yt[m], prt[m], labels=[0,1]).ravel()
    rows.append(dict(BIRADS=int(g), n=int(m.sum()), malig=int((yt[m]==1).sum()),
                     AUC=round(roc_auc_score(yt[m], p[iste.values][m]),3) if len(set(yt[m]))>1 else np.nan,
                     acc="%.1f%%" % (100*accuracy_score(yt[m], prt[m])), FP=b_, missed=c_))
print(pd.DataFrame(rows).to_string(index=False))

print("\nIMPORTANT — how to describe this in the thesis:")
print("  These are out-of-fold predictions from patient-grouped 5-fold CV, RESTRICTED to the")
print("  official test lesions, with thresholds fitted only on the official training lesions.")
print("  Every lesion is still scored by a model that never saw its patient, so the number is")
print("  leakage-free. It is NOT a reproduction of the official protocol (which would require")
print("  retraining on the official training set alone) — call it 'official test subset under")
print("  our CV protocol', not 'official split result'.")

official_split values: {'train': 1318, 'test': 378}

lesions: official-train 782 | official-test 223 | unmapped 0
patients in BOTH official train and test: 0  (must be 0)

thresholds fitted on official-train: {0: 0.54, 1: 0.02, 2: 0.8, 3: 0.59, 4: 0.48, 5: 0.02}

OFFICIAL CBIS-DDSM SPLIT
  official TRAIN (in-sample) n= 782  malig 48%  AUC 0.9150  acc 87.6%  sens 0.888  spec 0.865  FP  55  FN  42
  official TEST (held out)   n= 223  malig 39%  AUC 0.8791  acc 81.6%  sens 0.816  spec 0.816  FP  25  FN  16
  all lesions (5-fold CV)    n=1005  malig 46%  AUC 0.9077  acc 86.3%  sens 0.874  spec 0.853  FP  80  FN  58

--- official TEST, by BI-RADS ---
 BIRADS  n  malig   AUC   acc  FP  missed
      0 18      2 0.906 77.8%   3       1
      2 10      1 1.000 90.0%   0       1
      3 53      2 0.980 92.5%   4       0
      4 96     38 0.750 68.8%  16      14
      5 45     43 0.640 95.6%   2       0

IMPORTANT — how to describe this in the thesis:
  These are out-of-fold predictions from pati

In [2]:
import os, pandas as pd
D = "/root/autodl-tmp/CBIS"
m = pd.read_csv(os.path.join(D, "unified_folds_mass.csv"))
c = pd.read_csv(os.path.join(D, "unified_folds_calc.csv"))
mp, cp = set(m.patient_id), set(c.patient_id)
clean = c[~c.patient_id.isin(mp)]
print("mass patients %d | calc patients %d | overlap %d" % (len(mp), len(cp), len(mp & cp)))
print("calc lesions from NON-overlapping patients: %d of %d" % (len(clean), len(c)))
print("  malignant %.1f%%" % (100*clean.label.mean()))
print("calc columns:", [x for x in c.columns if x in ('img','msk','label','patient_id','lesion_key')])
print("predmasks_calc present:", os.path.isdir(os.path.join(D, "predmasks_calc")))

mass patients 892 | calc patients 753 | overlap 79
calc lesions from NON-overlapping patients: 1635 of 1866
  malignant 35.2%
calc columns: ['patient_id', 'img', 'msk', 'label', 'lesion_key']
predmasks_calc present: True


In [5]:
# ══════════════════════════════════════════════════════════════════════
# CELL 34 — CALCIFICATION PRETRAINING -> TWO-STREAM FINE-TUNE
#   Stage 1: DenseNet-121 trained on 1,635 calcification lesions whose
#            patients appear NOWHERE in the mass dataset (leak-free for
#            all folds). Backbone saved to disk.
#   Stage 2: both two-stream backbones initialised from it, fine-tuned
#            and evaluated on mass only. Zero calc lesions in any test set.
#   >>> QUICK_TEST = True FIRST <<<   full run ~5 h
# ══════════════════════════════════════════════════════════════════════
import os, gc, time
os.environ.setdefault("OMP_NUM_THREADS", "4")
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score
cv2.setNumThreads(0)

D = "/root/autodl-tmp/CBIS"
DEV = torch.device("cuda")
torch.backends.cudnn.benchmark = True; torch.backends.cuda.matmul.allow_tf32 = True

QUICK_TEST = False                     # <<<<<< set False for the real run

CKPT   = os.path.join(D, "calc_pretrained_backbone.pt")
OUTCSV = os.path.join(D, "cv_mass_twostream_calcpre_oof.csv")
ST, SW, BATCH = 512, 384, 8
SEED = 11
PRE_EPOCHS, PRE_BATCH, PRE_LR = 14, 12, 3e-4
EPOCHS, FREEZE = 20, 3
LR_HEAD, LR_HEAD_FT, LR_BACK = 1e-3, 3e-4, 3e-5
WD, GAMMA, AUX_W, MULT, PATIENCE, ATT = 1e-4, 2.0, 0.3, 3, 7, 2.0
FOLDS = [0, 1, 2, 3, 4]
if QUICK_TEST:
    PRE_EPOCHS, EPOCHS, FREEZE, MULT, FOLDS = 3, 4, 1, 1, [0]
    print(">>> QUICK TEST: 3 pretrain epochs, 1 fold, 4 epochs — error check only\n")

MEAN = np.array([0.485,0.456,0.406], np.float32).reshape(3,1,1)
STD  = np.array([0.229,0.224,0.225], np.float32).reshape(3,1,1)
cl = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
def load(p_, size, mask=False):
    im = cv2.imread(str(p_), cv2.IMREAD_GRAYSCALE)
    if im is None: im = np.zeros((size,size), np.uint8)
    if im.shape != (size,size):
        im = cv2.resize(im, (size,size),
                        interpolation=cv2.INTER_NEAREST if mask else cv2.INTER_AREA)
    return (im>127).astype(np.uint8) if mask else cl.apply(im)

def pool2(feat, m, att):
    mm = F.interpolate(m, size=feat.shape[2:], mode="bilinear", align_corners=False)
    w = 1.0 + att*mm
    return torch.cat([(feat*w).sum((2,3))/(w.sum((2,3))+1e-6), feat.mean((2,3))], 1)

def backbone():
    try:  return models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1).features
    except Exception: return models.densenet121(weights=None).features

def focal(lg, tg, al):
    ce = F.cross_entropy(lg.float(), tg, weight=al, reduction="none")
    return ((1-torch.exp(-ce))**GAMMA * ce).mean()

# ══════════════════════════════════════════════════════════════════
# STAGE 1 — pretrain on calcification
# ══════════════════════════════════════════════════════════════════
if os.path.exists(CKPT) and not QUICK_TEST:
    print("stage 1 skipped — checkpoint already exists (%s)" % os.path.basename(CKPT))
else:
    m_ = pd.read_csv(os.path.join(D, "unified_folds_mass.csv"))
    c  = pd.read_csv(os.path.join(D, "unified_folds_calc.csv"))
    c  = c[~c.patient_id.isin(set(m_.patient_id))].reset_index(drop=True)
    PMC = os.path.join(D, "predmasks_calc")
    c["stem"] = c["img"].apply(lambda q: os.path.basename(str(q)).replace("_img.png",""))
    c["pm"]   = c["stem"].apply(lambda s: os.path.join(PMC, s+"_pred.png"))
    c = c[c["pm"].apply(os.path.exists) & c["img"].apply(os.path.exists)].reset_index(drop=True)
    c["label"] = c["label"].astype(int)
    print("STAGE 1: %d calc images | %d patients | malignant %.1f%%"
          % (len(c), c.patient_id.nunique(), 100*c.label.mean()))
    assert len(c) > 500, "too few clean calc images"

    rng = np.random.default_rng(0)
    pats = np.array(sorted(c.patient_id.unique())); rng.shuffle(pats)
    vp = set(pats[:max(int(0.12*len(pats)), 10)])
    tr_i = np.where(~c.patient_id.isin(vp))[0]
    va_i = np.where( c.patient_id.isin(vp))[0]
    print("  pretrain split: %d train / %d val images (patient-disjoint)" % (len(tr_i), len(va_i)))

    CC, t0 = {}, time.time()
    for _, r in c.iterrows():
        CC[r["stem"]] = (load(r["img"], ST), load(r["pm"], ST, True))
    print("  cached %d calc images in %.0fs" % (len(CC), time.time()-t0))

    class CDS(Dataset):
        def __init__(self, idx, aug): self.idx, self.aug = np.asarray(idx), aug
        def __len__(self): return len(self.idx)
        def __getitem__(self, i):
            j = int(self.idx[i]); r = c.iloc[j]
            im, mk = [a.copy() for a in CC[r["stem"]]]
            if self.aug:
                if np.random.rand()<.5: im, mk = im[:,::-1], mk[:,::-1]
                if np.random.rand()<.5: im, mk = im[::-1,:], mk[::-1,:]
                kk = np.random.randint(4)
                if kk: im, mk = np.rot90(im,kk), np.rot90(mk,kk)
                im, mk = np.ascontiguousarray(im), np.ascontiguousarray(mk)
                if np.random.rand()<.7:
                    M = cv2.getRotationMatrix2D((ST/2,ST/2), np.random.uniform(-25,25),
                                                np.random.uniform(.9,1.12))
                    im = cv2.warpAffine(im, M, (ST,ST), flags=cv2.INTER_LINEAR,
                                        borderMode=cv2.BORDER_REFLECT)
                    mk = cv2.warpAffine(mk, M, (ST,ST), flags=cv2.INTER_NEAREST,
                                        borderMode=cv2.BORDER_CONSTANT)
                if np.random.rand()<.5:
                    im = np.clip(im.astype(np.float32)*np.random.uniform(.85,1.15)
                                 + np.random.uniform(-12,12), 0, 255).astype(np.uint8)
            im, mk = np.ascontiguousarray(im), np.ascontiguousarray(mk)
            f = im.astype(np.float32)/255.0
            x = ((np.stack([f]*3,0)-MEAN)/STD).astype(np.float32)
            return (torch.from_numpy(x),
                    torch.from_numpy(mk.astype(np.float32))[None],
                    torch.tensor(int(r["label"])))

    class PreNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.b = backbone()
            self.head = nn.Sequential(nn.Linear(2048,512), nn.BatchNorm1d(512), nn.ReLU(True),
                                      nn.Dropout(0.4), nn.Linear(512,2))
        def forward(self, x, m):
            return self.head(pool2(F.relu(self.b(x)), m, ATT))

    torch.manual_seed(SEED); np.random.seed(SEED)
    net = PreNet().to(DEV)
    yv = c["label"].values
    n0, n1 = float((yv[tr_i]==0).sum()), float((yv[tr_i]==1).sum())
    al = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV, dtype=torch.float32)
    opt = torch.optim.AdamW(net.parameters(), lr=PRE_LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=PRE_EPOCHS)
    scaler = torch.amp.GradScaler()
    tl = DataLoader(CDS(tr_i, True), batch_size=PRE_BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)
    vl = DataLoader(CDS(va_i, False), batch_size=16, shuffle=False, num_workers=0)

    best, bstate = -1.0, None
    for ep in range(1, PRE_EPOCHS+1):
        net.train()
        for x, m, t_ in tl:
            x, m, t_ = x.to(DEV, non_blocking=True), m.to(DEV, non_blocking=True), t_.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                loss = focal(net(x, m), t_, al)
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            scaler.step(opt); scaler.update()
        sch.step()
        net.eval(); ps = []
        with torch.no_grad():
            for x, m, _ in vl:
                with torch.amp.autocast(device_type="cuda"):
                    o = net(x.to(DEV), m.to(DEV))
                ps += list(torch.softmax(o.float(),1)[:,1].cpu().numpy())
        a = roc_auc_score(yv[va_i], np.array(ps)) if len(set(yv[va_i]))>1 else 0.0
        star = ""
        if a > best:
            best = a; star = " *"
            bstate = {k: v.detach().cpu().clone() for k, v in net.b.state_dict().items()}
        print("    pretrain ep %2d  calc val-AUC %.4f%s" % (ep, a, star), flush=True)

    torch.save(bstate, CKPT)
    print("\n  STAGE 1 done. best calc val-AUC %.4f  ->  %s" % (best, os.path.basename(CKPT)))
    print("  GATE: if this is below ~0.62 the backbone learned little and stage 2 will not help.")
    del net, CC; gc.collect(); torch.cuda.empty_cache()

# ══════════════════════════════════════════════════════════════════
# STAGE 2 — two-stream fine-tune on MASS, backbones pre-initialised
# ══════════════════════════════════════════════════════════════════
LES = "mass"
WIDE = os.path.join(D, "crops_wide_%s" % LES); PM = os.path.join(D, "predmasks_%s" % LES)
d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["label"] = d["label"].astype(int)
d["stem"]  = d["img"].apply(lambda q: os.path.basename(str(q)).replace("_img.png",""))
d["tmask"] = d["stem"].apply(lambda s: os.path.join(PM,   s+"_pred.png"))
d["wimg"]  = d["stem"].apply(lambda s: os.path.join(WIDE, s+"_img.png"))
d["wmask"] = d["stem"].apply(lambda s: os.path.join(WIDE, s+"_pred.png"))
assert (~d["wimg"].apply(os.path.exists)).sum() == 0, "wide crops missing — run Cell 29c"
print("\nSTAGE 2: %d mass images | %d patients | malignant %.1f%%"
      % (len(d), d.patient_id.nunique(), 100*d.label.mean()))

CACHE, t0 = {}, time.time()
for _, r in d.iterrows():
    CACHE[r["stem"]] = (load(r["img"], ST), load(r["tmask"], ST, True),
                        load(r["wimg"], SW), load(r["wmask"], SW, True))
print("  cached %d in %.0fs" % (len(CACHE), time.time()-t0))

def primary(x): return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()
aux, meta = {}, {}
for cc in ["subtlety","mass_shape","mass_margins"]:
    if cc not in d.columns or d[cc].notna().sum()==0: continue
    if cc == "subtlety":
        v = pd.to_numeric(d[cc], errors="coerce").where(lambda z:(z>=1)&(z<=5))
        codes, n = (v-1).fillna(-1).astype(int).values, 5
    else:
        pr = d[cc].map(primary)
        pr = pr.where(pr.isin(pr.value_counts().head(6).index.tolist()), "OTHER")
        cats = sorted([q for q in pr.unique() if q != "UNK"]); mp = {q:i for i,q in enumerate(cats)}
        codes, n = pr.map(lambda z: mp.get(z,-1)).astype(int).values, len(cats)
    if n > 1: aux[cc], meta[cc] = codes, n
AK = sorted(aux.keys()); print("  helper heads:", meta)

class DS(Dataset):
    def __init__(self, idx, aug, mult=1, tta=0):
        self.idx, self.aug, self.mult, self.tta = np.asarray(idx), aug, (mult if aug else 1), tta
    def __len__(self): return len(self.idx)*self.mult
    def __getitem__(self, i):
        j = int(self.idx[i % len(self.idx)]); r = d.iloc[j]
        ti, tm, wi, wm = [a.copy() for a in CACHE[r["stem"]]]
        if self.aug:
            fh, fv = np.random.rand()<.5, np.random.rand()<.5
            kk = np.random.randint(4); aff = np.random.rand()<.7
            ang, sc = np.random.uniform(-25,25), np.random.uniform(.9,1.12)
            itn = np.random.rand()<.5
            gg, bb = np.random.uniform(.85,1.15), np.random.uniform(-12,12)
            def T(im, mk, s):
                if fh: im, mk = im[:,::-1], mk[:,::-1]
                if fv: im, mk = im[::-1,:], mk[::-1,:]
                if kk: im, mk = np.rot90(im,kk), np.rot90(mk,kk)
                im, mk = np.ascontiguousarray(im), np.ascontiguousarray(mk)
                if aff:
                    M = cv2.getRotationMatrix2D((s/2,s/2), ang, sc)
                    im = cv2.warpAffine(im, M, (s,s), flags=cv2.INTER_LINEAR,
                                        borderMode=cv2.BORDER_REFLECT)
                    mk = cv2.warpAffine(mk, M, (s,s), flags=cv2.INTER_NEAREST,
                                        borderMode=cv2.BORDER_CONSTANT)
                if itn: im = np.clip(im.astype(np.float32)*gg+bb, 0, 255).astype(np.uint8)
                return im, mk
            ti, tm = T(ti, tm, ST); wi, wm = T(wi, wm, SW)
        else:
            t = self.tta
            def V(im, mk):
                if   t==1: return im[:,::-1], mk[:,::-1]
                elif t==2: return im[::-1,:], mk[::-1,:]
                elif t==3: return np.rot90(im,2), np.rot90(mk,2)
                return im, mk
            ti, tm = V(ti, tm); wi, wm = V(wi, wm)
        o = []
        for im, mk in [(ti,tm),(wi,wm)]:
            f = np.ascontiguousarray(im).astype(np.float32)/255.0
            o.append(torch.from_numpy(((np.stack([f]*3,0)-MEAN)/STD).astype(np.float32)))
            o.append(torch.from_numpy(np.ascontiguousarray(mk).astype(np.float32))[None])
        av = (np.array([aux[q][j] for q in AK], dtype=np.int64) if AK else np.zeros(0, np.int64))
        return o[0], o[1], o[2], o[3], torch.tensor(int(r["label"])), torch.from_numpy(av)

class TwoStream(nn.Module):
    def __init__(self, am, pre=None):
        super().__init__()
        self.bt, self.bw = backbone(), backbone()
        if pre is not None:
            self.bt.load_state_dict(pre); self.bw.load_state_dict(pre)
        Fd = 1024*4
        self.head = nn.Sequential(nn.Linear(Fd,512), nn.BatchNorm1d(512), nn.ReLU(True),
                                  nn.Dropout(0.4), nn.Linear(512,2))
        self.keys = sorted(am.keys())
        self.aux = nn.ModuleList([nn.Sequential(nn.Linear(Fd,128), nn.ReLU(True),
                                                nn.Dropout(0.3), nn.Linear(128, am[q]))
                                  for q in self.keys])
    def forward(self, xt, mt, xw, mw):
        g = torch.cat([pool2(F.relu(self.bt(xt)), mt, ATT),
                       pool2(F.relu(self.bw(xw)), mw, ATT)], 1)
        return self.head(g), [h(g) for h in self.aux]

PRE = torch.load(CKPT, map_location="cpu")
print("  loaded pretrained backbone (%d tensors)" % len(PRE))

@torch.no_grad()
def predict(net, idx, tta=True):
    net.eval(); tot = None
    for t in ([0,1,2,3] if tta else [0]):
        ld = DataLoader(DS(idx, False, 1, tta=t), batch_size=12, shuffle=False, num_workers=0)
        ps = []
        for xt, mt, xw, mw, _, _ in ld:
            with torch.amp.autocast(device_type="cuda"):
                o, _ = net(xt.to(DEV), mt.to(DEV), xw.to(DEV), mw.to(DEV))
            ps += list(torch.softmax(o.float(),1)[:,1].cpu().numpy())
        ps = np.array(ps); tot = ps if tot is None else tot+ps
    return tot/(4 if tta else 1)

def train_fold(tr, va, te):
    torch.manual_seed(SEED); np.random.seed(SEED)
    yy = d["label"].values
    n0, n1 = float((yy[tr]==0).sum()), float((yy[tr]==1).sum())
    al = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV, dtype=torch.float32)
    net = TwoStream(meta, PRE).to(DEV)
    back = list(net.bt.parameters()) + list(net.bw.parameters())
    for q in back: q.requires_grad = False
    hp = [q for n_, q in net.named_parameters()
          if not (n_.startswith("bt.") or n_.startswith("bw."))]
    scaler = torch.amp.GradScaler()
    opt = torch.optim.AdamW(hp, lr=LR_HEAD, weight_decay=WD); sch = None
    tl = DataLoader(DS(tr, True, MULT), batch_size=BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)
    best, bstate, bad = -1.0, None, 0
    for ep in range(1, EPOCHS+1):
        if ep == FREEZE+1:
            for q in back: q.requires_grad = True
            opt = torch.optim.AdamW([{"params": back, "lr": LR_BACK},
                                     {"params": hp,   "lr": LR_HEAD_FT}], weight_decay=WD)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS-FREEZE))
        net.train()
        if ep <= FREEZE: net.bt.eval(); net.bw.eval()
        for xt, mt, xw, mw, t_, a_ in tl:
            xt, mt = xt.to(DEV, non_blocking=True), mt.to(DEV, non_blocking=True)
            xw, mw = xw.to(DEV, non_blocking=True), mw.to(DEV, non_blocking=True)
            t_, a_ = t_.to(DEV), a_.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o, ax = net(xt, mt, xw, mw)
                loss = focal(o, t_, al)
                if len(ax):
                    loss = loss + AUX_W*sum(F.cross_entropy(q.float(), a_[:,h], ignore_index=-1)
                                            for h, q in enumerate(ax))/len(ax)
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            scaler.step(opt); scaler.update()
        if sch is not None: sch.step()
        pv = predict(net, va, tta=False)
        a = roc_auc_score(yy[va], pv) if len(set(yy[va]))>1 else 0.0
        star = ""
        if a > best:
            best, bad = a, 0; star = " *"
            bstate = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        else: bad += 1
        print("      ep %2d  val-AUC %.4f%s" % (ep, a, star), flush=True)
        if bad >= PATIENCE: print("      early stop"); break
    net.load_state_dict({k: v.to(DEV) for k, v in bstate.items()})
    pt = predict(net, te, tta=True)
    del net; gc.collect(); torch.cuda.empty_cache()
    return pt, best

y = d["label"].values; oof = np.full(len(d), np.nan)
for kf in FOLDS:
    role = d["role_f%d" % kf]
    tr, va, te = np.where(role=="train")[0], np.where(role=="val")[0], np.where(role=="test")[0]
    assert not (set(d.patient_id[tr]) & set(d.patient_id[te])), "LEAK"
    print("\n### fold %d | train %d val %d test %d" % (kf, len(tr), len(va), len(te)))
    t0 = time.time()
    p_, bv = train_fold(tr, va, te)
    oof[te] = p_
    print("  FOLD %d best val %.4f | test AUC %.4f  (%.0fs)"
          % (kf, bv, roc_auc_score(y[te], p_), time.time()-t0))

done = ~np.isnan(oof)
res = d.loc[done, ["img","lesion_key","label"]].copy(); res["prob"] = oof[done]
Lg = res.groupby("lesion_key").agg(y=("label","max"), p=("prob","mean")).reset_index()
print("\n" + "="*72); print("RESULT"); print("="*72)
print("  per-image  AUC %.4f  (n=%d)" % (roc_auc_score(y[done], oof[done]), done.sum()))
print("  per-lesion AUC %.4f" % roc_auc_score(Lg.y, Lg.p))
print("  COMPARE AGAINST: two-stream seed 11 WITHOUT pretraining = 0.8652")
print("  (same seed, same folds, same data — the only change is the initialisation)")
if not QUICK_TEST:
    res.rename(columns={"label":"true"}).to_csv(OUTCSV, index=False)
    print("\n  saved %s  -> rerun Cell 27" % os.path.basename(OUTCSV))
else:
    print("\n  QUICK TEST clean. Set QUICK_TEST = False and rerun (~5 h).")

STAGE 1: 1635 calc images | 674 patients | malignant 35.2%
  pretrain split: 1397 train / 238 val images (patient-disjoint)
  cached 1635 calc images in 8s
    pretrain ep  1  calc val-AUC 0.6417 *
    pretrain ep  2  calc val-AUC 0.7994 *
    pretrain ep  3  calc val-AUC 0.7573
    pretrain ep  4  calc val-AUC 0.7780
    pretrain ep  5  calc val-AUC 0.7576
    pretrain ep  6  calc val-AUC 0.8236 *
    pretrain ep  7  calc val-AUC 0.8176
    pretrain ep  8  calc val-AUC 0.7968
    pretrain ep  9  calc val-AUC 0.8312 *
    pretrain ep 10  calc val-AUC 0.8043
    pretrain ep 11  calc val-AUC 0.8481 *
    pretrain ep 12  calc val-AUC 0.8499 *
    pretrain ep 13  calc val-AUC 0.8498
    pretrain ep 14  calc val-AUC 0.8467

  STAGE 1 done. best calc val-AUC 0.8499  ->  calc_pretrained_backbone.pt
  GATE: if this is below ~0.62 the backbone learned little and stage 2 will not help.

STAGE 2: 1696 mass images | 892 patients | malignant 46.2%
  cached 1696 in 17s
  helper heads: {'subtlety': 5

In [4]:
import os
q = "/root/autodl-tmp/CBIS/calc_pretrained_backbone.pt"
if os.path.exists(q):
    os.remove(q); print("deleted the 3-epoch checkpoint — Stage 1 will retrain properly")
else:
    print("no checkpoint present, nothing to do")

deleted the 3-epoch checkpoint — Stage 1 will retrain properly


In [1]:
# ── find the TRAINING cells (the ones that WRITE the oof files) ──
import json, glob, re
NBS = sorted(glob.glob("**/*.ipynb", recursive=True))
print("notebooks found:")
for nb in NBS:
    try: n = len(json.load(open(nb)).get("cells", []))
    except Exception: n = "?"
    print(f"   {n:>4} cells   {nb}")

PAT_WRITE = re.compile(r"to_csv\([^)]*cv_mass_(v2|efficientnet_b0|resnet|convnext)", re.I)
PAT_ARCH  = re.compile(r"(densenet121|efficientnet_b0|resnet50|convnext|timm\.create_model)", re.I)

print("\n── cells that WRITE a backbone oof file ──")
for nb in NBS:
    try: cells = json.load(open(nb)).get("cells", [])
    except Exception: continue
    for i, c in enumerate(cells):
        t = "".join(c.get("source", []))
        if PAT_WRITE.search(t):
            head = [l for l in t.split("\n") if l.strip()][:3]
            print(f"\n  {nb}  cell {i}   ({len(t)} chars)")
            for l in head: print("     ", l[:100])

print("\n── cells that BUILD a backbone ──")
for nb in NBS:
    try: cells = json.load(open(nb)).get("cells", [])
    except Exception: continue
    for i, c in enumerate(cells):
        t = "".join(c.get("source", []))
        hits = sorted(set(m.group(0).lower() for m in PAT_ARCH.finditer(t)))
        if hits and ("def " in t or "class " in t):
            head = next((l for l in t.split("\n") if l.strip().startswith("#") and len(l) > 12), "")
            print(f"  {nb:<32} cell {i:>3}  {','.join(hits):<40} {head[:60]}")

notebooks found:
    382 cells   Breast_Cancer_Final (6).ipynb
      8 cells   Experiment.ipynb
     11 cells   INBreastfile.ipynb
     89 cells   Segementation .ipynb
     76 cells   the whole pipeline.ipynb
     55 cells   the whole pipeline_new (1).ipynb

── cells that WRITE a backbone oof file ──

── cells that BUILD a backbone ──
  Breast_Cancer_Final (6).ipynb    cell 117  efficientnet_b0                          #######Classification1 
  Breast_Cancer_Final (6).ipynb    cell 118  efficientnet_b0                          # CELL CLS-VIZ — Visualize classification results + ROC/AUC 
  Breast_Cancer_Final (6).ipynb    cell 119  efficientnet_b0                          # CELL CLS-CROP — Segmentation-GUIDED lesion-context CROP cl
  Breast_Cancer_Final (6).ipynb    cell 120  efficientnet_b0                          # CELL CLS-GT-UPPERBOUND — SAME classifier, but crop from GR
  Breast_Cancer_Final (6).ipynb    cell 121  efficientnet_b0                          #######Classification Thro

In [2]:
# ── fingerprint the candidate training cells ──
import json, re
NB = "the whole pipeline.ipynb"
CELLS = [18, 22, 23, 25, 26, 29]
KEEP = re.compile(
    r"(to_csv|create_model|densenet121|efficientnet_b0|resnet50|convnext"
    r"|EPOCH|epochs\s*=|\blr\s*=|LR\s*=|batch|BS\s*=|SIZE\s*=|IMG|512|384"
    r"|crops_|predmask|def forward|pool|concat|class \w+\(nn\.Module\))", re.I)

cells = json.load(open(NB))["cells"]
for i in CELLS:
    t = "".join(cells[i].get("source", []))
    lines = t.split("\n")
    head = [l for l in lines if l.strip()][:5]
    hits = [l.rstrip() for l in lines if KEEP.search(l) and l.strip()]
    print(f"\n{'='*74}\ncell {i}   ({len(t)} chars, {len(lines)} lines)\n{'='*74}")
    for l in head:  print("  ", l[:110])
    print("   ---")
    for l in hits[:22]: print("  ", l.strip()[:110])
    if len(hits) > 22: print(f"   ... {len(hits)-22} more matching lines")


cell 18   (13830 chars, 327 lines)
   # ══════════════════════════════════════════════════════════════════════
   # CELL 6 — MASS CLASSIFIER, RETRAINED (fixed LR + seed averaging)
   #          Self-contained. Uses PREDICTED masks only (end-to-end honest).
   #
   #   >>> RUN WITH QUICK_TEST = True FIRST (about 10 min) <<<
   ---
   S, BATCH   = 512, 12
   EPOCHS     = 22
   FREEZE     = 3             # epochs with backbone frozen
   SEEDS, EPOCHS, FREEZE, MULT, FOLDS = [11], 4, 1, 1, [0]
   print(">>> QUICK TEST: 1 fold, 1 seed, 4 epochs — checking for errors only\n")
   PM = os.path.join(D, "predmasks_%s" % LES)
   d["predmask"] = d["img"].apply(
   lambda p: os.path.join(PM, os.path.basename(str(p)).replace("_img.png", "") + "_pred.png"))
   missing = (~d["predmask"].apply(os.path.exists)).sum()
   _clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
   k = str(r["img"])
   pm = cv2.imread(str(r["predmask"]), cv2.IMREAD_GRAYSCALE)
   img, msk = CACHE[str(r["img"])]
   img, 

In [3]:
import json, difflib, re
cells = json.load(open("the whole pipeline.ipynb"))["cells"]
strip = lambda i: [l.rstrip() for l in "".join(cells[i]["source"]).split("\n")
                   if l.strip() and not l.strip().startswith("#")]
a, b = strip(18), strip(25)
d = [l for l in difflib.unified_diff(a, b, "cell18_densenet", "cell25_backbones",
                                     lineterm="", n=0)
     if l[:1] in "+-" and l[1:].strip() and not l.startswith(("---", "+++"))]
print(f"{len(a)} vs {len(b)} code lines, {len(d)} differing\n")
for l in d[:120]: print(l[:118])
if len(d) > 120: print(f"\n... {len(d)-120} more")

279 vs 251 code lines, 288 differing

-os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
-from sklearn.metrics import (roc_auc_score, accuracy_score,
-                             balanced_accuracy_score, confusion_matrix)
+from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, confusion_matrix
-D   = "/root/autodl-tmp/CBIS"
-LES = "mass"
-DEV = torch.device("cuda")
+D, DEV = "/root/autodl-tmp/CBIS", torch.device("cuda")
-S, BATCH   = 512, 12
-SEEDS      = [11, 22]
-EPOCHS     = 22
-FREEZE     = 3             # epochs with backbone frozen
-LR_HEAD    = 1e-3          # head, while backbone frozen
-LR_HEAD_FT = 3e-4          # head, after unfreeze   <-- was 1e-5 (the bug)
-LR_BACK    = 3e-5          # backbone, after unfreeze
-WD, GAMMA, AUX_W = 1e-4, 2.0, 0.3
-MULT, PATIENCE, ATT = 4, 7, 2.0
+BACKBONE   = "auto"        # "auto" picks the best AVAILABLE pretrained backbone
+S, BATCH = 512, 12
+SEEDS, EPOCHS, FREEZE = [11, 22], 22, 3
+LR_HEAD, LR_HEAD_FT, LR_

In [4]:
import json, re
cells = json.load(open("the whole pipeline.ipynb"))["cells"]

def classes(i):
    src = "".join(cells[i]["source"]).split("\n")
    blocks, cur = [], None
    for l in src:
        if re.match(r"^class\s+\w+\s*\(\s*nn\.Module", l):
            cur = [l]; blocks.append(cur); continue
        if cur is not None:
            if l.strip() and not l.startswith((" ", "\t")): cur = None
            else: cur.append(l)
    return ["\n".join(b).rstrip() for b in blocks]

for i in (18, 25):
    print(f"\n{'#'*72}\n#  cell {i}\n{'#'*72}")
    for b in classes(i):
        if "Dataset" in b.split("\n")[0]: continue      # skip the data loader
        print(b[:2000]); print("-"*58)


########################################################################
#  cell 18
########################################################################
class GuidedNet(nn.Module):
    """DenseNet-121 + mask-guided pooling AND whole-image pooling."""
    def __init__(self, aux_meta, att=2.0):
        super().__init__()
        try:
            dn = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        except Exception:
            dn = models.densenet121(weights=None)
            print("  (ImageNet weights unavailable — training from scratch)")
        self.b, self.att = dn.features, att
        Fdim = 1024
        self.head = nn.Sequential(
            nn.Linear(Fdim * 2, 512), nn.BatchNorm1d(512), nn.ReLU(True),
            nn.Dropout(0.4), nn.Linear(512, 2))
        self.keys = sorted(aux_meta.keys())
        self.aux = nn.ModuleList([
            nn.Sequential(nn.Linear(Fdim * 2, 128), nn.ReLU(True),
                          nn.Dropout(0.3), nn.Linear(1

In [6]:
# ══════ run cell 25 (multi-backbone trainer) with a chosen backbone ══════
import json, re, time

NB       = "the whole pipeline.ipynb"
IDX      = 25
QUICK    = False          # <<< True = timing probe first. Set False for the real run.
WANT     = "resnet50"    # "resnet50" | "convnext_tiny" | "efficientnet_b3"

src = "".join(json.load(open(NB))["cells"][IDX]["source"])
assert "def build(name)" in src and "CANDIDATES" in src, \
    "cell 25 is not the backbone trainer — check IDX"

patched = src
def sub(pat, rep, label, need=1):
    global patched
    patched, n = re.subn(pat, rep, patched, flags=re.M)
    print(f"  {label:<26} {n} replacement(s)")
    assert n >= need, f"FAILED: {label} — pattern not found, stopping before training"

print("patching:")
sub(r"^QUICK_TEST\s*=.*$", f"QUICK_TEST = {QUICK}", "QUICK_TEST")
sub(r"^BACKBONE\s*=.*$",   f'BACKBONE   = "{WANT}"', "BACKBONE")

print("\ncontrol lines after patch:")
for l in patched.split("\n"):
    if re.match(r"^\s*(QUICK_TEST|BACKBONE|S, BATCH|SEEDS|CANDIDATES|LR_HEAD)\b", l):
        print("   ", l.strip()[:95])

print(f"\n{'='*70}\nrunning: backbone={WANT}  quick={QUICK}\n{'='*70}\n")
t0 = time.time()
exec(compile(patched, f"{NB}::cell{IDX}", "exec"), globals())
el = time.time() - t0
print(f"\n{'='*70}\nelapsed {el/60:.1f} min")
if QUICK:
    print(f"full run estimate: {el*55/3600:.1f} hours  (5 folds x 2 seeds x 22 epochs)")
    print("if that is acceptable, set QUICK = False and re-run this cell")

patching:
  QUICK_TEST                 1 replacement(s)
  BACKBONE                   2 replacement(s)

control lines after patch:
    QUICK_TEST = False
    BACKBONE   = "resnet50"
    S, BATCH = 512, 12
    SEEDS, EPOCHS, FREEZE = [11, 22], 22, 3
    LR_HEAD, LR_HEAD_FT, LR_BACK = 1e-3, 3e-4, 3e-5
    SEEDS, EPOCHS, FREEZE, MULT, FOLDS = [11], 4, 1, 1, [0]
    CANDIDATES = ["efficientnet_b0", "convnext_tiny", "resnet50", "efficientnet_b3"]
    BACKBONE   = "resnet50"

running: backbone=resnet50  quick=False

checking which backbones have pretrained weights available offline:
   efficientnet_b0    OK   (feature dim 1280)
   convnext_tiny      OK   (feature dim 768)
   resnet50           OK   (feature dim 2048)
   efficientnet_b3    OK   (feature dim 1536)

using backbone: resnet50  (feature dim 2048)

cached 1696 in 8s
helper heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5}

### fold 0 | train 1202 val 155 test 339
    seed 11
      ep  1  val-AUC 0.6376 *
      ep  2  val-AU

In [8]:
# ══════ run cell 25 (multi-backbone trainer) with a chosen backbone ══════
import json, re, time

NB       = "the whole pipeline.ipynb"
IDX      = 25
QUICK    = False         # <<< True = timing probe first. Set False for the real run.
WANT     = "convnext_tiny"    # "resnet50" | "convnext_tiny" | "efficientnet_b3"

src = "".join(json.load(open(NB))["cells"][IDX]["source"])
assert "def build(name)" in src and "CANDIDATES" in src, \
    "cell 25 is not the backbone trainer — check IDX"

patched = src
def sub(pat, rep, label, need=1):
    global patched
    patched, n = re.subn(pat, rep, patched, flags=re.M)
    print(f"  {label:<26} {n} replacement(s)")
    assert n >= need, f"FAILED: {label} — pattern not found, stopping before training"

print("patching:")
sub(r"^QUICK_TEST\s*=.*$", f"QUICK_TEST = {QUICK}", "QUICK_TEST")
sub(r"^BACKBONE\s*=.*$",   f'BACKBONE   = "{WANT}"', "BACKBONE")

print("\ncontrol lines after patch:")
for l in patched.split("\n"):
    if re.match(r"^\s*(QUICK_TEST|BACKBONE|S, BATCH|SEEDS|CANDIDATES|LR_HEAD)\b", l):
        print("   ", l.strip()[:95])

print(f"\n{'='*70}\nrunning: backbone={WANT}  quick={QUICK}\n{'='*70}\n")
t0 = time.time()
exec(compile(patched, f"{NB}::cell{IDX}", "exec"), globals())
el = time.time() - t0
print(f"\n{'='*70}\nelapsed {el/60:.1f} min")
if QUICK:
    print(f"full run estimate: {el*55/3600:.1f} hours  (5 folds x 2 seeds x 22 epochs)")
    print("if that is acceptable, set QUICK = False and re-run this cell")

patching:
  QUICK_TEST                 1 replacement(s)
  BACKBONE                   2 replacement(s)

control lines after patch:
    QUICK_TEST = False
    BACKBONE   = "convnext_tiny"
    S, BATCH = 512, 12
    SEEDS, EPOCHS, FREEZE = [11, 22], 22, 3
    LR_HEAD, LR_HEAD_FT, LR_BACK = 1e-3, 3e-4, 3e-5
    SEEDS, EPOCHS, FREEZE, MULT, FOLDS = [11], 4, 1, 1, [0]
    CANDIDATES = ["efficientnet_b0", "convnext_tiny", "resnet50", "efficientnet_b3"]
    BACKBONE   = "convnext_tiny"

running: backbone=convnext_tiny  quick=False

checking which backbones have pretrained weights available offline:
   efficientnet_b0    OK   (feature dim 1280)
   convnext_tiny      OK   (feature dim 768)
   resnet50           OK   (feature dim 2048)
   efficientnet_b3    OK   (feature dim 1536)

using backbone: convnext_tiny  (feature dim 768)

cached 1696 in 8s
helper heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5}

### fold 0 | train 1202 val 155 test 339
    seed 11
      ep  1  val-AUC 0.6332 *


In [9]:
# ══ paired backbone test + does ConvNeXt improve the ensemble? ══
import os, itertools, numpy as np, pandas as pd
from scipy.stats import rankdata, ttest_rel
from sklearn.metrics import roc_auc_score

D = "CBIS"
M = pd.read_csv(f"{D}/unified_folds_mass.csv")
M["_f"] = -1
for k in range(5):
    M.loc[M[f"role_f{k}"].astype(str).str.lower().eq("test"), "_f"] = k
assert (M["_f"] >= 0).all()
K2L = dict(zip(M["img"], M["lesion_key"]))
F2  = dict(zip(M["img"], M["_f"]))

POOL = ["v2", "efficientnet_b0", "convnext_tiny", "resnet50", "handcrafted",
        "endtoend_fused", "twostream", "twostream_calcpre", "imageonly"]
cols, y_, f_ = {}, None, None
for m in POOL:
    fp = f"{D}/cv_mass_{m}_oof.csv"
    if not os.path.exists(fp): print("missing", m); continue
    d = pd.read_csv(fp)
    d["lk"] = d["img"].map(K2L); d["_f"] = d["img"].map(F2)
    d = d.dropna(subset=["lk", "_f"])
    g = d.groupby("lk").agg(p=("prob","mean"), y=("true","max"), f=("_f","max"))
    cols[m] = g["p"]
    if y_ is None: y_, f_ = g["y"], g["f"]
X = pd.concat(cols, axis=1).join(y_.rename("y")).join(f_.rename("f")).dropna()
y, fd = X["y"].values.astype(int), X["f"].values.astype(int)
names = [c for c in X.columns if c not in ("y","f")]
print(f"{len(X)} lesions, {len(names)} models\n")

print("── per-fold AUC ──")
per = {m: np.array([roc_auc_score(y[fd==k], X[m].values[fd==k]) for k in range(5)]) for m in names}
for m in sorted(names, key=lambda z: -per[z].mean()):
    print(f"  {m:<20} pooled {roc_auc_score(y, X[m]):.4f}   fold {per[m].mean():.4f} ± {per[m].std(ddof=1):.4f}")

print("\n── ConvNeXt vs DenseNet, paired over folds ──")
if "convnext_tiny" in per and "v2" in per:
    dlt = per["convnext_tiny"] - per["v2"]
    print(f"  per-fold Δ: {np.round(dlt,4).tolist()}")
    print(f"  mean {dlt.mean():+.4f}   p = {ttest_rel(per['convnext_tiny'], per['v2']).pvalue:.3f}"
          f"   ({(dlt>0).sum()}/5 folds)")

# ── nested greedy forward selection ──
R = {m: rankdata(X[m].values)/len(X) for m in names}
def auc_of(ms, mask): return roc_auc_score(y[mask], np.mean([R[m][mask] for m in ms], 0))
oof = np.zeros(len(X))
for k in range(5):
    inner, outer = fd != k, fd == k
    sel, best = [], -1
    while len(sel) < 4:
        cand = [(auc_of(sel+[m], inner), m) for m in names if m not in sel]
        s, m = max(cand)
        if s <= best + 1e-5: break
        best, _ = s, sel.append(m)
    oof[outer] = np.mean([R[m][outer] for m in sel], 0)
    print(f"  fold {k}: {' + '.join(sel)}   (inner {best:.4f})")
ens = [roc_auc_score(y[fd==k], oof[fd==k]) for k in range(5)]
print(f"\nENSEMBLE  pooled {roc_auc_score(y, oof):.4f}   fold {np.mean(ens):.4f} ± {np.std(ens,ddof=1):.4f}")
print("previous (without convnext/resnet50): pooled 0.9089")

1005 lesions, 9 models

── per-fold AUC ──
  endtoend_fused       pooled 0.8773   fold 0.8922 ± 0.0351
  twostream            pooled 0.8885   fold 0.8921 ± 0.0245
  convnext_tiny        pooled 0.8744   fold 0.8767 ± 0.0397
  v2                   pooled 0.8692   fold 0.8742 ± 0.0274
  twostream_calcpre    pooled 0.8656   fold 0.8717 ± 0.0459
  efficientnet_b0      pooled 0.8536   fold 0.8585 ± 0.0241
  resnet50             pooled 0.8301   fold 0.8339 ± 0.0343
  imageonly            pooled 0.7796   fold 0.7815 ± 0.0742
  handcrafted          pooled 0.6963   fold 0.7075 ± 0.0471

── ConvNeXt vs DenseNet, paired over folds ──
  per-fold Δ: [-0.0126, 0.0059, -0.0157, 0.0071, 0.0279]
  mean +0.0025   p = 0.764   (3/5 folds)
  fold 0: twostream + endtoend_fused + twostream_calcpre   (inner 0.9184)
  fold 1: twostream + endtoend_fused + twostream_calcpre   (inner 0.9113)
  fold 2: twostream + endtoend_fused + twostream_calcpre + convnext_tiny   (inner 0.9175)
  fold 3: twostream + endtoend_fus

In [1]:
# ══ Stage 2 parameter counts (no weights downloaded, no GPU) ══
import torch, torch.nn as nn
from torchvision import models

def build(name):
    if name == "densenet121":     m = models.densenet121(weights=None);     return m.features, 1024
    if name == "convnext_tiny":   m = models.convnext_tiny(weights=None);   return m.features, 768
    if name == "efficientnet_b0": m = models.efficientnet_b0(weights=None); return m.features, 1280
    if name == "resnet50":
        m = models.resnet50(weights=None); return nn.Sequential(*list(m.children())[:-2]), 2048
    raise ValueError(name)

AUX = {"subtlety": 5, "mass_shape": 6, "mass_margins": 6}
n = lambda m: sum(p.numel() for p in m.parameters())

def head_params(feat, streams=1, dual=True):
    d = feat * streams * (2 if dual else 1)
    main = nn.Sequential(nn.Linear(d, 512), nn.BatchNorm1d(512), nn.ReLU(True),
                         nn.Dropout(0.4), nn.Linear(512, 2))
    aux = nn.ModuleList([nn.Sequential(nn.Linear(d, 128), nn.ReLU(True),
                                       nn.Dropout(0.3), nn.Linear(128, k))
                         for k in AUX.values()])
    return n(main), n(aux)

print(f"{'configuration':<46}{'backbone':>10}{'head':>8}{'INFER M':>10}{'aux(train)':>12}")
print("-" * 86)
for name in ["convnext_tiny", "densenet121", "efficientnet_b0", "resnet50"]:
    f, fd = build(name); bp = n(f)
    for st, dual, lab in [(1, False, "single-stream, global pool only (control)"),
                          (1, True,  "single-stream, DUAL pooling"),
                          (2, True,  "two-stream, DUAL pooling")]:
        hp, ap = head_params(fd, st, dual)
        print(f"{name + '  ' + lab:<46}{bp*st/1e6:>10.2f}{hp/1e6:>8.2f}"
              f"{(bp*st+hp)/1e6:>10.2f}{ap/1e6:>12.2f}")

configuration                                   backbone    head   INFER M  aux(train)
--------------------------------------------------------------------------------------
convnext_tiny  single-stream, global pool only (control)     27.82    0.40     28.21        0.30
convnext_tiny  single-stream, DUAL pooling         27.82    0.79     28.61        0.59
convnext_tiny  two-stream, DUAL pooling            55.64    1.58     57.21        1.18
densenet121  single-stream, global pool only (control)      6.95    0.53      7.48        0.40
densenet121  single-stream, DUAL pooling            6.95    1.05      8.00        0.79
densenet121  two-stream, DUAL pooling              13.91    2.10     16.01        1.58
efficientnet_b0  single-stream, global pool only (control)      4.01    0.66      4.67        0.49
efficientnet_b0  single-stream, DUAL pooling        4.01    1.31      5.32        0.99
efficientnet_b0  two-stream, DUAL pooling           8.02    2.62     10.64        1.97
resnet50  sin

In [2]:
import json, glob, re
for nb in sorted(glob.glob("**/*.ipynb", recursive=True)):
    try: cells = json.load(open(nb)).get("cells", [])
    except Exception: continue
    for i, c in enumerate(cells):
        t = "".join(c.get("source", []))
        if "cv_mass_twostream" not in t or "to_csv" not in t: continue
        print(f"\n{nb}  cell {i}")
        for l in [x for x in t.split("\n") if x.strip()][:6]: print("  ", l[:100])
        for l in t.split("\n"):
            if re.search(r"densenet121|BACKBONE|WIDE|S2\s*=|S\s*,\s*BATCH|SEEDS", l):
                print("   >", l.strip()[:100])


the whole pipeline.ipynb  cell 46
   # ══════════════════════════════════════════════════════════════════════
   # CELL 30 — TWO-STREAM MASS CLASSIFIER  (tight lesion + wide context)
   #   Stream A: 512px tight crop  -> lesion texture & margin
   #   Stream B: 384px wide crop   -> surrounding architecture
   #   Predicted masks only. Saves cv_mass_twostream_oof.csv
   #   >>> QUICK_TEST = True FIRST <<<
   > SEEDS, EPOCHS, FREEZE = [11], 20, 3
   > SEEDS, EPOCHS, FREEZE, MULT, FOLDS = [11], 4, 1, 1, [0]
   > WIDE = os.path.join(D, "crops_wide_%s" % LES)
   > d["wimg"]  = d["stem"].apply(lambda s: os.path.join(WIDE, s+"_img.png"))
   > d["wmask"] = d["stem"].apply(lambda s: os.path.join(WIDE, s+"_pred.png"))
   > try:  return models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1).features
   > except Exception: return models.densenet121(weights=None).features
   > for sd in SEEDS:

the whole pipeline.ipynb  cell 47
   # ═══════════════════════════════════════════════════

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# BACKBONE ABLATION — is dual pooling backbone-agnostic?
#   Grid: {convnext_tiny, densenet121, efficientnet_b0, resnet50}
#         x {blind, dual}.  Identical folds, aug, TTA, loss, seeds.
#     blind : g = mean(f)
#     dual  : w=1+2m ; f_g=sum(w*f)/sum(w) ; f_u=mean(f) ; g=[f_g||f_u]
#   RESULT = the gain (dual - blind) per backbone.
#   QUICK=True absolute AUCs will NOT match your 0.8692 — never mix rows.
#   Self-contained. Resumable. LES forced to "mass".
# ══════════════════════════════════════════════════════════════════════
import os, time
os.environ["OMP_NUM_THREADS"] = "4"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score
cv2.setNumThreads(0)

D   = "/root/autodl-tmp/CBIS"
DEV = torch.device("cuda")
OUT = os.path.join(D, "backbone_ablation"); os.makedirs(OUT, exist_ok=True)

QUICK          = True     # True: S=448,MULT=4,EP=12 (~4h)  False: 512/8/20 (~12h)
USE_PRED_MASKS = True     # True = predicted masks (end-to-end, honest)

LES  = "mass"
AUXC = ["subtlety", "mass_shape", "mass_margins"]
S      = 448 if QUICK else 512
MULT   = 4   if QUICK else 8
EPOCHS = 12  if QUICK else 20
BATCH, LR_HEAD, LR_FT, FREEZE, GAMMA, AUX_W, PATIENCE = 12, 1e-3, 1e-5, 3, 2.0, 0.3, 5

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD  = np.array([0.229, 0.224, 0.225], np.float32)

# ctor, weights, channels, post-activation
SPECS = {
    "convnext_tiny":   (models.convnext_tiny,   models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1,    768, "none"),
    "densenet121":     (models.densenet121,     models.DenseNet121_Weights.IMAGENET1K_V1,     1024, "relu"),
    "efficientnet_b0": (models.efficientnet_b0, models.EfficientNet_B0_Weights.IMAGENET1K_V1, 1280, "none"),
    "resnet50":        (models.resnet50,        models.ResNet50_Weights.IMAGENET1K_V1,        2048, "none"),
}

# ══ PREFLIGHT — real ImageNet weights, or abort before burning hours ══
print("preflight - checking pretrained weights")
bad = []
for name, (fn, w, C, _) in SPECS.items():
    try:
        fn(weights=w); print("  OK   " + name)
    except Exception as e:
        bad.append(name); print("  FAIL " + name + "  -> " + str(e)[:130])
if bad:
    raise SystemExit(
        "\nABORT: no ImageNet weights for: " + ", ".join(bad) +
        "\nRandom init would invalidate the comparison.\n\n"
        "On any machine with internet:\n"
        "  python -c \"from torchvision import models as m; \\\n"
        "    m.convnext_tiny(weights=m.ConvNeXt_Tiny_Weights.IMAGENET1K_V1); \\\n"
        "    m.resnet50(weights=m.ResNet50_Weights.IMAGENET1K_V1); \\\n"
        "    m.efficientnet_b0(weights=m.EfficientNet_B0_Weights.IMAGENET1K_V1)\"\n"
        "then copy ~/.cache/torch/hub/checkpoints/ here.\n"
        "(HF_HUB_OFFLINE does not affect torchvision - it uses its own URL.)")
print("preflight passed\n")

# ══ DATA ══════════════════════════════════════════════════════════════
d = pd.read_csv(os.path.join(D, "unified_folds_" + LES + ".csv")).reset_index(drop=True)
d["label"] = d["label"].astype(int)
PM = os.path.join(D, "predmasks_" + LES)
d["pred"] = d["img"].apply(
    lambda p: os.path.join(PM, os.path.basename(p).replace("_img.png", "") + "_pred.png"))
assert d["pred"].apply(os.path.exists).all(), "missing predicted masks - rerun Phase 2"
print(LES.upper() + "   n=" + str(len(d)) + "   lesions=" + str(d.lesion_key.nunique()) +
      "   patients=" + str(d.patient_id.nunique()))

CACHE = {}; t0 = time.time()
for _, r in d.iterrows():
    k = r["img"]
    if k in CACHE: continue
    im = cv2.resize(cv2.imread(k, cv2.IMREAD_GRAYSCALE), (S, S))
    gt = cv2.imread(r["msk"], cv2.IMREAD_GRAYSCALE)
    gm = (cv2.resize(gt, (S, S), interpolation=cv2.INTER_NEAREST) > 127).astype(np.float32) \
         if gt is not None else np.zeros((S, S), np.float32)
    pdd = cv2.imread(r["pred"], cv2.IMREAD_GRAYSCALE)
    pmm = (cv2.resize(pdd, (S, S), interpolation=cv2.INTER_NEAREST) > 127).astype(np.float32) \
          if pdd is not None else np.zeros((S, S), np.float32)
    CACHE[k] = (_clahe.apply(im), gm, pmm)
print("cached " + str(len(CACHE)) + " images in " + format(time.time() - t0, ".0f") + "s")

def primary(x):
    return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()

aux, meta = {}, {}
for c in AUXC:
    if c not in d.columns or d[c].notna().sum() == 0: continue
    if c == "subtlety":
        v = pd.to_numeric(d[c], errors="coerce").where(lambda z: (z >= 1) & (z <= 5))
        codes = (v - 1).fillna(-1).astype(int); n = int(v.max()) if v.notna().any() else 0
    else:
        pr = d[c].map(primary); keep = pr.value_counts().head(6).index.tolist()
        pr = pr.where(pr.isin(keep), "OTHER")
        cats = sorted([q for q in pr.unique() if q != "UNK"]); mp = {q: i for i, q in enumerate(cats)}
        codes = pr.map(lambda z: mp.get(z, -1)).astype(int); n = len(cats)
    if n > 1: aux[c] = codes.values; meta[c] = n
AK = sorted(aux.keys()); print("aux heads:", meta, "\n")

class DS(Dataset):
    def __init__(s, idx, aug, tta=0):
        s.idx = np.array(idx); s.aug = aug; s.m = MULT if aug else 1; s.tta = tta
    def __len__(s): return len(s.idx) * s.m
    def __getitem__(s, i):
        j = s.idx[i % len(s.idx)]; v = i // len(s.idx); r = d.iloc[j]
        img, gm, pm = CACHE[r["img"]]
        mask = (pm if USE_PRED_MASKS else gm).copy(); img = img.copy()
        if s.aug and v > 0:
            if   v == 1: img = np.fliplr(img);   mask = np.fliplr(mask)
            elif v == 2: img = np.flipud(img);   mask = np.flipud(mask)
            elif v == 3: img = np.rot90(img, 1); mask = np.rot90(mask, 1)
            elif v == 4: img = np.rot90(img, 2); mask = np.rot90(mask, 2)
            elif v == 5: img = np.rot90(img, 3); mask = np.rot90(mask, 3)
            elif v == 6:
                M = cv2.getRotationMatrix2D((S/2, S/2), np.random.uniform(-20, 20),
                                            np.random.uniform(.9, 1.1))
                img  = cv2.warpAffine(img,  M, (S, S), borderMode=cv2.BORDER_REFLECT)
                mask = cv2.warpAffine(mask, M, (S, S), flags=cv2.INTER_NEAREST)
            elif v == 7:
                img = np.clip(img.astype(np.float32)*np.random.uniform(.85, 1.15), 0, 255).astype(np.uint8)
        if   s.tta == 1: img = np.fliplr(img);   mask = np.fliplr(mask)
        elif s.tta == 2: img = np.flipud(img);   mask = np.flipud(mask)
        elif s.tta == 3: img = np.rot90(img, 2); mask = np.rot90(mask, 2)
        im = np.ascontiguousarray(img).astype(np.float32) / 255.
        x  = np.stack([im, im, im], 0)
        x  = ((x.transpose(1, 2, 0) - MEAN) / STD).transpose(2, 0, 1).astype(np.float32)
        av = np.array([aux[c][j] for c in AK], dtype=np.int64) if AK else np.zeros(0, np.int64)
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(mask))[None],
                torch.tensor(int(r["label"])), torch.from_numpy(av))

class Net(nn.Module):
    def __init__(s, bk, mode, meta):
        super().__init__()
        fn, w, C, post = SPECS[bk]
        m = fn(weights=w); s.post = post; s.mode = mode
        s.b = nn.Sequential(*list(m.children())[:-2]) if bk == "resnet50" else m.features
        IN = 2 * C if mode == "dual" else C
        s.head = nn.Sequential(nn.Linear(IN, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, 2))
        s.keys = sorted(meta.keys())
        s.aux  = nn.ModuleList([nn.Sequential(nn.Linear(IN, 128), nn.ReLU(), nn.Dropout(0.3),
                                              nn.Linear(128, meta[k])) for k in s.keys])
    def forward(s, x, mask):
        f = s.b(x)
        if s.post == "relu": f = F.relu(f)
        if s.mode == "dual":
            mm = F.interpolate(mask, size=f.shape[2:], mode="bilinear", align_corners=False)
            w  = 1.0 + 2.0 * mm
            fg = (f * w).sum((2, 3)) / (w.sum((2, 3)) + 1e-6)
            fu = f.mean((2, 3))
            g  = torch.cat([fg, fu], 1)
        else:
            g = f.mean((2, 3))
        return s.head(g), [h(g) for h in s.aux]

y = d["label"].values

def run(bk, mode):
    tag = bk + "_" + mode; f_oof = os.path.join(OUT, tag + ".npy")
    if os.path.exists(f_oof):
        print("  [cached]  " + tag); return np.load(f_oof)
    print("  running   " + tag)
    oof = np.zeros(len(d)); T0 = time.time()
    for k in range(5):
        role = d["role_f" + str(k)]
        tr = np.where(role == "train")[0]; va = np.where(role == "val")[0]; te = np.where(role == "test")[0]
        assert not (set(d.patient_id[tr]) & set(d.patient_id[te])), "PATIENT LEAK"
        assert not (set(d.patient_id[va]) & set(d.patient_id[te])), "PATIENT LEAK"
        t0 = time.time(); torch.manual_seed(k); np.random.seed(k)
        n0 = float((y[tr] == 0).sum()); n1 = float((y[tr] == 1).sum())
        al = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV)
        def focal(lo, t):
            ce = F.cross_entropy(lo.float(), t, weight=al, reduction="none")
            pt = torch.exp(-ce); return ((1 - pt) ** GAMMA * ce).mean()
        net = Net(bk, mode, meta).to(DEV).to(memory_format=torch.channels_last)
        for p_ in net.b.parameters(): p_.requires_grad = False
        sc  = torch.amp.GradScaler()
        opt = torch.optim.AdamW([p_ for p_ in net.parameters() if p_.requires_grad],
                                lr=LR_HEAD, weight_decay=1e-3)
        tl  = DataLoader(DS(tr, True), batch_size=BATCH, shuffle=True, num_workers=0, pin_memory=True)
        @torch.no_grad()
        def col(idx, tta=True):
            net.eval(); reps = [0, 1, 2, 3] if tta else [0]; tot = None
            for t in reps:
                ld = DataLoader(DS(idx, False, tta=t), batch_size=20, shuffle=False, num_workers=0)
                ps = []
                for x, m, _, _ in ld:
                    x = x.to(DEV).to(memory_format=torch.channels_last); m = m.to(DEV)
                    with torch.amp.autocast(device_type="cuda"): o, _ = net(x, m)
                    ps += list(torch.softmax(o.float(), 1)[:, 1].cpu().numpy())
                ps = np.array(ps); tot = ps if tot is None else tot + ps
            return tot / len(reps)
        best, bs, ni = 0, None, 0
        for ep in range(1, EPOCHS + 1):
            if ep == FREEZE + 1:
                for p_ in net.b.parameters(): p_.requires_grad = True
                opt = torch.optim.AdamW(net.parameters(), lr=LR_FT, weight_decay=1e-3)
            net.train()
            if ep <= FREEZE: net.b.eval()
            for x, m, t2, a in tl:
                x = x.to(DEV).to(memory_format=torch.channels_last); m = m.to(DEV)
                t2 = t2.to(DEV); a = a.to(DEV)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast(device_type="cuda"):
                    o, ax = net(x, m); L = focal(o, t2)
                    if len(ax):
                        la = sum(F.cross_entropy(g.float(), a[:, h], ignore_index=-1)
                                 for h, g in enumerate(ax)) / len(ax)
                        L = L + AUX_W * la
                if not torch.isfinite(L): continue
                sc.scale(L).backward(); sc.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
                sc.step(opt); sc.update()
            pv = col(va, tta=False)
            au = roc_auc_score(y[va], pv) if len(set(y[va])) > 1 else 0
            if au > best:
                best = au; bs = {q: v.cpu().clone() for q, v in net.state_dict().items()}; ni = 0
            else: ni += 1
            if ni >= PATIENCE: break
        net.load_state_dict({q: v.to(DEV) for q, v in bs.items()})
        oof[te] = col(te)
        print("     fold " + str(k) + "  AUC " + format(roc_auc_score(y[te], oof[te]), ".4f") +
              "   (" + format(time.time() - t0, ".0f") + "s)")
        del net; torch.cuda.empty_cache()
    np.save(f_oof, oof)
    print("     -> " + tag + " done in " + format((time.time() - T0) / 60, ".1f") + " min")
    return oof

def per_lesion(p):
    L = d.assign(_p=p).groupby("lesion_key").agg(yy=("label", "max"), pp=("_p", "mean"))
    return roc_auc_score(L.yy, L.pp)

def nparams(bk, mode):
    return sum(p.numel() for p in Net(bk, mode, meta).parameters()) / 1e6

# ══ RUN ═══════════════════════════════════════════════════════════════
R = {}
for bk in SPECS:
    print("\n" + "#" * 70 + "\n#  " + bk + "\n" + "#" * 70)
    for mode in ["blind", "dual"]:
        R[(bk, mode)] = run(bk, mode)

# ══ RESULT TABLE ══════════════════════════════════════════════════════
rows = []
for bk in SPECS:
    rows.append(dict(backbone=bk,
                     params_M=nparams(bk, "dual"),
                     feat=SPECS[bk][2],
                     blind=per_lesion(R[(bk, "blind")]),
                     dual =per_lesion(R[(bk, "dual")])))
for r in rows: r["gain"] = r["dual"] - r["blind"]
pd.DataFrame(rows).to_csv(os.path.join(OUT, "backbone_ablation_summary.csv"), index=False)

print("\n" + "=" * 82)
print("BACKBONE ABLATION  -  " + ("QUICK" if QUICK else "FULL") + " settings,  " +
      ("predicted" if USE_PRED_MASKS else "reference") + " masks,  per-lesion AUC")
print("=" * 82)
print(format("backbone", "<18") + format("params(M)", ">10") + format("feat", ">7") +
      format("blind", ">9") + format("dual", ">9") + format("gain", ">10"))
print("-" * 82)
for r in rows:
    print(format(r["backbone"], "<18") + format(r["params_M"], ">10.2f") +
          format(r["feat"], ">7") + format(r["blind"], ">9.4f") +
          format(r["dual"], ">9.4f") + format(r["gain"], ">+10.4f"))
print("-" * 82)
g = [r["gain"] for r in rows]
print("mean gain " + format(np.mean(g), "+.4f") + "   sd " + format(np.std(g), ".4f") +
      "   min " + format(min(g), "+.4f") + "   max " + format(max(g), "+.4f"))
print("\nIf the gain holds across all four backbones, the mechanism is")
print("backbone-agnostic and not an artefact of DenseNet-121.")
print("=" * 82)

preflight - checking pretrained weights
  OK   convnext_tiny
  OK   densenet121
  OK   efficientnet_b0
  OK   resnet50
preflight passed

MASS   n=1696   lesions=1005   patients=892
cached 1696 images in 11s
aux heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5} 


######################################################################
#  convnext_tiny
######################################################################
  running   convnext_tiny_blind
     fold 0  AUC 0.7678   (492s)
     fold 1  AUC 0.8702   (408s)
     fold 2  AUC 0.8153   (355s)
     fold 3  AUC 0.8522   (412s)
     fold 4  AUC 0.8651   (380s)
     -> convnext_tiny_blind done in 34.1 min
  running   convnext_tiny_dual
     fold 0  AUC 0.7865   (416s)
     fold 1  AUC 0.8618   (409s)
     fold 2  AUC 0.8181   (411s)
     fold 3  AUC 0.8423   (422s)
     fold 4  AUC 0.8586   (344s)
     -> convnext_tiny_dual done in 33.4 min

######################################################################
#  densenet121